# Resumen de todo lo que se hizo: extracción, limpieza, NLP y dashboard (todos los scripts juntos)

Monitor de obras de infraestructura educativa paralizadas.

Por qué este cuaderno es tan extenso: reúne todos los scripts del proyecto en un solo archivo (la Parte A trae el código completo de cada módulo). Cada script también existe por separado en la carpeta src (por ejemplo src/nlp_causas.py o src/scraper_infobras.py), por si se quiere revisar por partes. El archivo proyecto_completo.py es la misma ejecución, pero solo con código.

Este cuaderno es autosuficiente: lleva adentro todo el código del proyecto (descarga, extracción, PDF, NLP de causas, producto final, CSV del dashboard, diccionario, auditorías y dashboard) y lo ejecuta de principio a fin. Primero escribe un solo CSV con todos los datos limpios y validados (data/final/dashboard_obras.csv) y con él arma el **dashboard en HTML** (resultados/dashboard.html): un solo archivo que se abre con doble clic en cualquier navegador, sin instalar nada y sin internet. Al terminar, el dashboard se abre solo en el navegador.

## Cómo ejecutarlo

No hay que cambiar ninguna ruta ni editar código: el programa encuentra solo la carpeta del proyecto, esté donde esté. Solo hay que instalar y ejecutar. Hay dos opciones: la opción rápida y la opción larga, donde se scrapean las páginas una por una desde la consola.

### Antes de empezar

1. Tener Python 3.10 o superior y VS Code con la extensión Jupyter.
2. Descargar el repositorio completo de GitHub (botón verde "Code", opción "Download ZIP") y abrir esa carpeta en VS Code. Se debe descargar la carpeta completa, porque el cuaderno necesita la carpeta data/processed, que ya viene incluida.
3. Abrir la carpeta donde se descomprimió el archivo, escribir cmd en la barra de direcciones del explorador y presionar Enter. En la consola que se abre, escribir ejecutar.bat --sin-descarga: crea el entorno .venv, instala las librerías, rehace el producto final y abre el dashboard.
4. Si se prefiere instalar las librerías sin el .bat, escribir en la terminal (una sola vez): py -m pip install -r requirements.txt
5. Arriba a la derecha, elegir Seleccionar kernel y luego el entorno de Python donde se instalaron las librerías (.venv si se usó el paso 3).
6. Solo para la opción larga: tener Google Chrome, internet y unos 3 GB de espacio libre.

### Opción rápida: rehacer el producto final y el dashboard

Qué hace: parte de las tablas ya procesadas que vienen en data/processed (no necesita datos crudos ni internet) y rehace la clasificación de causas con NLP, el CSV final, el diccionario, las tres auditorías y el dashboard en HTML.

Cómo hacerlo: pulsar Run All sin cambiar nada. Al no encontrar datos crudos, el cuaderno omite las descargas y los PDF, y corre solo los pasos 5 a 8. Los resultados quedan en data/final/dashboard_obras.csv y en resultados/dashboard.html, que se abre solo en el navegador.

Si prefiere la terminal, el mismo flujo se ejecuta con: python proyecto_completo.py (necesita la carpeta src al lado). También abre el dashboard al terminar.

### Opción larga (alrededor de 1 hora): correr todo desde cero

Qué hace: abre Chrome y descarga ESCALE, ENLA, Contraloría y MEF (unos 450 MB); arma las tablas; lee los 39 PDF; clasifica las causas con NLP; y genera el CSV final, el diccionario, las tres auditorías y el dashboard.

Cómo hacerlo: en la celda "0. Opciones" poner DESCARGAR = True y pulsar Run All. En Windows también se puede hacer con doble clic en ejecutar.bat.

Durante la ejecución: Chrome se abre y se maneja solo; no hay que cerrarlo ni usarlo hasta que termine. Las descargas tardan unos 40 minutos y la lectura de los PDF unos 20. Si un portal no responde, se vuelve a ejecutar y sigue donde quedó, porque lo ya descargado se salta.

Sobre INFOBRAS: no se vuelve a leer por defecto, porque es una foto de noviembre de 2026, leerlo tarda unas 2,5 horas y cambiaría los números. Sus datos ya están en data/processed.

### Si solo quiere ver o revisar

- Ver el resultado sin ejecutar nada: abrir resultados/dashboard.html con doble clic (los datos están en data/final/dashboard_obras.csv).
- Revisar el código por partes: cada módulo está por separado en la carpeta src.
- Leer qué se hizo y cuáles son sus límites: docs/producto_final.md y docs/limites_y_calidad.md (también en la página Límites del dashboard).

## Estructura del cuaderno

- Parte A, el código: una celda por módulo (empieza con %%modulo y el nombre). Se ejecuta una vez y deja los módulos cargados. Cada celda se puede leer y editar.
- Parte B, la ejecución paso a paso: solo el código que corre el proyecto, un paso por celda: 1 descarga, 2 ESCALE y ENLA, 3 datos personales, 4 proyecto PDF, 5 producto final (NLP de causas y CSV del dashboard), 6 diccionario, datos personales e inventario, 7 auditoría de calidad, 8 dashboard en HTML.

## Preparación (ejecutar primero)

In [ ]:
import sys
import types
from pathlib import Path

from IPython.core.magic import register_cell_magic

# La carpeta del proyecto es la que tiene src/ y notebooks/. Se busca desde el cuaderno (VS Code) y desde la carpeta actual, subiendo de
# nivel, para que funcione aunque VS Code se haya abierto sin la carpeta del proyecto.
_inicios = [Path(globals().get("__vsc_ipynb_file__", ".")).resolve().parent, Path.cwd().resolve()]
RAIZ_PROYECTO = next((d for inicio in _inicios for d in [inicio, *inicio.parents] if (d / "src").is_dir() and (d / "notebooks").is_dir()), None)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError("No encuentro la carpeta del proyecto (la que tiene src y notebooks). En VS Code: File -> Open Folder -> la carpeta "
                            "del proyecto, y vuelve a abrir este cuaderno.")


@register_cell_magic
def modulo(linea, celda):
    """%%modulo nombre: ejecuta la celda como un modulo de Python llamado «nombre», que las demas celdas pueden importar."""
    nombre = linea.strip()
    mod = types.ModuleType(nombre)
    mod.__file__ = str(RAIZ_PROYECTO / "src" / f"{nombre}.py")        # config.py deduce de aqui la carpeta del proyecto
    sys.modules[nombre] = mod
    try:
        exec(compile(celda, mod.__file__, "exec"), mod.__dict__)
    except BaseException:
        sys.modules.pop(nombre, None)
        raise
    print(f"módulo «{nombre}» cargado ({len(celda.splitlines())} líneas)")


print("Carpeta del proyecto:", RAIZ_PROYECTO.name)

# Parte A — El código del proyecto
Una celda por módulo, en orden de dependencias. Se ejecutan una vez y quedan cargados; después la Parte B los usa.

### `config` — Configuración
Configuracion central: rutas, URLs, selectores y parametros. Es el unico lugar a editar.

In [ ]:
%%modulo config
"""Configuracion central: rutas, URLs, selectores y parametros. Es el unico lugar a editar."""
from pathlib import Path

# --- Rutas (la raiz se define una sola vez) ---
RAIZ = Path(__file__).resolve().parent.parent   # carpeta del proyecto, donde sea que este (la que contiene src/)
RAW = RAIZ / "data" / "raw"
INTERIM = RAIZ / "data" / "interim"
PROCESSED = RAIZ / "data" / "processed"
FINAL = RAIZ / "data" / "final"                    # ENTREGABLE: el CSV unico para el dashboard (dashboard_obras.csv)
COMPONENTES = RAIZ / "data" / "interim" / "componentes"   # las 5 tablas con las que se arma el CSV unico (uso interno)
DOCS = RAIZ / "docs"
CAPTURAS = DOCS / "capturas"
LOG = DOCS / "log_descargas.txt"
REGISTRO = DOCS / "registro_fuentes.csv"

for carpeta in (RAW / "escale", RAW / "enla", INTERIM, PROCESSED, CAPTURAS):
    carpeta.mkdir(parents=True, exist_ok=True)

# --- Parametros de cortesia con los portales ---
USER_AGENT = "Proyecto academico USMP - Data Analytics"
PAUSA_MIN, PAUSA_MAX = 2, 5      # segundos entre acciones de descarga
REINTENTOS = 3
ESPERA_DESCARGA = 120            # segundos maximos para que termine una descarga
ESPERA_ELEMENTO = 30             # segundos maximos para que aparezca un elemento

ANIO_INICIO = 2022                # primer anio a descargar (se puede cambiar con --anio-inicio)
EXTENSIONES = r"\.(?:zip|pdf|xlsx?|sav|csv|dbf|rar|7z)"   # tipos de archivo que se reconocen en los listados

# --- ESCALE ---
ESCALE_URL_INICIO = "https://escale.minedu.gob.pe/uee"
ESCALE_CARPETA_CENSO = "A. Censo Educativo"     # texto del enlace a la carpeta del censo
ESCALE_SIGUIENTE = "Siguiente"                  # texto del boton de paginacion
ESCALE_BOTON_DESCARGAR = "//a[starts-with(normalize-space(.), 'Descargar (')]"
ESCALE_SELECTOR_ARCHIVOS = "table a"            # enlaces de archivos dentro de la carpeta de un anio
ESCALE_MAX_PAGINAS = 12
# Solo se bajan los archivos necesarios para infraestructura (los nombres cambian cada anio):
ESCALE_PATRONES = {
    "diccionario_local": r"(?i)diccionario.*[ _]LE[ _]",
    "local_lineal": r"(?i)loc(al)?_lineal",
    "servicios_basicos": r"(?i)servicios_basicos|servba",
    "padron_local": r"(?i)^\d+_(padron|padlocal)\.zip",
    "edificaciones": r"(?i)sec400|p401_edific|p53_edific",   # estado de las edificaciones (una fila por edificacion)
}

# --- ESCALE: como se leen los archivos de local en la extraccion ---
# Anios 2022-2024: tabla larga de servicios (una fila por local y servicio; la fila marcada "X" es la predominante)
# NUMERO 1 = red publica en agua, desague y luz (segun el diccionario de cada anio).
ESCALE_SERVICIOS_TABLA = {
    2022: {"zip": r"servicios_basicos", "marca": "TIPO", "cuadros": {"agua": "C202", "desague": "C206", "luz": "C207"}},
    2023: {"zip": r"servba", "marca": "TIPO", "cuadros": {"agua": "C202", "desague": "C206", "luz": "C207"}},
    2024: {"zip": r"servba", "marca": "PREDOMINA", "cuadros": {"agua": "C202", "desague": "C212", "luz": "C216"}},
}
ESCALE_RED_PUBLICA_TABLA = ["1"]
# Anio 2025 en adelante: los servicios son columnas de Loc_Lineal_1 (columna, codigos que cuentan como red publica).
# En luz, el diccionario 2025 llama "Red publica" a 1 (empresa distribuidora) y 2 (terceros).
ESCALE_SERVICIOS_COLUMNAS = {"agua": ("P321", ["1"]), "desague": ("P3212", ["1"]), "luz": ("P3217", ["1", "2"])}
# Archivo lineal 1 de cada anio (el nombre cambia: local_lineal_01 en 2022, loc_lineal_1 despues)
ESCALE_ZIP_LINEAL = r"lineal_0?1[_.]"
ESCALE_ZIP_PADRON = r"padron"
ESCALE_GESTION_PUBLICA = ["1", "2"]      # 1 publica de gestion directa, 2 publica de gestion privada; 3 privada
ESCALE_AREA_RURAL = "2"                  # AREA_CENSO: 1 urbana, 2 rural

# --- CONTRALORIA (gob.pe): informes de obras paralizadas y sus anexos ---
# Ruta de clics: portal -> categoria "Resultados de la Contraloria" -> compendio "Obras paralizadas - Documentos"
CGR_URL_PORTAL = "https://www.gob.pe/contraloria"
CGR_TEXTO_CATEGORIA = "Resultados de la Contraloría"
CGR_TEXTO_COMPENDIO = "Obras paralizadas - Documentos"
CGR_ID_POR_PAGINA = "filter_per_page"      # lista desplegable; asi no se usan las paginas ?sheet= que robots.txt no permite
CGR_OPCION_POR_PAGINA = "100 por página"
CGR_HREF_FICHA = "/informes-publicaciones/"
# Solo se descargan los informes o reportes de obras paralizadas (no infografias ni presentaciones)
CGR_REGEX_INFORME = r"(?i)^(informe|reporte) de obras paralizadas"
CGR_MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
             "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
CGR_TRIMESTRES = {"i": (3, 31), "ii": (6, 30), "iii": (9, 30), "iv": (12, 31)}   # fin de cada trimestre

# --- INFOBRAS (Contraloria): obras educativas paralizadas ---
INFOBRAS_BASE = "https://infobras.contraloria.gob.pe/InfobrasWeb"
INFOBRAS_URL_BUSCADOR = INFOBRAS_BASE + "/Mapa/Index?search-level=avanzada"
INFOBRAS_ID_NIVEL1 = "ID_ADVANCED_Nivel01"        # lista "Tipo de obra nivel 1"; se espera a que tenga opciones
INFOBRAS_FILTROS = {"ID_ADVANCED_Nivel01": "educaci", "ID_ADVANCED_Estado": "paraliz"}   # id de la lista -> texto de la opcion
INFOBRAS_ID_BOTON_BASICO = "searchBtn"             # boton "Buscar" de la busqueda basica: NO es el de la avanzada
INFOBRAS_ID_EXPORTAR_CSV = "btnExportCSV"
INFOBRAS_FILTROS_TEXTO = "educacion;paralizada"    # se anota en el registro de fuentes
INFOBRAS_FICHAS = INTERIM / "infobras_fichas.jsonl"   # una linea por obra leida (permite retomar si se corta)
INFOBRAS_SALIDA = PROCESSED / "fichas_infobras.csv"
INFOBRAS_DIAS_VIGENCIA = 30   # una ficha leida hace mas dias se vuelve a leer (el estado de la obra cambia)
INFOBRAS_MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
                  "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

# --- ESCALE: estado de las edificaciones (una fila por edificacion; los campos cambian de numero cada anio) ---
# uso = "la edificacion esta en uso"; riesgo = "esta en riesgo de colapso o presenta riesgo estructural";
# estado = estado de conservacion de muros, columnas, vigas y estructura de techos.
ESCALE_ZIP_EDIFICACIONES = r"sec400|p401_edific|p53_edific"
ESCALE_EDIFICACIONES = {
    2022: {"uso": "P401_3", "riesgo": "P401_20", "estado": ["P401_11_1C", "P401_11_2C", "P401_11_3C", "P401_11_4C"]},
    2023: {"uso": "P401_3", "riesgo": "P401_19", "estado": ["P401_10_12", "P401_10_22", "P401_10_32", "P401_10_42"]},
    2024: {"uso": "P401_3", "riesgo": "P401_20", "estado": ["P401_11_12", "P401_11_22", "P401_11_32", "P401_11_42"]},
    2025: {"uso": "P53_3", "riesgo": "P53_20", "estado": ["P53_11_12", "P53_11_22", "P53_11_32", "P53_11_42"]},
}
ESCALE_SI = ["SI", "1"]                 # 2022 usa texto SI/NO; los demas anios 1/2
ESCALE_NO = ["NO", "2"]
ESCALE_ESTADO_MALO = ["03", "MA"]       # 01 Bueno, 02 Regular, 03 Mal estado (2025: BU, RE, MA); 04, 05 y NA = no tiene / no aplica

# --- ENLA: como se leen las tablas regionales de los PDF en la extraccion ---
ENLA_TITULO_TABLA = r"Resultados? regionales seg[uú]n niveles de logro"
ENLA_GRADOS = {("4", "primaria"): "4.° primaria", ("2", "secundaria"): "2.° secundaria"}   # los demas grados no se piden
ENLA_AREAS = {"Lectura": "lectura", "Matemática": "matematica"}
ENLA_REGIONES = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Callao", "Cusco",
                 "Huancavelica", "Huánuco", "Ica", "Junín", "La Libertad", "Lambayeque", "Lima Metropolitana",
                 "Lima Provincias", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno", "San Martín",
                 "Tacna", "Tumbes", "Ucayali"]   # 23 departamentos sin Lima + Callao + Lima Metropolitana + Lima Provincias = 26

# --- ENLA: SICRECE (acceso publico). Pide un rol antes de entrar; el usuario autorizo "Estudiante de pregrado" ---
SICRECE_URL = "https://sicrece.minedu.gob.pe/#/home"
SICRECE_XPATH_ACCESO_PUBLICO = "//img[contains(@src,'quiensoy-publico')]/ancestor::a[1]"
SICRECE_SELECT_ROL = "app-popuppublico select"
SICRECE_ROL_VALOR = "1"                          # 1 = Estudiante de pregrado
SICRECE_BOTON_INGRESAR = "btnSesion"
SICRECE_PESTANA_BUSQUEDA = "Búsqueda avanzada"
SICRECE_TIPOS = ["MUESTRAL", "CENSAL"]            # tipos de evaluacion a recorrer
# posicion de cada lista desplegable entre las visibles del formulario (los ids mat-input-N cambian)
SICRECE_ORDEN_SELECTS = {"tipo": 0, "anio": 1, "nivel": 2, "grado": 3}
SICRECE_AREAS = ["LECTURA", "MATEMÁTICA"]
SICRECE_AMBITO = "DRE"                           # ambito territorial (una DRE por region); con DRE se desactiva urbana/rural
SICRECE_DRE_TODAS = "TODOS"
SICRECE_BOTON_BUSCAR = "//button[contains(.,'Buscar')]"
# respuesta que el navegador recibe al buscar (cada tipo de evaluacion usa un endpoint distinto)
SICRECE_API_REPORTE = {"MUESTRAL": "reporteGrafico_Publico_EMAmbitos_Nacional_DRE", "CENSAL": "reporteGrafico_Publico_ECE_DRE"}
# (nivel, grado) como aparecen en SICRECE -> etiqueta de grado de enla_regiones.csv
SICRECE_GRADOS = {("PRIMARIA", "SEGUNDO"): "2.° primaria", ("PRIMARIA", "CUARTO"): "4.° primaria", ("PRIMARIA", "SEXTO"): "6.° primaria",
                  ("SECUNDARIA", "SEGUNDO"): "2.° secundaria", ("SECUNDARIA", "QUINTO"): "5.° secundaria"}   # todos los grados que ofrece el portal
# Combinaciones que el menu de SICRECE lista pero cuya consulta nunca devuelve respuesta (3 intentos, 2026-11-12): no se vuelven a pedir
SICRECE_SIN_RESPUESTA = [("MUESTRAL", 2022, "PRIMARIA", "SEXTO"), ("MUESTRAL", 2023, "PRIMARIA", "SEGUNDO"), ("MUESTRAL", 2025, "SECUNDARIA", "QUINTO")]
SICRECE_ESPERA_RESPUESTA = 60

# --- ENLA (UMC). Por HTTPS el certificado esta roto: se usa HTTP ---
ENLA_ORIGEN_HTTP = "http://umc.minedu.gob.pe"   # Chrome bloquea descargas por http salvo que se confie en este origen
ENLA_URL_INICIO = "http://umc.minedu.gob.pe/enla/"
ENLA_REGEX_PAGINA_ANIO = r"/resultados(enla|em)(\d{4})/?$"    # enlaces "Resultados ENLA 2025", "Resultados EM 2022"
ENLA_REGEX_PDF = r"/wp-content/uploads/.*\.pdf$"
ENLA_EXCLUIR = r"(?i)catalogo|infograf|rm_n|_rm_|resoluci"    # catalogo del sitio, infografias y normas
# el HTML puede traer el nombre con tilde literal o codificado (%C3%B3): se prueban las dos formas
ENLA_XPATH_ENLACE_PDF = "//a[contains(@href, '{codificado}') or contains(@href, '{decodificado}')]"


# --- Proyecto PDF: inventario, tablas y texto de los informes ---
LOG_PDF = DOCS / "log_pdf.txt"
INVENTARIO_PDF = DOCS / "inventario_pdf.csv"
CATALOGO_TABLAS = DOCS / "catalogo_tablas.csv"
INTERIM_PDF = INTERIM / "pdf"
EXTENSIONES_DOCUMENTO = (".pdf", ".ppt", ".pptx")
MIN_CARACTERES_PAGINA = 50          # una pagina con menos letras que esto se considera sin texto seleccionable
PAGINAS_PRUEBA_TEXTO = 3            # se prueban las primeras paginas (y una del medio) para saber si el PDF es escaneado

# Departamentos del Peru con su codigo INEI de 2 digitos (el Callao se trata como departamento, como hace la Contraloria)
DEPARTAMENTOS = {
    "AMAZONAS": "01", "ANCASH": "02", "APURIMAC": "03", "AREQUIPA": "04", "AYACUCHO": "05", "CAJAMARCA": "06",
    "CALLAO": "07", "CUSCO": "08", "HUANCAVELICA": "09", "HUANUCO": "10", "ICA": "11", "JUNIN": "12",
    "LA LIBERTAD": "13", "LAMBAYEQUE": "14", "LIMA": "15", "LORETO": "16", "MADRE DE DIOS": "17", "MOQUEGUA": "18",
    "PASCO": "19", "PIURA": "20", "PUNO": "21", "SAN MARTIN": "22", "TACNA": "23", "TUMBES": "24", "UCAYALI": "25",
}
TOLERANCIA_SOLES = 30        # diferencia aceptada entre una suma de filas y el total del informe (los montos vienen redondeados)
UMBRAL_DIFERENCIA_PCT = 10   # diferencia maxima aceptada contra las cifras oficiales (criterio del proyecto)
CIFRAS_OFICIALES = PROCESSED / "cifras_oficiales_contraloria.csv"
CAUSALES_CGR = PROCESSED / "contraloria_causales.csv"
DISTRIBUCIONES_CGR = PROCESSED / "contraloria_distribuciones.csv"


# --- Fase 4 (texto): citas, terminos y grupos de causa por palabras clave ---
CITAS_REGEX = (r"\b(educaci[oó]n|educativ[oa]s?|escuelas?|colegios?|escolar(?:es)?|aulas?|docentes?|estudiantes?|"
               r"ense[ñn]anza|minedu|pronied)\b")
CITAS_MIN_PALABRAS = 7
CITAS_MAX_CARACTERES = 700
TERMINOS_POR_ANIO = 40
STOPWORDS_ES = set("""para como pero este esta estos estas entre sobre desde hasta cuando donde tambien ademas segun cada tiene tienen
puede pueden sido fueron sera seran parte otros otras todo todos todas mismo misma dicho dicha lugar tras ante bajo contra durante
mediante hacia sino porque aunque luego cual cuales quien cuyo cuya cuyos cuyas esto eso aquel aquella ellos ellas nosotros
tanto tanta tantos tantas mayor menor mayores menores primer primera segundo segunda tercer tercera cuatro cinco seis siete ocho
nueve diez anos fecha total cantidad numero nivel niveles forma manera caso casos vez veces asi aun dentro fuera
obras obra paralizadas paralizada paralizacion publicas publica informe informes territorio nacional siguiente cuadro tabla
fuente elaboracion subgerencia seguimiento evaluacion sistema contraloria general republica control entidades entidad""".split())

# Grupos de causa (se prueban en este orden; en empate gana el primero). Los textos se comparan en minusculas y sin tildes.
# Se evitan palabras que aparecen en simples notas de avance ("valorizacion del mes", "costo del expediente tecnico").
CAUSAS_REGLAS = {
    # Cada grupo es una lista de patrones sobre el texto ya normalizado (minusculas, sin tildes). El orden de los grupos desempata.
    "recursos_financieros": [r"recursos? (financier|economic|presupuest)", r"disponibilidad (oportuna |presupuestal |presupuestaria )?de (los )?recursos",
                             r"presupuest", r"financiamiento|falta de financ|problemas? (de )?financier|desabastecimiento financiero", r"liquidez", r"falta de pago|pagos? (de |por )?(las )?valoriza|valorizacion\w* (impaga|pendiente)",
                             r"deuda|adeud", r"\bfondos\b", r"ano fiscal", r"\bpia\b", r"transferencia (de (los )?recursos|financiera|presupuestal)",
                             r"sin (dinero|recursos)|no cuenta con (dinero|recursos)", r"asignacion (presupuest|de recursos)"],
    "incumplimiento_contractual": [r"incumpl", r"abandon", r"resuelt|resolvi|resuelve (el )?contrato|resolucion (total |parcial )?(de|del) contrato",
                                   r"rescin|rescis", r"carta notarial", r"penalidad", r"carta fianza", r"nulidad (de oficio )?del contrato",
                                   r"contratista.{0,30}(no (ejecut|cumpl|present)|retir|desaparec)"],
    "expediente_tecnico": [r"deficienc\w* (en|del) (el )?expediente", r"deficienc\w* tecnic", r"modificacion\w* (al|del) exp", r"mayores metrados",
                           r"observaciones? de caracter tecnico", r"incompatib", r"(elaboracion|elaborando|elabora|aprobacion|actualizacion|evaluacion|observaciones? (al|del)|absolucion de consultas).{0,40}expediente|expediente tecnico de(l)? saldo", r"adicional(es)? de obra|prestacion(es)? adicional|deductivo",
                           r"saldo de obra", r"reformulacion|error(es)? (en|de) (el )?(diseno|expediente|calculo)|vicios? ocultos?|replanteo|cambio de diseno"],
    "controversias_arbitraje": [r"arbitr", r"controversia", r"conciliacion", r"discrepancia", r"litigio", r"judicial", r"denunci", r"procuradur",
                                r"delito", r"investigacion", r"intervencion de (la )?contraloria", r"\bjrd\b|laudo|medida cautelar|demanda|peritaje|fiscalia"],
    "clima": [r"lluvia", r"clima", r"precipitac", r"pluvial", r"huaico", r"huayco", r"inundac", r"friaje", r"nevada", r"granizo", r"fenomeno",
              r"desastre", r"deslizamiento", r"talud", r"estiaje|sismo|terremoto|helada"],
    "conflictos_sociales": [r"conflicto", r"\bparo\b", r"protesta", r"bloqueo", r"huelga", r"dirigente",
                            r"reclamos? (de la |de los )?(poblacion|comunidad|vecinos|pobladores)",
                            r"(comunidad|poblacion|pobladores|vecinos).{0,40}(oposic|impid|reclam|rechaz|no permit)", r"rondas? campesinas?",
                            r"invasion|impedimento\w* (para el )?ingreso|no permiten? el ingreso", r"inseguridad|extorsi|amenaza"],
    "terreno_saneamiento": [r"terreno", r"saneamiento fisico", r"predio", r"posesion", r"propietari", r"expropiacion", r"titularidad|libre disponibilidad",
                            r"liberacion de (area|terreno|predio)|servidumbre|colindante|invasor"],
    "permisos_licencias": [r"licencia", r"permiso", r"autorizacion", r"\bcira\b", r"certificacion (ambiental|arqueologica)|arqueolog|ministerio de cultura",
                           r"factibilidad|conexion electrica"],
    "interferencias": [r"interferencia", r"reubicacion de (redes|postes|servicios)", r"postes|redes? (de agua|de desague|electricas)|linea de alta tension"],
    "materiales_logistica": [r"material(es)?", r"desabastec", r"abastecimiento", r"transporte", r"cemento|fierro|acero|ladrillo|agregado|asfalto",
                             r"combustible", r"insumo", r"adquisicion|proveedor|suministro|equipamiento|mobiliario|maquinaria|transformador|escasez|postores",
                             r"mano de obra"],
    "personal_clave": [r"personal clave", r"(sin|falta de) (residente|supervisor|supervision|inspector)",
                       r"(renuncia|ausencia|reemplazo|cambio)\w* (del|de la) (residente|supervisor|inspector)"],
    "pandemia": [r"covid", r"pandemia", r"emergencia sanitaria", r"coronavirus"],
    # grupos de ESTADO: no son causas; solo se asignan cuando el texto no trae ninguna causa real
    "obra_concluida_o_en_marcha": [r"concluid|culminad|finalizad", r"dentro de lo programado|adelantado|con normalidad|avance normal",
                                   r"liquidacion|liquidad", r"recepcion de (la )?obra|acta de recepcion|recepcionad",
                                   r"ha iniciado (su )?ejecucion|inicio de (la )?ejecucion|reinici|reanud", r"transferencia (del|de la) (proyecto|obra)"],
    "sin_restricciones": [r"\bno\b[^.]{0,40}\b(restriccion|problema|problematica|inconveniente|dificultad)", r"\bsin (restriccion|problema|problematica|inconveniente)",
                          r"\bninguna?\b"],
}
CAUSAS_PRESENCIA = ("clima", "conflictos_sociales", "controversias_arbitraje", "pandemia", "interferencias")   # si la palabra va precedida de «no», «sin» o «permite», no es causa
CAUSAS_ESTADO = ("obra_concluida_o_en_marcha", "sin_restricciones")     # grupos que no son causa de paralizacion
# Macro-categorias para el dashboard (pocas y de tamano comparable): cada grupo de causa pertenece a una sola
CAUSAS_MACRO = {
    "recursos_financieros": "financiera", "incumplimiento_contractual": "contractual_y_legal", "controversias_arbitraje": "contractual_y_legal",
    "expediente_tecnico": "tecnica", "personal_clave": "tecnica", "materiales_logistica": "logistica",
    "conflictos_sociales": "entorno_social_y_territorial", "terreno_saneamiento": "entorno_social_y_territorial",
    "permisos_licencias": "entorno_social_y_territorial", "interferencias": "entorno_social_y_territorial",
    "clima": "fuerza_mayor", "pandemia": "fuerza_mayor",
    "obra_concluida_o_en_marcha": "sin_causa_de_paralizacion", "sin_restricciones": "sin_causa_de_paralizacion",
    "otros": "sin_clasificar", "sin_informacion": "sin_clasificar"}
# Texto que no dice nada de la causa: frases tipicas y palabras vacias. Si solo queda eso, el grupo es 'sin_informacion'.
FRASES_SIN_INFORMACION = [r"no se (reportaron|registraron) comentarios", r"actualizado segun el inventario de obras publicas paralizadas del banco de inversiones",
                          r"catalogo de obra", r"\bninguno\b"]
PALABRAS_SIN_INFORMACION = set("""obra obras proyecto paralizada paralizado paralizadas paralizados paraliza suspendida suspendido suspension
se encuentra encuentran continua continuan esta viene vienen actualmente fecha actual hasta desde presente la el lo los las un una de del
al en por que con a y o es son su sus ya mes dias dia ano anos total el mismo misma estado situacion sin""".split())
CITAS_EDUCACION = PROCESSED / "citas_educacion.csv"
FRECUENCIA_TERMINOS = PROCESSED / "frecuencia_terminos.csv"


# --- INFOBRAS, pasada ampliada (A4 + B1 + B2): una sola lectura de fichas para las tres tareas ---
# No toca infobras_fichas.jsonl ni fichas_infobras.csv (datos del 2026-10-27): todo lo nuevo va en archivos aparte.
INFOBRAS_FILTROS_TODOS = {"ID_ADVANCED_Nivel01": "educaci"}         # A4: Educacion/Cultura con TODOS los estados (sin filtro de estado)
INFOBRAS_FILTROS_TODOS_TEXTO = "educacion;todos_los_estados"
INFOBRAS_NOMBRE_LISTADO_TODOS = "listado_educacion_todos.csv"       # -> infobras_{anio}_listado_educacion_todos_{fecha}.csv
INFOBRAS_AMPLIADAS = INTERIM / "infobras_fichas_ampliadas.jsonl"     # cache: una linea por obra con ficha, avances y paralizaciones
INFOBRAS_COORDENADAS = INTERIM / "infobras_coordenadas.jsonl"      # coordenadas completadas despues (la ficha las carga con retraso)
INFOBRAS_CANDADO = INTERIM / "infobras_navegador.lock"              # impide dos lecturas de INFOBRAS a la vez
# Detalle de los avances paralizados (B2): "todas" lee la pagina de cada mes paralizado; "por_tramo" solo el primer y
# el ultimo mes de cada tramo seguido de meses paralizados (los meses del medio repiten causal y fecha de paralizacion).
INFOBRAS_DETALLE_PARALIZACIONES = "por_tramo"
INFOBRAS_AVANCES = PROCESSED / "infobras_avances.csv"
INFOBRAS_PARALIZACIONES = PROCESSED / "infobras_paralizaciones.csv"
INFOBRAS_FICHAS_AMPLIADAS = PROCESSED / "infobras_fichas_ampliadas.csv"
INFOBRAS_VALIDACION_AMPLIADA = DOCS / "validacion_infobras_ampliada.csv"

# --- MEF (Portal de Datos Abiertos): Formato 12-B de inversiones, con el devengado por CUI ---
MEF_URL_F12B = "https://fs.datosabiertos.mef.gob.pe/datastorefiles/FORMATO_12B.csv"
MEF_URL_F12B_DICCIONARIO = "https://fs.datosabiertos.mef.gob.pe/datastorefiles/F12B_Diccionario.csv"
MEF_FILTROS_TEXTO = "formato_12b"

### `utils` — Utilidades (navegador, descargas, registro, datos personales)
Funciones de apoyo: navegador, log, registro de fuentes, descargas, lectura de .dbf y resumen.

In [ ]:
%%modulo utils
"""Funciones de apoyo: navegador, log, registro de fuentes, descargas, lectura de .dbf y resumen."""
import csv
import json
import random
import re
import shutil
import struct
import time
import unicodedata
import zipfile
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import ElementClickInterceptedException, ElementNotInteractableException

import config

COLUMNAS_REGISTRO = [
    "fuente", "url", "archivo", "anio", "filtros_usados", "fecha_descarga",
    "fecha_corte_del_dato", "tamano_mb", "estado", "notas",
]


# ---------------- log y pausas ----------------
def log(mensaje, archivo=None):
    """Escribe un evento con fecha y hora en pantalla y en el log (por defecto el de descargas; las fases de PDF usan el suyo)."""
    linea = f"{datetime.now():%Y-%m-%d %H:%M:%S} | {mensaje}"
    print(linea)
    with open(archivo or config.LOG, "a", encoding="utf-8") as f:
        f.write(linea + "\n")


def pausa():
    """Pausa aleatoria entre acciones de descarga."""
    time.sleep(random.uniform(config.PAUSA_MIN, config.PAUSA_MAX))


# ---------------- navegador ----------------
def crear_navegador(carpeta_descarga, headless=False, origen_http=None, registrar_red=False):
    """Abre Chrome (visible por defecto) y lo configura para descargar sin preguntar.

    origen_http: sitio que solo funciona por http y del que se confia para poder descargar.
    registrar_red: guarda el trafico de red para poder leer las respuestas que recibe la pagina.
    """
    carpeta_descarga = Path(carpeta_descarga)
    carpeta_descarga.mkdir(parents=True, exist_ok=True)
    opciones = webdriver.ChromeOptions()
    opciones.add_argument(f"--user-agent={config.USER_AGENT}")
    opciones.add_argument("--window-size=1400,1000")
    # evita que Chrome (perfil nuevo) descargue componentes internos dentro de la carpeta de descargas
    opciones.add_argument("--disable-component-update")
    opciones.add_argument("--disable-background-networking")
    if headless:
        opciones.add_argument("--headless=new")
    if origen_http:
        opciones.add_argument(f"--unsafely-treat-insecure-origin-as-secure={origen_http}")
    if registrar_red:
        opciones.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    opciones.add_experimental_option("prefs", {
        "download.default_directory": str(carpeta_descarga),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True,   # los PDF se descargan, no se abren
    })
    return webdriver.Chrome(options=opciones)


def cambiar_carpeta_descarga(driver, carpeta):
    """Cambia la carpeta de descarga del navegador abierto (una carpeta por anio)."""
    Path(carpeta).mkdir(parents=True, exist_ok=True)
    driver.execute_cdp_cmd("Browser.setDownloadBehavior", {"behavior": "allow", "downloadPath": str(carpeta)})


def hacer_clic(driver, elemento):
    """Clic normal; si el elemento esta oculto o tapado, clic por JavaScript."""
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", elemento)
        elemento.click()
    except (ElementNotInteractableException, ElementClickInterceptedException):
        driver.execute_script("arguments[0].click();", elemento)


def limpiar_registro_red(driver):
    """Vacia el registro de red acumulado (cada lectura de get_log lo borra)."""
    driver.get_log("performance")


def respuesta_post(driver, fragmento_url):
    """Espera la respuesta de un POST cuya URL contenga el fragmento y devuelve su cuerpo (texto)."""
    ids_post, vistos = set(), set()
    limite = time.time() + config.SICRECE_ESPERA_RESPUESTA
    while time.time() < limite:
        for entrada in driver.get_log("performance"):
            m = json.loads(entrada["message"])["message"]
            p = m["params"]
            if m["method"] == "Network.requestWillBeSent" and fragmento_url in p["request"]["url"] and p["request"]["method"] == "POST":
                ids_post.add(p["requestId"])
            elif m["method"] == "Network.loadingFinished" and p["requestId"] in ids_post:
                vistos.add(p["requestId"])
        for rid in vistos:
            try:
                return driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": rid})["body"]
            except Exception:
                pass   # la respuesta todavia no esta disponible: se reintenta
        time.sleep(0.5)
    raise TimeoutError("no llego la respuesta de la API")


def captura(driver, nombre):
    """Guarda una captura de pantalla en docs\\capturas."""
    ruta = config.CAPTURAS / f"{nombre}.png"
    try:
        driver.save_screenshot(str(ruta))
        log(f"captura guardada: {ruta.name}")
    except Exception as error:
        log(f"no se pudo guardar la captura {nombre}: {error}")


# ---------------- descargas ----------------
def archivos_actuales(carpeta):
    return {p.name for p in Path(carpeta).iterdir() if p.is_file()}


def limpiar_parciales(carpeta):
    """Borra descargas interrumpidas (.crdownload) que quedaron de un intento anterior."""
    for parcial in Path(carpeta).glob("*.crdownload"):
        parcial.unlink(missing_ok=True)


def esperar_descarga(carpeta, antes, extension):
    """Espera un archivo nuevo con esa extension (ej. '.pdf') y a que termine su .crdownload.

    Se ignoran los demas archivos nuevos: Chrome puede dejar archivos propios en la carpeta.
    """
    extensiones = (extension,) if isinstance(extension, str) else tuple(extension)   # una o varias (ej. .xlsx y .xls)
    terminaciones = tuple(e.lower() for e in extensiones) + tuple(e.lower() + ".crdownload" for e in extensiones)
    limite = time.time() + config.ESPERA_DESCARGA
    while time.time() < limite:
        nuevos = [n for n in archivos_actuales(carpeta) - antes if n.lower().endswith(terminaciones)]
        en_curso = [n for n in nuevos if n.lower().endswith(".crdownload")]
        terminados = [n for n in nuevos if n not in en_curso]
        if terminados and not en_curso:
            return Path(carpeta) / terminados[0]
        time.sleep(1)
    raise TimeoutError("la descarga no termino a tiempo")


def extension_real(ruta):
    """Extension segun el contenido: Chrome a veces guarda un .xlsx con la extension .xls."""
    ruta = Path(ruta)
    if ruta.suffix.lower() in (".xls", ".xlsx"):
        with open(ruta, "rb") as f:
            return ".xlsx" if f.read(2) == b"PK" else ".xls"   # un .xlsx es un zip; el .xls antiguo no
    return ruta.suffix.lower()


def archivo_valido(ruta):
    """Comprueba que un PDF, ZIP o JSON sea de verdad de ese tipo (otros tipos se aceptan)."""
    ruta = Path(ruta)
    tipo = ruta.suffix.lower()
    try:
        if tipo == ".pdf":
            with open(ruta, "rb") as f:
                return f.read(5) == b"%PDF-"
        if tipo in (".zip", ".xlsx"):   # un .xlsx es un zip por dentro
            return zipfile.is_zipfile(ruta)
        if tipo == ".json":
            json.loads(ruta.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return False
    return True


def nombre_limpio(nombre_original):
    """'51_Loc_lineal_1.zip' -> 'loc_lineal_1' (sin tildes, sin numero inicial, en minusculas)."""
    base = Path(nombre_original).stem
    base = unicodedata.normalize("NFKD", base).encode("ascii", "ignore").decode()
    base = re.sub(r"[^a-z0-9]+", "_", base.lower()).strip("_")
    return re.sub(r"^\d+_", "", base)[:60]


def nombre_destino(fuente, anio, descripcion, extension):
    """Formato: {fuente}_{anio}_{descripcion}_{AAAA-MM-DD}.{ext}"""
    return f"{fuente}_{anio}_{descripcion}_{date.today():%Y-%m-%d}.{extension.lstrip('.')}"


def ya_descargado(fuente, anio, filtros):
    """True si el registro tiene esa descarga como 'ok' y el archivo sigue en disco y es valido."""
    if not config.REGISTRO.exists():
        return False
    with open(config.REGISTRO, newline="", encoding="utf-8-sig") as f:
        for fila in csv.DictReader(f):
            if (fila["fuente"] == fuente and fila["anio"] == str(anio) and fila["estado"] == "ok"
                    and fila["filtros_usados"] == filtros):
                ruta = config.RAW / fuente / str(anio) / fila["archivo"]
                if ruta.exists() and archivo_valido(ruta):
                    return True
    return False


def registrar_fuente(**campos):
    """Agrega una fila a docs\\registro_fuentes.csv."""
    existe = config.REGISTRO.exists()
    fila = {c: campos.get(c, "") for c in COLUMNAS_REGISTRO}
    with open(config.REGISTRO, "a", newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS_REGISTRO)
        if not existe:
            escritor.writeheader()
        escritor.writerow(fila)


def guardar_descarga(descargado, carpeta, fuente, anio, nombre_original, url, filtros, fecha_corte="no confirmado", notas=""):
    """Renombra el archivo descargado, lo registra y devuelve (ruta_final, tamano_mb)."""
    if not archivo_valido(descargado):
        raise ValueError(f"el archivo descargado no es un {Path(nombre_original).suffix} valido")
    destino = Path(carpeta) / nombre_destino(
        fuente, anio, nombre_limpio(nombre_original), Path(nombre_original).suffix)
    destino.unlink(missing_ok=True)   # solo existe si se uso --forzar el mismo dia
    shutil.move(str(descargado), str(destino))
    mb = round(destino.stat().st_size / 1024 / 1024, 3)
    registrar_fuente(fuente=fuente, url=url, archivo=destino.name, anio=anio, filtros_usados=filtros,
                     fecha_descarga=str(date.today()), fecha_corte_del_dato=fecha_corte,
                     tamano_mb=mb, estado="ok", notas=notas)
    return destino, mb


def registrar_omitido(fuente, anio, nombre_original, filtros):
    registrar_fuente(fuente=fuente, archivo=nombre_original, anio=anio, filtros_usados=filtros,
                     fecha_descarga=str(date.today()), estado="omitido", notas="ya estaba descargado")


def registrar_error(fuente, anio, nombre_original, filtros, motivo):
    registrar_fuente(fuente=fuente, archivo=nombre_original, anio=anio, filtros_usados=filtros,
                     fecha_descarga=str(date.today()), estado="error", notas=str(motivo)[:200])


# ---------------- resumen ----------------
def nuevo_resultado(fuente):
    return {"fuente": fuente, "descargados": [], "omitidos": [], "errores": [],
            "anios_encontrados": [], "anios_faltantes": [], "anios_sin_archivos": []}


def imprimir_resumen(resultados):
    """Imprime y guarda en el log el resumen final de la corrida."""
    log("=" * 60)
    log("RESUMEN DE LA CORRIDA")
    total_mb = 0.0
    for r in resultados:
        mb = sum(m for _, m in r["descargados"])
        total_mb += mb
        log(f"[{r['fuente']}] descargados={len(r['descargados'])} ({mb:.2f} MB) | "
            f"omitidos (ya existian)={len(r['omitidos'])} | errores={len(r['errores'])}")
        log(f"[{r['fuente']}] anios encontrados desde el inicio: {r['anios_encontrados']}")
        if r["anios_faltantes"]:
            log(f"[{r['fuente']}] anios SIN carpeta en el portal: {r['anios_faltantes']}")
        if r["anios_sin_archivos"]:
            log(f"[{r['fuente']}] anios sin archivos que coincidan con la seleccion: {r['anios_sin_archivos']}")
        for nombre, motivo in r["errores"]:
            log(f"[{r['fuente']}] ERROR {nombre}: {motivo}")
    log(f"Peso total descargado en esta corrida: {total_mb:.2f} MB")
    log("=" * 60)


# ---------------- lectura de .dbf ----------------
def campos_dbf(ruta):
    """Lee la cabecera de un .dbf: devuelve (n_registros, largo_registro, inicio_datos, codigo_idioma, campos)."""
    with open(ruta, "rb") as f:
        cab = f.read(32)
        n_reg = struct.unpack("<I", cab[4:8])[0]
        largo_cab, largo_reg = struct.unpack("<HH", cab[8:12])
        campos = []
        while True:
            d = f.read(32)
            if d[:1] == b"\r":
                break
            nombre = d[:11].split(b"\x00")[0].decode("ascii")
            campos.append((nombre, chr(d[11]), d[16], d[17]))   # nombre, tipo, largo, decimales
    return n_reg, largo_reg, largo_cab, cab[29], campos


# byte de idioma de la cabecera del .dbf -> pagina de codigos (0x03 y otros: UTF-8 o cp1252, se prueba)
CODIFICACION_DBF = {0x01: "cp437", 0x02: "cp850", 0x57: "cp1252", 0xC8: "cp1250"}


def _decodificar(columna, codificacion):
    """Con codificacion=None prueba UTF-8 y, si falla, cp1252 (los archivos de ESCALE mezclan ambas)."""
    if codificacion:
        return np.char.decode(columna, codificacion, "replace")
    try:
        return np.char.decode(columna, "utf-8")
    except UnicodeDecodeError:
        # columna con valores mezclados: cada valor distinto se decodifica como UTF-8 si se puede y si no como cp1252
        unicos, posicion = np.unique(columna, return_inverse=True)
        textos = np.array([_texto(valor) for valor in unicos])
        return textos[posicion]


def _texto(valor):
    try:
        return valor.decode("utf-8")
    except UnicodeDecodeError:
        return valor.decode("cp1252", "replace")


def leer_dbf(ruta, columnas=None, max_filas=None, codificacion="auto"):
    """Lee un .dbf como texto (conserva ceros a la izquierda). columnas=None lee todas."""
    n_reg, largo_reg, largo_cab, idioma, campos = campos_dbf(ruta)
    if codificacion == "auto":
        codificacion = CODIFICACION_DBF.get(idioma)
    dtipo = np.dtype([("_borrado", "S1")] + [(n, f"S{largo}") for n, _, largo, _ in campos])
    assert dtipo.itemsize == largo_reg, "el largo de registro no coincide con la cabecera"
    filas = n_reg if max_filas is None else min(max_filas, n_reg)
    datos = np.fromfile(ruta, dtype=dtipo, count=filas, offset=largo_cab)
    datos = datos[datos["_borrado"] != b"*"]
    nombres = columnas or [n for n, *_ in campos]
    tabla = {n: np.char.strip(_decodificar(datos[n], codificacion)) for n in nombres}
    return pd.DataFrame(tabla)


# ---------------- datos personales ----------------
OMITIDO = "[NOMBRE OMITIDO]"
# una palabra de nombre nunca puede ser un cargo ni «consorcio»: asi un nombre no se come al cargo que le sigue
_NO_NOMBRE = r"(?!(?i:residente|supervisor|supervisora|inspector|inspectora|proyectista|consorcio|contratista|tipo|obra)\b)"
_PALABRA_MAYUS = _NO_NOMBRE + r"[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÑáéíóúñ'’]{1,}"
_PALABRA_TITULO = _NO_NOMBRE + r"[A-ZÁÉÍÓÚÑ][a-záéíóúñ'’]{1,}"
_NOMBRE_MAYUS = rf"{_PALABRA_MAYUS}(?:\s+(?:de|del|de\s+la|DE|DEL|DE\s+LA)\s+{_PALABRA_MAYUS}|\s+{_PALABRA_MAYUS}){{1,4}}"
_NOMBRE_TITULO = rf"{_PALABRA_TITULO}(?:\s+(?:de|del|de\s+la)\s+{_PALABRA_TITULO}|\s+{_PALABRA_TITULO}){{1,4}}"
_NOMBRE_SOLO_MAYUS = _NO_NOMBRE + r"[A-ZÁÉÍÓÚÑ]{3,}(?:\s+" + _NO_NOMBRE + r"[A-ZÁÉÍÓÚÑ]{2,}){1,3}"
_TITULOS = r"(?:ing|arq|abog|econ|lic|sr|sra|srta|cpc|bach)"   # el punto es opcional; «Dr.» y «Dra.» llevan punto obligatorio (DRA = Direccion Regional de Agricultura)
_CARGOS = r"(?:residente|supervisor|supervisora|inspector|inspectora|proyectista)(?:\s+de\s+obra)?"
_NO_ES_NOMBRE = ("consorcio", "empresa", "municipalidad", "gobierno", "unidad", "proyecto", "obra", "entidad", "contratista", "sociedad",
                 "universidad", "comunica", "presenta", "informa", "solicita", "indica", "cuenta", "no", "se", "el", "la", "de", "del", "y")


def omitir_nombres(texto):
    """Reemplaza nombres de personas naturales en un comentario libre por [NOMBRE OMITIDO].

    Reglas: (1) un titulo (Ing., Arq., Lic., Sr., Dr., ...) seguido de un nombre; (2) un cargo (residente, supervisor, inspector)
    seguido de un nombre con inicial mayuscula (no si va despues de «Consorcio»); (3) un cargo seguido de dos puntos y un nombre en
    mayusculas. Es una barrera simple: ante la duda borra de mas. datos_personales.auditar() la vigila con patrones propios.
    """
    if not isinstance(texto, str) or not texto:
        return texto
    texto = re.sub(rf"\b(?i:{_TITULOS})\.?\s+{_NOMBRE_MAYUS}", OMITIDO, texto)
    texto = re.sub(rf"\b(?i:dra?)\.\s+{_NOMBRE_MAYUS}", OMITIDO, texto)

    def por_cargo(m):
        primera = m.group("nombre").split()[0].lower()
        return m.group(0) if primera in _NO_ES_NOMBRE else f"{m.group('cargo')}{m.group('sep')}{OMITIDO}"

    texto = re.sub(rf"(?<!(?i:consorcio) )(?P<cargo>\b(?i:{_CARGOS}))(?P<sep>[,:]?\s+)(?P<nombre>{_NOMBRE_TITULO})", por_cargo, texto)
    return re.sub(rf"(?P<cargo>\b(?i:{_CARGOS}))(?P<sep>\s*:\s*)(?P<nombre>{_NOMBRE_SOLO_MAYUS})", por_cargo, texto)

### `scraper_escale` — Descarga ESCALE
ESCALE: abre el portal, recorre cada anio del Censo Educativo y descarga los archivos de local educativo.

In [ ]:
%%modulo scraper_escale
"""ESCALE: abre el portal, recorre cada anio del Censo Educativo y descarga los archivos de local educativo."""
import re
from pathlib import Path

from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

import config
import utils


def esperar(driver):
    return WebDriverWait(driver, config.ESPERA_ELEMENTO)


def abrir_portal(driver):
    driver.get(config.ESCALE_URL_INICIO)
    esperar(driver).until(EC.presence_of_element_located((By.LINK_TEXT, config.ESCALE_CARPETA_CENSO)))


def elegir_carpeta_censo(driver):
    driver.find_element(By.LINK_TEXT, config.ESCALE_CARPETA_CENSO).click()
    esperar(driver).until(EC.presence_of_element_located((By.PARTIAL_LINK_TEXT, "20")))


def listar_anios(driver):
    """Devuelve {anio: enlace_de_la_carpeta} leyendo las carpetas que hay en el portal."""
    anios = {}
    for enlace in driver.find_elements(By.TAG_NAME, "a"):
        href = enlace.get_attribute("href") or ""
        if re.fullmatch(r"\d{4}", enlace.text.strip()) and "document_library_display" in href:
            anios[int(enlace.text.strip())] = href
    return anios


def elegir_anio(driver, enlace_anio):
    driver.get(enlace_anio)
    esperar(driver).until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))


def listar_archivos(driver):
    """Recorre todas las paginas de la carpeta y devuelve {nombre_archivo: enlace_de_detalle}."""
    archivos = {}
    for _ in range(config.ESCALE_MAX_PAGINAS):
        for enlace in driver.find_elements(By.CSS_SELECTOR, config.ESCALE_SELECTOR_ARCHIVOS):
            # el texto a veces trae una descripcion pegada: se corta en la extension
            m = re.match(rf"^(.+?{config.EXTENSIONES})\b", enlace.text.strip(), re.I)
            if m:
                archivos.setdefault(m.group(1), enlace.get_attribute("href"))
        siguientes = [a for a in driver.find_elements(By.LINK_TEXT, config.ESCALE_SIGUIENTE) if a.is_displayed()]
        if not siguientes:
            break
        siguientes[0].click()
        esperar(driver).until(EC.staleness_of(siguientes[0]))
    return archivos


def elegir_objetivos(archivos):
    """Se queda solo con los archivos que coinciden con los patrones de config.py."""
    elegidos = {}
    for nombre, enlace in archivos.items():
        for patron in config.ESCALE_PATRONES.values():
            if re.search(patron, nombre):
                elegidos[nombre] = enlace
                break
    return elegidos


def descargar_archivo(driver, carpeta, anio, nombre, enlace_detalle, forzar, resultado):
    filtros = f"Censo Educativo > {anio} > {nombre}"
    if not forzar and utils.ya_descargado("escale", anio, filtros):
        utils.log(f"ESCALE {anio}: {nombre} ya estaba descargado, se omite")
        utils.registrar_omitido("escale", anio, nombre, filtros)
        resultado["omitidos"].append(nombre)
        return
    motivo = ""
    for intento in range(1, config.REINTENTOS + 1):
        try:
            utils.limpiar_parciales(carpeta)
            driver.get(enlace_detalle)
            boton = esperar(driver).until(EC.element_to_be_clickable((By.XPATH, config.ESCALE_BOTON_DESCARGAR)))
            url = boton.get_attribute("href")
            antes = utils.archivos_actuales(carpeta)
            boton.click()
            descargado = utils.esperar_descarga(carpeta, antes, Path(nombre).suffix)
            destino, mb = utils.guardar_descarga(descargado, carpeta, "escale", anio, nombre, url, filtros)
            utils.log(f"ESCALE {anio}: descargado {destino.name} ({mb} MB)")
            resultado["descargados"].append((destino.name, mb))
            return
        except (TimeoutException, TimeoutError, WebDriverException, ValueError) as error:
            motivo = f"{type(error).__name__}"
            utils.log(f"ESCALE {anio}: intento {intento}/{config.REINTENTOS} fallo para {nombre}: {motivo}")
            utils.captura(driver, f"error_escale_{anio}_{utils.nombre_limpio(nombre)}_intento{intento}")
            utils.pausa()
    utils.registrar_error("escale", anio, nombre, filtros, motivo)
    resultado["errores"].append((f"{anio} {nombre}", motivo))


def ejecutar(anio_inicio=config.ANIO_INICIO, forzar=False, headless=False):
    resultado = utils.nuevo_resultado("ESCALE")
    driver = utils.crear_navegador(config.RAW / "escale", headless)
    try:
        try:
            abrir_portal(driver)
            elegir_carpeta_censo(driver)
            anios = listar_anios(driver)
        except (TimeoutException, WebDriverException) as error:
            utils.log(f"ESCALE: el portal no responde o cambio de estructura ({type(error).__name__}). Se omite la fuente.")
            utils.captura(driver, "error_escale_portal")
            resultado["errores"].append(("portal", type(error).__name__))
            return resultado

        disponibles = sorted(a for a in anios if a >= anio_inicio)
        resultado["anios_encontrados"] = disponibles
        if anios:
            resultado["anios_faltantes"] = [a for a in range(anio_inicio, max(anios) + 1) if a not in anios]
        utils.log(f"ESCALE: anios disponibles desde {anio_inicio}: {disponibles}")

        for anio in disponibles:
            try:
                carpeta = config.RAW / "escale" / str(anio)
                utils.cambiar_carpeta_descarga(driver, carpeta)
                elegir_anio(driver, anios[anio])
                archivos = listar_archivos(driver)
                objetivos = elegir_objetivos(archivos)
                utils.log(f"ESCALE {anio}: {len(archivos)} archivos en el portal, {len(objetivos)} seleccionados")
                if not objetivos:
                    resultado["anios_sin_archivos"].append(anio)
                for nombre, enlace in objetivos.items():
                    descargar_archivo(driver, carpeta, anio, nombre, enlace, forzar, resultado)
                    utils.pausa()
            except Exception as error:   # un anio que falla no detiene a los demas
                utils.log(f"ESCALE {anio}: fallo el anio completo: {type(error).__name__} {error}")
                utils.captura(driver, f"error_escale_anio_{anio}")
                resultado["errores"].append((f"{anio} (anio completo)", type(error).__name__))
    finally:
        driver.quit()
    return resultado

### `scraper_enla` — Descarga ENLA
ENLA (UMC): descubre las paginas de resultados de cada anio y descarga los PDF de reportes.

In [ ]:
%%modulo scraper_enla
"""ENLA (UMC): descubre las paginas de resultados de cada anio y descarga los PDF de reportes."""
import json
import os
import re
import time
from urllib.parse import unquote, urlparse

from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

import config
import utils


def esperar(driver):
    return WebDriverWait(driver, config.ESPERA_ELEMENTO)


def abrir_portal(driver):
    driver.get(config.ENLA_URL_INICIO)
    esperar(driver).until(EC.presence_of_element_located((By.TAG_NAME, "a")))


def listar_anios(driver):
    """Devuelve {anio: (etiqueta, enlace)} leyendo los enlaces 'Resultados ...' del portal."""
    anios = {}
    for enlace in driver.find_elements(By.TAG_NAME, "a"):
        m = re.search(config.ENLA_REGEX_PAGINA_ANIO, enlace.get_attribute("href") or "")
        if m:
            anio = int(m.group(2))
            anios.setdefault(anio, (f"{m.group(1).upper()} {anio}", enlace.get_attribute("href")))
    return anios


def elegir_anio(driver, enlace_anio):
    driver.get(enlace_anio)
    esperar(driver).until(EC.presence_of_element_located((By.TAG_NAME, "a")))


def listar_pdfs(driver):
    """Devuelve {nombre_pdf: enlace} con los PDF de la pagina, sin catalogo, infografias ni normas."""
    pdfs = {}
    for enlace in driver.find_elements(By.TAG_NAME, "a"):
        href = enlace.get_attribute("href") or ""
        if re.search(config.ENLA_REGEX_PDF, href, re.I) and not re.search(config.ENLA_EXCLUIR, href):
            pdfs.setdefault(unquote(os.path.basename(urlparse(href).path)), href)
    return pdfs


def descargar_pdf(driver, carpeta, anio, etiqueta, nombre, href, forzar, resultado):
    filtros = f"Resultados {etiqueta} > {nombre}"
    if not forzar and utils.ya_descargado("enla", anio, filtros):
        utils.log(f"ENLA {anio}: {nombre} ya estaba descargado, se omite")
        utils.registrar_omitido("enla", anio, nombre, filtros)
        resultado["omitidos"].append(nombre)
        return
    nombre_en_url = os.path.basename(urlparse(href).path)   # tal como aparece en el enlace (con %)
    motivo = ""
    for intento in range(1, config.REINTENTOS + 1):
        try:
            utils.limpiar_parciales(carpeta)
            xpath = config.ENLA_XPATH_ENLACE_PDF.format(codificado=nombre_en_url, decodificado=nombre)
            enlace = esperar(driver).until(EC.presence_of_element_located((By.XPATH, xpath)))
            antes = utils.archivos_actuales(carpeta)
            utils.hacer_clic(driver, enlace)
            descargado = utils.esperar_descarga(carpeta, antes, ".pdf")
            destino, mb = utils.guardar_descarga(descargado, carpeta, "enla", anio, nombre, href, filtros)
            utils.log(f"ENLA {anio}: descargado {destino.name} ({mb} MB)")
            resultado["descargados"].append((destino.name, mb))
            return
        except (TimeoutException, TimeoutError, WebDriverException, ValueError) as error:
            motivo = f"{type(error).__name__}"
            utils.log(f"ENLA {anio}: intento {intento}/{config.REINTENTOS} fallo para {nombre}: {motivo}")
            utils.captura(driver, f"error_enla_{anio}_{utils.nombre_limpio(nombre)}_intento{intento}")
            utils.pausa()
    utils.registrar_error("enla", anio, nombre, filtros, motivo)
    resultado["errores"].append((f"{anio} {nombre}", motivo))


def ejecutar_umc(resultado, anio_inicio, forzar, headless):
    """PDF de reportes de la UMC."""
    driver = utils.crear_navegador(config.RAW / "enla", headless, origen_http=config.ENLA_ORIGEN_HTTP)
    try:
        try:
            abrir_portal(driver)
            anios = listar_anios(driver)
        except (TimeoutException, WebDriverException) as error:
            utils.log(f"ENLA: el portal no responde o cambio de estructura ({type(error).__name__}). Se omite la fuente.")
            utils.captura(driver, "error_enla_portal")
            resultado["errores"].append(("portal", type(error).__name__))
            return

        disponibles = sorted(a for a in anios if a >= anio_inicio)
        resultado["anios_encontrados"] = disponibles
        if anios:
            resultado["anios_faltantes"] = [a for a in range(anio_inicio, max(anios) + 1) if a not in anios]
        utils.log(f"ENLA: anios con pagina de resultados desde {anio_inicio}: {disponibles}")

        for anio in disponibles:
            try:
                etiqueta, enlace_anio = anios[anio]
                carpeta = config.RAW / "enla" / str(anio)
                utils.cambiar_carpeta_descarga(driver, carpeta)
                elegir_anio(driver, enlace_anio)
                pdfs = listar_pdfs(driver)
                utils.log(f"ENLA {anio} ({etiqueta}): {len(pdfs)} PDF de reportes encontrados")
                if not pdfs:
                    resultado["anios_sin_archivos"].append(anio)
                for nombre, href in pdfs.items():
                    descargar_pdf(driver, carpeta, anio, etiqueta, nombre, href, forzar, resultado)
                    utils.pausa()
            except Exception as error:   # un anio que falla no detiene a los demas
                utils.log(f"ENLA {anio}: fallo el anio completo: {type(error).__name__} {error}")
                utils.captura(driver, f"error_enla_anio_{anio}")
                resultado["errores"].append((f"{anio} (anio completo)", type(error).__name__))
    finally:
        driver.quit()


# ---------------- SICRECE (acceso publico): resultados por region ----------------
def abrir_sicrece(driver):
    """Entra por ACCESO PUBLICO con el rol autorizado (sin credenciales) y abre la Busqueda avanzada."""
    driver.get(config.SICRECE_URL)
    esperar(driver).until(EC.element_to_be_clickable((By.XPATH, config.SICRECE_XPATH_ACCESO_PUBLICO))).click()
    esperar(driver).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, f"{config.SICRECE_SELECT_ROL} option[value='{config.SICRECE_ROL_VALOR}']")))
    Select(driver.find_element(By.CSS_SELECTOR, config.SICRECE_SELECT_ROL)).select_by_value(config.SICRECE_ROL_VALOR)
    esperar(driver).until(EC.element_to_be_clickable((By.ID, config.SICRECE_BOTON_INGRESAR))).click()
    pestana = esperar(driver).until(EC.presence_of_element_located(
        (By.XPATH, f"//*[normalize-space(text())='{config.SICRECE_PESTANA_BUSQUEDA}']")))
    utils.hacer_clic(driver, pestana)
    esperar(driver).until(lambda d: len(selects_formulario(d)) >= 2)


def cuerpo_activo(driver):
    return [c for c in driver.find_elements(By.CSS_SELECTOR, "mat-tab-body")
            if "mat-tab-body-active" in (c.get_attribute("class") or "")][0]


def selects_formulario(driver):
    """Listas desplegables visibles del formulario (sin la lista de anios de la tarjeta del grafico)."""
    return [s for s in cuerpo_activo(driver).find_elements(By.TAG_NAME, "select")
            if s.is_displayed() and not driver.execute_script("return !!arguments[0].closest('app-horizontal-bar-chart')", s)]


def elegir_opcion(driver, campo, texto):
    """Espera a que la opcion exista (se carga despues del filtro anterior) y la elige. False si no existe."""
    posicion = config.SICRECE_ORDEN_SELECTS[campo]
    try:
        WebDriverWait(driver, 10).until(lambda d: len(selects_formulario(d)) > posicion and texto in
                                        [o.text.strip() for o in Select(selects_formulario(d)[posicion]).options])
    except TimeoutException:
        return False
    Select(selects_formulario(driver)[posicion]).select_by_visible_text(texto)
    time.sleep(1.5)
    return True


def opciones_de(driver, campo):
    posicion = config.SICRECE_ORDEN_SELECTS[campo]
    esperar(driver).until(lambda d: len(selects_formulario(d)) > posicion)
    time.sleep(1.5)
    return [o.text.strip() for o in Select(selects_formulario(driver)[posicion]).options if o.text.strip() != "SELECCIONAR"]


def marcar_casilla(driver, texto):
    """Marca la casilla con ese texto si no esta marcada (idempotente)."""
    esperar(driver).until(lambda d: any(" ".join(c.text.split()) == texto
                                        for c in cuerpo_activo(d).find_elements(By.TAG_NAME, "mat-checkbox")))
    casilla = [c for c in cuerpo_activo(driver).find_elements(By.TAG_NAME, "mat-checkbox") if " ".join(c.text.split()) == texto][0]
    if not casilla.find_element(By.CSS_SELECTOR, "input").is_selected():
        utils.hacer_clic(driver, casilla.find_element(By.CSS_SELECTOR, "label"))
        time.sleep(0.5)


def elegir_todas_las_dre(driver):
    desplegable = esperar(driver).until(lambda d: next(iter([m for m in cuerpo_activo(d).find_elements(By.TAG_NAME, "mat-select")
                                                             if m.is_displayed()][-1:]), False))
    utils.hacer_clic(driver, desplegable)
    opcion = esperar(driver).until(EC.presence_of_element_located(
        (By.XPATH, f"//div[contains(@class,'cdk-overlay-container')]//mat-option[normalize-space(.)='{config.SICRECE_DRE_TODAS}']")))
    if opcion.get_attribute("aria-selected") != "true":   # si ya estaba marcada, pulsarla la desmarcaria
        utils.hacer_clic(driver, opcion)
    time.sleep(1)
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
    time.sleep(1)
    esperar(driver).until(lambda d: "(+" in " ".join(" ".join(m.text for m in cuerpo_activo(d).find_elements(By.TAG_NAME, "mat-select")).split()))   # "DRE Amazonas (+25)" = las 26 DRE


def consultar(driver, tipo, anio, nivel, grado):
    """Rellena el formulario, pulsa Buscar y devuelve el JSON que recibe la pagina (texto)."""
    elegir_opcion(driver, "tipo", tipo)
    elegir_opcion(driver, "anio", str(anio))
    elegir_opcion(driver, "nivel", nivel)
    elegir_opcion(driver, "grado", grado)
    for area in config.SICRECE_AREAS:
        marcar_casilla(driver, area)
    marcar_casilla(driver, config.SICRECE_AMBITO)
    elegir_todas_las_dre(driver)
    utils.limpiar_registro_red(driver)
    boton = driver.find_element(By.XPATH, config.SICRECE_BOTON_BUSCAR)
    utils.hacer_clic(driver, boton)
    return utils.respuesta_post(driver, config.SICRECE_API_REPORTE[tipo])


def combinaciones_sicrece(driver, anio_inicio):
    """Recorre tipo > anio > nivel > grado y devuelve las combinaciones de (tipo, anio, nivel, grado) que interesan."""
    encontradas, anios_vistos = [], set()
    for tipo in config.SICRECE_TIPOS:
        elegir_opcion(driver, "tipo", tipo)
        for anio in [int(a) for a in opciones_de(driver, "anio") if a.isdigit() and int(a) >= anio_inicio]:
            anios_vistos.add(anio)
            elegir_opcion(driver, "anio", str(anio))
            for nivel in opciones_de(driver, "nivel"):
                elegir_opcion(driver, "nivel", nivel)
                for grado in opciones_de(driver, "grado"):
                    if (nivel, grado) in config.SICRECE_GRADOS and (tipo, anio, nivel, grado) not in config.SICRECE_SIN_RESPUESTA:
                        encontradas.append((tipo, anio, nivel, grado))
    return encontradas, sorted(anios_vistos)


def guardar_respuesta(texto, carpeta, anio, tipo, nivel, grado, filtros):
    """Guarda el JSON recibido como archivo crudo y lo registra."""
    json.loads(texto)   # debe ser JSON valido
    temporal = carpeta / "_sicrece_temporal.json"
    temporal.write_text(texto, encoding="utf-8")
    nombre_logico = f"sicrece_{tipo}_{nivel}_{grado}.json".lower()
    destino, mb = utils.guardar_descarga(temporal, carpeta, "enla", anio, nombre_logico, config.SICRECE_API_REPORTE[tipo], filtros)
    return destino, mb


def ejecutar_sicrece(resultado, anio_inicio, forzar, headless):
    carpeta_base = config.RAW / "enla"
    driver = utils.crear_navegador(carpeta_base, headless, registrar_red=True)
    try:
        try:
            abrir_sicrece(driver)
            combinaciones, anios = combinaciones_sicrece(driver, anio_inicio)
        except (TimeoutException, WebDriverException) as error:
            utils.log(f"SICRECE: el portal no responde o cambio de estructura ({type(error).__name__}). Se omite.")
            utils.captura(driver, "error_sicrece_portal")
            resultado["errores"].append(("SICRECE portal", type(error).__name__))
            return
        utils.log(f"SICRECE: anios {anios}; {len(combinaciones)} combinaciones de interes: {combinaciones}")
        for tipo, anio, nivel, grado in combinaciones:
            filtros = f"SICRECE > {tipo} > {anio} > {nivel} > {grado} > {'+'.join(config.SICRECE_AREAS)} > {config.SICRECE_AMBITO} {config.SICRECE_DRE_TODAS}"
            nombre = f"sicrece_{tipo}_{nivel}_{grado}".lower()
            carpeta = carpeta_base / str(anio)
            carpeta.mkdir(parents=True, exist_ok=True)
            if not forzar and utils.ya_descargado("enla", anio, filtros):
                utils.log(f"SICRECE {anio}: {tipo} {nivel} {grado} ya estaba descargado, se omite")
                utils.registrar_omitido("enla", anio, nombre, filtros)
                resultado["omitidos"].append(nombre)
                continue
            motivo = ""
            for intento in range(1, config.REINTENTOS + 1):
                try:
                    texto = consultar(driver, tipo, anio, nivel, grado)
                    destino, mb = guardar_respuesta(texto, carpeta, anio, tipo, nivel, grado, filtros)
                    utils.log(f"SICRECE {anio}: guardado {destino.name} ({mb} MB)")
                    resultado["descargados"].append((destino.name, mb))
                    break
                except Exception as error:
                    motivo = type(error).__name__
                    utils.log(f"SICRECE {anio}: intento {intento}/{config.REINTENTOS} fallo para {tipo} {nivel} {grado}: {motivo} {str(error)[:120]}")
                    utils.captura(driver, f"error_sicrece_{anio}_{tipo}_{nivel}_{grado}_intento{intento}".lower())
                    utils.pausa()
            else:
                utils.registrar_error("enla", anio, nombre, filtros, motivo)
                resultado["errores"].append((f"{anio} SICRECE {tipo} {nivel} {grado}", motivo))
            utils.pausa()
    finally:
        driver.quit()


def ejecutar(anio_inicio=config.ANIO_INICIO, forzar=False, headless=False):
    resultado = utils.nuevo_resultado("ENLA")
    for paso in (ejecutar_umc, ejecutar_sicrece):
        try:
            paso(resultado, anio_inicio, forzar, headless)
        except Exception as error:   # si un paso falla se sigue con el otro
            utils.log(f"ENLA: {paso.__name__} fallo: {type(error).__name__} {error}")
            resultado["errores"].append((paso.__name__, type(error).__name__))
    return resultado

### `scraper_contraloria` — Descarga Contraloría
Contraloria (gob.pe): descarga los informes de obras paralizadas y sus anexos (PDF y Excel).

In [ ]:
%%modulo scraper_contraloria
"""Contraloria (gob.pe): descarga los informes de obras paralizadas y sus anexos (PDF y Excel)."""
import re
import time
from datetime import date

from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

import config
import utils

NOMBRE_MES = {1: "enero", 2: "febrero", 3: "marzo", 4: "abril", 5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
              9: "setiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"}
MESES = "|".join(config.CGR_MESES)

# JavaScript: para cada boton "Descargar" devuelve su enlace y las lineas del bloque (etiqueta, formato, tamano)
JS_ARCHIVOS = """
return Array.from(document.querySelectorAll('main a'))
  .filter(a => (a.textContent || '').trim() === 'Descargar' && (a.href || '').includes('/uploads/'))
  .map(a => {
    let bloque = a;
    for (let i = 0; i < 6; i++) {
      bloque = bloque.parentElement;
      if (!bloque) break;
      const veces = (bloque.innerText.match(/Descargar/g) || []).length;
      const lineas = bloque.innerText.split('\\n').filter(x => x.trim());
      if (veces === 1 && lineas.length >= 3) break;
    }
    const lineas = bloque ? bloque.innerText.split('\\n').map(x => x.trim()).filter(x => x) : [];
    return {url: a.href, lineas: lineas};
  });
"""


def esperar(driver):
    return WebDriverWait(driver, config.ESPERA_ELEMENTO)


# ---------------- ruta de clics y listado ----------------
def abrir_portal(driver):
    driver.get(config.CGR_URL_PORTAL)
    esperar(driver).until(EC.presence_of_element_located((By.LINK_TEXT, config.CGR_TEXTO_CATEGORIA)))


def abrir_compendio(driver):
    """Portal > categoria 'Resultados de la Contraloria' > compendio 'Obras paralizadas - Documentos'."""
    utils.hacer_clic(driver, driver.find_element(By.LINK_TEXT, config.CGR_TEXTO_CATEGORIA))
    compendio = esperar(driver).until(EC.presence_of_element_located((By.LINK_TEXT, config.CGR_TEXTO_COMPENDIO)))
    utils.hacer_clic(driver, compendio)
    esperar(driver).until(EC.presence_of_element_located((By.ID, config.CGR_ID_POR_PAGINA)))


def mostrar_todos(driver):
    """Elige '100 por pagina' para ver la coleccion completa sin pasar a las paginas siguientes.

    Si la coleccion cabe en una pagina (hasta 25 resultados) esa opcion viene desactivada y no hace falta.
    """
    lista = Select(driver.find_element(By.ID, config.CGR_ID_POR_PAGINA))
    opcion = next(o for o in lista.options if o.text.strip() == config.CGR_OPCION_POR_PAGINA)
    if opcion.is_enabled():
        lista.select_by_visible_text(config.CGR_OPCION_POR_PAGINA)
        time.sleep(4)
    else:
        utils.log("CONTRALORIA: la coleccion cabe en una sola pagina; no hace falta cambiar 'por pagina'")
    esperar(driver).until(EC.presence_of_element_located((By.CSS_SELECTOR, f"main a[href*='{config.CGR_HREF_FICHA}']")))


def total_resultados(driver):
    """Numero de resultados que anuncia la pagina ('25 resultados:'); None si no se encuentra."""
    m = re.search(r"([\d.,]+) resultados", driver.find_element(By.TAG_NAME, "main").text)
    return int(re.sub(r"\D", "", m.group(1))) if m else None


def listar_items(driver):
    """Devuelve [(titulo, enlace_de_la_ficha)] de la coleccion, en el orden en que aparecen."""
    items = []
    for a in driver.find_elements(By.CSS_SELECTOR, f"main a[href*='{config.CGR_HREF_FICHA}']"):
        titulo = " ".join((a.text or "").split())
        href = a.get_attribute("href")
        if titulo and (titulo, href) not in items:
            items.append((titulo, href))
    return items


# ---------------- lectura de cada ficha ----------------
def fecha_en_texto(texto):
    """Primera fecha 'D de mes de AAAA' (o 'del AAAA') que aparece en el texto."""
    m = re.search(rf"(\d{{1,2}}) de ({MESES}) de(?:l)? (\d{{4}})", texto, re.I)
    return date(int(m.group(3)), config.CGR_MESES[m.group(2).lower()], int(m.group(1))) if m else None


def fecha_de_corte(titulo, descripcion):
    """Fecha de corte del informe: fin del mes o trimestre que dice el titulo; si no, una fecha 'al D de mes de AAAA' del texto."""
    m = re.search(rf"\ba (?:{MESES}) (?:de )?\d{{4}}", titulo, re.I)
    if m:
        mes, anio = re.search(rf"({MESES}) (?:de )?(\d{{4}})", m.group(0), re.I).groups()
        mes = config.CGR_MESES[mes.lower()]
        return date(int(anio), mes, [31, 29 if int(anio) % 4 == 0 else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][mes - 1])
    m = re.search(r"\b(I{1,3}|IV) trimestre (?:de )?(\d{4})", titulo, re.I)
    if m:
        mes, dia = config.CGR_TRIMESTRES[m.group(1).lower()]
        return date(int(m.group(2)), mes, dia)
    m = re.search(rf"\bal (\d{{1,2}}) de ({MESES}) de(?:l)? (\d{{4}})", descripcion, re.I)
    if m:
        return date(int(m.group(3)), config.CGR_MESES[m.group(2).lower()], int(m.group(1)))
    return None


def leer_ficha(driver, enlace):
    """Abre la ficha de un informe y devuelve titulo, fecha de publicacion, fecha de corte y sus archivos."""
    driver.get(enlace)
    esperar(driver).until(EC.presence_of_element_located((By.CSS_SELECTOR, "main h1, main h2")))
    time.sleep(2)
    texto = driver.find_element(By.TAG_NAME, "main").text
    lineas = [l.strip() for l in texto.splitlines() if l.strip()]
    titulo = next((l for l in lineas if re.match(config.CGR_REGEX_INFORME, l)), lineas[1] if len(lineas) > 1 else "")
    archivos = []
    for a in driver.execute_script(JS_ARCHIVOS):
        archivos.append({"url": a["url"], "etiqueta": a["lineas"][0] if a["lineas"] else "",
                         "formato": a["lineas"][1].lower() if len(a["lineas"]) > 1 else "",
                         "tamano": a["lineas"][2] if len(a["lineas"]) > 2 else ""})
    return {"titulo": titulo, "publicacion": fecha_en_texto(texto), "corte": fecha_de_corte(titulo, texto), "archivos": archivos}


def descripcion_archivo(archivo, ficha):
    """Nombre corto del archivo: 'informe' o 'anexo_NN', mas el periodo de corte (ej. informe_obras_paralizadas_junio_2026)."""
    m = re.search(r"anexo\s*n?\.?\s*[°º]?\s*(\d+)", archivo["etiqueta"], re.I)
    base = f"anexo_{int(m.group(1)):02d}" if m else "informe"
    fecha = ficha["corte"] or ficha["publicacion"]
    periodo = f"{NOMBRE_MES[fecha.month]}_{fecha.year}" if fecha else "sin_fecha"
    return f"{base}_obras_paralizadas_{periodo}"


# ---------------- descarga ----------------
def descargar_archivo(driver, ficha, archivo, enlace_ficha, forzar, resultado):
    anio = (ficha["corte"] or ficha["publicacion"]).year
    carpeta = config.RAW / "contraloria" / str(anio)
    carpeta.mkdir(parents=True, exist_ok=True)
    extension = "." + archivo["formato"]
    extensiones = (".xlsx", ".xls") if archivo["formato"] in ("xlsx", "xls") else (extension,)
    nombre = descripcion_archivo(archivo, ficha) + extension
    filtros = f"gob.pe > Contraloría > {config.CGR_TEXTO_COMPENDIO} > {ficha['titulo']} > {archivo['etiqueta']}"
    if not forzar and utils.ya_descargado("contraloria", anio, filtros):
        utils.log(f"CONTRALORIA {anio}: {nombre} ya estaba descargado, se omite")
        utils.registrar_omitido("contraloria", anio, nombre, filtros)
        resultado["omitidos"].append(nombre)
        return
    id_archivo = re.search(r"/file/(\d+)/", archivo["url"]).group(1)
    motivo = ""
    for intento in range(1, config.REINTENTOS + 1):
        try:
            utils.cambiar_carpeta_descarga(driver, carpeta)
            utils.limpiar_parciales(carpeta)
            boton = esperar(driver).until(EC.presence_of_element_located(
                (By.XPATH, f"//main//a[contains(@href, '/file/{id_archivo}/') and normalize-space(.)='Descargar']")))
            antes = utils.archivos_actuales(carpeta)
            utils.hacer_clic(driver, boton)
            descargado = utils.esperar_descarga(carpeta, antes, extensiones)
            nombre_real = descripcion_archivo(archivo, ficha) + utils.extension_real(descargado)
            corte = ficha["corte"].isoformat() if ficha["corte"] else "no confirmado"
            destino, mb = utils.guardar_descarga(
                descargado, carpeta, "contraloria", anio, nombre_real, archivo["url"], filtros, fecha_corte=corte,
                notas=f"publicado {ficha['publicacion']}; ficha {enlace_ficha}")
            utils.log(f"CONTRALORIA {anio}: descargado {destino.name} ({mb} MB)")
            resultado["descargados"].append((destino.name, mb))
            return
        except (TimeoutException, TimeoutError, WebDriverException, ValueError) as error:
            motivo = type(error).__name__
            utils.log(f"CONTRALORIA {anio}: intento {intento}/{config.REINTENTOS} fallo para {nombre}: {motivo}")
            utils.captura(driver, f"error_contraloria_{anio}_{utils.nombre_limpio(nombre)}_intento{intento}")
            utils.pausa()
    utils.registrar_error("contraloria", anio, nombre, filtros, motivo)
    resultado["errores"].append((f"{anio} {nombre}", motivo))


def ejecutar(anio_inicio=config.ANIO_INICIO, forzar=False, headless=False):
    resultado = utils.nuevo_resultado("CONTRALORIA")
    driver = utils.crear_navegador(config.RAW / "contraloria", headless)
    try:
        try:
            abrir_portal(driver)
            abrir_compendio(driver)
            mostrar_todos(driver)
            items = listar_items(driver)
            anunciados = total_resultados(driver)
            if anunciados is not None and anunciados != len(items):
                utils.log(f"AVISO CONTRALORIA: la pagina anuncia {anunciados} resultados y se leyeron {len(items)}; la lista puede estar incompleta")
                resultado["errores"].append(("lista incompleta", f"{anunciados} anunciados, {len(items)} leidos"))
        except (TimeoutException, WebDriverException) as error:
            utils.log(f"CONTRALORIA: el portal no responde o cambio de estructura ({type(error).__name__}). Se omite la fuente.")
            utils.captura(driver, "error_contraloria_portal")
            resultado["errores"].append(("portal", type(error).__name__))
            return resultado

        informes = [i for i in items if re.search(config.CGR_REGEX_INFORME, i[0])]
        otros = [i for i in items if i not in informes]
        utils.log(f"CONTRALORIA: {len(items)} entradas en el compendio: {len(informes)} informes/reportes y {len(otros)} de otro tipo")
        for titulo, enlace in otros:
            utils.registrar_fuente(fuente="contraloria", url=enlace, archivo=titulo, filtros_usados=f"gob.pe > {config.CGR_TEXTO_COMPENDIO} > {titulo}",
                                   fecha_descarga=str(date.today()), estado="omitido", notas="no es un informe de obras paralizadas (infografia, presentacion u otro): no se descarga")

        cortes = set()
        for titulo, enlace in informes:
            try:
                ficha = leer_ficha(driver, enlace)
                fecha_ref = ficha["publicacion"] or ficha["corte"]
                if fecha_ref and fecha_ref.year < anio_inicio:
                    utils.log(f"CONTRALORIA: '{titulo}' es anterior a {anio_inicio}, se omite")
                    continue
                if not ficha["archivos"]:
                    utils.log(f"CONTRALORIA: '{titulo}' no tiene archivos descargables")
                    resultado["errores"].append((titulo, "sin archivos en la ficha"))
                    continue
                if ficha["corte"]:
                    cortes.add(ficha["corte"].year)
                utils.log(f"CONTRALORIA: {titulo} | corte {ficha['corte']} | publicado {ficha['publicacion']} | "
                          f"{', '.join(a['formato'].upper() for a in ficha['archivos'])}")
                for archivo in ficha["archivos"]:
                    descargar_archivo(driver, ficha, archivo, enlace, forzar, resultado)
                    utils.pausa()
            except Exception as error:   # una ficha que falla no detiene a las demas
                utils.log(f"CONTRALORIA: fallo la ficha '{titulo}': {type(error).__name__} {error}")
                utils.captura(driver, f"error_contraloria_ficha_{utils.nombre_limpio(titulo)[:40]}")
                resultado["errores"].append((titulo, type(error).__name__))
        resultado["anios_encontrados"] = sorted(cortes)
        if cortes:
            resultado["anios_faltantes"] = [a for a in range(min(cortes), max(cortes) + 1) if a not in cortes]
    finally:
        driver.quit()
    return resultado

### `scraper_infobras` — Descarga INFOBRAS
INFOBRAS (Contraloria): listado de obras educativas paralizadas y lectura de la ficha publica de cada obra.

In [ ]:
%%modulo scraper_infobras
"""INFOBRAS (Contraloria): listado de obras educativas paralizadas y lectura de la ficha publica de cada obra.

No se leen los datos de preparacion (residente e inspector) para no guardar nombres de personas.
"""
import re
import time
from datetime import date

from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

import config
import utils


def esperar(driver, segundos=None):
    return WebDriverWait(driver, segundos or config.ESPERA_ELEMENTO)


# ---------------- listado: buscador avanzado y exportacion a CSV ----------------
def abrir_buscador(driver):
    """Abre el buscador avanzado y espera a que el sistema llene las listas del formulario (a veces tarda)."""
    driver.get(config.INFOBRAS_URL_BUSCADOR)
    esperar(driver, 90).until(lambda d: len(Select(d.find_element(By.ID, config.INFOBRAS_ID_NIVEL1)).options) > 1)
    time.sleep(3)


JS_FILTRAR = """
const [filtros] = arguments;
const resultado = {};
for (const [id, texto] of Object.entries(filtros)) {
  const lista = document.getElementById(id);
  const opcion = Array.from(lista.options).find(o => o.text.trim().toLowerCase().includes(texto));
  if (!opcion) { resultado[id] = null; continue; }
  lista.value = opcion.value;
  lista.dispatchEvent(new Event('change', {bubbles: true}));
  if (window.jQuery) { jQuery(lista).val(opcion.value).trigger('change'); if (jQuery(lista).selectpicker) jQuery(lista).selectpicker('refresh'); }
  resultado[id] = opcion.text.trim();
}
return resultado;
"""

JS_BUSCAR = """
const boton = Array.from(document.querySelectorAll('button')).find(b =>
  /^buscar$/i.test((b.innerText || '').trim()) && b.offsetParent !== null && b.id !== arguments[0]);
boton.scrollIntoView({block: 'center'}); boton.click();
"""


def aplicar_filtros_y_buscar(driver, filtros=None):
    elegidos = driver.execute_script(JS_FILTRAR, filtros or config.INFOBRAS_FILTROS)
    if None in elegidos.values():
        raise LookupError(f"no se encontro alguna opcion del filtro: {elegidos}")
    utils.log(f"INFOBRAS: filtros aplicados {elegidos}")
    time.sleep(3)
    driver.execute_script(JS_BUSCAR, config.INFOBRAS_ID_BOTON_BASICO)
    esperar(driver, 60).until(EC.presence_of_element_located((By.ID, config.INFOBRAS_ID_EXPORTAR_CSV)))
    time.sleep(3)


# ---------------- lectura de una ficha ----------------
def lineas_de_pagina(driver):
    return [l.strip() for l in driver.find_element(By.TAG_NAME, "body").text.splitlines() if l.strip()]


def esperar_carga(driver, segundos=12):
    """La ficha dice «Cargando coordenadas...» y las rellena con JavaScript 1 a 3 s despues: se espera a que desaparezca."""
    limite = time.time() + segundos
    while time.time() < limite:
        if "cargando" not in driver.find_element(By.TAG_NAME, "body").text.lower():
            return
        time.sleep(0.5)


def abrir_pagina(driver, nombre, parametro, codigo):
    driver.get(f"{config.INFOBRAS_BASE}/Mapa/{nombre}?{parametro}={codigo}")
    esperar(driver).until(lambda d: d.execute_script("return document.readyState") == "complete")
    esperar(driver).until(EC.presence_of_element_located((By.XPATH, "//*[contains(text(),'FICHA DE OBRA')]")))
    esperar_carga(driver)
    return lineas_de_pagina(driver)


def avance_fisico_estable(driver):
    """El porcentaje de avance del encabezado es un contador animado (sube de 0 al valor final):
    se espera a que dos lecturas seguidas den lo mismo antes de guardarlo."""
    anterior, actual = None, None
    for _ in range(15):
        actual = numero(valor(lineas_de_pagina(driver), "PORCENTAJE DE AVANCE FÍSICO"))
        if actual is not None and actual == anterior:
            return actual
        anterior = actual
        time.sleep(1.2)
    return actual


def valor(lineas, etiqueta):
    """Texto de la linea que sigue a la etiqueta (las fichas escriben 'ETIQUETA' y debajo su valor)."""
    for i, linea in enumerate(lineas[:-1]):
        if linea.upper() == etiqueta.upper():
            return lineas[i + 1]
    return None


def numero(texto):
    """'S/ 1,213,383.66' -> 1213383.66 ; '5.32%' -> 5.32 ; vacio o sin dato -> None."""
    if not isinstance(texto, str) or not texto.strip():   # vacio, None o NaN (celda vacia del CSV exportado)
        return None
    m = re.search(r"-?\d[\d,]*\.?\d*", texto)
    return float(m.group(0).replace(",", "")) if m else None


def fecha(texto):
    """'01/09/2026' -> '2026-09-01' ; sin fecha -> None."""
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", (texto or "").strip())
    return f"{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}" if m else None


def periodo(texto):
    """'2026 / SEPTIEMBRE' -> (2026, 9); no reconocido -> (0, 0)."""
    m = re.match(r"(\d{4})\s*/\s*([A-Za-zÁÉÍÓÚáéíóú]+)", (texto or "").strip())
    return (int(m.group(1)), config.INFOBRAS_MESES.get(m.group(2).lower(), 0)) if m else (0, 0)


def leer_avances(driver):
    """Tabla 'Avances de obra' de la pagina de ejecucion: lista de dicts con periodo, avances, estado y enlace al detalle."""
    avances = []
    for tabla in driver.find_elements(By.TAG_NAME, "table"):
        if "AVANCE FÍSICO" not in tabla.text.upper():
            continue
        for fila in tabla.find_elements(By.CSS_SELECTOR, "tbody tr"):
            celdas = [c.text.strip() for c in fila.find_elements(By.TAG_NAME, "td")]
            enlaces = [a.get_attribute("href") for a in fila.find_elements(By.TAG_NAME, "a") if "Avance?codigo=" in (a.get_attribute("href") or "")]
            enlaces += [f"{config.INFOBRAS_BASE}/Mapa/Avance?codigo={b.get_attribute('data-codigo')}"   # otras fichas usan un boton
                        for b in fila.find_elements(By.CSS_SELECTOR, "button[data-codigo]")]
            if len(celdas) >= 9 and periodo(celdas[1]) != (0, 0):
                avances.append({"periodo": celdas[1], "fisico_programado": numero(celdas[2]), "fisico_real": numero(celdas[3]),
                                "estado": celdas[8], "enlace": enlaces[0] if enlaces else None})
    return avances


def leer_detalle_avance(driver, enlace):
    """Pagina de detalle de un avance: causal, fecha de paralizacion, dias paralizado y comentarios."""
    driver.get(enlace)
    esperar(driver).until(EC.presence_of_element_located((By.XPATH, "//*[contains(text(),'DETALLE DE AVANCE')]")))
    lineas = [l.strip() for l in driver.find_element(By.TAG_NAME, "body").text.splitlines() if l.strip()]
    comentario = lambda inicio: next((l.split(":", 1)[1].strip() for l in lineas if l.lower().startswith(inicio)), None)
    return {"causal_paralizacion": valor(lineas, "CAUSAL"), "fecha_paralizacion": fecha(valor(lineas, "FECHA DE PARALIZACIÓN")),
            "dias_paralizado": numero(valor(lineas, "DÍAS PARALIZADO")),
            "comentario_fisico": utils.omitir_nombres(comentario("comentario físico")),
            "comentario_valorizado": utils.omitir_nombres(comentario("comentario valorizado"))}


def leer_ficha(driver, codigo):
    """Lee las paginas Datos generales, Datos de ejecucion y el detalle del ultimo avance paralizado de una obra."""
    return _leer_ficha(driver, codigo)[0]


def _leer_ficha(driver, codigo):
    """Lo mismo que leer_ficha, pero devuelve tambien lo que la pasada ampliada reutiliza:
    (fila, todos los avances, avance paralizado elegido, su detalle). Asi no se abre dos veces la misma pagina."""
    generales = abrir_pagina(driver, "DatosGenerales", "obraId", codigo)
    utils.pausa()
    ubicacion = (valor(generales, "UBICACIÓN GEOGRÁFICA") or "").split(" - ")   # DEPARTAMENTO - PROVINCIA - DISTRITO
    coordenadas = valor(generales, "COORDENADAS GEOGRÁFICAS") or ""
    lat = re.search(r"Latitud:\s*(-?[\d.]+)", coordenadas)
    lon = re.search(r"Longitud:\s*(-?[\d.]+)", coordenadas)
    fila = {
        "codigo_infobras": valor(generales, "CÓDIGO INFOBRAS"), "cui": valor(generales, "CÓDIGO ÚNICO DE INVERSIÓN"),
        "nombre_obra": valor(generales, "NOMBRE DE OBRA"), "entidad": valor(generales, "ENTIDAD RESPONSABLE DE LA OBRA"),
        "estado": valor(generales, "ESTADO DE OBRA"), "avance_fisico_pct": avance_fisico_estable(driver),
        "modalidad": valor(generales, "MODALIDAD DE EJECUCIÓN"), "naturaleza": valor(generales, "NATURALEZA"),
        "tipo_obra": valor(generales, "TIPO DE OBRA"),
        "departamento": ubicacion[0] if len(ubicacion) == 3 else None, "provincia": ubicacion[1] if len(ubicacion) == 3 else None,
        "distrito": ubicacion[2] if len(ubicacion) == 3 else None,
        "latitud": float(lat.group(1)) if lat else None, "longitud": float(lon.group(1)) if lon else None,
    }
    ejecucion = abrir_pagina(driver, "DatosEjecucion", "obraId", codigo)
    fila["monto_ejecucion_acumulado"] = numero(valor(ejecucion, "MONTO DE EJECUCIÓN ACUMULADO"))
    fila["fecha_inicio"] = fecha(valor(ejecucion, "FECHA DE INICIO DE LA OBRA"))
    fila["fecha_fin_prevista"] = fecha(valor(ejecucion, "FECHA DE FIN DE LA OBRA"))
    avances = leer_avances(driver)
    fila["n_avances"] = len(avances)
    ultimo = max(avances, key=lambda a: periodo(a["periodo"])) if avances else None
    fila["periodo_ultimo_avance"] = ultimo["periodo"] if ultimo else None
    fila["avance_fisico_tabla"] = ultimo["fisico_real"] if ultimo else None   # para cruzar con el del encabezado
    if ultimo and fila["avance_fisico_pct"] is not None and ultimo["fisico_real"] is not None \
            and abs(fila["avance_fisico_pct"] - ultimo["fisico_real"]) > 0.01:
        utils.log(f"AVISO INFOBRAS {codigo}: avance del encabezado {fila['avance_fisico_pct']} y de la tabla {ultimo['fisico_real']} no coinciden")
    paralizados = [a for a in avances if a["estado"].lower().startswith("paraliz") and a["enlace"]]
    elegido = max(paralizados, key=lambda a: periodo(a["periodo"])) if paralizados else None
    fila["periodo_avance_paralizado"] = elegido["periodo"] if elegido else None
    detalle = {"causal_paralizacion": None, "fecha_paralizacion": None, "dias_paralizado": None,
               "comentario_fisico": None, "comentario_valorizado": None}
    if elegido:
        utils.pausa()
        detalle = leer_detalle_avance(driver, elegido["enlace"])
    fila.update(detalle)
    fila["url_ficha"] = f"{config.INFOBRAS_BASE}/Mapa/Sumario?ObraId={codigo}"
    fila["fecha_lectura"] = str(date.today())
    return fila, avances, elegido, detalle


# ---------------- listado de las obras (CSV exportado) ----------------
def descargar_listado(driver, forzar=False, filtros_buscador=None, filtros_texto=None, nombre="listado_obras.csv"):
    """Busca Educacion + Paralizada (o los filtros indicados) y baja el CSV del buscador a data/raw/infobras/{anio}.
    Devuelve la ruta del CSV.

    El listado cambia con el tiempo (una obra se reanuda o se paraliza), por eso la fecha de hoy va en los filtros:
    el mismo dia no se vuelve a bajar; otro dia, si.
    """
    filtros_buscador = filtros_buscador or config.INFOBRAS_FILTROS
    filtros_texto = filtros_texto or config.INFOBRAS_FILTROS_TEXTO
    anio = date.today().year
    carpeta = config.RAW / "infobras" / str(anio)
    carpeta.mkdir(parents=True, exist_ok=True)
    filtros = f"{filtros_texto};{date.today()}"
    if not forzar and utils.ya_descargado("infobras", anio, filtros):
        csvs = sorted(carpeta.glob(f"infobras_{anio}_{utils.nombre_limpio(nombre)}_{date.today():%Y-%m-%d}.csv"))
        if csvs:
            utils.log(f"INFOBRAS: el listado de hoy ya estaba descargado ({csvs[-1].name})")
            utils.registrar_omitido("infobras", anio, csvs[-1].name, filtros)
            return csvs[-1]
    utils.cambiar_carpeta_descarga(driver, carpeta)
    utils.limpiar_parciales(carpeta)
    abrir_buscador(driver)
    aplicar_filtros_y_buscar(driver, filtros_buscador)
    antes = utils.archivos_actuales(carpeta)
    utils.hacer_clic(driver, driver.find_element(By.ID, config.INFOBRAS_ID_EXPORTAR_CSV))
    descargado = utils.esperar_descarga(carpeta, antes, ".csv")
    destino, mb = utils.guardar_descarga(descargado, carpeta, "infobras", anio, nombre,
                                         config.INFOBRAS_URL_BUSCADOR, filtros, fecha_corte=str(date.today()),
                                         notas="fecha de corte = dia de la descarga (INFOBRAS muestra el estado actual)")
    utils.log(f"INFOBRAS: listado descargado {destino.name} ({mb} MB)")
    return destino


def leer_listado(ruta):
    """Lee el CSV exportado: codigos INFOBRAS (texto) y el monto del expediente tecnico de cada obra."""
    import pandas as pd
    for codificacion in ("utf-8-sig", "cp1252"):
        try:
            tabla = pd.read_csv(ruta, dtype=str, encoding=codificacion)
            break
        except UnicodeDecodeError:
            continue
    tabla.columns = [c.strip().lower() for c in tabla.columns]
    col_codigo = next(c for c in tabla.columns if "infobras" in c)
    col_monto = next((c for c in tabla.columns if "monto" in c), None)
    col_estado = next((c for c in tabla.columns if "estado" in c), None)
    salida = pd.DataFrame({"codigo_infobras": tabla[col_codigo].str.strip()})
    salida["monto_listado"] = tabla[col_monto].map(numero) if col_monto else None
    salida["estado_listado"] = tabla[col_estado].str.strip() if col_estado else None
    salida = salida[salida.codigo_infobras.notna() & (salida.codigo_infobras != "")]   # filas sin codigo (vacias o de pie)
    return salida.drop_duplicates("codigo_infobras")


# ---------------- lectura de todas las fichas ----------------
def codigos_leidos():
    """Codigos con una ficha leida hace menos de config.INFOBRAS_DIAS_VIGENCIA dias (las mas viejas se vuelven a leer)."""
    if not config.INFOBRAS_FICHAS.exists():
        return set()
    import json
    from datetime import timedelta
    limite = str(date.today() - timedelta(days=config.INFOBRAS_DIAS_VIGENCIA))
    vigentes = set()
    with open(config.INFOBRAS_FICHAS, encoding="utf-8") as f:
        for linea in f:
            if linea.strip():
                ficha = json.loads(linea)
                if str(ficha.get("fecha_lectura", "")) >= limite:
                    vigentes.add(str(ficha["codigo_infobras"]))
    return vigentes


def guardar_ficha(fila):
    import json
    config.INFOBRAS_FICHAS.parent.mkdir(parents=True, exist_ok=True)
    with open(config.INFOBRAS_FICHAS, "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")


def leer_todas(driver, codigos, forzar=False):
    """Lee la ficha de cada codigo con pausas y hasta 3 intentos. Devuelve (leidas, omitidas, errores)."""
    if forzar:
        config.INFOBRAS_FICHAS.unlink(missing_ok=True)
    hechos = codigos_leidos()
    leidas, omitidas, errores = 0, 0, []
    from tqdm import tqdm
    for codigo in tqdm(codigos, desc="fichas INFOBRAS"):
        if str(codigo) in hechos:
            omitidas += 1
            continue
        for intento in range(1, config.REINTENTOS + 1):
            try:
                guardar_ficha(leer_ficha(driver, codigo))
                leidas += 1
                break
            except (TimeoutException, WebDriverException, ValueError) as e:
                utils.log(f"INFOBRAS {codigo}: intento {intento} fallo ({type(e).__name__}: {str(e)[:80]})")
                if intento == config.REINTENTOS:
                    errores.append((codigo, type(e).__name__))
                    utils.registrar_error("infobras", date.today().year, f"ficha_{codigo}", "ficha", e)
                else:
                    time.sleep(5 * intento)
        utils.pausa()
    return leidas, omitidas, errores


def sin_tildes_mayusculas(texto):
    import unicodedata
    if texto is None:
        return None
    return unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper().strip()


def construir_csv(listado):
    """Une las fichas leidas con el listado, valida y guarda data/processed/fichas_infobras.csv (UTF-8)."""
    import json
    import pandas as pd
    with open(config.INFOBRAS_FICHAS, encoding="utf-8") as f:
        filas = [json.loads(l) for l in f if l.strip()]
    fichas = pd.DataFrame(filas).drop_duplicates("codigo_infobras", keep="last")
    fichas["codigo_infobras"] = fichas["codigo_infobras"].astype(str)
    for col in ("departamento", "provincia", "distrito"):
        fichas[col] = fichas[col].map(sin_tildes_mayusculas)
    fichas["departamento"] = fichas.departamento.map(lambda d: "CALLAO" if d and "CALLAO" in d else d)   # 'P. C. DEL CALLAO' -> CALLAO
    fichas["cod_departamento"] = fichas.departamento.map(config.DEPARTAMENTOS)   # codigo INEI de 2 digitos, como texto
    fichas = fichas[fichas.codigo_infobras.isin(listado.codigo_infobras)]   # solo las obras del listado vigente (otras ya no estan paralizadas)
    fichas = fichas.merge(listado, on="codigo_infobras", how="left")
    fichas = fichas.rename(columns={"monto_listado": "monto_expediente_tecnico"})
    fichas["fecha_corte"] = fichas["fecha_lectura"]
    orden = ["codigo_infobras", "cui", "nombre_obra", "entidad", "cod_departamento", "departamento", "provincia", "distrito", "estado",
             "modalidad", "naturaleza", "tipo_obra", "monto_expediente_tecnico", "monto_ejecucion_acumulado",
             "avance_fisico_pct", "fecha_inicio", "fecha_fin_prevista", "periodo_ultimo_avance", "n_avances",
             "periodo_avance_paralizado", "causal_paralizacion", "fecha_paralizacion", "dias_paralizado",
             "comentario_fisico", "comentario_valorizado", "latitud", "longitud", "avance_fisico_tabla",
             "estado_listado", "url_ficha", "fecha_lectura", "fecha_corte"]
    fichas = fichas[orden].sort_values("codigo_infobras")
    config.INFOBRAS_SALIDA.parent.mkdir(parents=True, exist_ok=True)
    fichas.to_csv(config.INFOBRAS_SALIDA, index=False, encoding="utf-8-sig")
    return fichas


def ejecutar(anio_inicio=None, forzar=False, headless=False):
    """(anio_inicio no se usa: INFOBRAS muestra solo el estado actual.) Descarga el listado, lee las fichas de las obras y arma fichas_infobras.csv. Devuelve el resultado de la corrida."""
    resultado = utils.nuevo_resultado("infobras")
    driver = utils.crear_navegador(config.RAW / "infobras", headless=headless)
    try:
        ruta = descargar_listado(driver, forzar)
        listado = leer_listado(ruta)
        utils.log(f"INFOBRAS: {len(listado)} obras en el listado")
        leidas, omitidas, errores = leer_todas(driver, list(listado["codigo_infobras"]), forzar)
    finally:
        driver.quit()
    fichas = construir_csv(listado)
    utils.log(f"INFOBRAS: fichas leidas {leidas}, omitidas {omitidas}, con error {len(errores)}; filas en el CSV final {len(fichas)}")
    resultado["descargados"] = [f"ficha_{c}" for c in fichas["codigo_infobras"]][:leidas]
    resultado["omitidos"] = ["ficha"] * omitidas
    resultado["errores"] = errores
    return resultado


# =====================================================================================================
# Pasada ampliada: A4 + B1 + B2 en UNA sola lectura de las fichas
#   A4  listado de Educacion/Cultura con todos los estados + ficha de las obras de la Contraloria que ya no estan
#       como Paralizada en INFOBRAS
#   B1  todos los avances mensuales de cada obra           -> data/processed/infobras_avances.csv
#   B2  detalle de los avances en estado Paralizado         -> data/processed/infobras_paralizaciones.csv
# No modifica infobras_fichas.jsonl ni fichas_infobras.csv (datos del 2026-10-27): usa su propio cache
# (config.INFOBRAS_AMPLIADAS) y escribe tablas nuevas. Se corre como proceso aparte:
#   .venv\Scripts\python.exe src\scraper_infobras.py --ampliado --codigos 561092        (prueba con 1 obra)
#   .venv\Scripts\python.exe src\scraper_infobras.py --ampliado --primeras 5            (prueba con 5)
#   .venv\Scripts\python.exe src\scraper_infobras.py --ampliado                         (todas)
# Si se corta, al volver a lanzarlo retoma: las obras ya guardadas en el cache no se vuelven a leer.
# =====================================================================================================
def codigo_avance(enlace):
    m = re.search(r"codigo=(\d+)", enlace or "")
    return m.group(1) if m else None


def tramos_paralizados(avances):
    """Avances en estado Paralizado, en orden cronologico, con su tramo (meses paralizados seguidos en la tabla) y su
    posicion en el tramo: unico, inicio, medio o fin."""
    ordenados = sorted(avances, key=lambda a: periodo(a["periodo"]))
    salida, tramo, anterior_paralizado = [], 0, False
    for i, a in enumerate(ordenados):
        paralizado = (a.get("estado") or "").lower().startswith("paraliz")
        if paralizado and not anterior_paralizado:
            tramo += 1
        if paralizado:
            sigue = i + 1 < len(ordenados) and (ordenados[i + 1].get("estado") or "").lower().startswith("paraliz")
            empieza = not anterior_paralizado
            posicion = "unico" if empieza and not sigue else "inicio" if empieza else "fin" if not sigue else "medio"
            salida.append((a, tramo, posicion))
        anterior_paralizado = paralizado
    return salida


def leer_ficha_ampliada(driver, codigo, modo=None):
    """Ficha + todos los avances + detalle de los avances paralizados de una obra (dict listo para el cache)."""
    modo = modo or config.INFOBRAS_DETALLE_PARALIZACIONES
    fila, avances, elegido, detalle_elegido = _leer_ficha(driver, codigo)
    paralizaciones = []
    for a, tramo, posicion in tramos_paralizados(avances):
        base = {"periodo": a["periodo"], "codigo_avance": codigo_avance(a["enlace"]), "tramo": tramo, "posicion_tramo": posicion,
                "detalle_leido": False, "causal_paralizacion": None, "fecha_paralizacion": None, "dias_paralizado": None,
                "comentario_fisico": None, "comentario_valorizado": None}
        if a["enlace"] and (modo == "todas" or posicion != "medio"):
            if elegido is not None and a["enlace"] == elegido["enlace"]:
                detalle = detalle_elegido          # ya se leyo en _leer_ficha
            else:
                utils.pausa()
                detalle = leer_detalle_avance(driver, a["enlace"])
            base.update(detalle)
            base["detalle_leido"] = True
        paralizaciones.append(base)
    avances_guardar = [{"periodo": a["periodo"], "fisico_programado": a["fisico_programado"], "fisico_real": a["fisico_real"],
                        "estado": a["estado"], "codigo_avance": codigo_avance(a["enlace"])} for a in avances]
    return {"codigo_infobras": str(fila["codigo_infobras"] or codigo), "codigo_pedido": str(codigo), "fecha_lectura": fila["fecha_lectura"],
            "modo_detalle": modo, "ficha": fila, "avances": avances_guardar, "paralizaciones": paralizaciones}


# ---------------- candado: una sola lectura de INFOBRAS a la vez ----------------
def tomar_candado():
    import os
    config.INFOBRAS_CANDADO.parent.mkdir(parents=True, exist_ok=True)
    try:
        fd = os.open(config.INFOBRAS_CANDADO, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError(f"ya hay una lectura de INFOBRAS en curso (existe {config.INFOBRAS_CANDADO}). Si estas seguro de que no "
                           "hay otra ventana leyendo INFOBRAS (revisa el Administrador de tareas), borra ese archivo y vuelve a lanzar.")
    with os.fdopen(fd, "w") as f:
        f.write(f"pid={os.getpid()} inicio={date.today()}\n")


def soltar_candado():
    config.INFOBRAS_CANDADO.unlink(missing_ok=True)


# ---------------- cache de la pasada ampliada ----------------
def leer_cache_ampliado():
    import json
    if not config.INFOBRAS_AMPLIADAS.exists():
        return []
    with open(config.INFOBRAS_AMPLIADAS, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def guardar_ampliada(registro):
    import json
    config.INFOBRAS_AMPLIADAS.parent.mkdir(parents=True, exist_ok=True)
    with open(config.INFOBRAS_AMPLIADAS, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")


def codigos_pasada():
    """Obras a leer: (codigo, origen). Primero las 198 de fichas_infobras.csv (paralizadas al 2026-10-27); luego las obras de
    Educacion/Cultura del anexo de la Contraloria de junio de 2026 que no estan en esa lista (A4, ~231)."""
    import glob
    import pandas as pd
    import tablas_finales
    fichas = pd.read_csv(config.INFOBRAS_SALIDA, dtype={"codigo_infobras": str}, encoding="utf-8-sig")
    actuales = list(fichas.codigo_infobras)
    x = pd.read_excel(glob.glob(str(config.RAW / "contraloria/2026/*anexo_01*junio*.xlsx"))[0], sheet_name="Data", dtype=str)
    x.columns = [tablas_finales.normalizar(c) for c in x.columns]
    tipo = x[next(c for c in x.columns if c.startswith("TIPO DE OBRA") and "NIVEL" not in c)].map(tablas_finales.normalizar)
    cgr = [c for c in x.loc[tipo.str.contains("EDUC"), "CODIGO INFOBRAS"].str.strip().drop_duplicates() if c not in set(actuales)]
    return [(c, "infobras_paralizada_2026-10-27") for c in actuales] + [(c, "contraloria_educacion_2026-06") for c in cgr]


def leer_todas_ampliadas(driver, codigos, forzar=False):
    """Lee la ficha ampliada de cada (codigo, origen) con pausas y hasta 3 intentos; guarda cada obra al terminarla.
    Devuelve (leidas, omitidas, errores)."""
    if forzar:
        config.INFOBRAS_AMPLIADAS.unlink(missing_ok=True)
    hechos = {r["codigo_pedido"] for r in leer_cache_ampliado()}
    leidas, omitidas, errores = 0, 0, []
    from tqdm import tqdm
    for codigo, origen in tqdm(codigos, desc="fichas INFOBRAS (ampliada)"):
        if str(codigo) in hechos:
            omitidas += 1
            continue
        for intento in range(1, config.REINTENTOS + 1):
            try:
                registro = leer_ficha_ampliada(driver, codigo)
                registro["origen"] = origen
                guardar_ampliada(registro)
                leidas += 1
                break
            except (TimeoutException, WebDriverException, ValueError) as e:
                utils.log(f"INFOBRAS ampliada {codigo}: intento {intento} fallo ({type(e).__name__}: {str(e)[:80]})")
                if intento == config.REINTENTOS:
                    errores.append((codigo, type(e).__name__))
                    utils.registrar_error("infobras", date.today().year, f"ficha_ampliada_{codigo}", "ficha ampliada", e)
                else:
                    time.sleep(5 * intento)
        utils.pausa()
    return leidas, omitidas, errores


def listado_todos_mas_reciente():
    """Ruta del ultimo listado de Educacion con todos los estados en data/raw/infobras, o None."""
    rutas = sorted(config.RAW.glob(f"infobras/*/infobras_*_{utils.nombre_limpio(config.INFOBRAS_NOMBRE_LISTADO_TODOS)}_*.csv"))
    return rutas[-1] if rutas else None


# ---------------- tablas finales de la pasada ampliada ----------------
def construir_csv_ampliados():
    """Arma infobras_avances.csv, infobras_paralizaciones.csv, infobras_fichas_ampliadas.csv y la validacion,
    solo con el cache (sin navegador). Si no hay cache, no hace nada. Devuelve (avances, paralizaciones, fichas) o None."""
    import pandas as pd
    import ubigeo_obras
    registros = leer_cache_ampliado()
    if not registros:
        utils.log("INFOBRAS ampliada: todavia no hay fichas ampliadas leidas; se omite")
        return None
    ultimos = {}
    for r in registros:                       # si una obra se leyo dos veces, vale la ultima lectura
        ultimos[r["codigo_pedido"]] = r
    registros = list(ultimos.values())
    coordenadas = leer_coordenadas_cache()   # las completadas despues (la ficha las cargaba con retraso)
    for r in registros:
        c = coordenadas.get(str(r["codigo_pedido"]))
        if c and r["ficha"].get("latitud") in (None, ""):
            r["ficha"]["latitud"], r["ficha"]["longitud"] = c.get("latitud"), c.get("longitud")

    def con_periodo(fila, extra):
        anio, mes = periodo(fila["periodo"])
        return {"codigo_infobras": extra["codigo_infobras"], "cui": extra["cui"], "periodo": fila["periodo"],
                "anio": anio or None, "mes": mes or None}

    filas_av, filas_par, filas_fi = [], [], []
    for r in registros:
        ficha = r["ficha"]
        extra = {"codigo_infobras": r["codigo_infobras"], "cui": ficha.get("cui")}
        for a in r["avances"]:
            filas_av.append({**con_periodo(a, extra), "fisico_programado": a["fisico_programado"], "fisico_real": a["fisico_real"],
                             "estado": a["estado"], "codigo_avance": a["codigo_avance"], "fecha_lectura": r["fecha_lectura"]})
        for p in r["paralizaciones"]:
            filas_par.append({**con_periodo(p, extra), "codigo_avance": p["codigo_avance"], "tramo": p["tramo"],
                              "posicion_tramo": p["posicion_tramo"], "detalle_leido": p["detalle_leido"],
                              "causal_paralizacion": p["causal_paralizacion"], "fecha_paralizacion": p["fecha_paralizacion"],
                              "dias_paralizado": p["dias_paralizado"], "comentario_fisico": p["comentario_fisico"],
                              "comentario_valorizado": p["comentario_valorizado"], "fecha_lectura": r["fecha_lectura"]})
        filas_fi.append({**{k: v for k, v in ficha.items() if not k.startswith("comentario")},
                         "origen": r.get("origen"), "n_paralizados": len(r["paralizaciones"])})

    texto = ["codigo_infobras", "cui", "codigo_avance"]
    avances = pd.DataFrame(filas_av).sort_values(["codigo_infobras", "anio", "mes"])
    paral = pd.DataFrame(filas_par, columns=["codigo_infobras", "cui", "periodo", "anio", "mes", "codigo_avance", "tramo", "posicion_tramo",
                                             "detalle_leido", "causal_paralizacion", "fecha_paralizacion", "dias_paralizado",
                                             "comentario_fisico", "comentario_valorizado", "fecha_lectura"]).sort_values(["codigo_infobras", "anio", "mes"])
    for t in (avances, paral):
        for c in texto:
            t[c] = t[c].astype("string")
        for c in ("anio", "mes", "tramo"):
            if c in t:
                t[c] = t[c].astype("Int64")
    paral["dias_paralizado"] = paral.dias_paralizado.astype("Float64").round().astype("Int64")
    # con el detalle "por_tramo" solo el primer y el ultimo mes de cada tramo traen causal y fecha: se repiten en los meses del medio
    for c in ("causal_paralizacion", "fecha_paralizacion"):
        paral[c + "_tramo"] = paral.groupby(["codigo_infobras", "tramo"])[c].transform(lambda s: s.ffill().bfill())

    fichas = pd.DataFrame(filas_fi)
    fichas["codigo_infobras"] = fichas.codigo_infobras.astype(str)
    for col in ("departamento", "provincia", "distrito"):
        fichas[col] = fichas[col].map(sin_tildes_mayusculas)
    fichas["departamento"] = fichas.departamento.map(lambda d: "CALLAO" if d and "CALLAO" in d else d)
    fichas["cod_departamento"] = fichas.departamento.map(config.DEPARTAMENTOS)
    ruta_todos = listado_todos_mas_reciente()
    if ruta_todos is not None:
        todos = leer_listado(ruta_todos).rename(columns={"monto_listado": "monto_expediente_tecnico", "estado_listado": "estado_listado_todos"})
        fichas = fichas.merge(todos, on="codigo_infobras", how="left")
    else:
        fichas["monto_expediente_tecnico"], fichas["estado_listado_todos"] = None, None
    fichas = ubigeo_obras.asignar_ubigeo(fichas)
    orden = ["codigo_infobras", "cui", "origen", "nombre_obra", "entidad", "cod_departamento", "departamento", "provincia", "distrito",
             "ubigeo", "ubigeo_metodo", "departamento_infobras", "estado", "estado_listado_todos", "modalidad", "naturaleza", "tipo_obra", "monto_expediente_tecnico",
             "monto_ejecucion_acumulado", "avance_fisico_pct", "fecha_inicio", "fecha_fin_prevista", "periodo_ultimo_avance", "n_avances",
             "n_paralizados", "periodo_avance_paralizado", "causal_paralizacion", "fecha_paralizacion", "dias_paralizado",
             "latitud", "longitud", "avance_fisico_tabla", "url_ficha", "fecha_lectura"]
    fichas = fichas[orden].sort_values(["origen", "codigo_infobras"])

    for tabla, ruta in ((avances, config.INFOBRAS_AVANCES), (paral, config.INFOBRAS_PARALIZACIONES), (fichas, config.INFOBRAS_FICHAS_AMPLIADAS)):
        tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    validar_ampliados(avances, paral, fichas, ruta_todos)
    utils.log(f"INFOBRAS ampliada: {len(fichas)} fichas, {len(avances)} avances, {len(paral)} avances paralizados "
              f"({int(paral.detalle_leido.sum())} con detalle leido); estados: {fichas.estado.value_counts().to_dict()}")
    return avances, paral, fichas


def validar_ampliados(avances, paral, fichas, ruta_todos):
    """Comprobaciones de la pasada ampliada -> docs/validacion_infobras_ampliada.csv (comprobacion, valor, esperado, ok, detalle)."""
    import pandas as pd
    filas = []

    def agregar(nombre, valor, esperado, detalle=""):
        ok = None if esperado is None else bool(valor == esperado)
        filas.append({"comprobacion": nombre, "valor": valor, "esperado": esperado, "ok": ok, "detalle": detalle})

    pedidos = codigos_pasada()
    agregar("obras pedidas (198 paralizadas + Contraloria jun-2026)", len(pedidos), None)
    agregar("obras leidas en el cache", len(fichas), len(pedidos), "si falta alguna: ver errores en docs/log_descargas.txt y relanzar")
    n_ficha = pd.to_numeric(fichas.set_index("codigo_infobras").n_avances, errors="coerce")
    n_tabla = avances.groupby("codigo_infobras").size().reindex(n_ficha.index, fill_value=0)
    agregar("obras cuyo numero de avances guardados = n_avances de la ficha", int((n_tabla == n_ficha).sum()), len(fichas))
    n_par = avances[avances.estado.str.lower().str.startswith("paraliz", na=False)].groupby("codigo_infobras").size()
    agregar("avances en estado Paralizado (tabla de avances) = filas de infobras_paralizaciones", len(paral), int(n_par.sum()))
    ultimo = avances.sort_values(["anio", "mes"]).groupby("codigo_infobras").fisico_real.last()
    enc = pd.to_numeric(fichas.set_index("codigo_infobras").avance_fisico_pct, errors="coerce")
    agregar("obras con avance fisico del encabezado = ultimo avance de la tabla (±0.01)",
            int(((enc - ultimo.reindex(enc.index)).abs() <= 0.01).sum()), int(enc.notna().sum()))
    agregar("obras con ubigeo", int(fichas.ubigeo.notna().sum()), len(fichas), "las que no: ubigeo_metodo = sin_ubigeo")
    if ruta_todos is not None:
        en_listado = fichas.estado_listado_todos.notna()
        agregar("obras leidas que aparecen en el listado de Educacion con todos los estados", int(en_listado.sum()), len(fichas), ruta_todos.name)
        distinto = (fichas.estado_listado_todos.map(sin_tildes_mayusculas) != fichas.estado.map(sin_tildes_mayusculas)) & en_listado
        agregar("obras con estado del listado distinto al de la ficha", int(distinto.sum()), 0, "el listado y la ficha se leen en momentos distintos")
    else:
        agregar("listado de Educacion con todos los estados", "no descargado", None)
    for origen, grupo in fichas.groupby("origen"):
        agregar(f"estados en la ficha ({origen})", "; ".join(f"{k}: {v}" for k, v in grupo.estado.value_counts().items()), None)
    pd.DataFrame(filas).to_csv(config.INFOBRAS_VALIDACION_AMPLIADA, index=False, encoding="utf-8-sig")


def ejecutar_ampliado(codigos=None, primeras=None, forzar=False, headless=False, modo=None):
    """Pasada ampliada completa: listado con todos los estados (A4) + fichas con avances (B1) y paralizaciones (B2)."""
    if modo:
        config.INFOBRAS_DETALLE_PARALIZACIONES = modo
    pedidos = codigos_pasada()
    if codigos:
        origen = dict(pedidos)
        pedidos = [(str(c), origen.get(str(c), "prueba")) for c in codigos]
    if primeras:
        pedidos = pedidos[:primeras]
    utils.log(f"INFOBRAS ampliada: inicio con {len(pedidos)} obras (detalle de paralizaciones: {config.INFOBRAS_DETALLE_PARALIZACIONES})")
    tomar_candado()
    try:
        driver = utils.crear_navegador(config.RAW / "infobras", headless=headless)
        try:
            ruta = descargar_listado(driver, forzar=False, filtros_buscador=config.INFOBRAS_FILTROS_TODOS,
                                     filtros_texto=config.INFOBRAS_FILTROS_TODOS_TEXTO, nombre=config.INFOBRAS_NOMBRE_LISTADO_TODOS)
            listado = leer_listado(ruta)
            estados = listado.estado_listado.value_counts().to_dict()
            utils.log(f"INFOBRAS ampliada: listado con todos los estados: {len(listado)} obras; {estados}")
            if len(estados) < 2:
                utils.log("AVISO INFOBRAS ampliada: el listado trae un solo estado; revisar que el filtro de estado quedo en 'todos'")
            utils.pausa()
            leidas, omitidas, errores = leer_todas_ampliadas(driver, pedidos, forzar)
        finally:
            driver.quit()
    finally:
        soltar_candado()
    utils.log(f"INFOBRAS ampliada: leidas {leidas}, omitidas (ya en el cache) {omitidas}, con error {len(errores)} {errores[:10]}")
    construir_csv_ampliados()
    return leidas, omitidas, errores


# ---------------- coordenadas: la ficha las rellena con JavaScript unos segundos despues de cargar ----------------
def leer_coordenadas_cache():
    """codigo -> registro del cache de coordenadas (data/interim/infobras_coordenadas.jsonl); vale el ultimo de cada obra."""
    import json
    if not config.INFOBRAS_COORDENADAS.exists():
        return {}
    with open(config.INFOBRAS_COORDENADAS, encoding="utf-8") as f:
        return {str(r["codigo_infobras"]): r for r in (json.loads(l) for l in f if l.strip())}


def guardar_coordenadas(registro):
    import json
    config.INFOBRAS_COORDENADAS.parent.mkdir(parents=True, exist_ok=True)
    with open(config.INFOBRAS_COORDENADAS, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")


def completar_coordenadas(headless=False):
    """Completa latitud y longitud de las obras de la pasada ampliada que no las tienen.

    1) Primero copia las de fichas_infobras.csv (lectura del 27-oct) cuando existen: no abre el navegador.
    2) Para las que siguen sin coordenadas abre SOLO la pagina de datos generales (esperando a que termine de cargar) y las lee.
    Un registro con latitud vacia despues de esperar significa que INFOBRAS no tiene coordenadas de esa obra.
    """
    import pandas as pd
    registros = {r["codigo_pedido"]: r for r in leer_cache_ampliado()}
    hechos = leer_coordenadas_cache()
    sin = [c for c, r in registros.items() if r["ficha"].get("latitud") in (None, "") and c not in hechos]
    if not sin:
        utils.log("INFOBRAS coordenadas: no falta ninguna")
        return 0, 0
    octubre = pd.read_csv(config.INFOBRAS_SALIDA, dtype=str, keep_default_na=False, encoding="utf-8-sig").set_index("codigo_infobras")
    copiadas = 0
    for c in list(sin):
        if c in octubre.index and octubre.loc[c, "latitud"] != "":
            guardar_coordenadas({"codigo_infobras": c, "latitud": float(octubre.loc[c, "latitud"]), "longitud": float(octubre.loc[c, "longitud"]),
                                 "fuente": "fichas_infobras_2026-10-27", "fecha_lectura": str(date.today())})
            sin.remove(c)
            copiadas += 1
    utils.log(f"INFOBRAS coordenadas: {copiadas} copiadas de fichas_infobras.csv; {len(sin)} por leer en la web")
    leidas = 0
    if sin:
        tomar_candado()
        try:
            driver = utils.crear_navegador(config.RAW / "infobras", headless=headless)
            try:
                from tqdm import tqdm
                for codigo in tqdm(sin, desc="coordenadas INFOBRAS"):
                    for intento in range(1, config.REINTENTOS + 1):
                        try:
                            lineas = abrir_pagina(driver, "DatosGenerales", "obraId", codigo)
                            texto = valor(lineas, "COORDENADAS GEOGRÁFICAS") or ""
                            if "cargando" in texto.lower():
                                raise TimeoutException("las coordenadas siguen cargando")
                            lat = re.search(r"Latitud:\s*(-?[\d.]+)", texto)
                            lon = re.search(r"Longitud:\s*(-?[\d.]+)", texto)
                            guardar_coordenadas({"codigo_infobras": codigo, "latitud": float(lat.group(1)) if lat else None,
                                                 "longitud": float(lon.group(1)) if lon else None, "fuente": "web",
                                                 "texto_coordenadas": texto[:60], "fecha_lectura": str(date.today())})
                            leidas += 1
                            break
                        except (TimeoutException, WebDriverException) as e:
                            utils.log(f"INFOBRAS coordenadas {codigo}: intento {intento} fallo ({type(e).__name__})")
                            time.sleep(5 * intento)
                    utils.pausa()
            finally:
                driver.quit()
        finally:
            soltar_candado()
    utils.log(f"INFOBRAS coordenadas: {leidas} leidas en la web, {copiadas} copiadas")
    construir_csv_ampliados()
    return leidas, copiadas


if __name__ == "__main__":
    import argparse
    import sys
    sys.stdout.reconfigure(encoding="utf-8")
    p = argparse.ArgumentParser(description="Pasada ampliada por INFOBRAS (A4 + B1 + B2). Abre Chrome: solo una persona a la vez.")
    p.add_argument("--ampliado", action="store_true", help="obligatorio: confirma que se quiere abrir el navegador")
    p.add_argument("--codigos", help="solo estos codigos INFOBRAS, separados por coma (para probar)")
    p.add_argument("--primeras", type=int, help="solo las N primeras obras de la lista (para probar)")
    p.add_argument("--modo", choices=["por_tramo", "todas"], help="detalle de paralizaciones (por defecto, el de config.py)")
    p.add_argument("--solo-tablas", action="store_true", help="no abre el navegador: rearma los CSV con lo que ya esta en el cache")
    p.add_argument("--coordenadas", action="store_true", help="completa las coordenadas que faltan (abre Chrome solo para las que no estaban en octubre)")
    p.add_argument("--headless", action="store_true", help="navegador sin ventana")
    a = p.parse_args()
    if a.coordenadas:
        completar_coordenadas(headless=a.headless)
    elif a.solo_tablas:
        construir_csv_ampliados()
    elif a.ampliado:
        ejecutar_ampliado([c.strip() for c in a.codigos.split(",")] if a.codigos else None, a.primeras, headless=a.headless, modo=a.modo)
    else:
        p.error("usa --ampliado para leer INFOBRAS, --coordenadas para completar coordenadas o --solo-tablas para rearmar los CSV")

### `descarga_mef` — Descarga MEF
MEF (Portal de Datos Abiertos): Formato 12-B de inversiones, con el devengado de cada inversion identificada por su CUI.

In [ ]:
%%modulo descarga_mef
"""MEF (Portal de Datos Abiertos): Formato 12-B de inversiones, con el devengado de cada inversion identificada por su CUI.

Es un archivo CSV publico (~180 MB) que se baja con una sola peticion HTTP, sin navegador. Se renueva una vez al mes (el archivo del MEF
se actualiza seguido, pero para este proyecto basta una foto mensual). Solo se guardan en data/processed las inversiones (CUI) que
aparecen en nuestras tablas de obras.
"""
import urllib.request
from datetime import date

import pandas as pd

import config
import utils

COLUMNAS_MEF = {   # columna del MEF -> nombre en nuestra tabla
    "CODIGO_UNICO": "cui", "ESTADO": "estado_mef", "PRIMER_DEVENGADO": "primer_devengado", "ULTIMO_DEVENGADO": "ultimo_devengado",
    "DEVENGADO_ACUMULADO": "devengado_hasta_anio_anterior_soles", "DEV_AÑO_ACTUAL": "devengado_anio_actual_soles",
    "PIM_AÑO_ACTUAL": "pim_anio_actual_soles", "AVANCE_FISICO": "avance_fisico_mef_pct", "AVANCE_EJECUCION": "avance_ejecucion_mef_pct",
    "ULT_PROBLEMA": "ult_problema",
    "FECHA_ULT_ACT_F12B": "fecha_ult_act_f12b", "ULT_PERIODO_REG_F12B": "ult_periodo_reg_f12b",
    "INICIO_EJEC_FISICA": "inicio_ejec_fisica", "CULMINA_EJEC_FISICA": "culmina_ejec_fisica"}


def _bajar(url, destino):
    """Descarga una URL a un archivo, en bloques, con el User-Agent del proyecto. Devuelve los bytes bajados."""
    pedido = urllib.request.Request(url, headers={"User-Agent": config.USER_AGENT})
    total = 0
    with urllib.request.urlopen(pedido, timeout=120) as respuesta, open(destino, "wb") as f:
        while True:
            bloque = respuesta.read(1024 * 1024)
            if not bloque:
                break
            f.write(bloque)
            total += len(bloque)
    return total


def _es_f12b_valido(ruta):
    """El archivo debe ser un CSV con la columna CODIGO_UNICO (no una pagina de error)."""
    cabecera = pd.read_csv(ruta, nrows=0, encoding="utf-8-sig").columns
    return "CODIGO_UNICO" in [c.strip() for c in cabecera]


def descargar(forzar=False):
    """Baja FORMATO_12B.csv a data/raw/mef/{anio}/ una vez por mes. Devuelve (ruta del archivo vigente, True si se bajo ahora)."""
    anio = date.today().year
    carpeta = config.RAW / "mef" / str(anio)
    carpeta.mkdir(parents=True, exist_ok=True)
    filtros = f"{config.MEF_FILTROS_TEXTO};mes={date.today():%Y-%m}"
    existentes = sorted(carpeta.glob(f"mef_{anio}_formato_12b_*.csv"))
    if not forzar and utils.ya_descargado("mef", anio, filtros) and existentes:
        utils.log(f"MEF: el Formato 12-B de este mes ya estaba descargado ({existentes[-1].name})")
        utils.registrar_omitido("mef", anio, existentes[-1].name, filtros)
        return existentes[-1], False
    utils.limpiar_parciales(carpeta)
    temporal = carpeta / "descarga_en_curso.crdownload"
    utils.log(f"MEF: descargando {config.MEF_URL_F12B} (unos 180 MB, una sola peticion)")
    bytes_bajados = _bajar(config.MEF_URL_F12B, temporal)
    if not _es_f12b_valido(temporal):
        temporal.unlink(missing_ok=True)
        raise ValueError("el archivo descargado del MEF no tiene la columna CODIGO_UNICO")
    destino = carpeta / utils.nombre_destino("mef", anio, "formato_12b", ".csv")
    destino.unlink(missing_ok=True)
    temporal.rename(destino)
    utils.registrar_fuente(fuente="mef", url=config.MEF_URL_F12B, archivo=destino.name, anio=anio, filtros_usados=filtros,
                           fecha_descarga=str(date.today()), fecha_corte_del_dato=str(date.today()),
                           tamano_mb=round(bytes_bajados / 1024 / 1024, 3), estado="ok",
                           notas="foto del dia de la descarga; el MEF actualiza el archivo con frecuencia")
    # el diccionario de datos del MEF acompana al archivo si esta disponible (el 2026-11-12 su servidor lo tenia vacio: 404)
    dic = carpeta / utils.nombre_destino("mef", anio, "formato_12b_diccionario", ".csv")
    if not dic.exists():
        try:
            _bajar(config.MEF_URL_F12B_DICCIONARIO, dic)
        except Exception as error:
            dic.unlink(missing_ok=True)
            utils.log(f"MEF: aviso, no se pudo bajar el diccionario del MEF ({type(error).__name__}); sus columnas estan en el diccionario de este proyecto")
    utils.log(f"MEF: guardado {destino.name} ({bytes_bajados / 1024 / 1024:.0f} MB)")
    return destino, True


def ejecutar(anio_inicio=None, forzar=False, headless=False):
    """Misma firma que los demas scrapers (main.descargar). anio_inicio y headless no se usan: no hay navegador."""
    resultado = utils.nuevo_resultado("mef")
    try:
        ruta, nuevo = descargar(forzar)
        resultado["descargados" if nuevo else "omitidos"].append(ruta.name)
    except Exception as error:
        utils.log(f"MEF: no se pudo descargar ({type(error).__name__}: {error})")
        resultado["errores"].append(("formato_12b", type(error).__name__))
    return resultado


def construir_tabla(cuis):
    """Filtra el Formato 12-B a los CUI indicados y guarda data/processed/mef_f12b_obras.csv (una fila por CUI)."""
    existentes = sorted((config.RAW / "mef").glob("*/mef_*_formato_12b_*.csv"))
    existentes = [e for e in existentes if "diccionario" not in e.name]
    if not existentes:
        utils.log("MEF: no hay Formato 12-B descargado; se omite mef_f12b_obras.csv")
        return None
    ruta = existentes[-1]
    cuis = {str(c).strip() for c in cuis if str(c).strip() not in ("", "nan")}
    piezas = []
    for bloque in pd.read_csv(ruta, dtype=str, encoding="utf-8-sig", chunksize=100_000, usecols=list(COLUMNAS_MEF)):
        bloque["CODIGO_UNICO"] = bloque["CODIGO_UNICO"].str.strip()
        piezas.append(bloque[bloque.CODIGO_UNICO.isin(cuis)])
    tabla = pd.concat(piezas).rename(columns=COLUMNAS_MEF).drop_duplicates("cui", keep="last")
    for c in ("devengado_hasta_anio_anterior_soles", "devengado_anio_actual_soles", "pim_anio_actual_soles", "avance_fisico_mef_pct", "avance_ejecucion_mef_pct"):
        tabla[c] = pd.to_numeric(tabla[c].str.replace(",", ""), errors="coerce")
    tabla["ult_problema"] = tabla.ult_problema.map(utils.omitir_nombres)   # texto libre de la entidad: sin nombres de personas
    tabla["devengado_total_soles"] = tabla.devengado_hasta_anio_anterior_soles.fillna(0) + tabla.devengado_anio_actual_soles.fillna(0)
    tabla["archivo_mef"] = ruta.name
    tabla = tabla.sort_values("cui").reset_index(drop=True)
    tabla.to_csv(config.PROCESSED / "mef_f12b_obras.csv", index=False, encoding="utf-8-sig")
    utils.log(f"MEF: mef_f12b_obras.csv con {len(tabla)} de {len(cuis)} CUI encontrados en el Formato 12-B")
    return tabla

### `extraccion` — Tablas de ESCALE y ENLA
Extraccion: de los archivos crudos (data\raw) a tablas limpias (data\processed).

In [ ]:
%%modulo extraccion
"""Extraccion: de los archivos crudos (data\\raw) a tablas limpias (data\\processed)."""
import json
import re
import unicodedata
import zipfile

import numpy as np
import pandas as pd
import pdfplumber

import config
import utils


# ---------------- utilidades de archivos ----------------
def dbf_de_zip(fuente, anio, patron_zip):
    """Busca el zip mas reciente del anio que coincida con el patron, lo descomprime en interim y devuelve su .dbf."""
    carpeta = config.RAW / fuente / str(anio)
    candidatos = sorted(p for p in carpeta.glob("*.zip") if re.search(patron_zip, p.name, re.I))
    if not candidatos:
        raise FileNotFoundError(f"no hay zip '{patron_zip}' en {carpeta}")
    ruta_zip = candidatos[-1]
    destino = config.INTERIM / fuente / str(anio) / ruta_zip.stem
    if not destino.exists():
        with zipfile.ZipFile(ruta_zip) as z:
            z.extractall(destino)
    return sorted(destino.rglob("*.dbf"))[0]


# ---------------- ESCALE: lectura por anio ----------------
def cargar_lineal(anio):
    """Una fila por local que respondio el censo: codigo, ubigeo y nombres (y servicios si el anio los trae en columnas)."""
    columnas = ["CODLOCAL", "CODGEO", "DPTO", "PROV", "DISTRITO"]
    if anio not in config.ESCALE_SERVICIOS_TABLA:
        columnas += [c for c, _ in config.ESCALE_SERVICIOS_COLUMNAS.values()]
    lineal = utils.leer_dbf(dbf_de_zip("escale", anio, config.ESCALE_ZIP_LINEAL), columnas=columnas)
    lineal = lineal.drop_duplicates("CODLOCAL")
    vacios = lineal.CODGEO == ""
    if vacios.any():   # algun local viene sin ubigeo: se toma el de otro local con el mismo departamento/provincia/distrito
        clave = ["DPTO", "PROV", "DISTRITO"]
        mapa = lineal[~vacios].drop_duplicates(clave).set_index(clave).CODGEO
        lineal.loc[vacios, "CODGEO"] = [mapa.get(k, "") for k in lineal.loc[vacios, clave].itertuples(index=False, name=None)]
        sin_ubigeo = int((lineal.CODGEO == "").sum())
        utils.log(f"ESCALE {anio}: {int(vacios.sum())} local(es) sin ubigeo, {int(vacios.sum()) - sin_ubigeo} recuperado(s) por nombre, {sin_ubigeo} excluido(s)")
        lineal = lineal[lineal.CODGEO != ""]
    return lineal


def cargar_locales_padron(anio):
    """Del padron (una fila por servicio educativo) saca, por local: si es publico y si es rural."""
    padron = utils.leer_dbf(dbf_de_zip("escale", anio, config.ESCALE_ZIP_PADRON),
                            columnas=["CODLOCAL", "GESTION", "AREA_CENSO"])
    padron = padron[padron.CODLOCAL != ""]   # los programas no escolarizados no tienen local
    padron["publico"] = padron.GESTION.isin(config.ESCALE_GESTION_PUBLICA)
    padron["rural"] = padron.AREA_CENSO == config.ESCALE_AREA_RURAL
    # un local es publico si tiene al menos un servicio publico (coincide con GES_LOC del Padlocal 2025)
    return padron.groupby("CODLOCAL").agg(publico=("publico", "any"), rural=("rural", "first"))


def _es_red(codigos, validos):
    """Serie de codigos -> booleano con NA cuando no hay dato ('' o '-')."""
    return codigos.isin(validos).astype("boolean").mask(codigos.isin(["", "-"]))


def servicios_por_local(anio, lineal):
    """Devuelve (tabla, notas): por local, agua/desague/luz de red publica (True/False/NA)."""
    notas = {}
    if anio in config.ESCALE_SERVICIOS_TABLA:
        cfg = config.ESCALE_SERVICIOS_TABLA[anio]
        t = utils.leer_dbf(dbf_de_zip("escale", anio, cfg["zip"]), columnas=["CODLOCAL", "CUADRO", "NUMERO", cfg["marca"]])
        t = t[t[cfg["marca"]].str.upper() == "X"]   # solo la fila predominante
        columnas = {}
        for servicio, cuadro in cfg["cuadros"].items():
            s = t[t.CUADRO == cuadro]
            repetidos = s.CODLOCAL.value_counts()
            notas[f"{servicio}_con_varias_predominantes"] = int((repetidos > 1).sum())
            s = s[~s.CODLOCAL.isin(repetidos[repetidos > 1].index)]   # ambiguas: se tratan como sin dato
            codigo = s.set_index("CODLOCAL").NUMERO.reindex(lineal.CODLOCAL)
            codigo.index = lineal.index
            columnas[servicio] = _es_red(codigo.fillna(""), config.ESCALE_RED_PUBLICA_TABLA)
        return pd.DataFrame(columnas), notas
    if anio != 2025:
        utils.log(f"AVISO: el anio {anio} no tiene formato configurado; se usa el formato de columnas de 2025")
    return pd.DataFrame({servicio: _es_red(lineal[columna], validos)
                         for servicio, (columna, validos) in config.ESCALE_SERVICIOS_COLUMNAS.items()}), notas


def edificaciones_por_local(anio):
    """Por local: si hay dato de edificaciones, si alguna en uso declara riesgo y si alguna tiene estructura en mal estado."""
    cfg = config.ESCALE_EDIFICACIONES.get(anio)
    if cfg is None:
        utils.log(f"AVISO: el anio {anio} no tiene campos de edificaciones configurados; se usan los de 2025")
        cfg = config.ESCALE_EDIFICACIONES[2025]
    t = utils.leer_dbf(dbf_de_zip("escale", anio, config.ESCALE_ZIP_EDIFICACIONES),
                       columnas=["CODLOCAL", cfg["uso"], cfg["riesgo"]] + cfg["estado"])
    t = t[t[cfg["uso"]].isin(config.ESCALE_SI)].copy()   # solo edificaciones en uso
    t["dato"] = t[cfg["riesgo"]].isin(config.ESCALE_SI + config.ESCALE_NO)
    t["riesgo"] = t[cfg["riesgo"]].isin(config.ESCALE_SI)
    t["mal"] = t[cfg["estado"]].isin(config.ESCALE_ESTADO_MALO).any(axis=1)
    return t.groupby("CODLOCAL").agg(edif_dato=("dato", "any"), edif_riesgo=("riesgo", "any"), edif_mal=("mal", "any"))


def tabla_escale_anio(anio):
    """Una fila por distrito para un anio, mas un diccionario con el recuento de depuracion."""
    lineal = cargar_lineal(anio)
    locales = cargar_locales_padron(anio)
    servicios, notas = servicios_por_local(anio, lineal)
    df = pd.concat([lineal[["CODLOCAL", "CODGEO", "DPTO", "PROV", "DISTRITO"]], servicios], axis=1)
    df = df.merge(locales, left_on="CODLOCAL", right_index=True, how="left")
    df = df.merge(edificaciones_por_local(anio), left_on="CODLOCAL", right_index=True, how="left")
    for columna in ["edif_dato", "edif_riesgo", "edif_mal"]:
        df[columna] = df[columna].fillna(False).astype(bool)
    depuracion = {"anio": anio, "locales_censo": len(df), "sin_padron": int(df.publico.isna().sum()),
                  "privados_excluidos": int((df.publico == False).sum())}   # noqa: E712
    df = df[df.publico == True].copy()   # noqa: E712  (solo gestion publica y locales activos del padron)
    df["tres"] = df.agua & df.desague & df.luz   # logica de tres valores: False si ya falta un servicio con certeza
    df["rural"] = df.rural.astype(bool)
    df["riesgo_si"] = df.edif_dato & df.edif_riesgo
    df["mal_si"] = df.edif_dato & df.edif_mal
    depuracion.update(publicos=len(df), tres_sin_dato=int(df.tres.isna().sum()),
                      con_dato_edif=int(df.edif_dato.sum()), **notas)

    grupos = df.groupby("CODGEO")
    con_dato = grupos.tres.apply(lambda s: s.notna().sum())
    tres_si = grupos.tres.apply(lambda s: (s == True).sum())   # noqa: E712
    con_edif = grupos.edif_dato.sum()
    salida = pd.DataFrame({
        "anio": anio,
        "departamento": grupos.DPTO.agg(lambda s: s.mode().iat[0]),
        "provincia": grupos.PROV.agg(lambda s: s.mode().iat[0]),
        "distrito": grupos.DISTRITO.agg(lambda s: s.mode().iat[0]),
        "n_locales_publicos": grupos.size(),
        "n_locales_sin_dato": grupos.tres.apply(lambda s: s.isna().sum()),
        "pct_3_servicios": (100 * tres_si / con_dato.replace(0, np.nan)).round(2),
        "pct_rural": (100 * grupos.rural.mean()).round(2),
        "n_locales_con_dato_edif": con_edif,
        "pct_riesgo_estructural": (100 * grupos.riesgo_si.sum() / con_edif.replace(0, np.nan)).round(2),
        "pct_estructura_mal_estado": (100 * grupos.mal_si.sum() / con_edif.replace(0, np.nan)).round(2),
    })
    salida["pct_sin_3_servicios"] = (100 - salida.pct_3_servicios).round(2)
    salida.index.name = "ubigeo"
    return salida.reset_index(), depuracion


NOMBRE_VALIDO = re.compile(r"^[A-ZÁÉÍÓÚÑÜ0-9 .,'\-/()_]+$")   # mayusculas, tildes, Ñ y signos normales (sin caracteres daniados)


def reparar_texto(texto):
    """Revierte el texto doblemente codificado de la fuente ('CONSTITUCIÃ\x93N' -> 'CONSTITUCIÓN'); si no aplica, lo deja igual."""
    try:
        return texto.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto


def armonizar_nombres(tabla):
    """Cada ubigeo conserva el mismo nombre en todos los anios.

    La fuente de 2023 trae nombres cortados ("MADRE DE DIO") y con la Ñ daniada. Para cada codigo se elige,
    entre sus nombres sin caracteres daniados, el mas largo (el mas reciente si empatan).
    """
    tabla = tabla.sort_values("anio", ascending=False).copy()   # primero el anio mas reciente
    for columna in ("departamento", "provincia", "distrito"):
        tabla[columna] = tabla[columna].map({n: reparar_texto(n) for n in tabla[columna].unique()})

    def mejor(nombres):
        distintos = list(dict.fromkeys(nombres))
        limpios = [n for n in distintos if NOMBRE_VALIDO.match(n)] or distintos
        return max(limpios, key=len)

    cambios = 0
    for columna, digitos in (("departamento", 2), ("provincia", 4), ("distrito", 6)):
        clave = tabla.ubigeo.str[:digitos]
        canonico = tabla.groupby(clave)[columna].agg(mejor)
        nuevo = clave.map(canonico)
        cambios += int((nuevo != tabla[columna]).sum())
        tabla[columna] = nuevo
    if cambios:
        utils.log(f"EXTRACCION ESCALE: nombres armonizados por ubigeo ({cambios} valores corregidos)")
    return tabla.sort_values(["anio", "ubigeo"])


def anios_escale_disponibles(anio_inicio):
    carpetas = [int(p.name) for p in (config.RAW / "escale").iterdir() if p.is_dir() and p.name.isdigit()]
    return sorted(a for a in carpetas if a >= anio_inicio)


def construir_escale(anio_inicio=config.ANIO_INICIO):
    """Genera data\\processed\\escale_infraestructura.csv"""
    tablas, resumen = [], []
    for anio in anios_escale_disponibles(anio_inicio):
        try:
            tabla, depuracion = tabla_escale_anio(anio)
            tablas.append(tabla)
            resumen.append(depuracion)
            utils.log(f"EXTRACCION ESCALE {anio}: {len(tabla)} distritos, {depuracion['publicos']} locales publicos")
        except Exception as error:   # un anio con problemas no detiene a los demas
            utils.log(f"EXTRACCION ESCALE {anio}: fallo ({type(error).__name__}: {error})")
    if not tablas:
        utils.log("EXTRACCION ESCALE: no hubo datos para procesar")
        return None, resumen
    final = armonizar_nombres(pd.concat(tablas))
    columnas = ["anio", "ubigeo", "departamento", "provincia", "distrito", "n_locales_publicos",
                "n_locales_sin_dato", "pct_3_servicios", "pct_sin_3_servicios", "pct_rural",
                "n_locales_con_dato_edif", "pct_riesgo_estructural", "pct_estructura_mal_estado"]
    ruta = config.PROCESSED / "escale_infraestructura.csv"
    final[columnas].to_csv(ruta, index=False, encoding="utf-8")
    utils.log(f"EXTRACCION ESCALE: guardado {ruta.name} ({len(final)} filas)")
    return final[columnas], resumen


# ---------------- ENLA: tablas regionales dentro de los PDF ----------------
def _sin_tildes(texto):
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower().strip()


REGIONES_NORMALIZADAS = {_sin_tildes(r): r for r in config.ENLA_REGIONES}
FILA_REGION = re.compile(r"^(?P<region>[^\d\n]+?)\s+(?P<numeros>\d+(?:,\d+)?\*?(?:\s+\d+(?:,\d+)?\*?)+)\s*$")


def leer_pagina_regional(texto, ambos_anios=False):
    """Si la pagina es una tabla regional (titulo conocido) devuelve (area, grado, anio, {region: satisfactorio}).

    Con ambos_anios=True devuelve (area, grado, {anio: {region: satisfactorio}}) con los dos anios que compara la tabla.

    La tabla trae pares 2019/2022 (o los dos anios que se comparan) para cada nivel de logro y la medida promedio:
    4.o primaria y 2.o secundaria tienen 5 columnas por par (previo, inicio, proceso, satisfactorio, medida).
    """
    if not re.search(config.ENLA_TITULO_TABLA, texto):
        return None
    lineas = [l.strip() for l in texto.splitlines()]
    area = next((config.ENLA_AREAS[l] for l in lineas if l in config.ENLA_AREAS), None)
    m = re.search(r"(\d)\.\S\s*grado de (primaria|secundaria)", texto)
    grado = config.ENLA_GRADOS.get((m.group(1), m.group(2))) if m else None
    if not area or not grado:
        return None
    anios = [int(a) for l in lineas if re.fullmatch(r"(?:(?:19|20)\d\d\s*){4,10}", l) for a in l.split()]
    if not anios:
        return None
    valores, previos, niveles_suman = {}, {}, True
    for linea in lineas:
        f = FILA_REGION.match(linea)
        region = REGIONES_NORMALIZADAS.get(_sin_tildes(f.group("region"))) if f else None
        if not region:
            continue
        nums = [float(n.replace("*", "").replace(",", ".")) for n in f.group("numeros").split()]
        if len(nums) != 10:   # previo, inicio, proceso, satisfactorio, medida; cada uno con dos anios
            return None
        niveles = [nums[1], nums[3], nums[5], nums[7]]   # columnas del anio mas reciente
        niveles_suman &= abs(sum(niveles) - 100) <= 0.6
        valores[region] = nums[7]
        previos[region] = nums[6]   # satisfactorio del anio anterior (los pares son anio anterior / anio reciente)
    if len(valores) != len(config.ENLA_REGIONES) or not niveles_suman:
        utils.log(f"AVISO ENLA: tabla {area} {grado} descartada ({len(valores)} regiones, niveles suman 100: {niveles_suman})")
        return None
    if ambos_anios:
        return area, grado, {max(anios): valores, min(anios): previos}
    return area, grado, max(anios), valores


ARCHIVO_SICRECE = re.compile(r"^enla_(\d{4})_sicrece_(muestral|censal)_.+_(\d{4}-\d{2}-\d{2})\.json$")


def region_de_sicrece(descripcion):
    """'Dre La libertad' o 'La Libertad' -> nombre de region de config.ENLA_REGIONES (None si no se reconoce)."""
    nombre = re.sub(r"^dre\s+", "", _sin_tildes(descripcion))
    return REGIONES_NORMALIZADAS.get(nombre)


def leer_json_sicrece(ruta):
    """De un JSON de SICRECE devuelve {(area, grado): {region: satisfactorio}} y una lista de problemas."""
    resultados, problemas = {}, []
    for entrada in json.loads(ruta.read_text(encoding="utf-8")):
        grado = config.SICRECE_GRADOS.get((entrada["nivel_educativo"].upper(), entrada["grado"].upper()))
        area = next((v for k, v in config.ENLA_AREAS.items() if _sin_tildes(k) == _sin_tildes(entrada["competencia"])), None)
        if not grado or not area:
            continue   # grados o areas que no se piden
        valores = {}
        for ambito in entrada["ambito"]:
            descripcion = ambito.get("descripcion_nivel_resultado") or ambito.get("descripcion_sede") or ""
            region = region_de_sicrece(descripcion)
            if not region:
                problemas.append(f"{ruta.name}: region no reconocida '{descripcion}'")
                continue
            niveles = {d["logro"]: d["porcentaje"] for d in ambito["detalle"]}
            if abs(sum(niveles.values()) - 100) > 0.6:
                problemas.append(f"{ruta.name}: {region} {area} niveles suman {sum(niveles.values()):.1f}")
            valores[region] = niveles.get("Satisfactorio", np.nan)
        if len(valores) != len(config.ENLA_REGIONES):
            problemas.append(f"{ruta.name}: {area} {grado} trae {len(valores)} regiones en vez de {len(config.ENLA_REGIONES)}")
        resultados[(area, grado)] = valores
    return resultados, problemas


def verificar_contra_pdf(tabla):
    """Compara los valores de SICRECE de 2022 con las tablas en texto del reporte nacional EM 2022 (si esta descargado)."""
    pdfs = sorted((config.RAW / "enla" / "2022").glob("*reporte_nacional*.pdf"))
    if not pdfs:
        return
    comparados, distintos = 0, []
    with pdfplumber.open(pdfs[-1]) as pdf:
        for pagina in pdf.pages:
            resultado = leer_pagina_regional(pagina.extract_text() or "")
            if not resultado:
                continue
            area, grado, anio, valores = resultado
            columna = f"{area}_satisfactorio"
            for region, valor_pdf in valores.items():
                fila = tabla[(tabla.anio == anio) & (tabla.grado == grado) & (tabla.region == region) & (tabla.evaluacion == "MUESTRAL")]
                if len(fila):
                    comparados += 1
                    if abs(fila.iloc[0][columna] - valor_pdf) > 0.05:
                        distintos.append((region, grado, area, fila.iloc[0][columna], valor_pdf))
    utils.log(f"VERIFICACION ENLA 2022 contra el PDF: {comparados} valores comparados, {len(distintos)} distintos {distintos[:5]}")


def construir_enla(verificar_pdf=False):
    """Genera data/processed/enla_regiones.csv con los resultados por region (DRE) que se obtuvieron de SICRECE."""
    encontrados, problemas = {}, []
    for ruta in sorted((config.RAW / "enla").glob("*/enla_*_sicrece_*.json")):
        m = ARCHIVO_SICRECE.match(ruta.name)
        if not m:
            continue
        anio, tipo = int(m.group(1)), m.group(2).upper()
        resultados, p = leer_json_sicrece(ruta)
        problemas += p
        for (area, grado), valores in resultados.items():
            encontrados[(anio, tipo, grado, area)] = valores   # si hay varias fechas gana la mas reciente (orden alfabetico)
    for problema in problemas:
        utils.log(f"AVISO ENLA: {problema}")
    filas = []
    for anio, tipo, grado in sorted({(a, t, g) for a, t, g, _ in encontrados}):
        lectura = encontrados.get((anio, tipo, grado, "lectura"), {})
        matematica = encontrados.get((anio, tipo, grado, "matematica"), {})
        for region in config.ENLA_REGIONES:
            filas.append({"region": region, "anio": anio, "grado": grado, "evaluacion": tipo,
                          "lectura_satisfactorio": lectura.get(region, np.nan),
                          "matematica_satisfactorio": matematica.get(region, np.nan)})
    if not filas:
        utils.log("EXTRACCION ENLA: no hay respuestas de SICRECE en data/raw/enla")
        return None
    tabla = pd.DataFrame(filas)
    ruta = config.PROCESSED / "enla_regiones.csv"
    tabla.to_csv(ruta, index=False, encoding="utf-8")
    utils.log(f"EXTRACCION ENLA: guardado {ruta.name} ({len(tabla)} filas)")
    if verificar_pdf:
        verificar_contra_pdf(tabla)
    return tabla


def construir_enla_niveles():
    """Genera data/processed/enla_niveles_logro.csv: los 4 niveles de logro por region (no solo Satisfactorio).

    Sale de los mismos JSON de SICRECE que enla_regiones.csv. Cada fila suma 100 (con tolerancia de 0.6 por el redondeo).
    """
    filas = []
    for ruta in sorted((config.RAW / "enla").glob("*/enla_*_sicrece_*.json")):
        m = ARCHIVO_SICRECE.match(ruta.name)
        if not m:
            continue
        anio, tipo = int(m.group(1)), m.group(2).upper()
        for entrada in json.loads(ruta.read_text(encoding="utf-8")):
            grado = config.SICRECE_GRADOS.get((entrada["nivel_educativo"].upper(), entrada["grado"].upper()))
            area = next((v for k, v in config.ENLA_AREAS.items() if _sin_tildes(k) == _sin_tildes(entrada["competencia"])), None)
            if not grado or not area:
                continue
            for ambito in entrada["ambito"]:
                region = region_de_sicrece(ambito.get("descripcion_nivel_resultado") or ambito.get("descripcion_sede") or "")
                if not region:
                    continue
                niveles = {d["logro"]: d["porcentaje"] for d in ambito["detalle"]}
                clave = unicodedata.normalize("NFKD", region).encode("ascii", "ignore").decode().upper()
                filas.append({"region": region, "region_clave": clave,
                              "cod_departamento": config.DEPARTAMENTOS.get("LIMA" if clave.startswith("LIMA") else clave),
                              "anio": anio, "evaluacion": tipo, "grado": grado, "area": area,
                              "previo_al_inicio": niveles.get("Previo al inicio", np.nan), "en_inicio": niveles.get("En inicio", np.nan),
                              "en_proceso": niveles.get("En proceso", np.nan), "satisfactorio": niveles.get("Satisfactorio", np.nan),
                              "archivo_json": ruta.name})
    if not filas:
        utils.log("EXTRACCION ENLA NIVELES: no hay respuestas de SICRECE en data/raw/enla")
        return None
    tabla = pd.DataFrame(filas).drop_duplicates(["region", "anio", "evaluacion", "grado", "area"], keep="last")
    tabla = tabla.sort_values(["anio", "grado", "area", "region"]).reset_index(drop=True)
    suma = tabla[["previo_al_inicio", "en_inicio", "en_proceso", "satisfactorio"]].sum(axis=1)
    malas = tabla[(suma - 100).abs() > 0.6]
    if len(malas):
        utils.log(f"AVISO ENLA NIVELES: {len(malas)} fila(s) no suman 100")
    tabla.to_csv(config.PROCESSED / "enla_niveles_logro.csv", index=False, encoding="utf-8-sig")
    utils.log(f"EXTRACCION ENLA NIVELES: guardado enla_niveles_logro.csv ({len(tabla)} filas; {len(malas)} que no suman 100)")
    return tabla


# ---------------- validacion ----------------
def validar(escale, enla):
    """Imprime filas por anio, nulos por columna, rangos 0-100 y 10 filas de ejemplo de cada tabla."""
    pd.set_option("display.width", 250, "display.max_columns", 30)
    for nombre, tabla, columnas_pct in [
        ("escale_infraestructura", escale, ["pct_3_servicios", "pct_sin_3_servicios", "pct_rural",
                                            "pct_riesgo_estructural", "pct_estructura_mal_estado"]),
        ("enla_regiones", enla, ["lectura_satisfactorio", "matematica_satisfactorio"]),
    ]:
        utils.log(f"VALIDACION {nombre}")
        if tabla is None:
            print("   (no se genero en esta corrida)")
            continue
        clave = ["anio"] if nombre.startswith("escale") else ["anio", "evaluacion", "grado"]
        print("filas por", "/".join(clave), ":", tabla.groupby(clave).size().to_dict())
        print("nulos por columna:", tabla.isna().sum().to_dict())
        fuera = {c: int(((tabla[c] < 0) | (tabla[c] > 100)).sum()) for c in columnas_pct}
        print("valores fuera de 0-100:", fuera, "| minimo/maximo:", {c: (tabla[c].min(), tabla[c].max()) for c in columnas_pct})
        print(tabla.sample(min(10, len(tabla)), random_state=1).sort_index().to_string(index=False))
    if enla is not None:
        por_anio = enla.groupby(["anio", "evaluacion", "grado"]).region.nunique()
        print("\nENLA regiones por anio, evaluacion y grado (deben ser 26):", por_anio.to_dict())


if __name__ == "__main__":
    escale, _ = construir_escale()
    enla = construir_enla()
    validar(escale, enla)

### `catalogo_escale` — Catálogo de variables de ESCALE
C2: catalogo de las variables de los archivos de ESCALE que ya estan descargados (sin descargar nada mas).

In [ ]:
%%modulo catalogo_escale
"""C2: catalogo de las variables de los archivos de ESCALE que ya estan descargados (sin descargar nada mas).

Lee los .dbf de data/interim/escale/{anio} (nombres, tipo y largo de cada campo) y busca la descripcion de cada campo en el
diccionario de datos oficial (texto del PDF en data/interim/pdf). Marca con palabras clave las variables que podrian servir
para describir la infraestructura de un local. Salida: docs/catalogo_variables_escale.csv
"""
import re

import pandas as pd

import config
import utils

PALABRAS_INFRAESTRUCTURA = ("internet", "conectividad", "aula", "conservaci", "mantenimiento", "riesgo", "terreno", "saneamiento",
                            "servicios higi", "agua", "energ", "electric", "estado de", "techo", "pared", "piso", "biblioteca",
                            "laboratorio", "discapacidad", "rampa", "cerco", "titulo", "propiedad", "inversion", "construcc")
LINEA_CAMPO = re.compile(r"^(?P<campo>[A-Za-z][A-Za-z0-9_]*)\s+(?P<tipo>[CNDL])\s+\((?P<largo>[\d,]+)\)\s*(?P<resto>.*)$")


def descripciones(texto):
    """campo -> descripcion (la primera linea del diccionario mas las siguientes hasta la lista de opciones o el siguiente campo)."""
    lineas = texto.splitlines()
    salida, i = {}, 0
    while i < len(lineas):
        m = LINEA_CAMPO.match(lineas[i].strip())
        if m:
            partes = [m.group("resto").strip()]
            j = i + 1
            while j < len(lineas) and len(partes) < 4:
                s = lineas[j].strip()
                if not s or LINEA_CAMPO.match(s) or re.match(r"^\d+:", s):
                    break
                partes.append(s)
                j += 1
            salida.setdefault(m.group("campo").upper(), re.sub(r"\s+", " ", " ".join(partes)).strip())
            i = j
        else:
            i += 1
    return salida


def construir(anio=None):
    """Escribe docs/catalogo_variables_escale.csv para el anio indicado (por defecto el mas reciente descargado)."""
    anios = sorted(p.name for p in (config.INTERIM / "escale").iterdir() if p.is_dir() and p.name.isdigit())
    anio = str(anio or anios[-1])
    diccionarios = list(config.INTERIM_PDF.glob(f"escale_{anio}_diccionario*/texto_completo.txt"))
    textos = descripciones(diccionarios[0].read_text(encoding="utf-8")) if diccionarios else {}
    usadas = {"CODLOCAL", "CODGEO", "DPTO", "PROV", "DISTRITO", "AREA_CENSO", "GESTION", "D_GESTION", "ESTADO"}   # llaves y filtros
    usadas |= {col for col, _ in config.ESCALE_SERVICIOS_COLUMNAS.values()}
    for campos in config.ESCALE_EDIFICACIONES.values():
        usadas |= {campos["uso"], campos["riesgo"], *campos["estado"]}
    filas = []
    for ruta in sorted((config.INTERIM / "escale" / anio).rglob("*.dbf")):
        n_registros, _, _, _, campos = utils.campos_dbf(ruta)
        for nombre, tipo, largo, _ in campos:
            descripcion = textos.get(nombre.upper(), "")
            clave = f"{nombre} {descripcion}".lower()
            filas.append({"anio": int(anio), "archivo_dbf": ruta.name, "registros": n_registros, "campo": nombre, "tipo": tipo, "largo": largo,
                          "descripcion": descripcion, "se_usa_en_escale_infraestructura": nombre.upper() in usadas,
                          "posible_infraestructura": any(p in clave for p in PALABRAS_INFRAESTRUCTURA)})
    tabla = pd.DataFrame(filas)
    tabla.to_csv(config.DOCS / "catalogo_variables_escale.csv", index=False, encoding="utf-8-sig")
    utils.log(f"CATALOGO ESCALE {anio}: {len(tabla)} variables en {tabla.archivo_dbf.nunique()} archivos; "
              f"{int(tabla.posible_infraestructura.sum())} posibles de infraestructura; {int((tabla.descripcion != '').sum())} con descripcion")
    return tabla

### `escale_obras` — ESCALE: locales en construcción
C2: locales educativos que declaran estar en construccion/reconstruccion (ESCALE 2025), con el CUI de la obra.

In [ ]:
%%modulo escale_obras
"""C2: locales educativos que declaran estar en construccion/reconstruccion (ESCALE 2025), con el CUI de la obra.

ESCALE le pregunta a cada local si se encuentra en construccion (P211), desde y hasta cuando (P212_FI, P212_FC; no se usan: vienen cortadas) y el CUI de la obra
(P212_CUI). Solo el Censo Educativo 2025 trae el CUI. Sirve para vincular una obra con su local exacto (CODLOCAL) y su distrito.
Salida: data/processed/escale_obras_en_local.csv (una fila por local que declara estar en construccion).
"""
import glob

import pandas as pd

import config
import utils

COLUMNAS = {"CODLOCAL": "codlocal", "CODGEO": "ubigeo", "DPTO": "departamento", "PROV": "provincia", "DISTRITO": "distrito",
            "P211_SI": "tipo_construccion", "P212_CUI": "cui"}
# Las fechas de la obra (P212_FI y P212_FC) vienen cortadas en el propio archivo del Censo ('25-08-2'): no se usan.


def sin_tildes_conservando_enie(texto):
    """'Junín' -> 'JUNIN'; la Ñ se conserva (CAÑETE) porque es una letra del alfabeto, no una tilde."""
    import unicodedata
    texto = unicodedata.normalize("NFD", str(texto).upper())
    salida = []
    for i, ch in enumerate(texto):
        if unicodedata.combining(ch) and not (ch == "\u0303" and i > 0 and texto[i - 1] == "N"):
            continue
        salida.append(ch)
    return unicodedata.normalize("NFC", "".join(salida)).strip()


def construir(anio=2025):
    """Genera escale_obras_en_local.csv a partir del archivo de local (Loc_Lineal_1) del censo indicado."""
    rutas = sorted(glob.glob(str(config.INTERIM / "escale" / str(anio) / "*lineal*" / "*.dbf")))
    if not rutas:
        utils.log(f"ESCALE OBRAS: no hay archivo de local de {anio} en data/interim/escale; se omite")
        return None
    locales = utils.leer_dbf(rutas[0])
    if "P212_CUI" not in locales:
        utils.log(f"ESCALE OBRAS: el censo {anio} no trae P212_CUI; se omite")
        return None
    en_obra = locales[locales["P211"].astype(str).str.strip() == "1"].copy()   # 1 = si, el local esta en construccion/reconstruccion
    tabla = en_obra[list(COLUMNAS)].rename(columns=COLUMNAS)
    for c in tabla.columns:
        tabla[c] = tabla[c].astype(str).str.strip()
    tabla["cui"] = tabla.cui.str.replace(r"\.0$", "", regex=True)
    tabla["tipo_construccion"] = tabla.tipo_construccion.map({"1": "TOTAL", "2": "PARCIAL"}).fillna("")
    for c in ("departamento", "provincia", "distrito"):
        tabla[c] = tabla[c].map(sin_tildes_conservando_enie)
    tabla.insert(0, "anio", anio)
    tabla = tabla.sort_values(["ubigeo", "codlocal"]).reset_index(drop=True)
    tabla.to_csv(config.PROCESSED / "escale_obras_en_local.csv", index=False, encoding="utf-8-sig")
    con_cui = int((tabla.cui != "").sum())
    utils.log(f"ESCALE OBRAS: {len(tabla)} locales en construccion en {anio}, {con_cui} con CUI de la obra ({tabla.loc[tabla.cui != '', 'cui'].nunique()} CUI distintos)")
    return tabla

### `escale_ampliado` — ESCALE ampliado (2025)
ESCALE ampliado (Censo Educativo 2025): mas indicadores de infraestructura por distrito, solo locales publicos.

In [ ]:
%%modulo escale_ampliado
"""ESCALE ampliado (Censo Educativo 2025): mas indicadores de infraestructura por distrito, solo locales publicos.

Los indicadores de escale_infraestructura.csv (servicios basicos, ruralidad, riesgo y estado de la estructura) existen para 2022-2025.
Estos son adicionales y se calculan solo para 2025, porque los nombres de las variables cambian cada anio:
  Del cuestionario del local (loc_lineal_1): internet, modulo de conectividad, agua y luz todos los dias, centro poblado con internet, area del terreno.
  De las edificaciones en uso (Loc_P53_Edific): pisos, antiguedad, red interna de agua y luz, rampa de acceso, estado del techo, la cobertura y el tablero electrico.
Salida: data/processed/escale_distrito_ampliado.csv (una fila por distrito).
"""
import numpy as np
import pandas as pd

import config
import extraccion
import utils

ANIO = 2025
SI, NO = config.ESCALE_SI, config.ESCALE_NO
COLUMNAS_LINEAL = ["CODLOCAL", "CODGEO", "P3230", "P3236", "P327", "P3224", "P311_5", "P232"]
COLUMNAS_EDIF = ["CODLOCAL", "P53_3", "P53_6", "P53_8_1", "P53_13_1", "P53_13_3", "P53_14_21", "P53_11_42", "P53_12_12", "P53_18_22"]

DICCIONARIO = [
    ("anio", "entero", "Año del Censo Educativo (2025).", "año"),
    ("ubigeo", "texto de 6 dígitos", "Código de ubigeo del distrito.", "código"),
    ("n_locales_publicos_ampliado", "entero", "Locales públicos del distrito en el Censo 2025 usados para estos indicadores.", "locales"),
    ("pct_internet", "decimal", "% de locales públicos con servicio de internet (de los que respondieron sí o no).", "%"),
    ("pct_modulo_conectividad", "decimal", "% de locales públicos con módulo de conectividad.", "%"),
    ("pct_agua_todos_los_dias", "decimal", "% de locales con agua potable todos los días de la semana (de los que respondieron la pregunta, que solo se hace si hay servicio).", "%"),
    ("pct_energia_todos_los_dias", "decimal", "% de locales con energía eléctrica todos los días de la semana.", "%"),
    ("pct_centro_poblado_internet", "decimal", "% de locales cuyo centro poblado cuenta con internet.", "%"),
    ("area_terreno_mediana_m2", "decimal", "Mediana del área del terreno del local (m²), sin contar los que declaran 0.", "m²"),
    ("n_edificaciones_en_uso", "entero", "Edificaciones en uso de los locales públicos del distrito.", "edificaciones"),
    ("pisos_promedio", "decimal", "Promedio de pisos de las edificaciones en uso.", "pisos"),
    ("antiguedad_mediana_anios", "decimal", "Mediana de años desde el término de la construcción (2025 − año de término; solo años entre 1900 y 2025).", "años"),
    ("pct_edif_red_interna_agua", "decimal", "% de edificaciones con red interna de agua.", "%"),
    ("pct_edif_red_interna_energia", "decimal", "% de edificaciones con red interna de energía eléctrica.", "%"),
    ("pct_edif_rampa_primer_piso", "decimal", "% de edificaciones con rampa de acceso al primer piso (de las que respondieron).", "%"),
    ("pct_edif_techo_mal_estado", "decimal", "% de edificaciones con la estructura de techos en mal estado (de las que informan su estado).", "%"),
    ("pct_edif_cobertura_mal_estado", "decimal", "% de edificaciones con la cobertura de techos en mal estado.", "%"),
    ("pct_edif_tablero_mal_estado", "decimal", "% de edificaciones con el tablero eléctrico en mal estado.", "%"),
]
INDICADORES = [c for c, *_ in DICCIONARIO if c not in ("anio", "ubigeo")]


def _pct_si(serie, validos_si=SI, validos_no=NO):
    """% de 'si' entre los que respondieron si o no (los vacios y '-' no cuentan). Devuelve (porcentaje, respondieron)."""
    respondio = serie.isin(validos_si + validos_no)
    return serie[respondio].isin(validos_si).sum(), respondio.sum()


def _pct_por_distrito(df, columna, validos_si, validos_no):
    """Serie distrito -> % de filas con 'si' entre las que respondieron."""
    marca = df[columna].isin(validos_si + validos_no)
    si = df[columna].isin(validos_si) & marca
    g = pd.DataFrame({"ubigeo": df.ubigeo, "si": si, "resp": marca}).groupby("ubigeo").sum()
    return (100 * g.si / g.resp.replace(0, np.nan)).round(2)


def construir():
    """Calcula y guarda escale_distrito_ampliado.csv; devuelve la tabla."""
    try:
        publicos = extraccion.cargar_locales_padron(ANIO)
    except FileNotFoundError:
        utils.log("ESCALE AMPLIADO: no hay datos crudos de ESCALE; se usa escale_distrito_ampliado.csv ya calculado")
        return pd.read_csv(config.PROCESSED / "escale_distrito_ampliado.csv", dtype={"ubigeo": str}, encoding="utf-8-sig")
    publicos = publicos[publicos.publico].index
    lineal = utils.leer_dbf(extraccion.dbf_de_zip("escale", ANIO, config.ESCALE_ZIP_LINEAL), columnas=COLUMNAS_LINEAL)
    lineal = lineal.drop_duplicates("CODLOCAL")
    lineal = lineal[lineal.CODLOCAL.isin(publicos) & (lineal.CODGEO != "")].rename(columns={"CODGEO": "ubigeo"})
    edif = utils.leer_dbf(extraccion.dbf_de_zip("escale", ANIO, config.ESCALE_ZIP_EDIFICACIONES), columnas=COLUMNAS_EDIF)
    edif = edif[edif.P53_3.isin(SI) & edif.CODLOCAL.isin(publicos)]
    edif = edif.merge(lineal[["CODLOCAL", "ubigeo"]], on="CODLOCAL", how="inner")   # el distrito de cada edificacion es el de su local

    salida = pd.DataFrame(index=sorted(lineal.ubigeo.unique()))
    salida["n_locales_publicos_ampliado"] = lineal.groupby("ubigeo").size()
    salida["pct_internet"] = _pct_por_distrito(lineal, "P3230", ["1"], ["2"])
    salida["pct_modulo_conectividad"] = _pct_por_distrito(lineal, "P3236", ["1"], ["2"])
    salida["pct_agua_todos_los_dias"] = _pct_por_distrito(lineal, "P327", ["1"], ["2"])
    salida["pct_energia_todos_los_dias"] = _pct_por_distrito(lineal, "P3224", ["1"], ["2"])
    salida["pct_centro_poblado_internet"] = _pct_por_distrito(lineal, "P311_5", ["1"], ["2"])
    area = pd.to_numeric(lineal.P232.str.replace(",", "."), errors="coerce")
    salida["area_terreno_mediana_m2"] = area[area > 0].groupby(lineal.ubigeo).median().round(1)

    salida["n_edificaciones_en_uso"] = edif.groupby("ubigeo").size()
    pisos = pd.to_numeric(edif.P53_6, errors="coerce")
    salida["pisos_promedio"] = pisos[pisos.between(1, 20)].groupby(edif.ubigeo).mean().round(2)
    termino = pd.to_numeric(edif.P53_8_1, errors="coerce")
    antig = (ANIO - termino)[termino.between(1900, ANIO)]
    salida["antiguedad_mediana_anios"] = antig.groupby(edif.ubigeo).median().round(1)
    salida["pct_edif_red_interna_agua"] = _pct_por_distrito(edif, "P53_13_1", ["1"], ["2"])
    salida["pct_edif_red_interna_energia"] = _pct_por_distrito(edif, "P53_13_3", ["1"], ["2"])
    salida["pct_edif_rampa_primer_piso"] = _pct_por_distrito(edif, "P53_14_21", ["1"], ["2"])
    for destino, columna in (("pct_edif_techo_mal_estado", "P53_11_42"), ("pct_edif_cobertura_mal_estado", "P53_12_12"), ("pct_edif_tablero_mal_estado", "P53_18_22")):
        salida[destino] = _pct_por_distrito(edif, columna, ["MA"], ["BU", "RE"])
    salida.insert(0, "anio", ANIO)
    salida.index.name = "ubigeo"
    salida = salida.reset_index()
    salida.to_csv(config.PROCESSED / "escale_distrito_ampliado.csv", index=False, encoding="utf-8-sig", na_rep="")
    utils.log(f"ESCALE AMPLIADO {ANIO}: {len(salida)} distritos, {len(lineal)} locales publicos, {len(edif)} edificaciones en uso; escale_distrito_ampliado.csv")
    return salida

### `leer_pdf` — PDF: texto y tablas
Fase 1 del proyecto PDF: inventario de los PDF/PPT descargados y lectura segura de paginas.

In [ ]:
%%modulo leer_pdf
"""Fase 1 del proyecto PDF: inventario de los PDF/PPT descargados y lectura segura de paginas.

Nunca se modifica un PDF original: solo se abre en modo lectura.
"""
import csv
from datetime import date
from pathlib import Path

import pdfplumber
import pypdf
from tqdm import tqdm

import config
import utils

COLUMNAS_INVENTARIO = ["archivo", "fuente", "anio", "fecha_corte", "paginas", "tipo_texto", "escaneado",
                       "n_tablas", "peso_mb", "ruta", "lectura", "fecha_inventario"]


def log(mensaje):
    utils.log(mensaje, config.LOG_PDF)


# ---------------- lectura de paginas (pdfplumber y, si falla, pypdf) ----------------
def texto_pagina(ruta, numero):
    """Texto de una pagina (1 = primera). Usa pdfplumber y, si falla, pypdf. Devuelve '' si no hay texto."""
    try:
        with pdfplumber.open(ruta) as pdf:
            return pdf.pages[numero - 1].extract_text() or ""
    except Exception as e:
        log(f"AVISO {Path(ruta).name} p.{numero}: pdfplumber fallo ({type(e).__name__}); se usa pypdf")
        try:
            return pypdf.PdfReader(str(ruta)).pages[numero - 1].extract_text() or ""
        except Exception as e2:
            log(f"ERROR {Path(ruta).name} p.{numero}: pypdf tambien fallo ({type(e2).__name__})")
            return ""


def numero_de_paginas(ruta):
    try:
        with pdfplumber.open(ruta) as pdf:
            return len(pdf.pages)
    except Exception:
        return len(pypdf.PdfReader(str(ruta)).pages)


# ---------------- datos de cada documento ----------------
def fuente_y_anio(ruta):
    """data/raw/{fuente}/{anio}/archivo.pdf -> ('contraloria', '2023'); fuera de ese esquema -> ('otro', '')."""
    partes = Path(ruta).relative_to(config.RAW).parts
    return (partes[0], partes[1]) if len(partes) >= 3 else ("otro", "")


def fechas_de_corte():
    """archivo -> fecha de corte del dato, leida del registro de fuentes."""
    if not config.REGISTRO.exists():
        return {}
    with open(config.REGISTRO, newline="", encoding="utf-8-sig") as f:
        return {fila["archivo"]: fila["fecha_corte_del_dato"] for fila in csv.DictReader(f) if fila["estado"] == "ok"}


def clasificar_texto(ruta, paginas):
    """'seleccionable', 'escaneado' o 'mixto', probando las primeras paginas y una del medio.

    Un PDF escaneado es una imagen: no tiene letras que se puedan copiar y necesitaria OCR.
    """
    muestra = sorted({*range(1, min(config.PAGINAS_PRUEBA_TEXTO, paginas) + 1), (paginas + 1) // 2})
    con_texto = sum(len(texto_pagina(ruta, n).strip()) >= config.MIN_CARACTERES_PAGINA for n in muestra)
    if con_texto == len(muestra):
        return "seleccionable"
    return "escaneado" if con_texto == 0 else "mixto"


def contar_tablas(ruta):
    """Numero de tablas que detecta pdfplumber en todo el documento."""
    total = 0
    with pdfplumber.open(ruta) as pdf:
        for pagina in pdf.pages:
            try:
                total += len(pagina.find_tables())
            except Exception:
                pass
    return total


def describir(ruta, cortes):
    ruta = Path(ruta)
    fuente, anio = fuente_y_anio(ruta)
    fila = {"archivo": ruta.name, "fuente": fuente, "anio": anio, "fecha_corte": cortes.get(ruta.name, ""),
            "peso_mb": round(ruta.stat().st_size / 1024 / 1024, 3), "ruta": str(ruta.relative_to(config.RAIZ)),
            "fecha_inventario": str(date.today())}
    if ruta.suffix.lower() != ".pdf":
        fila.update(paginas="", tipo_texto="no analizado", escaneado="", n_tablas="", lectura="presentacion: sin lector")
        return fila
    try:
        paginas = numero_de_paginas(ruta)
        tipo = clasificar_texto(ruta, paginas)
        fila.update(paginas=paginas, tipo_texto=tipo, escaneado=(tipo != "seleccionable"),
                    n_tablas=contar_tablas(ruta) if tipo != "escaneado" else 0, lectura="ok")
    except Exception as e:
        fila.update(paginas="", tipo_texto="ilegible", escaneado="", n_tablas="", lectura=f"error: {type(e).__name__}")
        log(f"ERROR inventario {ruta.name}: {type(e).__name__}: {str(e)[:100]}")
    return fila


# ---------------- inventario ----------------
def leer_inventario():
    if not config.INVENTARIO_PDF.exists():
        return {}
    with open(config.INVENTARIO_PDF, newline="", encoding="utf-8-sig") as f:
        return {fila["ruta"]: fila for fila in csv.DictReader(f)}


def listar_documentos():
    return sorted(p for p in config.RAW.rglob("*") if p.is_file() and p.suffix.lower() in config.EXTENSIONES_DOCUMENTO)


def construir_inventario(forzar=False):
    """Crea o completa docs/inventario_pdf.csv. Un documento ya inventariado (misma ruta y mismo peso) no se vuelve a analizar."""
    previo = {} if forzar else leer_inventario()
    cortes = fechas_de_corte()
    filas, nuevos = [], 0
    for ruta in tqdm(listar_documentos(), desc="inventario PDF"):
        clave = str(ruta.relative_to(config.RAIZ))
        peso = round(ruta.stat().st_size / 1024 / 1024, 3)
        if clave in previo and float(previo[clave]["peso_mb"]) == peso and previo[clave]["lectura"] != "":
            filas.append(previo[clave])
            continue
        filas.append(describir(ruta, cortes))
        nuevos += 1
    filas.sort(key=lambda f: (f["fuente"], f["anio"], f["archivo"]))
    config.INVENTARIO_PDF.parent.mkdir(parents=True, exist_ok=True)
    with open(config.INVENTARIO_PDF, "w", newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS_INVENTARIO)
        escritor.writeheader()
        escritor.writerows(filas)
    escaneados = [f["archivo"] for f in filas if str(f["escaneado"]) == "True"]
    log(f"Fase 1: {len(filas)} documentos en el inventario ({nuevos} analizados ahora, {len(filas) - nuevos} ya estaban); "
        f"escaneados o mixtos: {len(escaneados)}")
    return filas

### `tablas_pdf` — PDF: catálogo de tablas
Fase 2 del proyecto PDF: extrae el texto y todas las tablas de cada documento a data/interim/pdf/{nombre}/.

In [ ]:
%%modulo tablas_pdf
"""Fase 2 del proyecto PDF: extrae el texto y todas las tablas de cada documento a data/interim/pdf/{nombre}/.

Criterios de limpieza (se documentan tambien en docs/hallazgos_pdf.md):
  1. Las celdas combinadas de los PDF llegan como None: se quitan y cada fila queda "compactada".
  2. Solo se guardan tablas con numeros; las demas (vinetas, cuadros de texto) quedan en el catalogo como descartadas.
  3. El titulo es la linea "Tabla N. ..." mas cercana por encima de la tabla.
  4. Una tabla sin titulo que empieza en la pagina siguiente a otra sin fila 'Total' es su continuacion y se une a ella.
  5. Los encabezados no van en el CSV (van en el catalogo); en las celdas se quitan saltos de linea, comas de miles, 'S/' y '%'.
Los PDF originales no se modifican.
"""
import csv
import json
import re
import unicodedata
from pathlib import Path

import pdfplumber
from tqdm import tqdm

import config
import leer_pdf
from leer_pdf import log

COLUMNAS_CATALOGO = ["archivo", "fuente", "anio", "pagina", "pagina_fin", "tabla_n", "titulo", "tipo", "descripcion",
                     "filas", "columnas", "encabezado", "estado", "archivo_csv"]

REGEX_TITULO = re.compile(r"^(tabla|cuadro|gr[aá]fico|figura|mapa)\s*n?[°º.]?\s*(\d+)\s*[.:\-–]?\s*(.*)$", re.I)
REGEX_NUMERO = re.compile(r"^(S/\.?\s*)?-?\d[\d,]*(\.\d+)?\s*%?$")

# (patron sobre el titulo sin tildes y en minusculas, tipo); se usa la primera regla que coincide
REGLAS_TIPO = [
    (r"servicios de control.*departamento", "servicios_control_departamento"),
    (r"servicios de control.*nivel", "servicios_control_nivel_gobierno"),
    (r"servicios de control", "servicios_control_total"),
    (r"principales obras|paralizadas en (?!marcad)(?!el )(?!territorio)", "principales_obras_departamento"),
    (r"ubicacion geografica", "obras_por_departamento"),
    (r"evento registrado", "evento_infobras"),
    (r"nivel de avance", "sectores_avance_alto"),
    (r"nivel de gobierno", "nivel_gobierno"),
    (r"sector(es)? de gobierno", "sector_gobierno"),
    (r"marco normativo (segun|para|que)", "marco_normativo"),
    (r"^modalidades de ejecucion", "definicion_modalidades"),
    (r"modalidad de ejecucion", "modalidad_ejecucion"),
    (r"rango de costo", "rango_costo"),
    (r"rango de avance", "rango_avance_fisico"),
    (r"causal", "causal_paralizacion"),
    (r"asignacion de pim.*departamento", "pim_departamento"),
    (r"asignacion de pim", "pim_sector"),
    (r"estado situacional|resolucion de paralizacion", "estado_situacional"),
    (r"marco normativo|d\.?u\.? n|ley n", "obras_marco_normativo"),
]


def sin_tildes(texto):
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()


def clasificar(titulo):
    """Tipo de tabla segun las palabras de su titulo ('sin_titulo' si no tiene)."""
    if not titulo:
        return "sin_titulo"
    limpio = sin_tildes(titulo).lower()
    for patron, tipo in REGLAS_TIPO:
        if re.search(patron, limpio):
            return tipo
    return "otro"


# ---------------- limpieza de celdas y filas ----------------
def convertir_celda(celda):
    """'2,392,448,145' -> '2392448145'; '16.36%' -> '16.36'; 'S/ 1,234.50' -> '1234.5'; el texto solo pierde saltos de linea.

    Los numeros con ceros a la izquierda ('010101', ubigeos o codigos) se dejan como texto.
    """
    texto = re.sub(r"\s+", " ", str(celda)).strip()
    if REGEX_NUMERO.match(texto):
        numero = re.sub(r"^S/\.?\s*|\s*%$", "", texto).replace(",", "")
        if re.match(r"^-?0\d", numero):
            return texto
        valor = float(numero)
        return str(int(valor)) if valor == int(valor) and "." not in numero else numero
    return texto


def compactar(fila):
    """Quita las celdas vacias o None de una fila (vienen de celdas combinadas del PDF)."""
    return [c for c in (convertir_celda(x) for x in fila if x is not None) if c != ""]


def es_numerica(celda):
    return bool(REGEX_NUMERO.match(celda))


def es_fila_de_datos(fila):
    return sum(es_numerica(c) for c in fila) >= 2


def separar_encabezado(filas):
    """Filas compactadas -> (encabezado, datos): el encabezado son las filas antes de la primera fila con >= 2 numeros."""
    for i, fila in enumerate(filas):
        if es_fila_de_datos(fila):
            return filas[:i], filas[i:]
    return filas, []


def texto_del_encabezado(encabezado):
    celdas = []
    for fila in encabezado:
        celdas += [c for c in fila if c not in celdas]
    return " | ".join(celdas)


# ---------------- lectura de una pagina ----------------
def lineas_de_pagina(pagina):
    try:
        return pagina.extract_text_lines(return_chars=False)
    except Exception:
        return []


def titulo_sobre_tabla(lineas, tabla_top, limite_superior):
    """Titulo (Tabla N. ...) mas cercano por encima de la tabla; une la linea siguiente si el titulo continua en otra linea."""
    candidatas = [i for i, l in enumerate(lineas)
                  if l["bottom"] <= tabla_top + 3 and l["top"] >= limite_superior - 3 and REGEX_TITULO.match(l["text"].strip())]
    if not candidatas:
        return None
    i = candidatas[-1]
    partes = [lineas[i]["text"].strip()]
    for siguiente in lineas[i + 1:i + 3]:
        if siguiente["bottom"] <= tabla_top + 3 and not REGEX_TITULO.match(siguiente["text"].strip()):
            partes.append(siguiente["text"].strip())
    return " ".join(partes)


def tablas_de_pagina(pagina, numero):
    """Tablas de una pagina como lista de dicts (titulo, encabezado, filas compactadas, estado)."""
    lineas = lineas_de_pagina(pagina)
    salida, limite = [], 0
    try:
        encontradas = sorted(pagina.find_tables(), key=lambda t: t.bbox[1])
    except Exception as e:
        log(f"AVISO p.{numero}: no se pudieron buscar tablas ({type(e).__name__})")
        return salida
    for t in encontradas:
        try:
            filas = [compactar(f) for f in t.extract()]
        except Exception as e:
            log(f"AVISO p.{numero}: una tabla no se pudo leer ({type(e).__name__})")
            continue
        filas = [f for f in filas if f]
        encabezado, datos = separar_encabezado(filas)
        titulo = titulo_sobre_tabla(lineas, t.bbox[1], limite)
        if not datos:
            # fragmento de encabezado o cuadro de texto: queda anotado pero no se guarda
            salida.append({"pagina": numero, "titulo": titulo, "encabezado": encabezado, "datos": [], "estado": "descartada: sin numeros"})
            continue
        limite = t.bbox[3]
        salida.append({"pagina": numero, "titulo": titulo, "encabezado": encabezado, "datos": datos, "estado": "guardada"})
    return salida


def unir_continuaciones(tablas):
    """Une una tabla sin titulo con la tabla guardada de la pagina anterior si esta no terminaba en 'Total'."""
    resultado = []
    for t in tablas:
        if t["estado"] != "guardada":
            resultado.append(t)
            continue
        previa = next((r for r in reversed(resultado) if r["estado"] == "guardada"), None)
        if (previa and not t["titulo"] and t["pagina"] == previa["pagina_fin"] + 1 and not previa["cerrada"]
                and len(t["datos"][0]) == len(previa["datos"][-1])):
            previa["datos"] += t["datos"]
            previa["paginas_filas"] += [t["pagina"]] * len(t["datos"])
            previa["pagina_fin"] = t["pagina"]
            previa["cerrada"] = any(f[0].lower().startswith("total") for f in t["datos"])
            continue
        t["pagina_fin"] = t["pagina"]
        t["paginas_filas"] = [t["pagina"]] * len(t["datos"])
        t["cerrada"] = any(f[0].lower().startswith("total") for f in t["datos"])
        resultado.append(t)
    return resultado


# ---------------- un documento completo ----------------
def carpeta_de(ruta):
    return config.INTERIM_PDF / Path(ruta).stem


def escribir_tabla(carpeta, t, tabla_n, nombre_pdf):
    """Guarda una tabla como CSV con columnas de procedencia y devuelve el nombre del archivo."""
    ancho = max(len(f) for f in t["datos"])
    nombre = f"tablas_p{t['pagina']:02d}_t{tabla_n}.csv"
    with open(carpeta / nombre, "w", newline="", encoding="utf-8-sig") as f:
        escritor = csv.writer(f)
        escritor.writerow(["fuente_pdf", "pagina", "tabla_n"] + [f"col_{i}" for i in range(1, ancho + 1)])
        for fila, pagina in zip(t["datos"], t["paginas_filas"]):
            escritor.writerow([nombre_pdf, pagina, tabla_n] + fila + [""] * (ancho - len(fila)))
    return nombre, ancho


def procesar_documento(ruta, fila_inventario):
    """Extrae texto y tablas de un PDF. Devuelve (filas_del_catalogo, paginas_sin_texto)."""
    carpeta = carpeta_de(ruta)
    carpeta.mkdir(parents=True, exist_ok=True)
    for viejo in list(carpeta.glob("tablas_p*.csv")) + list(carpeta.glob("texto_p*.txt")):
        viejo.unlink()
    todas, sin_texto, textos = [], [], []
    with pdfplumber.open(ruta) as pdf:
        for numero, pagina in enumerate(tqdm(pdf.pages, desc=Path(ruta).name[:40], leave=False), 1):
            try:
                texto = pagina.extract_text() or ""
            except Exception:
                texto = leer_pdf.texto_pagina(ruta, numero)   # respaldo con pypdf
            (carpeta / f"texto_p{numero:02d}.txt").write_text(texto, encoding="utf-8")
            textos.append(f"=== pagina {numero} ===\n{texto}\n")
            if len(texto.strip()) < config.MIN_CARACTERES_PAGINA:
                sin_texto.append(numero)
            todas += tablas_de_pagina(pagina, numero)
            pagina.flush_cache()
    (carpeta / "texto_completo.txt").write_text("\n".join(textos), encoding="utf-8")
    filas_catalogo, por_pagina = [], {}
    for t in unir_continuaciones(todas):
        por_pagina[t["pagina"]] = por_pagina.get(t["pagina"], 0) + 1
        n = por_pagina[t["pagina"]]
        if t["estado"] == "guardada":
            nombre_csv, ancho = escribir_tabla(carpeta, t, n, Path(ruta).name)
        else:
            nombre_csv, ancho = "", max([len(f) for f in t["encabezado"]] or [0])
        tipo = clasificar(t["titulo"]) if t["titulo"] else ("continuacion" if t["estado"] == "guardada" else "sin_titulo")
        filas_catalogo.append({
            "archivo": Path(ruta).name, "fuente": fila_inventario["fuente"], "anio": fila_inventario["anio"],
            "pagina": t["pagina"], "pagina_fin": t.get("pagina_fin", t["pagina"]), "tabla_n": n, "titulo": t["titulo"] or "",
            "tipo": tipo, "descripcion": re.sub(r"^\S+\s*N?[°º.]?\s*\d+[.:]?\s*", "", t["titulo"] or "").strip(),
            "filas": len(t["datos"]), "columnas": ancho, "encabezado": texto_del_encabezado(t["encabezado"]),
            "estado": t["estado"], "archivo_csv": nombre_csv})
    with open(carpeta / "catalogo.csv", "w", newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS_CATALOGO)
        escritor.writeheader()
        escritor.writerows(filas_catalogo)
    return filas_catalogo, sin_texto


def extraer_todo(forzar=False):
    """Procesa cada PDF del inventario que no tenga su marca _ok.json (o todos con forzar). Reune docs/catalogo_tablas.csv."""
    inventario = leer_pdf.leer_inventario()
    procesados = omitidos = 0
    for clave, fila in tqdm(sorted(inventario.items()), desc="documentos"):
        ruta = config.RAIZ / clave
        if fila["lectura"] != "ok" or ruta.suffix.lower() != ".pdf":
            continue
        marca = carpeta_de(ruta) / "_ok.json"
        peso = ruta.stat().st_size
        if not forzar and marca.exists() and json.loads(marca.read_text(encoding="utf-8")).get("peso_bytes") == peso:
            omitidos += 1
            continue
        try:
            catalogo, sin_texto = procesar_documento(ruta, fila)
        except Exception as e:
            log(f"ERROR {ruta.name}: {type(e).__name__}: {str(e)[:120]}")
            continue
        guardadas = sum(c["estado"] == "guardada" for c in catalogo)
        log(f"Fase 2: {ruta.name}: {guardadas} tablas guardadas de {len(catalogo)} detectadas; paginas sin texto: {len(sin_texto)}"
            + (f" {sin_texto[:15]}" if sin_texto else ""))
        marca.write_text(json.dumps({"peso_bytes": peso, "tablas_guardadas": guardadas, "paginas_sin_texto": sin_texto}), encoding="utf-8")
        procesados += 1
    return reunir_catalogo(), procesados, omitidos


def reunir_catalogo():
    """Junta los catalogo.csv de cada documento en docs/catalogo_tablas.csv."""
    filas = []
    for archivo in sorted(config.INTERIM_PDF.glob("*/catalogo.csv")):
        with open(archivo, newline="", encoding="utf-8-sig") as f:
            filas += list(csv.DictReader(f))
    with open(config.CATALOGO_TABLAS, "w", newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS_CATALOGO)
        escritor.writeheader()
        escritor.writerows(filas)
    return filas

### `tablas_finales` — PDF: tablas finales y validación
Fase 3 del proyecto PDF: tablas finales de los informes de la Contraloria (cifras oficiales, causales, distribuciones).

In [ ]:
%%modulo tablas_finales
"""Fase 3 del proyecto PDF: tablas finales de los informes de la Contraloria (cifras oficiales, causales, distribuciones).

De donde sale cada dato:
  * Informes 2023-2026: las tablas que extrajo tablas_pdf.py (data/interim/pdf/.../tablas_pNN_tM.csv).
  * Informes de 2022: pdfplumber no detecta bien sus tablas, asi que se lee el texto de la pagina (texto_pNN.txt).
Criterios (tambien en docs/hallazgos_pdf.md):
  * Las filas partidas en dos lineas fisicas se unen (la segunda fila empieza con un numero).
  * El orden de las columnas obras / costo / saldo cambia entre informes: se asigna por magnitud
    (obras = la suma mas pequena, costo = la mas grande, saldo = la restante).
  * Cada cifra lleva fuente_pdf, pagina y tabla_n para poder volver al PDF.
"""
import re
import unicodedata
from pathlib import Path

import pandas as pd

import config
import leer_pdf
import tablas_pdf
from leer_pdf import log

COLUMNAS_PROCEDENCIA = ["fuente_pdf", "pagina", "tabla_n"]


# ---------------- utilidades ----------------
def normalizar(texto):
    """'Áncash ' -> 'ANCASH' (mayusculas, sin tildes, sin espacios repetidos)."""
    sin = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", sin).strip().upper()


def departamento_canonico(nombre):
    n = re.sub(r"[\d*]+$", "", normalizar(nombre)).strip()
    if "CALLAO" in n:
        return "CALLAO"
    return n if n in config.DEPARTAMENTOS else None


def a_numero(celda):
    """'1,234.5' -> 1234.5; '2,849,716 ,176' (espacio suelto del PDF) -> 2849716176; '-' (sin monto) -> 0."""
    texto = str(celda).replace(",", "").replace(" ", "")
    if texto == "-":
        return 0.0
    try:
        return float(texto)
    except ValueError:
        return None


def informes_contraloria():
    """Informes PDF de la Contraloria con su fecha de corte (leida del registro de fuentes)."""
    filas = []
    for fila in leer_pdf.leer_inventario().values():
        if fila["fuente"] == "contraloria" and fila["archivo"].endswith(".pdf"):
            filas.append({"archivo": fila["archivo"], "fecha_corte": fila["fecha_corte"], "anio": int(fila["fecha_corte"][:4]),
                          "carpeta": config.INTERIM_PDF / Path(fila["archivo"]).stem})
    return sorted(filas, key=lambda f: f["fecha_corte"])


def tablas_de(informe, tipo):
    """Tablas guardadas de un tipo en un informe: lista de (meta, filas) con las columnas de procedencia ya quitadas."""
    catalogo = pd.read_csv(config.CATALOGO_TABLAS, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    seleccion = catalogo[(catalogo.archivo == informe["archivo"]) & (catalogo.tipo == tipo) & (catalogo.estado == "guardada")]
    resultado = []
    for _, meta in seleccion.iterrows():
        tabla = pd.read_csv(informe["carpeta"] / meta.archivo_csv, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        filas = [[c for c in fila[3:] if c != ""] for fila in tabla.values.tolist()]
        resultado.append((meta, filas))
    return resultado


def procedencia(informe, meta):
    return {"fuente_pdf": informe["archivo"], "pagina": int(meta["pagina"]), "tabla_n": int(meta["tabla_n"])}


# ---------------- tablas de pdfplumber (2023-2026) ----------------
def reparar_filas(filas):
    """Une las filas partidas: si una fila empieza con un numero y la anterior quedo corta, es su continuacion."""
    ancho = max(len(f) for f in filas)
    salida = []
    for fila in filas:
        if salida and tablas_pdf.es_numerica(fila[0]) and len(salida[-1]) < ancho:
            salida[-1] = salida[-1] + fila
        else:
            salida.append(list(fila))
    return salida


def asignar_bloques(registros):
    """registros: lista de (etiqueta, [5 o 6 numeros]) -> DataFrame con obras, costo y saldo (y sus porcentajes).

    Los numeros vienen como [A, %A, B, %B, C, (%C)]. Cual es obras, costo y saldo se decide por la suma de cada columna.
    """
    arreglados = []
    for etiqueta, n in registros:
        if len(n) == 6 and n[4] <= 100 < n[5]:   # a veces el porcentaje y el monto del saldo vienen intercambiados
            n = n[:4] + [n[5], n[4]]
        arreglados.append((etiqueta, n))
    cuerpo = [n for e, n in arreglados if not e.lower().startswith("total")] or [n for e, n in arreglados]
    sumas = [sum(n[i] for n in cuerpo) for i in (0, 2, 4)]
    i_obras = sumas.index(min(sumas))
    resto = [i for i in range(3) if i != i_obras]
    i_costo, i_saldo = (resto[0], resto[1]) if sumas[resto[0]] >= sumas[resto[1]] else (resto[1], resto[0])
    filas = []
    for etiqueta, n in arreglados:
        pct = lambda i: n[2 * i + 1] if 2 * i + 1 < len(n) else None
        filas.append({"etiqueta": etiqueta, "obras": n[2 * i_obras], "pct_obras": pct(i_obras), "costo": n[2 * i_costo],
                      "pct_costo": pct(i_costo), "saldo": n[2 * i_saldo], "pct_saldo": pct(i_saldo)})
    return pd.DataFrame(filas)


ETIQUETA_FALSO_TOTAL = "FILA ROTULADA TOTAL EN EL PDF (NO ES EL TOTAL)"


def bloques_de_filas(filas, obras_nacional=None):
    """Filas de una tabla -> DataFrame con obras, costo y saldo.

    Si una fila 'Total' es menor que la suma de las demas filas hay dos casos (se distinguen con el total nacional):
      * las demas filas ya suman el total nacional -> el 'Total' del PDF esta mal (p. ej. copiado de otro informe): se descarta;
      * no suman                                    -> esa fila es otra categoria mal rotulada 'Total': se conserva aparte.
    """
    registros = []
    for fila in reparar_filas(filas):
        numeros = [a_numero(c) for c in fila[1:]]
        if len(numeros) in (5, 6) and None not in numeros:
            registros.append((fila[0], numeros))
    if not registros:
        return pd.DataFrame()
    tabla = asignar_bloques(registros)
    es_t = tabla.etiqueta.map(lambda e: str(e).strip().lower().startswith("total"))
    if es_t.any() and obras_nacional is not None and tabla[es_t].obras.iloc[0] < tabla[~es_t].obras.sum():
        if tabla[~es_t].obras.sum() == obras_nacional:
            log("AVISO Fase 3: el 'Total' de una tabla no coincide con la suma de sus filas (que si suma el total nacional); se descarta")
            tabla = tabla[~es_t].reset_index(drop=True)
        else:
            log("AVISO Fase 3: una fila rotulada 'Total' no es el total; se conserva como categoria aparte")
            tabla.loc[es_t, "etiqueta"] = ETIQUETA_FALSO_TOTAL
    return tabla


def sector_de_filas(filas):
    """Tabla por sector: [nacional, regional, local, total, % obras, costo, % costo]."""
    registros = []
    for fila in filas:
        n = [a_numero(c) for c in fila[1:]]
        if len(n) == 7 and None not in n:
            registros.append({"etiqueta": fila[0], "obras_nacional": n[0], "obras_regional": n[1], "obras_local": n[2],
                              "obras": n[3], "pct_obras": n[4], "costo": n[5], "pct_costo": n[6]})
    return pd.DataFrame(registros)


# ---------------- texto de los informes de 2022 ----------------
def pagina_con(informe, patron, desde=8):
    """(numero, texto) de la primera pagina cuyo texto cumple el patron."""
    for archivo in sorted(informe["carpeta"].glob("texto_p*.txt")):
        numero = int(archivo.stem.split("_p")[1])
        texto = archivo.read_text(encoding="utf-8")
        if numero >= desde and re.search(patron, texto):
            return numero, texto
    return None, ""


def es_porcentaje(token):
    return "%" in token or "." in token


def partir_en_dos(grupos):
    """['258','824','516','136','132','368'] -> (258824516, 136132368): se elige el corte donde costo >= saldo y los largos se parecen."""
    mejor = None
    for k in range(1, len(grupos)):
        izq, der = grupos[:k], grupos[k:]
        if len(izq[0]) > 3 or len(der[0]) > 3 or any(len(g) != 3 for g in izq[1:] + der[1:]):
            continue
        a, b = int("".join(izq)), int("".join(der))
        if a >= b:
            diferencia = abs(len("".join(izq)) - len("".join(der)))
            if mejor is None or diferencia < mejor[0]:
                mejor = (diferencia, a, b)
    return (mejor[1], mejor[2]) if mejor else (None, None)


def numeros_de_tokens(tokens):
    """Tokens de una fila de 2022 -> (enteros, porcentajes). Los montos usan espacio como separador de miles.

    Un grupo de 1 o 2 digitos siempre inicia un numero nuevo (los demas grupos tienen 3 digitos). Si quedan solo 2 enteros
    (obras + un bloque con costo y saldo pegados y todos de 3 digitos) se parte el bloque donde costo >= saldo.
    """
    corridas, porcentajes, actual = [], [], []
    for t in tokens:
        limpio = t.replace(",", "")
        if es_porcentaje(t):
            if actual:
                corridas.append(actual)
                actual = []
            try:
                porcentajes.append(float(limpio.rstrip("%")))
            except ValueError:
                pass
        elif limpio.isdigit():
            actual.append(limpio)
    if actual:
        corridas.append(actual)
    enteros = []
    for grupos in corridas:
        piezas = [[grupos[0]]]
        for g in grupos[1:]:
            (piezas[-1].append(g) if len(g) == 3 else piezas.append([g]))
        enteros.append([int("".join(p)) for p in piezas])
    planos = [n for lista in enteros for n in lista]
    if len(planos) == 2:
        larga = max(range(len(corridas)), key=lambda i: len(corridas[i]))
        a, b = partir_en_dos(corridas[larga])
        if a is not None:
            enteros[larga] = [a, b]
            planos = [n for lista in enteros for n in lista]
    return planos, porcentajes


def fila_2022(linea):
    """'Incumplimiento de contrato 314 13.4 9 568 323 713 4 840 583 487 33.0' -> (etiqueta, [3 enteros], porcentajes) o None."""
    linea = re.sub(r"(?<=\d),(?=\d{3})", " ", linea.strip())   # '5,788 631 244' es 5 788 631 244
    linea = re.sub(r"^\d+\s+(?=\D)", "", linea)                    # numero suelto de pie de pagina delante de la etiqueta
    m = re.match(r"^(\D+?(?:-\d+)?)\s+(\d.*)$", linea)              # la etiqueta puede terminar en '-19' (Covid-19)
    if not m:
        return None
    tokens = m.group(2).split()
    for _ in range(2):
        enteros, porcentajes = numeros_de_tokens(tokens)
        if len(enteros) == 3 and porcentajes and max(enteros) < 10 ** 12:
            return m.group(1).strip(" ."), enteros, porcentajes
        if not tokens[-1].isdigit():
            break
        tokens = tokens[:-1]   # a veces el ultimo porcentaje viene sin punto ('780' es 78.0%): se descarta
    return None


def tabla_de_texto(texto, titulo, fin=r"^(\(\*\)|\*|FUENTE|Total)"):
    """Filas (etiqueta, [obras, costo, saldo]) entre la linea del titulo y la fila Total (incluida)."""
    lineas = texto.splitlines()
    inicio = next((i for i, l in enumerate(lineas) if re.search(titulo, l, re.I)), None)
    if inicio is None:
        return []
    salida = []
    for linea in lineas[inicio + 1:]:
        fila = fila_2022(linea)
        if fila:
            salida.append(fila)
            if fila[0].lower().startswith("total"):
                break
    return salida


def bloques_de_texto(texto, titulo):
    """Tabla de 2022 como DataFrame (etiqueta, obras, costo, saldo); roles por magnitud (obras < saldo <= costo)."""
    filas = tabla_de_texto(texto, titulo)
    registros = []
    for etiqueta, enteros, porcentajes in filas:
        ordenados = sorted(enteros)
        registros.append({"etiqueta": etiqueta, "obras": ordenados[0], "saldo": ordenados[1], "costo": ordenados[2]})
    return pd.DataFrame(registros)


def sector_de_texto(texto):
    """Tabla 4 de 2022: 'Educacion 44 55 132 231 9.8 1 376 070 230 [3.5]' -> nacional, regional, local, total, costo."""
    lineas = texto.splitlines()
    inicio = next((i for i, l in enumerate(lineas) if re.search(r"TABLA 4", l)), None)
    registros = []
    for linea in lineas[inicio + 1:] if inicio is not None else []:
        m = re.match(r"^(\D+?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\S+)\s+(.*)$", linea.strip())
        if not m or "Cantidad" in linea:
            continue
        nac, reg, loc, total = (int(m.group(i)) for i in range(2, 6))
        if nac + reg + loc != total:
            continue
        resto = m.group(7).split()
        if resto and es_porcentaje(resto[-1]):
            resto = resto[:-1]
        if not all(t.isdigit() for t in resto):
            continue
        registros.append({"etiqueta": m.group(1).strip(), "obras_nacional": nac, "obras_regional": reg, "obras_local": loc,
                          "obras": total, "pct_obras": float(m.group(6).rstrip("%")), "costo": float("".join(resto)) if resto else None})
        if m.group(1).strip().lower().startswith("total"):
            break
    return pd.DataFrame(registros)


NOMBRES_MAPA = sorted([*config.DEPARTAMENTOS, "P. C. DEL CALLAO", "PROVINCIA CONSTITUCIONAL DEL CALLAO"], key=len, reverse=True)


def nombres_en_linea(linea):
    """Nombres de departamento de una linea del mapa de 2022; None si la linea trae otras cosas."""
    resto = normalizar(linea)
    encontrados = []
    while resto:
        for nombre in NOMBRES_MAPA:
            if resto.startswith(nombre):
                encontrados.append(departamento_canonico(nombre))
                resto = resto[len(nombre):].strip()
                break
        else:
            return None
    return encontrados


def montos_en_millones(linea):
    """'S/ 18 656 3 S/ 40.7' -> [18656.3, 40.7]: se lee cada tramo 'S/ ...' por separado; un token con punto cierra el numero
    y '18 656 3' es una errata de 18 656.3."""
    montos = []
    for tramo in linea.split("S/"):
        acumulado = []
        for token in tramo.split():
            if "." in token:
                montos.append(float("".join(acumulado) + token))
                acumulado = []
            else:
                acumulado.append(token)
        if acumulado:
            if len(acumulado) >= 2 and len(acumulado[-1]) == 1:
                montos.append(float("".join(acumulado[:-1]) + "." + acumulado[-1]))
            else:
                montos.append(float("".join(acumulado)))
    return montos


def departamentos_de_texto(texto):
    """Figura 1 de 2022 (mapa): bloques de nombres + linea de costo con % + linea de obras con % + linea de saldo.

    En junio los montos vienen en soles; en setiembre y diciembre vienen en millones con un decimal (precision de 100 mil soles).
    """
    lineas = [l.strip() for l in texto.splitlines()]
    registros = []
    for i, linea in enumerate(lineas[:-3]):
        nombres = nombres_en_linea(linea)
        if not nombres:
            continue
        costos = re.findall(r"((?:S/)?\s*\d[\d ]*(?:\.\d+)?)\s*\(?([\d.]+)%\)", lineas[i + 1])   # a veces falta el '('  
        obras = re.findall(r"(\d+)\s*\(([\d.]+)%\)", lineas[i + 2])
        en_millones = bool(costos) and "." in costos[0][0]
        saldos = montos_en_millones(lineas[i + 3]) if en_millones else [s for s in re.split(r"S/", lineas[i + 3]) if s.strip()]
        if not (len(nombres) == len(costos) == len(obras) == len(saldos)):
            continue
        for nombre, (c, pc), (o, po), s in zip(nombres, costos, obras, saldos):
            monto = (lambda x: round(float(re.sub(r"[^\d.]", "", x)) * 1_000_000)) if en_millones else (lambda x: float(re.sub(r"\D", "", x)))
            registros.append({"etiqueta": nombre, "obras": int(o), "pct_obras": float(po), "costo": monto(c), "pct_costo": float(pc),
                              "saldo": round(s * 1_000_000) if en_millones else monto(s), "pct_saldo": None, "unidad_original": "millones de soles (1 decimal)" if en_millones else "soles"})
    return pd.DataFrame(registros)


# ---------------- ensamblado de las tablas finales ----------------
COLUMNAS_CIFRAS = ["anio", "fecha_corte", "ambito", "categoria", "cod_departamento", "obras_paralizadas", "pct_obras",
                   "costo_actualizado_soles", "pct_costo", "saldo_inversion_soles", "obras_g_nacional", "obras_g_regional",
                   "obras_g_local", "unidad_original", "fuente_pdf", "pagina", "tabla_n"]
COLUMNAS_CAUSALES = ["anio", "fecha_corte", "causal", "obras_paralizadas", "pct_obras", "costo_actualizado_soles", "pct_costo",
                     "saldo_inversion_soles", "pct_saldo", "fuente_pdf", "pagina", "tabla_n"]
COLUMNAS_DISTRIBUCIONES = ["anio", "fecha_corte", "dimension", "categoria", "obras_paralizadas", "pct_obras",
                           "costo_actualizado_soles", "pct_costo", "saldo_inversion_soles", "pct_saldo", "fuente_pdf", "pagina", "tabla_n"]


def es_total(etiqueta):
    return str(etiqueta).strip().lower().startswith("total")


def tabla_unica(informe, tipo, convertir):
    """Primera tabla del tipo que se pueda convertir; devuelve (DataFrame, procedencia) o (vacio, None)."""
    for meta, filas in tablas_de(informe, tipo):
        tabla = convertir(filas)
        if len(tabla):
            return tabla, procedencia(informe, meta)
    return pd.DataFrame(), None


def nivel_2023_en_adelante(informe, obras_nacional):
    """La tabla de nivel de gobierno es la que tiene filas Nacional/Regional/Local y un Total igual al total nacional."""
    for meta, filas in tablas_de(informe, "nivel_gobierno"):
        tabla = bloques_de_filas(filas)
        if len(tabla) and any("nacional" in e.lower() for e in tabla.etiqueta) and es_total(tabla.etiqueta.iloc[-1]) \
                and tabla.obras.iloc[-1] == obras_nacional:
            return tabla, procedencia(informe, meta)
    return pd.DataFrame(), None


def leer_informe(informe):
    """Todas las tablas finales de un informe: dict con departamentos, sector, causales, nivel, rangos y su procedencia (_p)."""
    r = {"informe": informe}
    if informe["anio"] >= 2023:
        meta, filas = tablas_de(informe, "obras_por_departamento")[0]
        r["dep"], r["dep_p"] = bloques_de_filas(recuperar_departamentos(informe, meta, filas)), procedencia(informe, meta)
        total = r["dep"][r["dep"].etiqueta.map(es_total)]
        obras_nacional = total.obras.iloc[0] if len(total) else r["dep"].obras.sum()
        con_nacional = lambda filas: bloques_de_filas(filas, obras_nacional)
        r["sector"], r["sector_p"] = tabla_unica(informe, "sector_gobierno", sector_de_filas)
        r["causal"], r["causal_p"] = tabla_unica(informe, "causal_paralizacion", con_nacional)
        r["rango_costo"], r["rango_costo_p"] = tabla_unica(informe, "rango_costo", con_nacional)
        r["rango_avance_fisico"], r["rango_avance_fisico_p"] = tabla_unica(informe, "rango_avance_fisico", con_nacional)
        r["nivel"], r["nivel_p"] = nivel_2023_en_adelante(informe, obras_nacional)
        return r

    def desde_texto(patron, numero_tabla, convertir):
        n, texto = pagina_con(informe, patron)
        return convertir(texto), {"fuente_pdf": informe["archivo"], "pagina": n, "tabla_n": numero_tabla}

    r["nivel"], r["nivel_p"] = desde_texto(r"TABLA 3\.", 3, lambda t: bloques_de_texto(t, r"TABLA 3\."))
    r["sector"], r["sector_p"] = desde_texto(r"TABLA 4\.", 4, sector_de_texto)
    r["causal"], r["causal_p"] = desde_texto(r"TABLA 8\.", 8, lambda t: bloques_de_texto(t, r"TABLA 8\."))
    r["dep"], r["dep_p"] = desde_texto(r"FIGURA 1\.", 1, departamentos_de_texto)
    rangos = rangos_2022(informe)   # B5: rangos de costo y de avance fisico, leidos por posicion de las palabras
    r["rango_costo"], r["rango_costo_p"] = rangos["rango_costo"]
    r["rango_avance_fisico"], r["rango_avance_fisico_p"] = rangos["rango_avance_fisico"]
    return r


def nacional_de(r):
    """(obras, costo, saldo) nacionales y su procedencia: fila Total de la tabla por departamento (en 2022, de nivel de gobierno)."""
    tabla, prov = (r["dep"], r["dep_p"]) if r["informe"]["anio"] >= 2023 else (r["nivel"], r["nivel_p"])
    total = tabla[tabla.etiqueta.map(es_total)]
    if len(total):
        t = total.iloc[0]
        return {"obras": t.obras, "costo": t.costo, "saldo": t.saldo}, prov
    return {"obras": tabla.obras.sum(), "costo": tabla.costo.sum(), "saldo": tabla.saldo.sum()}, prov   # sin fila Total (febrero 2023)


def construir_tablas_finales():
    """Devuelve (cifras, causales, distribuciones, avisos, lecturas); lecturas guarda lo leido de cada informe para validar."""
    cifras, causales, distribuciones, avisos, lecturas = [], [], [], [], []
    for informe in informes_contraloria():
        r = leer_informe(informe)
        lecturas.append(r)
        base = {"anio": informe["anio"], "fecha_corte": informe["fecha_corte"]}
        nac, prov = nacional_de(r)
        cifras.append({**base, "ambito": "nacional", "categoria": "NACIONAL", "obras_paralizadas": nac["obras"], "pct_obras": 100.0,
                       "costo_actualizado_soles": nac["costo"], "pct_costo": 100.0, "saldo_inversion_soles": nac["saldo"],
                       "unidad_original": "soles", **prov})
        for _, f in r["dep"][~r["dep"].etiqueta.map(es_total)].iterrows():
            dep = departamento_canonico(f.etiqueta)
            if dep is None:
                avisos.append(f"{informe['fecha_corte']}: departamento no reconocido '{f.etiqueta}'")
                continue
            cifras.append({**base, "ambito": "departamento", "categoria": dep, "cod_departamento": config.DEPARTAMENTOS[dep],
                           "obras_paralizadas": f.obras, "pct_obras": f.pct_obras, "costo_actualizado_soles": f.costo,
                           "pct_costo": f.pct_costo, "saldo_inversion_soles": f.saldo,
                           "unidad_original": f.get("unidad_original", "soles"), **r["dep_p"]})
        for _, f in r["sector"][~r["sector"].etiqueta.map(es_total)].iterrows():
            cifras.append({**base, "ambito": "sector", "categoria": normalizar(f.etiqueta), "obras_paralizadas": f.obras,
                           "pct_obras": f.pct_obras, "costo_actualizado_soles": f.costo, "pct_costo": f.get("pct_costo"),
                           "obras_g_nacional": f.obras_nacional, "obras_g_regional": f.obras_regional,
                           "obras_g_local": f.obras_local, "unidad_original": "soles", **r["sector_p"]})
        for _, f in r["causal"][~r["causal"].etiqueta.map(es_total)].iterrows():
            causales.append({**base, "causal": re.sub(r"\s*(\(\*\)|\*)", "", normalizar(f.etiqueta)), "obras_paralizadas": f.obras, "pct_obras": f.get("pct_obras"),
                             "costo_actualizado_soles": f.costo, "pct_costo": f.get("pct_costo"), "saldo_inversion_soles": f.saldo,
                             "pct_saldo": f.get("pct_saldo"), **r["causal_p"]})
        for dimension in ("nivel", "rango_costo", "rango_avance_fisico"):
            tabla = r[dimension]
            if not len(tabla):
                if informe["anio"] >= 2023:
                    avisos.append(f"{informe['fecha_corte']}: no se encontro la tabla '{dimension}'")
                continue
            for _, f in tabla[~tabla.etiqueta.map(es_total)].iterrows():
                distribuciones.append({**base, "dimension": "nivel_gobierno" if dimension == "nivel" else dimension,
                                       "categoria": normalizar(f.etiqueta), "obras_paralizadas": f.obras, "pct_obras": f.get("pct_obras"),
                                       "costo_actualizado_soles": f.costo, "pct_costo": f.get("pct_costo"),
                                       "saldo_inversion_soles": f.saldo, "pct_saldo": f.get("pct_saldo"), **r[dimension + "_p"]})
    return (pd.DataFrame(cifras).reindex(columns=COLUMNAS_CIFRAS), pd.DataFrame(causales).reindex(columns=COLUMNAS_CAUSALES),
            pd.DataFrame(distribuciones).reindex(columns=COLUMNAS_DISTRIBUCIONES), avisos, lecturas)


# ---------------- recuperacion de filas que pdfplumber no incluyo ----------------
def recuperar_departamentos(informe, meta, filas):
    """Si a la tabla por departamento le falta alguno, busca su linea en el texto de las paginas de la tabla y la agrega."""
    presentes = {departamento_canonico(f[0]) for f in filas}
    faltan = [d for d in config.DEPARTAMENTOS if d not in presentes]
    if not faltan:
        return filas
    nuevas = []
    for pagina in range(int(meta["pagina"]), int(meta["pagina_fin"]) + 1):
        texto = (informe["carpeta"] / f"texto_p{pagina:02d}.txt").read_text(encoding="utf-8")
        for linea in texto.splitlines():
            partes = linea.split()
            for dep in faltan:
                nombre = dep.split()
                if [normalizar(p) for p in partes[:len(nombre)]] == nombre:
                    numeros = [t.rstrip("%") for t in partes[len(nombre):]]
                    if len(numeros) in (5, 6) and all(a_numero(n) is not None for n in numeros):
                        nuevas.append([linea.split(numeros[0])[0].strip()] + numeros)
    if nuevas:
        log(f"Fase 3: {informe['fecha_corte']}: se recuperaron del texto {len(nuevas)} fila(s) de departamento que faltaban en la tabla")
    # la fila Total debe quedar al final
    cuerpo = [f for f in filas if not es_total(f[0])]
    total = [f for f in filas if es_total(f[0])]
    return cuerpo + nuevas + total


def validar(cifras, causales, distribuciones, lecturas):
    """Comprobaciones por informe. Devuelve un DataFrame (una fila por comprobacion) con diferencia y si pasa."""
    checks = []

    def agregar(corte, nombre, a, b, tolerancia):
        diferencia = None if a is None or b is None else float(a) - float(b)
        checks.append({"fecha_corte": corte, "comprobacion": nombre, "valor_a": a, "valor_b": b, "diferencia": diferencia,
                       "tolerancia": tolerancia, "ok": diferencia is not None and abs(diferencia) <= tolerancia})

    for corte, c in cifras.groupby("fecha_corte"):
        nac = c[c.ambito == "nacional"].iloc[0]
        dep = c[c.ambito == "departamento"]
        sec = c[c.ambito == "sector"]
        tol_costo = 1_500_000 if (dep.unidad_original.astype(str).str.startswith("millones").any()) else config.TOLERANCIA_SOLES
        agregar(corte, "departamentos presentes (25)", len(dep), 25, 0)
        agregar(corte, "suma obras departamentos = total nacional", dep.obras_paralizadas.sum(), nac.obras_paralizadas, 0)
        agregar(corte, "suma costo departamentos = total nacional", dep.costo_actualizado_soles.sum(), nac.costo_actualizado_soles, tol_costo)
        agregar(corte, "suma saldo departamentos = total nacional", dep.saldo_inversion_soles.sum(), nac.saldo_inversion_soles, tol_costo)
        agregar(corte, "suma obras sectores = total nacional", sec.obras_paralizadas.sum(), nac.obras_paralizadas, 0)
        agregar(corte, "suma costo sectores = total nacional", sec.costo_actualizado_soles.sum(), nac.costo_actualizado_soles, config.TOLERANCIA_SOLES * 100)
        cs = causales[causales.fecha_corte == corte]
        agregar(corte, "suma obras causales = total nacional", cs.obras_paralizadas.sum(), nac.obras_paralizadas, 0)
        agregar(corte, "suma costo causales = total nacional", cs.costo_actualizado_soles.sum(), nac.costo_actualizado_soles, config.TOLERANCIA_SOLES * 100)
        ds = distribuciones[distribuciones.fecha_corte == corte]
        for dimension, grupo in ds.groupby("dimension"):
            agregar(corte, f"suma obras {dimension} = total nacional", grupo.obras_paralizadas.sum(), nac.obras_paralizadas, 0)
    return pd.DataFrame(checks)


# ---------------- validacion independiente con los anexos Excel de la Contraloria ----------------
def hoja_base(ruta):
    """(nombre de hoja, fila del encabezado) de la hoja obra por obra del anexo, o None si no tiene (la hoja trae SECTOR y DEPARTAMENTO)."""
    import openpyxl
    libro = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    for hoja in libro.worksheets:
        if hoja.max_row < 1500:
            continue
        for i, fila in enumerate(hoja.iter_rows(min_row=1, max_row=8, values_only=True)):
            columnas = [normalizar(c) for c in fila if c is not None]
            if "SECTOR" in columnas and any(c.startswith("DEPARTAMENTO") for c in columnas):
                return hoja.title, i
    return None


def leer_base_excel(ruta):
    """Base obra por obra con columnas simples: codigo, departamento, sector, sector_agrupado, costo. None si el anexo no la trae.

    El informe agrupa los sectores (columna 'SECTOR 2' / 'SECTOR2'; en jun-2026 'EDUCACION' + 'EDUCACION/CULTURA'),
    por eso se guardan el sector estricto y el agrupado.
    """
    ubicada = hoja_base(ruta)
    if ubicada is None:
        return None
    hoja, fila_encabezado = ubicada
    base = pd.read_excel(ruta, sheet_name=hoja, header=fila_encabezado, dtype=str)
    base.columns = [normalizar(c) for c in base.columns]
    col = lambda patron: next(c for c in base.columns if re.match(patron, c))
    sector = base[col(r"SECTOR$")].map(normalizar)
    agrupada = next((c for c in base.columns if re.fullmatch(r"SECTOR ?2", c)), None)
    sector_agrupado = base[agrupada].map(normalizar) if agrupada else sector.replace({"EDUCACION/CULTURA": "EDUCACION"})
    salida = pd.DataFrame({
        "codigo": base[col(r"CODIGO INFOBRAS")], "departamento": base[col(r"DEPARTAMENTO")].map(departamento_canonico),
        "sector": sector, "sector_agrupado": sector_agrupado,
        "costo": pd.to_numeric(base[col(r"COSTO ACTUALIZADO")], errors="coerce")})
    return salida[salida.codigo.notna() & (salida.codigo.str.strip() != "")].assign(hoja=hoja)


def validar_con_excel(cifras):
    """Compara las cifras del PDF con el anexo Excel del mismo corte.

    Devuelve (comparacion, educacion_por_departamento, detalle); detalle tiene una fila por cifra comparada (nacional, cada
    departamento y la fila Educacion) para marcar en las cifras cuales coinciden con el Excel.
    """
    cortes = leer_pdf.fechas_de_corte()
    comparacion, educacion, detalle = [], [], []
    for ruta in sorted(config.RAW.glob("contraloria/*/*.xlsx")):
        if "anexo_03" in ruta.name or ruta.name not in cortes:
            continue
        base = leer_base_excel(ruta)
        if base is None:
            continue
        corte = cortes[ruta.name]
        pdf = cifras[cifras.fecha_corte == corte]
        if pdf.empty:
            continue
        nac = pdf[pdf.ambito == "nacional"].iloc[0]
        edu_pdf = pdf[(pdf.ambito == "sector") & (pdf.categoria == "EDUCACION")]
        edu = base[base.sector_agrupado == "EDUCACION"]
        for nombre, v_pdf, v_xls in [
                ("obras nacional", nac.obras_paralizadas, len(base)),
                ("costo actualizado nacional (S/)", nac.costo_actualizado_soles, base.costo.sum()),
                ("obras sector Educacion", edu_pdf.obras_paralizadas.iloc[0] if len(edu_pdf) else None, len(edu)),
                ("costo sector Educacion (S/)", edu_pdf.costo_actualizado_soles.iloc[0] if len(edu_pdf) else None, edu.costo.sum())]:
            dif = None if v_pdf is None or not v_pdf else (v_xls - v_pdf) / v_pdf * 100
            comparacion.append({"fecha_corte": corte, "anexo_excel": ruta.name, "hoja": base.hoja.iloc[0], "comprobacion": nombre,
                                "valor_pdf": v_pdf, "valor_excel": v_xls, "diferencia_pct": None if dif is None else round(dif, 3),
                                "ok_10pct": dif is not None and abs(dif) <= config.UMBRAL_DIFERENCIA_PCT})
        # detalle por cifra (para la columna coincide_excel)
        detalle.append({"fecha_corte": corte, "ambito": "nacional", "categoria": "NACIONAL", "obras_excel": len(base)})
        for dep, grupo in base.groupby("departamento"):
            detalle.append({"fecha_corte": corte, "ambito": "departamento", "categoria": dep, "obras_excel": len(grupo)})
        detalle.append({"fecha_corte": corte, "ambito": "sector", "categoria": "EDUCACION", "obras_excel": len(edu)})
        # educacion por departamento: la del informe (agrupada) y la estricta (solo SECTOR = EDUCACION)
        estricta = base[base.sector == "EDUCACION"]
        por_dep = edu.groupby("departamento").agg(obras_educacion=("codigo", "count"), costo_educacion_soles=("costo", "sum"))
        por_dep = por_dep.join(estricta.groupby("departamento").agg(obras_educacion_estricto=("codigo", "count"),
                                                                    costo_educacion_estricto_soles=("costo", "sum")), how="outer").fillna(0).reset_index()
        por_dep["anio"], por_dep["fecha_corte"] = int(corte[:4]), corte
        por_dep["cod_departamento"] = por_dep.departamento.map(config.DEPARTAMENTOS)
        por_dep["fuente_excel"], por_dep["hoja"] = ruta.name, base.hoja.iloc[0]
        educacion.append(por_dep)
    columnas = ["anio", "fecha_corte", "cod_departamento", "departamento", "obras_educacion", "costo_educacion_soles",
                "obras_educacion_estricto", "costo_educacion_estricto_soles", "fuente_excel", "hoja"]
    return (pd.DataFrame(comparacion), pd.concat(educacion)[columnas] if educacion else pd.DataFrame(columns=columnas),
            pd.DataFrame(detalle))


def marcar_coincidencia(cifras, detalle):
    """Agrega coincide_excel a las cifras: True/False si el anexo Excel del corte confirma (o no) el numero de obras; vacio si no hay Excel."""
    cifras = cifras.merge(detalle, on=["fecha_corte", "ambito", "categoria"], how="left")
    cifras["coincide_excel"] = (cifras.obras_excel == cifras.obras_paralizadas).where(cifras.obras_excel.notna())
    return cifras.drop(columns="obras_excel")


# ---------------- cruce de la base propia (INFOBRAS) con la lista oficial de la Contraloria ----------------
def cruzar_infobras_con_contraloria():
    """Compara las fichas de INFOBRAS con el anexo Excel mas reciente de la Contraloria que trae la base obra por obra.

    Devuelve una tabla de comprobaciones (cuantas obras hay en cada lado, cuantas en comun, si coinciden monto y departamento).
    """
    if not config.INFOBRAS_SALIDA.exists():
        return pd.DataFrame()
    fichas = pd.read_csv(config.INFOBRAS_SALIDA, dtype={"codigo_infobras": str}, encoding="utf-8-sig", parse_dates=["fecha_paralizacion"])
    cortes = leer_pdf.fechas_de_corte()
    anexos = [(cortes[r.name], r) for r in config.RAW.glob("contraloria/*/*.xlsx") if r.name in cortes and "anexo_03" not in r.name
              and hoja_base(r) is not None]
    if not anexos:
        return pd.DataFrame()
    corte, ruta = max(anexos, key=lambda a: a[0])
    hoja, fila = hoja_base(ruta)
    x = pd.read_excel(ruta, sheet_name=hoja, header=fila, dtype=str)
    x.columns = [normalizar(c) for c in x.columns]
    x["codigo"] = x[next(c for c in x.columns if c.startswith("CODIGO INFOBRAS"))].str.strip()
    tipo = x[next(c for c in x.columns if c.startswith("TIPO DE OBRA") and "NIVEL" not in c)].map(normalizar)
    educacion = x[tipo.str.contains("EDUC")]
    comunes = fichas.merge(x[["codigo", next(c for c in x.columns if c.startswith("COSTO DE OBRA")),
                              next(c for c in x.columns if c.startswith("DEPARTAMENTO"))]], left_on="codigo_infobras", right_on="codigo")
    costo_xlsx = pd.to_numeric(comunes[next(c for c in comunes.columns if c.startswith("COSTO DE OBRA"))], errors="coerce")
    dep_xlsx = comunes[next(c for c in comunes.columns if c.startswith("DEPARTAMENTO"))].map(departamento_canonico)
    n_comunes = len(comunes)
    fuera = fichas[~fichas.codigo_infobras.isin(x.codigo)]
    filas = [
        ("obras Educacion/Cultura en la lista de la Contraloria", len(educacion), f"{ruta.name} (corte {corte})"),
        ("obras paralizadas de Educacion/Cultura en INFOBRAS", len(fichas), f"fichas leidas el {fichas.fecha_lectura.iloc[0]}"),
        ("diferencia INFOBRAS vs Contraloria (%)", round((len(fichas) - len(educacion)) / len(educacion) * 100, 1), "supera el 10 %: son poblaciones distintas (ver hallazgos)"),
        ("obras presentes en ambas", n_comunes, "mismo codigo INFOBRAS"),
        ("de las comunes, monto del expediente tecnico igual (<=0.5 %)", int((((comunes.monto_expediente_tecnico - costo_xlsx).abs() / costo_xlsx) <= 0.005).sum()), f"de {n_comunes}"),
        ("de las comunes, mismo departamento", int((comunes.departamento == dep_xlsx).sum()), f"de {n_comunes}"),
        ("obras de INFOBRAS que no estan en la lista de la Contraloria", len(fuera), "motivo no verificado; hipotesis: la lista de la Contraloria sigue su propio criterio (inventario MEF / Invierte.pe) y no el estado de INFOBRAS"),
        ("  de ellas, paralizadas despues del corte", int((fuera.fecha_paralizacion > corte).sum()), f"fecha_paralizacion > {corte}"),
        ("obras de la lista de la Contraloria que INFOBRAS ya no marca Paralizada", int((~educacion.codigo.isin(fichas.codigo_infobras)).sum()), "motivo no verificado (reactivadas, concluidas u otro estado en INFOBRAS)"),
    ]
    return pd.DataFrame(filas, columns=["comprobacion", "valor", "detalle"])


# ---------------- Fase 3: guardar ----------------
def a_enteros(tabla, columnas):
    for c in columnas:
        if c in tabla:
            tabla[c] = pd.to_numeric(tabla[c], errors="coerce").round().astype("Int64")
    return tabla


def guardar_csv(tabla, ruta):
    """Guarda en UTF-8; los decimales se redondean a 4 cifras para que 56.91 no salga como 56.910000000000004."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla = tabla.copy()
    for columna in tabla.select_dtypes("float").columns:
        tabla[columna] = tabla[columna].round(4)
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig", na_rep="")


def ejecutar_fase3():
    """Construye y guarda cifras_oficiales_contraloria, contraloria_causales, contraloria_distribuciones y
    contraloria_educacion_departamento (data/processed), y docs/validacion_pdf.csv y docs/validacion_excel.csv."""
    cifras, causales, distribuciones, avisos, lecturas = construir_tablas_finales()
    for aviso in avisos:
        log(f"AVISO Fase 3: {aviso}")
    comparacion, educacion, detalle = validar_con_excel(cifras)
    cifras = marcar_coincidencia(cifras, detalle)
    cifras["observacion"] = ""
    cifras.loc[cifras.coincide_excel == False, "observacion"] = "no coincide con el anexo Excel del mismo corte (ver docs/hallazgos_pdf.md)"
    cifras.loc[cifras.unidad_original.astype(str).str.startswith("millones"), "observacion"] = \
        "monto leido de un mapa en millones de soles con 1 decimal (precision de 100 mil soles)"
    checks = validar(cifras, causales, distribuciones, lecturas)
    enla = construir_enla_pdf()
    checks = pd.concat([checks, validar_enla_pdf(enla).assign(fecha_corte="ENLA")], ignore_index=True)

    enteros = ["obras_paralizadas", "obras_g_nacional", "obras_g_regional", "obras_g_local"]
    guardar_csv(a_enteros(cifras, enteros + ["pagina", "tabla_n"]), config.CIFRAS_OFICIALES)
    guardar_csv(a_enteros(causales, ["obras_paralizadas", "pagina", "tabla_n"]), config.CAUSALES_CGR)
    guardar_csv(a_enteros(distribuciones, ["obras_paralizadas", "pagina", "tabla_n"]), config.DISTRIBUCIONES_CGR)
    guardar_csv(a_enteros(educacion, ["obras_educacion", "obras_educacion_estricto"]), config.PROCESSED / "contraloria_educacion_departamento.csv")
    guardar_csv(a_enteros(enla, ["pagina", "tabla_n"]), config.PROCESSED / "enla_pdf.csv")
    cruce = cruzar_infobras_con_contraloria()
    if len(cruce):
        guardar_csv(cruce, config.DOCS / "validacion_infobras.csv")
    guardar_csv(checks, config.DOCS / "validacion_pdf.csv")
    guardar_csv(comparacion, config.DOCS / "validacion_excel.csv")
    log(f"Fase 3: cifras {len(cifras)} filas, causales {len(causales)}, distribuciones {len(distribuciones)}, "
        f"educacion por departamento {len(educacion)}, enla_pdf {len(enla)}; comprobaciones internas {len(checks)} (fallan {(~checks.ok).sum()}); "
        f"comparaciones con Excel {len(comparacion)} (fuera de 10%: {(~comparacion.ok_10pct).sum()})")
    return cifras, causales, distribuciones, checks, comparacion


# ---------------- ENLA por region desde los PDF ----------------
COLUMNAS_ENLA = ["region", "cod_departamento", "anio", "grado", "area", "pct_satisfactorio", "fuente_pdf", "pagina", "tabla_n"]


def construir_enla_pdf():
    """% de estudiantes en nivel satisfactorio por region, leido de las tablas regionales de los PDF de ENLA/EM.

    Solo el reporte nacional de la EM 2022 trae esas tablas; compara 2019 con 2022 (ambos anios se guardan).
    Los demas PDF solo traen tablas de cobertura (instituciones y estudiantes evaluados) y no se usan aqui.
    """
    import extraccion
    filas = []
    for ficha in leer_pdf.leer_inventario().values():
        if ficha["fuente"] != "enla" or not ficha["archivo"].endswith(".pdf"):
            continue
        carpeta = config.INTERIM_PDF / Path(ficha["archivo"]).stem
        for archivo in sorted(carpeta.glob("texto_p*.txt")):
            resultado = extraccion.leer_pagina_regional(archivo.read_text(encoding="utf-8"), ambos_anios=True)
            if not resultado:
                continue
            area, grado, por_anio = resultado
            for anio, valores in por_anio.items():
                for region, valor in valores.items():
                    nombre = normalizar(region)
                    cod = config.DEPARTAMENTOS.get("LIMA" if nombre.startswith("LIMA") else nombre)
                    filas.append({"region": nombre, "cod_departamento": cod, "anio": anio, "grado": grado, "area": area,
                                  "pct_satisfactorio": valor, "fuente_pdf": ficha["archivo"],
                                  "pagina": int(archivo.stem.split("_p")[1]), "tabla_n": 1})   # una tabla por pagina
    tabla = pd.DataFrame(filas, columns=COLUMNAS_ENLA)
    return tabla.sort_values(["anio", "grado", "area", "region"]).reset_index(drop=True)


def validar_enla_pdf(tabla):
    """Comprobaciones de enla_pdf: 26 regiones por anio/grado/area, valores entre 0 y 100, sin nulos y igual a SICRECE en 2022."""
    checks = []
    for (anio, grado, area), grupo in tabla.groupby(["anio", "grado", "area"]):
        checks.append({"comprobacion": f"{anio} {grado} {area}: 26 regiones", "valor_a": len(grupo), "valor_b": 26, "ok": len(grupo) == 26})
        checks.append({"comprobacion": f"{anio} {grado} {area}: valores entre 0 y 100 y sin nulos", "valor_a": grupo.pct_satisfactorio.isna().sum(),
                       "valor_b": 0, "ok": grupo.pct_satisfactorio.between(0, 100).all()})
    ruta = config.PROCESSED / "enla_regiones.csv"
    if ruta.exists():
        sicrece = pd.read_csv(ruta, encoding="utf-8")
        sicrece["region"] = sicrece.region.map(normalizar)
        largo = sicrece.melt(id_vars=["region", "anio", "grado"], value_vars=["lectura_satisfactorio", "matematica_satisfactorio"],
                             var_name="area", value_name="sicrece")
        largo["area"] = largo.area.str.replace("_satisfactorio", "")
        union = tabla.merge(largo, on=["region", "anio", "grado", "area"], how="inner")
        diferencias = (union.pct_satisfactorio - union.sicrece).abs()
        checks.append({"comprobacion": f"PDF vs SICRECE ({len(union)} valores comparables, diferencia maxima en puntos)",
                       "valor_a": round(diferencias.max(), 3) if len(union) else None, "valor_b": 0.0, "ok": len(union) > 0 and diferencias.max() <= 0.15})
    return pd.DataFrame(checks)


# ---------------- B5: rangos de costo y de avance fisico de 2022, leidos por posicion de las palabras ----------------
# En 2022 pdfplumber no detecta las tablas y el texto plano mezcla los grupos de miles con las columnas ('946 408 258 969 194 409 594'
# son tres numeros). Cada columna esta alineada a la derecha, asi que se leen las palabras con su posicion: los grupos de una cifra
# estan separados por menos de 10 puntos y las columnas por mas.
HUECO_MISMA_CIFRA = 10     # puntos PDF: separacion maxima entre dos grupos de la misma cifra
HUECO_ETIQUETA = 15        # separacion minima entre la etiqueta de la fila y su primera cifra
RANGO_COSTO_2022 = {"HASTA 1 MILLON": "HASTA 1 MILLON", "1 - 3 MILLONES": "1 - 3 MILLONES", "3 - 10 MILLONES": "3 - 10 MILLONES",
                    "10 A 100 MILLONES": "10 A 100 MILLONES", "100 A 1000 MILLONES": "100 A 1000 MILLONES",
                    "MAS DE 1000 MILLONES": "DE MAS DE 1000 MILLONES"}   # misma etiqueta que usan los informes de 2023


def _cifra(texto):
    t = texto.replace(",", "").replace("%", "")
    try:
        return float(t) if "." in t else int(t)
    except ValueError:
        return None


ESQUEMAS_RANGOS_2022 = {   # numero de cifras por fila -> sus nombres (junio trae menos porcentajes que setiembre y diciembre)
    "rango_costo": {3: ["obras", "costo", "saldo"], 6: ["obras", "pct_obras", "costo", "pct_costo", "saldo", "pct_saldo"]},
    "rango_avance_fisico": {5: ["obras", "pct_obras", "costo", "saldo", "pct_saldo"],
                            6: ["obras", "pct_obras", "costo", "pct_costo", "saldo", "pct_saldo"]},
}


def filas_por_posicion(ruta_pdf, pagina, patron_titulo, esquemas):
    """Filas de una tabla de 2022 como (etiqueta, {columna: cifra}); parte de la linea del titulo y termina en la fila 'Total' (incluida).

    esquemas = {numero de cifras de la fila: [nombres de columna]}. Las filas con otro numero de cifras no se guardan (se avisa en el log).
    """
    import pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        palabras = pdf.pages[pagina - 1].extract_words()
    lineas = {}
    for w in palabras:
        lineas.setdefault(round(w["top"] / 3), []).append(w)
    ordenadas = [sorted(lineas[k], key=lambda w: w["x0"]) for k in sorted(lineas)]
    inicio = next((i for i, l in enumerate(ordenadas) if re.search(patron_titulo, " ".join(w["text"] for w in l), re.I)), None)
    if inicio is None:
        return []
    filas = []
    for linea in ordenadas[inicio + 1:]:
        if not any(ch.isdigit() for w in linea for ch in w["text"]):
            if linea and linea[0]["text"].upper().startswith("FUENTE"):
                break
            continue
        corte = next((i for i in range(1, len(linea)) if linea[i]["x0"] - linea[i - 1]["x1"] > HUECO_ETIQUETA), None)
        if corte is None:
            continue
        etiqueta = " ".join(w["text"] for w in linea[:corte])
        cifras, actual, previo = [], [], None
        for w in linea[corte:]:
            if previo is not None and w["x0"] - previo["x1"] > HUECO_MISMA_CIFRA:
                cifras.append(actual)
                actual = []
            actual.append(w["text"])
            previo = w
        cifras.append(actual)
        valores = [_cifra("".join(grupo)) for grupo in cifras]
        if len(valores) not in esquemas or None in valores:
            log(f"AVISO B5: {Path(ruta_pdf).name} p.{pagina}: fila '{etiqueta}' con {len(valores)} cifras (se esperaban {sorted(esquemas)}); no se guarda")
            continue
        filas.append((etiqueta, dict(zip(esquemas[len(valores)], valores))))
        if etiqueta.strip().lower().startswith("total"):
            break
    return filas


def rangos_2022(informe):
    """Rangos de costo y de avance fisico de un informe de 2022: {nombre: (DataFrame con etiqueta, obras, costo, saldo y porcentajes, procedencia)}."""
    ruta = config.RAW / "contraloria" / str(informe["anio"]) / informe["archivo"]
    resultado = {}
    for nombre, patron_titulo, tabla_n in (("rango_costo", r"TABLA 6\.", 6), ("rango_avance_fisico", r"TABLA 7\.", 7)):
        pagina, _ = pagina_con(informe, patron_titulo)
        filas = filas_por_posicion(ruta, pagina, patron_titulo, ESQUEMAS_RANGOS_2022[nombre]) if pagina else []
        registros = []
        for etiqueta, v in filas:
            clave = normalizar(etiqueta)
            if nombre == "rango_costo":
                etiqueta = "Total" if es_total(clave) else RANGO_COSTO_2022.get(clave, clave)
            registros.append({"etiqueta": etiqueta, **v})
        resultado[nombre] = (pd.DataFrame(registros), {"fuente_pdf": informe["archivo"], "pagina": pagina, "tabla_n": tabla_n})
    return resultado

### `contraloria_excel` — Contraloría: anexos Excel
Tareas de los anexos Excel: datos de la Contraloria desde los anexos Excel y los PDF ya extraidos (sin navegador).

In [ ]:
%%modulo contraloria_excel
"""Tareas de los anexos Excel: datos de la Contraloria desde los anexos Excel y los PDF ya extraidos (sin navegador).

  A2  leer_obras_corte / construir_obras_corte  -> data/processed/contraloria_obras_corte.csv
  A3  construir_panel                           -> data/processed/contraloria_obras_panel.csv
  B7  corregir_educacion                        -> columna obras_educacion_corregida en cifras_oficiales_contraloria.csv
  B3  construir_pim                             -> data/processed/contraloria_pim.csv
  B5  construir_rangos_2022                     -> filas nuevas en contraloria_distribuciones.csv
  B4  construir_modalidad / evento / servicios  -> un CSV por tipo en data/processed
  B6  construir_principales_obras               -> data/processed/contraloria_principales_obras.csv
Reglas: data/raw no se modifica; sin datos personales; lugares en MAYUSCULAS sin tildes; codigos como texto.
"""
import re

import pandas as pd

import config
import leer_pdf
import tablas_finales as tf
from leer_pdf import log

OBRAS_CORTE = config.PROCESSED / "contraloria_obras_corte.csv"
OBRAS_PANEL = config.PROCESSED / "contraloria_obras_panel.csv"

# campo de salida -> patron (sobre el nombre de columna normalizado). El primero que coincida gana;
# asi se sobrevive a los cambios de nombre entre cortes ('SECTOR 2'/'SECTOR2', 'SALDO DE INVERSION'/'SALDO DE LA INVERSION', ...).
PATRONES = {
    "codigo_infobras": r"CODIGO INFOBRAS",
    "cui": r"CUI$",
    "entidad": r"ENTIDAD$",
    "evento": r"EVENTO$",
    "modalidad_ejecucion": r"MODALIDAD DE EJECUCION",
    "tipo_obra": r"TIPO DE OBRA$",
    "tipo_obra_nivel2": r"TIPO DE OBRA \(NIVEL 2\)",
    "tipo_obra_nivel3": r"TIPO DE OBRA \(NIVEL 3\)",
    "nivel_gobierno": r"NIVEL DE GOBIERNO",
    "sector": r"SECTOR$",
    "sector_agrupado": r"SECTOR ?2$",
    "causal_paralizacion": r"CAUSAL PARALIZACION",
    "avance_fisico": r"AVANCE FISICO$",
    "rango_avance_fisico": r"RANGO AVANCE FISICO",
    "anio_inicio": r"ANO INICIO OBRA",
    "plazo_ejecucion_dias": r"PLAZO DE EJECUCION",
    "departamento": r"DEPARTAMENTO",
    "provincia": r"PROVINCIA",
    "distrito": r"DISTRITO",
    "costo_actualizado_soles": r"COSTO ACTUALIZADO",
    "saldo_inversion_soles": r"SALDO DE (LA )?INVERSION",
    "rango_inversion_mll": r"RANGO DE INVERSION \(",
}
COLUMNAS_OBRAS_CORTE = ["corte", "codigo_infobras", "cui", "entidad", "evento", "modalidad_ejecucion", "tipo_obra", "tipo_obra_nivel2",
                        "tipo_obra_nivel3", "nivel_gobierno", "sector", "sector_agrupado", "causal_codigo", "causal_paralizacion",
                        "avance_fisico", "rango_avance_fisico", "anio_inicio", "plazo_ejecucion_dias", "cod_departamento", "departamento",
                        "provincia", "distrito", "costo_actualizado_soles", "saldo_inversion_soles", "rango_inversion_mll",
                        "fuente_excel", "hoja"]
ALIAS_MODALIDAD = {"CONTRATA": "POR CONTRATA", "ASOCIACION PUBLICA PRIVADA - APP": "ASOCIACION PUBLICA PRIVADA APP",
                   "APP / CONCESIONES": "ASOCIACION PUBLICA PRIVADA APP",
                   "ADM. DIRECTA": "ADMINISTRACION DIRECTA", "OBRAS POR IMPUESTO": "OBRAS POR IMPUESTOS"}


def _texto_codigo(serie):
    """'2077997.0' -> '2077997'; vacio -> NA. Los codigos son texto."""
    return serie.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True).replace("", pd.NA)


def leer_obras_corte(ruta):
    """Una fila por obra de un anexo Excel (hoja base), con las columnas de PATRONES. None si el anexo no trae la base.

    Cada columna se busca por patron; las que un corte no tiene quedan vacias. El corte sale de leer_pdf.fechas_de_corte().
    """
    ubicada = tf.hoja_base(ruta)
    if ubicada is None:
        return None
    hoja, fila = ubicada
    base = pd.read_excel(ruta, sheet_name=hoja, header=fila, dtype=str)
    base.columns = [tf.normalizar(c) for c in base.columns]
    base = base.loc[:, ~base.columns.duplicated()]

    def columna(campo):
        return next((c for c in base.columns if re.match(PATRONES[campo], c)), None)

    salida = pd.DataFrame(index=base.index)
    for campo in PATRONES:
        c = columna(campo)
        salida[campo] = base[c] if c else pd.NA
    # solo filas con codigo numerico: el anexo de dic-2024 trae 3 notas al pie ('* Obras culminadas con observaciones', ...)
    salida = salida[salida.codigo_infobras.str.strip().str.fullmatch(r"\d+(\.0+)?", na=False)].copy()

    for c in ("codigo_infobras", "cui"):
        salida[c] = _texto_codigo(salida[c])
    for c in ("avance_fisico", "plazo_ejecucion_dias", "costo_actualizado_soles", "saldo_inversion_soles"):
        salida[c] = pd.to_numeric(salida[c], errors="coerce")
    salida["anio_inicio"] = pd.to_numeric(salida.anio_inicio, errors="coerce").astype("Int64")
    salida["plazo_ejecucion_dias"] = salida.plazo_ejecucion_dias.round().astype("Int64")
    for c in ("entidad", "evento", "modalidad_ejecucion", "tipo_obra", "tipo_obra_nivel2", "tipo_obra_nivel3", "nivel_gobierno", "sector",
              "causal_paralizacion", "rango_avance_fisico", "provincia", "distrito", "rango_inversion_mll"):
        salida[c] = salida[c].map(lambda x: tf.normalizar(x) if pd.notna(x) else pd.NA)
    salida["modalidad_ejecucion"] = salida.modalidad_ejecucion.replace(ALIAS_MODALIDAD)
    # sector agrupado: el informe cuenta Educacion y Educacion/Cultura juntos (columna 'SECTOR 2'; si no existe, se deriva)
    agrupado = salida.sector_agrupado.map(lambda x: tf.normalizar(x) if pd.notna(x) else pd.NA)
    salida["sector_agrupado"] = agrupado.where(agrupado.notna(), salida.sector.replace({"EDUCACION/CULTURA": "EDUCACION"}))
    # causal: '3 DISCREPANCIAS, ...' -> codigo 3 + texto; algunos cortes traen la causal sin numero
    partes = salida.causal_paralizacion.str.extract(r"^(\d+)\s+(.*)$")
    salida["causal_codigo"] = partes[0].astype("string")
    salida["causal_paralizacion"] = partes[1].where(partes[1].notna(), salida.causal_paralizacion)
    # lugares: departamento canonico (misma regla del resto del proyecto)
    dep_crudo = salida.departamento
    salida["departamento"] = dep_crudo.map(tf.departamento_canonico)
    sin_reconocer = salida.departamento.isna() & dep_crudo.notna()
    if sin_reconocer.any():
        log(f"AVISO A2: {ruta.name}: {sin_reconocer.sum()} obra(s) con departamento no reconocido: {sorted(set(dep_crudo[sin_reconocer]))[:5]}")
    salida["cod_departamento"] = salida.departamento.map(config.DEPARTAMENTOS)
    salida["fuente_excel"], salida["hoja"] = ruta.name, hoja
    salida["corte"] = leer_pdf.fechas_de_corte()[ruta.name]
    return salida.reindex(columns=COLUMNAS_OBRAS_CORTE).reset_index(drop=True)


def rutas_con_base():
    """Los anexos Excel que traen la base obra por obra (13 cortes), una ruta por corte."""
    cortes = leer_pdf.fechas_de_corte()
    rutas = {}
    for ruta in sorted(config.RAW.glob("contraloria/*/*.xlsx")):
        if "anexo_03" in ruta.name or ruta.name not in cortes or tf.hoja_base(ruta) is None:
            continue
        rutas.setdefault(cortes[ruta.name], ruta)
    return [rutas[c] for c in sorted(rutas)]


def construir_obras_corte():
    """Apila los 13 cortes: una fila por obra y corte."""
    return pd.concat([leer_obras_corte(r) for r in rutas_con_base()], ignore_index=True)


def validar_obras_corte(obras):
    """Compara filas por corte contra leer_base_excel (la lectura ya validada del proyecto). Devuelve DataFrame con ok."""
    filas = []
    for ruta in rutas_con_base():
        base = tf.leer_base_excel(ruta)
        base = base[base.codigo.str.strip().str.fullmatch(r"\d+(\.0+)?", na=False)]   # sin notas al pie
        o = obras[obras.fuente_excel == ruta.name]
        edu_a = int((o.sector_agrupado == "EDUCACION").sum())
        edu_b = int((base.sector_agrupado == "EDUCACION").sum())
        filas.append({"fecha_corte": o.corte.iloc[0], "obras_nuevo": len(o), "obras_leer_base_excel": len(base),
                      "educacion_nuevo": edu_a, "educacion_leer_base_excel": edu_b,
                      "costo_nuevo": round(o.costo_actualizado_soles.sum(), 2), "costo_leer_base_excel": round(base.costo.sum(), 2),
                      "codigos_duplicados": int(o.codigo_infobras.duplicated().sum()),
                      "ok": len(o) == len(base) and edu_a == edu_b and abs(o.costo_actualizado_soles.sum() - base.costo.sum()) < 1})
    return pd.DataFrame(filas)


# ---------------- A3: panel historico por obra ----------------
def construir_panel(obras):
    """Una fila por codigo_infobras: primer y ultimo corte, numero de cortes, si sigue en el ultimo corte, reingresos y avance por corte.

    Limites: el panel empieza en sep-2023 (los cortes de 2022 y feb-ago 2023 no tienen anexo con base), asi que 'primer_corte'
    de una obra presente desde sep-2023 es solo 'desde antes o igual a esa fecha' (primer_corte_censurado = True).
    reingresos = veces que la obra salio de la lista y volvio a entrar (cortes intermedios faltantes en bloques).
    """
    cortes = sorted(obras.corte.unique())
    posicion = {c: i for i, c in enumerate(cortes)}
    ultimo_global = cortes[-1]
    o = obras.assign(pos=obras.corte.map(posicion)).sort_values(["codigo_infobras", "pos"])
    g = o.groupby("codigo_infobras")
    panel = g.agg(primer_corte=("corte", "min"), ultimo_corte=("corte", "max"), n_cortes=("corte", "nunique"),
                  pos_min=("pos", "min"), pos_max=("pos", "max")).reset_index()
    panel["sigue_en_ultimo_corte"] = panel.ultimo_corte == ultimo_global
    panel["cortes_faltantes_intermedios"] = (panel.pos_max - panel.pos_min + 1) - panel.n_cortes
    saltos = o.assign(salto=g.pos.diff().fillna(1) > 1).groupby("codigo_infobras").salto.sum()
    panel["reingresos"] = panel.codigo_infobras.map(saltos).astype(int)
    panel["primer_corte_censurado"] = panel.primer_corte == cortes[0]
    panel = panel.drop(columns=["pos_min", "pos_max"])
    ultima = g.tail(1).set_index("codigo_infobras")
    for c in ("cui", "sector_agrupado", "nivel_gobierno", "causal_paralizacion", "cod_departamento", "departamento", "provincia", "distrito"):
        panel[f"{c}_ultimo"] = panel.codigo_infobras.map(ultima[c])
    primera = g.head(1).set_index("codigo_infobras")
    panel["avance_primer_corte"] = panel.codigo_infobras.map(primera.avance_fisico)
    panel["avance_ultimo_corte"] = panel.codigo_infobras.map(ultima.avance_fisico)
    ancho = o.pivot(index="codigo_infobras", columns="corte", values="avance_fisico")
    ancho.columns = [f"avance_{c}" for c in ancho.columns]
    panel = panel.merge(ancho.reset_index(), on="codigo_infobras", how="left")
    panel["fuente"] = "contraloria_obras_corte.csv (anexos Excel de la Contraloria, 13 cortes sep-2023 a jun-2026)"
    return panel


def validar_panel(panel, obras):
    """Comprobaciones del panel contra obras_corte (todas deben dar ok)."""
    ultimo = sorted(obras.corte.unique())[-1]
    filas = [("obras distintas", len(panel), obras.codigo_infobras.nunique()),
             ("suma de n_cortes = filas de obras_corte", int(panel.n_cortes.sum()), len(obras)),
             (f"obras que siguen en el ultimo corte ({ultimo})", int(panel.sigue_en_ultimo_corte.sum()), int((obras.corte == ultimo).sum())),
             ("codigos repetidos dentro de un mismo corte", int(obras.duplicated(["corte", "codigo_infobras"]).sum()), 0)]
    t = pd.DataFrame(filas, columns=["comprobacion", "panel", "obras_corte"])
    t["ok"] = t.panel == t.obras_corte
    return t


# ---------------- lectura de tablas de los PDF conservando las celdas vacias ----------------
def filas_posicionales(informe, meta):
    """Filas de una tabla del catalogo con las celdas vacias en su sitio.

    El CSV de data/interim compacta las filas (quita las celdas vacias), y entonces 'Por nucleo ejecutor 27 33134819 27 ...' no dice a que
    nivel de gobierno pertenece. Aqui se vuelve a abrir la pagina del PDF y se recupera la fila con '' donde la celda esta vacia; cada fila
    se empareja con la del CSV por su contenido. Devuelve (filas, n_sin_emparejar); las que no se emparejan quedan como en el CSV.
    """
    import pdfplumber
    import tablas_pdf
    ruta = config.RAW / "contraloria" / str(informe["anio"]) / informe["archivo"]
    csv_filas = [[c for c in fila[3:] if c != ""] for fila in pd.read_csv(
        informe["carpeta"] / meta.archivo_csv, dtype=str, keep_default_na=False, encoding="utf-8-sig").values.tolist()]
    candidatas = []
    with pdfplumber.open(ruta) as pdf:
        for n in range(int(meta["pagina"]), int(meta["pagina_fin"]) + 1):
            for t in pdf.pages[n - 1].find_tables():
                for fila in t.extract():
                    crudas = [tablas_pdf.convertir_celda(c) for c in fila if c is not None]
                    candidatas.append((crudas, [c for c in crudas if c != ""]))
    resultado, sin_emparejar, usadas = [], 0, set()
    for fila in csv_filas:
        i = next((k for k, (_, compacta) in enumerate(candidatas) if k not in usadas and compacta == fila), None)
        if i is None:
            resultado.append(fila)
            sin_emparejar += 1
        else:
            usadas.add(i)
            resultado.append(candidatas[i][0])
    return resultado, sin_emparejar


# ---------------- B7: serie corregida de Educacion ----------------
def corregir_educacion(cifras, educacion_departamento):
    """Agrega obras_educacion_corregida a las cifras oficiales (solo filas ambito='sector', categoria='EDUCACION').

    Donde hay anexo Excel del corte (13 cortes), el valor es la suma de contraloria_educacion_departamento (Excel); en los demas
    cortes (2022 y feb-ago 2023) es el valor del PDF, que no se puede contrastar. fuente_educacion_corregida dice cual se uso.
    En set-2023 y set-2025 el PDF trae la cifra mal (324 vs 283 y 73 vs 196).
    """
    cifras = cifras.drop(columns=[c for c in ("obras_educacion_corregida", "fuente_educacion_corregida") if c in cifras.columns]).copy()
    excel = educacion_departamento.groupby("fecha_corte").obras_educacion.sum()
    es_edu = (cifras.ambito == "sector") & (cifras.categoria == "EDUCACION")
    del_excel = cifras.fecha_corte.map(excel)
    cifras["obras_educacion_corregida"] = pd.Series(pd.NA, index=cifras.index, dtype="Int64")
    cifras["fuente_educacion_corregida"] = ""
    usa_excel = es_edu & del_excel.notna()
    cifras.loc[usa_excel, "obras_educacion_corregida"] = del_excel[usa_excel].round().astype("Int64")
    cifras.loc[usa_excel, "fuente_educacion_corregida"] = "anexo Excel (contraloria_educacion_departamento.csv)"
    solo_pdf = es_edu & del_excel.isna()
    cifras.loc[solo_pdf, "obras_educacion_corregida"] = cifras.loc[solo_pdf, "obras_paralizadas"].astype("Int64")
    cifras.loc[solo_pdf, "fuente_educacion_corregida"] = "informe PDF (sin anexo Excel para contrastar)"
    return cifras


def validar_educacion(cifras):
    """Serie corregida: debe coincidir con el PDF salvo en los cortes que el PDF trae mal (set-2023 y set-2025)."""
    e = cifras[(cifras.ambito == "sector") & (cifras.categoria == "EDUCACION")]
    t = e[["fecha_corte", "obras_paralizadas", "obras_educacion_corregida", "fuente_educacion_corregida"]].copy()
    t["difiere_del_pdf"] = t.obras_paralizadas != t.obras_educacion_corregida
    return t


# ---------------- B3: presupuesto (PIM) ----------------
def _num(celda):
    """Numero de una celda del PDF; '-' o el menos tipografico ('−') = 0; vacio = None."""
    texto = str(celda).strip().replace("−", "-")
    if texto == "":
        return None
    return tf.a_numero(texto)


def construir_pim(cifras):
    """PIM por departamento y para el total nacional de los 4 informes que lo traen (set-2025, dic-2025, mar-2026, jun-2026).

    Departamento: obras con PIM, sin PIM, % y total. Nacional: ademas costo actualizado con y sin PIM y el PIM en soles.
    Valida: 25 departamentos, con+sin = total por fila, suma de departamentos = fila Total, total = obras nacionales de cifras oficiales,
    nacional coincide con la suma de departamentos y costo total = costo nacional oficial.
    """
    filas, checks = [], []
    for informe in tf.informes_contraloria():
        deps = tf.tablas_de(informe, "pim_departamento")
        if not deps:
            continue
        corte, anio = informe["fecha_corte"], informe["anio"]
        meta, crudas = deps[0]
        anio_pim = int(re.search(r"PIM (\d{4})", meta.titulo).group(1))
        crudas = tf.recuperar_departamentos(informe, meta, crudas)
        base = {"anio": anio, "fecha_corte": corte, "anio_pim": anio_pim}
        prov = {"fuente_pdf": informe["archivo"], "pagina": int(meta.pagina), "tabla_n": int(meta.tabla_n)}
        deps_ok, total = [], None
        for f in crudas:
            n = [_num(c) for c in f[1:]]
            if len(n) != 5 or None in n:
                continue
            fila = {"obras_con_pim": n[0], "pct_obras_con_pim": n[1], "obras_sin_pim": n[2], "pct_obras_sin_pim": n[3], "obras_total": n[4]}
            if tf.es_total(f[0]):
                total = fila
                continue
            dep = tf.departamento_canonico(f[0])
            if dep:
                deps_ok.append({**base, "ambito": "departamento", "categoria": dep, "cod_departamento": config.DEPARTAMENTOS[dep], **fila, **prov})
        dep_df = pd.DataFrame(deps_ok)
        # nacional (tabla 'Asignacion de PIM' sin departamento)
        nacional = {}
        sectores = tf.tablas_de(informe, "pim_sector")
        if sectores:
            m2, filas2 = sectores[0]
            por_etiqueta = {re.sub(r"[^A-Z]", "", f[0].upper()): [_num(c) for c in f[1:]] for f in filas2}
            if {"SI", "NO", "TOTAL"} <= set(por_etiqueta):
                si, no, tot = por_etiqueta["SI"], por_etiqueta["NO"], por_etiqueta["TOTAL"]
                nacional = {"obras_con_pim": si[0], "pct_obras_con_pim": si[1], "obras_sin_pim": no[0], "pct_obras_sin_pim": no[1], "obras_total": tot[0],
                            "costo_con_pim_soles": si[2], "costo_sin_pim_soles": no[2], "costo_total_soles": tot[2], "pim_soles": si[4]}
                nac_prov = {"fuente_pdf": informe["archivo"], "pagina": int(m2.pagina), "tabla_n": int(m2.tabla_n)}
                filas.append({**base, "ambito": "nacional", "categoria": "NACIONAL", **nacional, **nac_prov})
        filas += deps_ok
        # ---- validaciones ----
        oficial = cifras[(cifras.fecha_corte == corte) & (cifras.ambito == "nacional")].iloc[0]
        agregar = lambda nombre, a, b, tol=0: checks.append({"fecha_corte": corte, "comprobacion": nombre, "valor_a": a, "valor_b": b,
                                                              "diferencia": None if a is None or b is None else float(a) - float(b), "tolerancia": tol,
                                                              "ok": a is not None and b is not None and abs(float(a) - float(b)) <= tol})
        agregar("departamentos presentes (25)", len(dep_df), 25)
        agregar("con + sin = total en cada departamento (filas que fallan)", int((dep_df.obras_con_pim + dep_df.obras_sin_pim != dep_df.obras_total).sum()), 0)
        if total:
            agregar("suma obras con PIM departamentos = fila Total", dep_df.obras_con_pim.sum(), total["obras_con_pim"])
            agregar("suma obras sin PIM departamentos = fila Total", dep_df.obras_sin_pim.sum(), total["obras_sin_pim"])
            agregar("fila Total de la tabla = obras nacionales oficiales", total["obras_total"], oficial.obras_paralizadas)
        if nacional:
            agregar("nacional: obras con PIM = suma de departamentos", nacional["obras_con_pim"], dep_df.obras_con_pim.sum())
            agregar("nacional: obras total = obras nacionales oficiales", nacional["obras_total"], oficial.obras_paralizadas)
            agregar("nacional: costo total = costo nacional oficial (S/)", nacional["costo_total_soles"], oficial.costo_actualizado_soles, 30)
            agregar("nacional: con + sin = total (costo, S/)", nacional["costo_con_pim_soles"] + nacional["costo_sin_pim_soles"], nacional["costo_total_soles"], 30)
    cols = ["anio", "fecha_corte", "anio_pim", "ambito", "categoria", "cod_departamento", "obras_con_pim", "pct_obras_con_pim", "obras_sin_pim",
            "pct_obras_sin_pim", "obras_total", "costo_con_pim_soles", "costo_sin_pim_soles", "costo_total_soles", "pim_soles",
            "fuente_pdf", "pagina", "tabla_n"]
    pim = pd.DataFrame(filas).reindex(columns=cols)
    return pim, pd.DataFrame(checks)


def validar_pim_con_excel(pim):
    """Contrasta obras con PIM (nacional y por departamento) con la columna TIENE PIM del anexo Excel de los mismos cortes."""
    filas = []
    for ruta in rutas_con_base():
        corte = leer_pdf.fechas_de_corte()[ruta.name]
        p = pim[(pim.fecha_corte == corte)]
        if p.empty:
            continue
        hoja, fila = tf.hoja_base(ruta)
        base = pd.read_excel(ruta, sheet_name=hoja, header=fila, dtype=str)
        base.columns = [tf.normalizar(c) for c in base.columns]
        col = next((c for c in base.columns if re.match(r"TIENE PIM", c)), None)
        if col is None:
            continue
        base = base[base["CODIGO INFOBRAS"].str.strip().str.fullmatch(r"\d+(\.0+)?", na=False)]
        con = base[col].map(lambda x: tf.normalizar(x) if pd.notna(x) else "") == "SI"
        base = base.assign(con_pim=con, dep=base[next(c for c in base.columns if c.startswith("DEPARTAMENTO"))].map(tf.departamento_canonico))
        nac = p[p.ambito == "nacional"]
        if len(nac):
            filas.append({"fecha_corte": corte, "ambito": "nacional", "categoria": "NACIONAL", "obras_con_pim_pdf": nac.obras_con_pim.iloc[0],
                          "obras_con_pim_excel": int(con.sum())})
        por_dep = base.groupby("dep").con_pim.sum()
        for _, f in p[p.ambito == "departamento"].iterrows():
            filas.append({"fecha_corte": corte, "ambito": "departamento", "categoria": f.categoria, "obras_con_pim_pdf": f.obras_con_pim,
                          "obras_con_pim_excel": int(por_dep.get(f.categoria, 0))})
    t = pd.DataFrame(filas)
    if len(t):
        t["diferencia"] = t.obras_con_pim_excel - t.obras_con_pim_pdf
    return t


# ---------------- B6: principales obras por departamento (anexos Excel) ----------------
PATRONES_PRINCIPALES = {
    "departamento": r"DEPARTAMENTO$", "codigo_infobras": r"CODIGO INFOBRAS", "cui": r"CUI$", "descripcion_obra": r"DESCRIPCION OBRA",
    "entidad": r"ENTIDAD$", "costo_actualizado_soles": r"COSTO ACTUALIZADO", "nivel_gobierno": r"NIVEL DE GOBIERNO",
    "causal_paralizacion": r"CAUSAL", "modalidad_ejecucion": r"MODALIDAD DE EJECUCION"}
COLUMNAS_PRINCIPALES = ["anio", "fecha_corte", "cod_departamento", "departamento", "orden_en_hoja", "cui", "codigo_infobras", "descripcion_obra",
                        "entidad", "nivel_gobierno", "modalidad_ejecucion", "causal_codigo", "causal_paralizacion", "costo_actualizado_soles",
                        "codigo_infobras_de_base", "es_lista_completa", "fuente_excel", "hoja"]
UMBRAL_LISTA_COMPLETA = 600   # una hoja de 'principales obras' con mas filas que esto es la lista completa (pasa en mar-2024)


def hoja_principales(ruta):
    """(hoja, fila del encabezado) de la lista de principales obras de un anexo, o None. La hoja cambia de nombre
    ('Principales obras por Dpto', 'Princ_Obras', 'ANEXO N°01', 'Anexo 01', ...); se reconoce por sus columnas CUI + DESCRIPCION OBRA + ENTIDAD."""
    import openpyxl
    libro = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
    for hoja in libro.worksheets:
        if not re.search(r"princ|anexo", hoja.title, re.I) or re.search(r"cuadros", hoja.title, re.I):
            continue
        for i, fila in enumerate(hoja.iter_rows(min_row=1, max_row=8, values_only=True)):
            columnas = [tf.normalizar(c) for c in fila if c is not None]
            if "CUI" in columnas and "DESCRIPCION OBRA" in columnas and "ENTIDAD" in columnas:
                return hoja.title, i
    return None


def leer_principales_obras(ruta):
    """Principales obras por departamento de un anexo Excel. None si el anexo no tiene esa lista."""
    ubicada = hoja_principales(ruta)
    if ubicada is None:
        return None
    hoja, fila = ubicada
    base = pd.read_excel(ruta, sheet_name=hoja, header=fila, dtype=str)
    base.columns = [tf.normalizar(c) for c in base.columns]
    salida = pd.DataFrame(index=base.index)
    for campo, patron in PATRONES_PRINCIPALES.items():
        c = next((c for c in base.columns if re.match(patron, c)), None)
        salida[campo] = base[c] if c else pd.NA
    salida["cui"] = _texto_codigo(salida.cui)
    salida = salida[salida.cui.str.fullmatch(r"\d+", na=False)].copy()   # sin filas vacias ni notas al pie
    salida["codigo_infobras"] = _texto_codigo(salida.codigo_infobras)
    salida["costo_actualizado_soles"] = pd.to_numeric(salida.costo_actualizado_soles, errors="coerce")
    for c in ("departamento", "entidad", "nivel_gobierno", "modalidad_ejecucion", "causal_paralizacion"):
        salida[c] = salida[c].map(lambda x: tf.normalizar(x) if pd.notna(x) else pd.NA)
    salida["departamento"] = salida.departamento.map(tf.departamento_canonico)
    salida["nivel_gobierno"] = salida.nivel_gobierno.str.replace(r"^G\. ", "G. ", regex=True)
    salida["modalidad_ejecucion"] = salida.modalidad_ejecucion.replace(ALIAS_MODALIDAD)
    partes = salida.causal_paralizacion.str.extract(r"^(\d+)\s+(.*)$")
    salida["causal_codigo"] = partes[0].astype("string")
    salida["causal_paralizacion"] = partes[1].where(partes[1].notna(), salida.causal_paralizacion)
    salida["descripcion_obra"] = salida.descripcion_obra.str.replace(r"\s+", " ", regex=True).str.strip()
    salida["cod_departamento"] = salida.departamento.map(config.DEPARTAMENTOS)
    corte = leer_pdf.fechas_de_corte()[ruta.name]
    salida["fecha_corte"], salida["anio"] = corte, int(corte[:4])
    salida["orden_en_hoja"] = range(1, len(salida) + 1)
    salida["es_lista_completa"] = len(salida) > UMBRAL_LISTA_COMPLETA
    salida["codigo_infobras_de_base"] = False
    salida["fuente_excel"], salida["hoja"] = ruta.name, hoja
    return salida.reindex(columns=COLUMNAS_PRINCIPALES).reset_index(drop=True)


def construir_principales_obras(obras=None):
    """Apila las listas de principales obras de todos los anexos que la traen (un anexo por corte).

    Los anexos de 2025 en adelante no traen CODIGO INFOBRAS (solo CUI): si se pasa `obras` (contraloria_obras_corte), el codigo se completa
    desde la base obra por obra del mismo corte y la fila queda marcada con codigo_infobras_de_base = True.
    """
    cortes = leer_pdf.fechas_de_corte()
    por_corte = {}
    for ruta in sorted(config.RAW.glob("contraloria/*/*.xlsx")):
        if "anexo_03" in ruta.name or ruta.name not in cortes:
            continue
        tabla = leer_principales_obras(ruta)
        if tabla is not None and len(tabla):
            por_corte.setdefault(cortes[ruta.name], tabla)
    principales = pd.concat([por_corte[c] for c in sorted(por_corte)], ignore_index=True)
    if obras is not None:
        base = obras.dropna(subset=["cui"]).drop_duplicates(["corte", "cui"]).set_index(["corte", "cui"]).codigo_infobras
        llave = pd.MultiIndex.from_arrays([principales.fecha_corte, principales.cui])
        encontrado = pd.Series(base.reindex(llave).to_numpy(), index=principales.index, dtype="string")
        falta = principales.codigo_infobras.isna() & encontrado.notna()
        principales.loc[falta, "codigo_infobras"] = encontrado[falta]
        principales["codigo_infobras_de_base"] = falta
    return principales


def validar_principales_obras(principales, obras):
    """Por corte: cuantas obras hay, cuantas CUI estan en la base obra por obra del mismo corte, y en cuantas coincide el costo (<= 0,5 %)."""
    filas = []
    for corte, g in principales.groupby("fecha_corte"):
        o = obras[obras.corte == corte].dropna(subset=["cui"]).drop_duplicates("cui").set_index("cui")
        if o.empty:
            filas.append({"fecha_corte": corte, "obras": len(g), "departamentos": g.departamento.nunique(), "cui_en_base": None,
                          "costo_coincide": None, "nota": "el corte no tiene base obra por obra para contrastar"})
            continue
        union = g.join(o[["costo_actualizado_soles"]].rename(columns={"costo_actualizado_soles": "costo_base"}), on="cui")
        en_base = union.costo_base.notna() | union.cui.isin(o.index)
        cercano = ((union.costo_actualizado_soles - union.costo_base).abs() <= 0.005 * union.costo_base.abs()) & en_base
        filas.append({"fecha_corte": corte, "obras": len(g), "departamentos": g.departamento.nunique(), "cui_en_base": int(en_base.sum()),
                      "costo_coincide": int(cercano.sum()), "nota": ""})
    return pd.DataFrame(filas)


# ---------------- B4: modalidad de ejecucion y evento INFOBRAS (tablas nivel de gobierno x categoria) ----------------
NIVELES = ("NACIONAL", "REGIONAL", "LOCAL")
ALIAS_CATEGORIA = {"CONTRATA": "POR CONTRATA", "NUCLEOS EJECUTORES": "POR NUCLEO EJECUTOR", "APP / CONCESIONES": "ASOCIACION PUBLICA PRIVADA APP",
                   "ASOCIACION PUBLICA PRIVADA - APP": "ASOCIACION PUBLICA PRIVADA APP", "ASOCIACIONES PUBLICO PRIVADAS": "ASOCIACION PUBLICA PRIVADA APP"}
CELDAS_POR_TABLA = {"modalidad_ejecucion": 10, "evento_infobras": 9}   # 6 de niveles (N y costo x 3) + cola (N, %, costo, %) o (N, costo, %)
COLUMNAS_NIVEL = ["anio", "fecha_corte", "tabla", "categoria", "nivel_gobierno", "obras_paralizadas", "costo_actualizado_soles", "pct_obras",
                  "pct_costo", "orden_niveles", "fuente_pdf", "pagina", "tabla_n"]


def _es_numero(celda):
    return bool(re.fullmatch(r"[-−]?\d[\d ,.]*%?|[-−]", str(celda).strip()))


def _categoria(etiqueta):
    """'Reconstruccion con Cambios8' -> 'RECONSTRUCCION CON CAMBIOS' (se quita la nota al pie pegada a la palabra)."""
    texto = tf.normalizar(re.sub(r"(?<=[A-Za-zÁÉÍÓÚáéíóúñÑ])\d+$", "", str(etiqueta).strip()))
    return ALIAS_CATEGORIA.get(texto, texto)


def _ensamblar_filas(filas_crudas, esperadas):
    """Filas crudas (con '' en las celdas vacias) -> [(etiqueta, [celdas])] con `esperadas` celdas cada una.

    Resuelve lo que el PDF hace con las tablas: la etiqueta puede venir despues de celdas vacias (['', 'Total', '', '356', ...]); una fila
    partida en dos lineas (una fila corta, con o sin etiqueta, que completa los huecos de la anterior); y filas con mas celdas vacias de las
    esperadas (columnas desplazadas), de las que se quitan las vacias.
    """
    filas = []
    for r in filas_crudas:
        i_etiqueta = next((i for i, c in enumerate(r) if c != "" and not _es_numero(c)), None)
        etiqueta = r[i_etiqueta] if i_etiqueta is not None else ""
        resto = list(r[i_etiqueta + 1:]) if i_etiqueta is not None else (list(r[1:]) if r and r[0] == "" else list(r))
        n_numeros = sum(1 for c in resto if c != "")
        previa = filas[-1] if filas else None
        if n_numeros <= 3 and previa is not None and (previa[0] == "" or "" in previa[1] or len(previa[1]) < esperadas):
            if etiqueta and previa[0] == "":
                previa[0] = etiqueta
            valores = [c for c in resto if c != ""]
            for k, c in enumerate(previa[1]):
                if c == "" and valores:
                    previa[1][k] = valores.pop(0)
        elif n_numeros > 3:
            filas.append([etiqueta, resto])
    salida = []
    for etiqueta, celdas in filas:
        if len(celdas) > esperadas:
            celdas = [c for c in celdas if c != ""]
        if len(celdas) == esperadas + 1:
            celdas = _unir_cifra_partida(celdas, esperadas)
        if len(celdas) == esperadas:
            salida.append((etiqueta, celdas))
    return salida


def _unir_cifra_partida(celdas, esperadas):
    """A veces pdfplumber parte una cifra en dos celdas ('31' y '0' por '310'; '1' y '0,646,852,188'). Si sobra una celda se prueba unir
    cada par contiguo y se deja la union con la que las obras de los tres niveles suman el total de la fila; si ninguna cuadra, no se toca."""
    for i in range(len(celdas) - 1):
        candidata = celdas[:i] + [celdas[i] + celdas[i + 1]] + celdas[i + 2:]
        v = [_num(c) for c in candidata]
        if None not in v and v[0] + v[2] + v[4] == v[6]:
            return candidata
    return celdas


def _registros_nivel(filas, esperadas):
    """[(etiqueta, celdas)] -> lista de dicts: categoria, pares (obras, costo) por columna de nivel, y el total (obras, %, costo, %)."""
    registros = []
    for etiqueta, celdas in filas:
        v = [_num(c) for c in celdas]
        pares = [(v[0], v[1]), (v[2], v[3]), (v[4], v[5])]
        pares = [(0 if p[0] is None and p[1] is None else p[0], p[1]) for p in pares]
        cola = v[6:]
        if esperadas == 10:
            total = {"obras": cola[0], "pct_obras": cola[1], "costo": cola[2], "pct_costo": cola[3]}
        else:
            total = {"obras": cola[0], "pct_obras": None, "costo": cola[1], "pct_costo": cola[2]}
        registros.append({"etiqueta": etiqueta, "pares": pares, "total": total})
    return registros


def _niveles_oficiales(distribuciones, corte):
    d = distribuciones[(distribuciones.fecha_corte == corte) & (distribuciones.dimension == "nivel_gobierno")]
    salida = {}
    for _, f in d.iterrows():
        for nivel in NIVELES:
            if str(f.categoria).endswith(nivel):
                salida[nivel] = int(f.obras_paralizadas)
    return salida


def _orden_de_niveles(fila_total, oficiales, referencia=None, filas_ref_buscar=()):
    """Orden real de las 3 columnas de nivel. El rotulo del PDF no es confiable (en el evento de jun-2026 el PDF dice Nacional-Regional-Local
    pero las cifras estan en Local-Nacional-Regional), asi que se decide por los totales: se busca la permutacion de (NACIONAL, REGIONAL, LOCAL)
    cuyas obras coinciden con las del cuadro 'nivel de gobierno' del mismo informe. Sin coincidencia exacta se prueba contra una tabla de referencia
    ya resuelta del mismo informe (pares obras/costo) y, si no, se deja el orden Nacional-Regional-Local marcado como no verificado.
    Devuelve (orden, verificado)."""
    import itertools
    ns = [p[0] for p in fila_total["pares"]]
    if len(oficiales) == 3:
        for perm in itertools.permutations(NIVELES):
            if all(ns[i] == oficiales[perm[i]] for i in range(3)):
                return list(perm), True
    if referencia:
        conteo = {}
        for nivel, par in referencia:
            conteo.setdefault(par, set()).add(nivel)
        pares_ref = {par: next(iter(n)) for par, n in conteo.items() if len(n) == 1}
        for fila in filas_ref_buscar:
            encontrados = [pares_ref.get(tuple(p)) for p in fila["pares"]]
            if None not in encontrados and len(set(encontrados)) == 3:
                return encontrados, True
    return list(NIVELES), False


def _validar_nivel(registros, orden, nacional_obras):
    """Comprobaciones de una tabla: suma de filas = fila Total por columna (obras y costo) y total de obras = nacional (si se conoce)."""
    es_t = [tf.es_total(r["etiqueta"]) for r in registros]
    if not any(es_t):
        return [("tiene fila Total", 0, 1, 0)]
    total = registros[es_t.index(True)]
    cuerpo = [r for r, t in zip(registros, es_t) if not t]
    checks = []
    for i, nivel in enumerate(orden):
        checks.append((f"suma obras {nivel} = Total", sum((r["pares"][i][0] or 0) for r in cuerpo), total["pares"][i][0] or 0, 0))
        checks.append((f"suma costo {nivel} = Total (S/)", sum((r["pares"][i][1] or 0) for r in cuerpo), total["pares"][i][1] or 0, 10 * len(cuerpo)))
    checks.append(("suma obras total de filas = Total", sum((r["total"]["obras"] or 0) for r in cuerpo), total["total"]["obras"] or 0, 0))
    checks.append(("obras por nivel suman el total de la fila (filas que fallan)",
                   sum(sum((p[0] or 0) for p in r["pares"]) != (r["total"]["obras"] or 0) for r in registros), 0, 0))
    if nacional_obras is not None:
        checks.append(("fila Total = obras nacionales oficiales", total["total"]["obras"], nacional_obras, 0))
    return checks


def construir_modalidad_evento(cifras, distribuciones):
    """Tablas de modalidad de ejecucion (dos por informe) y evento INFOBRAS, en formato largo (una fila por categoria y nivel de gobierno).

    Devuelve (modalidad, evento, validacion). Una tabla solo se guarda si todas sus comprobaciones pasan; las que fallan quedan en la
    validacion con el motivo (no se corrigen a mano).
    """
    modalidad, evento, validacion = [], [], []
    for informe in tf.informes_contraloria():
        corte, anio = informe["fecha_corte"], informe["anio"]
        oficiales = _niveles_oficiales(distribuciones, corte)
        nacional = cifras[(cifras.fecha_corte == corte) & (cifras.ambito == "nacional")].obras_paralizadas.iloc[0]
        referencia = None
        trabajos = [("modalidad_ejecucion", "por_modalidad"), ("modalidad_ejecucion", "administracion_directa_y_contrata"), ("evento_infobras", "por_evento")]
        for tipo, nombre in trabajos:
            tablas = tf.tablas_de(informe, tipo)
            if tipo == "modalidad_ejecucion":
                tablas = [t for t in tablas if (re.search(r"seg[uú]n", t[0].titulo, re.I) is not None) == (nombre != "por_modalidad")]
            if not tablas:
                validacion.append({"fecha_corte": corte, "tabla": nombre, "comprobacion": "tabla encontrada en el catalogo", "valor_a": 0, "valor_b": 1,
                                   "diferencia": -1, "tolerancia": 0, "ok": False})
                continue
            meta, _ = tablas[0]
            crudas, _sin = filas_posicionales(informe, meta)
            esperadas = CELDAS_POR_TABLA[tipo]
            registros = _registros_nivel(_ensamblar_filas(crudas, esperadas), esperadas)
            if not registros:
                validacion.append({"fecha_corte": corte, "tabla": nombre, "comprobacion": "filas legibles", "valor_a": 0, "valor_b": len(crudas),
                                   "diferencia": None, "tolerancia": 0, "ok": False})
                continue
            fila_total = next((r for r in registros if tf.es_total(r["etiqueta"])), None)
            if fila_total is None:
                validacion.append({"fecha_corte": corte, "tabla": nombre, "comprobacion": "tiene fila Total", "valor_a": 0, "valor_b": 1,
                                   "diferencia": -1, "tolerancia": 0, "ok": False})
                continue
            orden, verificado = _orden_de_niveles(fila_total, oficiales if nombre != "administracion_directa_y_contrata" else {}, referencia, registros)
            checks = _validar_nivel(registros, orden, nacional if nombre != "administracion_directa_y_contrata" else None)
            ok_tabla = all(abs(a - b) <= tol for _, a, b, tol in checks)
            for c, a, b, tol in checks:
                validacion.append({"fecha_corte": corte, "tabla": nombre, "comprobacion": c, "valor_a": a, "valor_b": b, "diferencia": a - b,
                                   "tolerancia": tol, "ok": abs(a - b) <= tol})
            validacion.append({"fecha_corte": corte, "tabla": nombre, "comprobacion": "orden de los niveles verificado con los totales del informe",
                               "valor_a": int(verificado), "valor_b": 1, "diferencia": int(verificado) - 1, "tolerancia": 0, "ok": verificado})
            if nombre == "por_modalidad" and ok_tabla and verificado:
                referencia = [(nivel, tuple(par)) for r in registros for nivel, par in zip(orden, r["pares"]) if par[0]]
            if not ok_tabla:
                continue
            base = {"anio": anio, "fecha_corte": corte, "tabla": nombre, "orden_niveles": "verificado" if verificado else "no verificado",
                    "fuente_pdf": informe["archivo"], "pagina": int(meta.pagina), "tabla_n": int(meta.tabla_n)}
            destino = evento if tipo == "evento_infobras" else modalidad
            for r in registros:
                if tf.es_total(r["etiqueta"]):
                    continue
                categoria = _categoria(r["etiqueta"])
                if destino is modalidad:
                    categoria = ALIAS_MODALIDAD.get(categoria, categoria)
                for nivel, (n, costo) in zip(orden, r["pares"]):
                    destino.append({**base, "categoria": categoria, "nivel_gobierno": nivel, "obras_paralizadas": n, "costo_actualizado_soles": costo})
                destino.append({**base, "categoria": categoria, "nivel_gobierno": "TOTAL", "obras_paralizadas": r["total"]["obras"],
                                "costo_actualizado_soles": r["total"]["costo"], "pct_obras": r["total"]["pct_obras"], "pct_costo": r["total"]["pct_costo"]})
    return (pd.DataFrame(modalidad).reindex(columns=COLUMNAS_NIVEL), pd.DataFrame(evento).reindex(columns=COLUMNAS_NIVEL), pd.DataFrame(validacion))


def validar_modalidad_evento_con_excel(modalidad, evento, obras):
    """Contrasta las obras por categoria y nivel de gobierno de cada tabla del PDF con la base obra por obra del mismo corte (13 cortes)."""
    filas = []
    for nombre, tabla, columna in (("por_modalidad", modalidad, "modalidad_ejecucion"), ("por_evento", evento, "evento")):
        for corte, g in tabla[tabla.tabla == nombre].groupby("fecha_corte"):
            o = obras[obras.corte == corte]
            if o.empty:
                continue
            o = o.assign(nivel=o.nivel_gobierno.str.replace("G. ", "", regex=False))
            excel = o.groupby([columna, "nivel"]).size()
            excel_total = o.groupby(columna).size()
            for _, f in g.iterrows():
                v = excel_total.get(f.categoria, 0) if f.nivel_gobierno == "TOTAL" else excel.get((f.categoria, f.nivel_gobierno), 0)
                filas.append({"fecha_corte": corte, "tabla": nombre, "categoria": f.categoria, "nivel_gobierno": f.nivel_gobierno,
                              "obras_pdf": f.obras_paralizadas, "obras_excel": int(v),
                              "diferencia": pd.NA if pd.isna(f.obras_paralizadas) else int(v) - int(f.obras_paralizadas)})
    t = pd.DataFrame(filas)
    return t


def validar_subset_modalidad(modalidad):
    """La tabla 'administracion directa y contrata' es un subconjunto de la de modalidades: sus obras por categoria y nivel deben ser iguales a las de la
    tabla principal del mismo informe. Si no lo son (p. ej. jun-2026, donde el PDF intercambia los rotulos Administracion directa y Por contrata),
    la fila queda marcada con coincide = False."""
    principal = modalidad[modalidad.tabla == "por_modalidad"].set_index(["fecha_corte", "categoria", "nivel_gobierno"]).obras_paralizadas
    sub = modalidad[modalidad.tabla == "administracion_directa_y_contrata"]
    t = sub[["fecha_corte", "categoria", "nivel_gobierno", "obras_paralizadas"]].rename(columns={"obras_paralizadas": "obras_subtabla"}).copy()
    t["obras_tabla_principal"] = [principal.get((a, b, c)) for a, b, c in zip(t.fecha_corte, t.categoria, t.nivel_gobierno)]
    t["coincide"] = t.obras_subtabla == t.obras_tabla_principal
    return t


# ---------------- B4: servicios de control (texto de las paginas del PDF) ----------------
COLUMNAS_SC_TOTAL = ["anio", "fecha_corte", "tipo_servicio", "cantidad", "pct", "fuente_pdf", "pagina", "tabla_n"]
COLUMNAS_SC_NIVEL = ["anio", "fecha_corte", "tipo_servicio", "nivel_gobierno", "cantidad", "pct", "orden_niveles", "fuente_pdf", "pagina", "tabla_n"]
COLUMNAS_SC_DEP = ["anio", "fecha_corte", "cod_departamento", "departamento", "control_previo", "control_simultaneo", "control_posterior",
                   "total_servicios", "fuente_pdf", "pagina", "tabla_n"]
TIPOS_SC = ("PREVIO", "SIMULTANEO", "POSTERIOR")


def _texto_paginas(informe, meta):
    return "\n".join((informe["carpeta"] / f"texto_p{p:02d}.txt").read_text(encoding="utf-8")
                      for p in range(int(meta.pagina), int(meta.pagina_fin) + 1))


def _filas_de_texto(texto, titulo, minimo_cifras):
    """Lineas 'Control previo 24 0.6% 31 0.8% ...' entre el titulo y la fila Total: [(tipo o 'TOTAL', [cifras])]."""
    lineas = texto.splitlines()
    inicio = next((i for i, l in enumerate(lineas) if re.search(titulo, l, re.I)), None)
    if inicio is None:
        return []
    filas = []
    for linea in lineas[inicio + 1:]:
        m = re.match(r"^(Control (?:previo|simult[aá]neo|posterior)|Total(?: general)?)\s+([-−\d].*)$", linea.strip(), re.I)
        if not m:
            continue
        crudos = m.group(2).split()
        tokens = [t.rstrip("%").replace("−", "-") for t in crudos]
        if minimo_cifras == 6 and len(tokens) == 7:   # falta una cifra: la cantidad vacia de un nivel (se lee como 0)
            if tokens.count("-") == 1:                 # un solo guion vale por la pareja (cantidad, %) vacia
                i = tokens.index("-")
                tokens[i:i + 1] = ["-", "-"]
            elif crudos[0].endswith("%"):              # '0.0% 18 1.3% ...': la fila empieza por el % de la cantidad vacia
                tokens.insert(0, "0")
        cifras = [tf.a_numero(t) for t in tokens]
        if None in cifras or len(cifras) < minimo_cifras:
            continue
        etiqueta = "TOTAL" if m.group(1).lower().startswith("total") else tf.normalizar(m.group(1)).replace("CONTROL ", "")
        filas.append((etiqueta, cifras))
        if etiqueta == "TOTAL":
            break
    return filas


def _orden_niveles_texto(informe, meta, totales):
    """Orden de las columnas de nivel de la tabla 'servicios de control por nivel de gobierno': el del encabezado, contrastado con la narracion
    del informe (p. ej. '1 977 (49.3%) corresponden al nivel de Gobierno Local'). Devuelve (orden, verificado)."""
    pagina = (informe["carpeta"] / f"texto_p{int(meta.pagina):02d}.txt").read_text(encoding="utf-8")
    encabezado = " ".join(pagina.split("Tabla 1")[1].split("Control Previo")[0].split("Control previo")[0].split()) if re.search(r"Tabla 1", pagina) else pagina
    orden = [tf.normalizar(m.group(1)) for m in re.finditer(r"(Nacional|Regional|Local)", encabezado)]
    orden = [o for i, o in enumerate(orden) if o not in orden[:i]]
    if sorted(orden) != sorted(NIVELES):
        orden = list(NIVELES)
    texto = re.sub(r"\s+", " ", (informe["carpeta"] / "texto_completo.txt").read_text(encoding="utf-8"))
    por_narracion = {}
    for n in totales:
        miles = f"{n // 1000}\\s?{n % 1000:03d}" if n >= 1000 else str(n)
        m = re.search(rf"(?<![\d.,]){miles}\b.{{0,70}}?Gobierno (Local|Regional|Nacional)", texto, re.I)
        if m:
            por_narracion[n] = tf.normalizar(m.group(1))
    por_encabezado = dict(zip(totales, orden))
    if len(por_narracion) == 3 and len(set(por_narracion.values())) == 3:
        return [por_narracion[n] for n in totales], por_narracion == por_encabezado
    return [por_encabezado[n] for n in totales], False


def _filas_departamento_pdf(informe, meta, paginas_extra=2):
    """Filas por departamento de la tabla de servicios de control, leidas del PDF con las celdas vacias en su sitio.

    Hay informes cuya tabla sigue en la pagina siguiente sin que el catalogo la una (mar-2023, set-2024, mar-2026 traian solo 6, 18 y 19 filas),
    asi que se recorren la pagina de la tabla y las dos siguientes y se toma la primera fila de cada departamento."""
    import pdfplumber
    import tablas_pdf
    ruta = config.RAW / "contraloria" / str(informe["anio"]) / informe["archivo"]
    vistas, filas = set(), []
    with pdfplumber.open(ruta) as pdf:
        for n in range(int(meta.pagina), min(int(meta.pagina) + paginas_extra, len(pdf.pages)) + 1):
            for t in pdf.pages[n - 1].find_tables():
                for fila in t.extract():
                    celdas = [tablas_pdf.convertir_celda(c) for c in fila if c is not None]
                    dep = tf.departamento_canonico(celdas[0]) if celdas else None
                    if dep and dep not in vistas and len(celdas) >= 4:
                        vistas.add(dep)
                        filas.append(celdas)
                    elif celdas and tf.es_total(celdas[0]) and "TOTAL" not in vistas and len(vistas) >= 20:
                        vistas.add("TOTAL")
                        filas.append(celdas)
    return filas


def construir_servicios_control(cifras):
    """Tres tablas: total, por nivel de gobierno y por departamento. Devuelve (total, nivel, departamento, validacion).

    total y nivel salen del texto de la pagina (formato estable en los 21 informes); departamento sale de la tabla del PDF con sus celdas vacias
    en su sitio (una celda vacia = ese tipo de servicio no se hizo en el departamento). 2022 solo trae total y nivel (por departamento es un mapa).
    """
    total, nivel, dep, validacion = [], [], [], []
    check = lambda corte, nombre, a, b, tol=0: validacion.append({"fecha_corte": corte, "tabla": nombre[0], "comprobacion": nombre[1], "valor_a": a, "valor_b": b,
                                                                  "diferencia": a - b, "tolerancia": tol, "ok": abs(a - b) <= tol})
    for informe in tf.informes_contraloria():
        corte, anio = informe["fecha_corte"], informe["anio"]
        base = {"anio": anio, "fecha_corte": corte}
        for meta, _ in tf.tablas_de(informe, "servicios_control_total")[:1]:
            prov = {"fuente_pdf": informe["archivo"], "pagina": int(meta.pagina), "tabla_n": int(meta.tabla_n)}
            filas = _filas_de_texto(_texto_paginas(informe, meta), r"total de servicios de control efectuados", 2)
            cuerpo = [(t, c) for t, c in filas if t != "TOTAL"]
            if cuerpo and any(t == "TOTAL" for t, _ in filas):
                check(corte, ("total", "suma de los tres tipos = Total"), sum(c[0] for _, c in cuerpo), next(c[0] for t, c in filas if t == "TOTAL"))
                for t, c in cuerpo:
                    total.append({**base, "tipo_servicio": t, "cantidad": c[0], "pct": c[1], **prov})
        for meta, _ in tf.tablas_de(informe, "servicios_control_nivel_gobierno")[:1]:
            prov = {"fuente_pdf": informe["archivo"], "pagina": int(meta.pagina), "tabla_n": int(meta.tabla_n)}
            filas = _filas_de_texto(_texto_paginas(informe, meta), r"servicios de control por nivel de gobierno", 6)
            cuerpo = [(t, c) for t, c in filas if t != "TOTAL"]
            fila_total = next((c for t, c in filas if t == "TOTAL"), None)
            if not cuerpo or fila_total is None or len(fila_total) < 8:
                check(corte, ("nivel", "tabla legible"), 0, 1)
                continue
            totales_nivel = [int(fila_total[0]), int(fila_total[2]), int(fila_total[4])]
            orden, verificado = _orden_niveles_texto(informe, meta, totales_nivel)
            for t, c in cuerpo:
                check(corte, ("nivel", f"{t}: niveles suman el total de la fila"), c[0] + c[2] + c[4], c[6])
                for i, n in enumerate(orden):
                    nivel.append({**base, "tipo_servicio": t, "nivel_gobierno": n, "cantidad": c[2 * i], "pct": c[2 * i + 1],
                                  "orden_niveles": "verificado con la narracion" if verificado else "segun el encabezado del PDF", **prov})
                nivel.append({**base, "tipo_servicio": t, "nivel_gobierno": "TOTAL", "cantidad": c[6], "pct": c[7],
                              "orden_niveles": "verificado con la narracion" if verificado else "segun el encabezado del PDF", **prov})
            for i, n in enumerate(orden):
                check(corte, ("nivel", f"suma de tipos {n} = Total"), sum(c[2 * i] for _, c in cuerpo), fila_total[2 * i])
            ref = [r for r in total if r["fecha_corte"] == corte]
            if ref:
                check(corte, ("nivel", "total de servicios = tabla total"), fila_total[6], sum(r["cantidad"] for r in ref))
        tablas_dep = tf.tablas_de(informe, "servicios_control_departamento")
        if tablas_dep:   # la tabla puede venir partida en varias tablas del catalogo (una por pagina)
            meta = tablas_dep[0][0]
            prov = {"fuente_pdf": informe["archivo"], "pagina": int(meta.pagina), "tabla_n": int(meta.tabla_n)}
            crudas = _filas_departamento_pdf(informe, meta)
            filas = []
            for r in crudas:
                if not r or not r[0] or _es_numero(r[0]):
                    continue
                celdas = list(r[1:])
                if len(celdas) != 4:
                    celdas = (celdas + [""] * 4)[:4] if len(celdas) < 4 else [c for c in celdas if c != ""][:4] if len([c for c in celdas if c != ""]) <= 4 else celdas[:4]
                filas.append((r[0], [(_num(c) or 0) for c in celdas]))
            registros = [(f, v) for f, v in filas if tf.departamento_canonico(f)]
            total_fila = next((v for f, v in filas if tf.es_total(f)), None)
            for f, v in registros:
                d = tf.departamento_canonico(f)
                dep.append({**base, "cod_departamento": config.DEPARTAMENTOS[d], "departamento": d, "control_previo": v[0], "control_simultaneo": v[1],
                            "control_posterior": v[2], "total_servicios": v[3], **prov})
            check(corte, ("departamento", "departamentos presentes (25)"), len(registros), 25)
            check(corte, ("departamento", "previo + simultaneo + posterior = total (filas que fallan)"), sum(sum(v[:3]) != v[3] for _, v in registros), 0)
            if total_fila:
                for j, nombre in enumerate(("previo", "simultaneo", "posterior", "total")):
                    check(corte, ("departamento", f"suma {nombre} departamentos = Total"), sum(v[j] for _, v in registros), total_fila[j])
            ref = [r for r in total if r["fecha_corte"] == corte]
            if ref:
                check(corte, ("departamento", "total de servicios = tabla total"), sum(v[3] for _, v in registros), sum(r["cantidad"] for r in ref))
    return (pd.DataFrame(total).reindex(columns=COLUMNAS_SC_TOTAL), pd.DataFrame(nivel).reindex(columns=COLUMNAS_SC_NIVEL),
            pd.DataFrame(dep).reindex(columns=COLUMNAS_SC_DEP), pd.DataFrame(validacion))


# ---------------- ejecucion de todas las tareas de los anexos Excel ----------------
VALIDACION = config.DOCS / "validacion_anexos_excel.csv"
VALIDACION_B4_EXCEL = config.DOCS / "validacion_b4_excel.csv"
ENTEROS_CIFRAS = ["obras_paralizadas", "obras_g_nacional", "obras_g_regional", "obras_g_local", "pagina", "tabla_n", "obras_educacion_corregida"]


def _checks(tarea, tabla):
    """Pasa una tabla de comprobaciones al formato comun de docs/validacion_anexos_excel.csv."""
    if tabla is None or tabla.empty:
        return pd.DataFrame()
    t = tabla.copy()
    t.insert(0, "tarea", tarea)
    for c in ("fecha_corte", "tabla"):
        if c not in t:
            t[c] = ""
    for c in ("valor_a", "valor_b", "diferencia", "tolerancia"):
        if c not in t:
            t[c] = pd.NA
    return t[["tarea", "fecha_corte", "tabla", "comprobacion", "valor_a", "valor_b", "diferencia", "tolerancia", "ok"]]


def ejecutar_anexos_excel():
    """Corre A2, A3, B7, B3, B4 y B6 y guarda sus tablas en data/processed y sus comprobaciones en docs/validacion_anexos_excel.csv.

    Se ejecuta en la Fase 3 del proyecto PDF (despues de tablas_finales.ejecutar_fase3, de la que depende, y antes del diccionario).
    B5 (rangos de 2022) vive en tablas_finales.leer_informe, porque agrega filas a contraloria_distribuciones.csv.
    """
    guardar = tf.guardar_csv
    P = config.PROCESSED
    cifras = tf.a_enteros(pd.read_csv(config.CIFRAS_OFICIALES, encoding="utf-8-sig", dtype={"cod_departamento": str}), ENTEROS_CIFRAS[:-1])
    distribuciones = pd.read_csv(config.DISTRIBUCIONES_CGR, encoding="utf-8-sig")
    checks = []

    # A2 y A3
    obras = construir_obras_corte()
    guardar(obras, OBRAS_CORTE)
    v = validar_obras_corte(obras)
    checks.append(_checks("A2", pd.DataFrame({
        "fecha_corte": v.fecha_corte, "tabla": "contraloria_obras_corte", "comprobacion": "obras y costo = leer_base_excel; educacion igual",
        "valor_a": v.obras_nuevo, "valor_b": v.obras_leer_base_excel, "diferencia": v.obras_nuevo - v.obras_leer_base_excel, "tolerancia": 0, "ok": v.ok})))
    panel = construir_panel(obras)
    guardar(panel, OBRAS_PANEL)
    vp = validar_panel(panel, obras)
    checks.append(_checks("A3", pd.DataFrame({"fecha_corte": "", "tabla": "contraloria_obras_panel", "comprobacion": vp.comprobacion, "valor_a": vp.panel,
                                              "valor_b": vp.obras_corte, "diferencia": vp.panel - vp.obras_corte, "tolerancia": 0, "ok": vp.ok})))

    # B7: la columna nueva va en cifras_oficiales_contraloria.csv (se vuelve a escribir en cada corrida de la Fase 3)
    educacion = pd.read_csv(P / "contraloria_educacion_departamento.csv", encoding="utf-8-sig")
    cifras = corregir_educacion(cifras, educacion)
    guardar(tf.a_enteros(cifras, ENTEROS_CIFRAS), config.CIFRAS_OFICIALES)
    ve = validar_educacion(cifras)
    checks.append(_checks("B7", pd.DataFrame({"fecha_corte": ve.fecha_corte, "tabla": "cifras_oficiales_contraloria",
                                              "comprobacion": "Educacion corregida = Excel (y = PDF salvo set-2023 y set-2025)",
                                              "valor_a": ve.obras_educacion_corregida, "valor_b": ve.obras_paralizadas,
                                              "diferencia": ve.obras_educacion_corregida - ve.obras_paralizadas, "tolerancia": 0,
                                              "ok": ~ve.difiere_del_pdf | ve.fecha_corte.isin(["2023-09-30", "2025-09-30"])})))

    # B3
    pim, vpim = construir_pim(cifras)
    guardar(tf.a_enteros(pim, ["obras_con_pim", "obras_sin_pim", "obras_total", "pagina", "tabla_n"]), P / "contraloria_pim.csv")
    checks.append(_checks("B3", vpim.assign(tabla="contraloria_pim")))
    xp = validar_pim_con_excel(pim)
    if len(xp):
        checks.append(_checks("B3", xp.groupby("fecha_corte").agg(valor_a=("obras_con_pim_pdf", "sum"), valor_b=("obras_con_pim_excel", "sum"),
                                                                  diferencia=("diferencia", lambda s: s.abs().sum())).reset_index().assign(
            tabla="contraloria_pim", comprobacion="obras con PIM del PDF = columna TIENE PIM del Excel (nacional + 25 departamentos)", tolerancia=0).assign(
            ok=lambda d: d.diferencia == 0)))

    # B4
    modalidad, evento, vmod = construir_modalidad_evento(cifras, distribuciones)
    for nombre, tabla in (("contraloria_modalidad_ejecucion", modalidad), ("contraloria_evento_infobras", evento)):
        guardar(tf.a_enteros(tabla, ["obras_paralizadas", "pagina", "tabla_n"]), P / f"{nombre}.csv")
    checks.append(_checks("B4", vmod.assign(tabla=vmod.tabla.map(lambda x: f"contraloria_{'evento_infobras' if x == 'por_evento' else 'modalidad_ejecucion'} / {x}"))))
    vx = pd.concat([validar_modalidad_evento_con_excel(modalidad, evento, obras)], ignore_index=True)
    guardar(vx, VALIDACION_B4_EXCEL)
    if len(vx):
        resumen = vx.groupby(["fecha_corte", "tabla"]).agg(valor_a=("obras_pdf", "sum"), valor_b=("obras_excel", "sum"),
                                                          diferencia=("diferencia", lambda s: s.abs().sum())).reset_index()
        checks.append(_checks("B4", resumen.assign(comprobacion="obras por categoria y nivel: PDF vs base Excel (detalle en validacion_b4_excel.csv)",
                                                   tolerancia=0).assign(ok=lambda d: d.diferencia == 0)))
    sub = validar_subset_modalidad(modalidad)
    if len(sub):
        g = sub.groupby("fecha_corte").coincide.all().reset_index()
        checks.append(_checks("B4", pd.DataFrame({"fecha_corte": g.fecha_corte, "tabla": "contraloria_modalidad_ejecucion / administracion_directa_y_contrata",
                                                  "comprobacion": "obras de la subtabla = obras de la tabla por modalidad (mismos rotulos)",
                                                  "valor_a": g.coincide.astype(int), "valor_b": 1, "diferencia": g.coincide.astype(int) - 1,
                                                  "tolerancia": 0, "ok": g.coincide})))
    sc_total, sc_nivel, sc_dep, vsc = construir_servicios_control(cifras)
    guardar(tf.a_enteros(sc_total, ["cantidad", "pagina", "tabla_n"]), P / "contraloria_servicios_control_total.csv")
    guardar(tf.a_enteros(sc_nivel, ["cantidad", "pagina", "tabla_n"]), P / "contraloria_servicios_control_nivel_gobierno.csv")
    guardar(tf.a_enteros(sc_dep, ["control_previo", "control_simultaneo", "control_posterior", "total_servicios", "pagina", "tabla_n"]),
            P / "contraloria_servicios_control_departamento.csv")
    checks.append(_checks("B4", vsc.assign(tabla=vsc.tabla.map(lambda x: f"contraloria_servicios_control / {x}"))))

    # B6
    principales = construir_principales_obras(obras)
    guardar(principales, P / "contraloria_principales_obras.csv")
    vpr = validar_principales_obras(principales, obras)
    checks.append(_checks("B6", pd.DataFrame({"fecha_corte": vpr.fecha_corte, "tabla": "contraloria_principales_obras",
                                              "comprobacion": "CUI de la lista presentes en la base obra por obra del mismo corte",
                                              "valor_a": vpr.cui_en_base, "valor_b": vpr.obras, "diferencia": vpr.cui_en_base - vpr.obras, "tolerancia": 0,
                                              "ok": vpr.cui_en_base.fillna(vpr.obras) == vpr.obras})))

    todo = pd.concat([c for c in checks if len(c)], ignore_index=True)
    guardar(todo, VALIDACION)
    log(f"Anexos Excel (A2, A3, B7, B3, B4, B6): obras_corte {len(obras)} filas, panel {len(panel)} obras, pim {len(pim)}, modalidad {len(modalidad)}, "
        f"evento {len(evento)}, servicios de control {len(sc_total)}/{len(sc_nivel)}/{len(sc_dep)}, principales obras {len(principales)}; "
        f"comprobaciones {len(todo)} (no pasan: {int((~todo.ok.astype(bool)).sum())}; detalle en docs/validacion_anexos_excel.csv)")
    return todo

### `contraloria_anexo03` — Contraloría: anexo 03 (servicios de control)
Anexo 03 de la Contraloria: servicios de control (control simultaneo, previo y posterior) que se hicieron a cada obra paralizada.

In [ ]:
%%modulo contraloria_anexo03
"""Anexo 03 de la Contraloria: servicios de control (control simultaneo, previo y posterior) que se hicieron a cada obra paralizada.

El anexo trae una fila por servicio de control y obra. Se usa el anexo mas reciente (marzo de 2026; el de junio de 2026 no publica anexo 03)
y se cuenta por obra. Salida: data/processed/contraloria_servicios_control_obra.csv (una fila por codigo de INFOBRAS).
No se guardan titulos ni direcciones de los informes: solo conteos y el ultimo anio de emision.
"""
import re

import pandas as pd

import config
import utils

MESES = {m: i for i, m in enumerate("enero febrero marzo abril mayo junio julio agosto setiembre octubre noviembre diciembre".split(), 1)}

DICCIONARIO = [
    ("codigo_infobras", "texto", "Código de la obra en INFOBRAS.", "código"),
    ("corte_anexo03", "fecha AAAA-MM-DD", "Corte del anexo 03 de donde salen los conteos (el más reciente publicado).", "fecha"),
    ("n_servicios_control", "entero", "Servicios de control que la Contraloría registra para la obra (todos los tipos).", "servicios"),
    ("n_control_simultaneo", "entero", "Servicios de control simultáneo (concurrente, visitas, orientación de oficio).", "servicios"),
    ("n_control_previo", "entero", "Servicios de control previo.", "servicios"),
    ("n_control_posterior", "entero", "Servicios de control posterior (auditorías).", "servicios"),
    ("n_servicios_concluidos", "entero", "Servicios de control con estado «Concluida».", "servicios"),
    ("ultimo_anio_informe_control", "entero", "Año de emisión del informe más reciente.", "año"),
]


def _archivo_mas_reciente():
    candidatos = []
    for ruta in config.RAW.glob("contraloria/*/*anexo_03*.xlsx"):
        m = re.search(r"obras_paralizadas_([a-z]+)_(\d{4})_", ruta.name)
        if m and m.group(1) in MESES:
            candidatos.append(((int(m.group(2)), MESES[m.group(1)]), ruta))
    return max(candidatos) if candidatos else (None, None)


def construir():
    """Cuenta los servicios de control por obra y guarda contraloria_servicios_control_obra.csv; devuelve la tabla."""
    clave, ruta = _archivo_mas_reciente()
    if ruta is None:
        utils.log("ANEXO 03: no hay archivos de anexo 03 en data/raw/contraloria; se omite")
        return None
    anio, mes = clave
    t = pd.read_excel(ruta, dtype=str).rename(columns=lambda c: re.sub(r"\s+", "_", str(c).strip().upper().replace("Ñ", "N")))
    columna = lambda *opciones: next(c for c in opciones if c in t.columns)
    codigo, tipo = columna("CODIGO_INFOBRAS"), columna("TIPO_CONTROL")
    estado, emision = columna("ESTADO_SERVICIO_DE_CONTROL"), columna("ANO_EMISION_DE_INFORME")
    t = t[t[codigo].fillna("").str.fullmatch(r"\d+")].copy()
    tipo_norm = t[tipo].fillna("").str.lower()
    t["simultaneo"] = tipo_norm.str.contains("simult")
    t["previo"] = tipo_norm.str.contains("previo")
    t["posterior"] = tipo_norm.str.contains("posterior")
    t["concluido"] = t[estado].fillna("").str.lower().str.startswith("concluid")
    t["anio_informe"] = pd.to_numeric(t[emision], errors="coerce")
    g = t.groupby(codigo)
    salida = pd.DataFrame({"corte_anexo03": pd.Timestamp(anio, mes, 1).to_period("M").to_timestamp("M").strftime("%Y-%m-%d"),
                           "n_servicios_control": g.size(), "n_control_simultaneo": g.simultaneo.sum(), "n_control_previo": g.previo.sum(),
                           "n_control_posterior": g.posterior.sum(), "n_servicios_concluidos": g.concluido.sum(),
                           "ultimo_anio_informe_control": g.anio_informe.max().astype("Int64")}).reset_index().rename(columns={codigo: "codigo_infobras"})
    for c in ("n_control_simultaneo", "n_control_previo", "n_control_posterior", "n_servicios_concluidos"):
        salida[c] = salida[c].astype(int)
    salida.to_csv(config.PROCESSED / "contraloria_servicios_control_obra.csv", index=False, encoding="utf-8-sig")
    utils.log(f"ANEXO 03 ({anio}-{mes:02d}): {len(t)} servicios de control en {len(salida)} obras; contraloria_servicios_control_obra.csv")
    return salida

### `ubigeo_obras` — Ubigeo de cada obra
A1: ubigeo de distrito (INEI, 6 digitos, como texto) de cada obra de fichas_infobras.csv.

In [ ]:
%%modulo ubigeo_obras
"""A1: ubigeo de distrito (INEI, 6 digitos, como texto) de cada obra de fichas_infobras.csv.

Como se asigna, en este orden:
  1. exacto: departamento, provincia y distrito iguales a los de escale_infraestructura.csv.
  2. normalizado: iguales despues de quitar tildes y enies, guiones bajos y el nombre alterno entre parentesis
     (ESCALE escribe 'LOS BAÑOS DEL INCA', 'ANCO_HUALLO', 'QUISQUI (KICHKI)'; INFOBRAS 'LOS BANOS DEL INCA', 'ANCO HUALLO', 'QUISQUI')
     y de llevar la provincia del Callao a 'CALLAO'.
  3. manual: obras cuya ubicacion en INFOBRAS es incoherente (departamento o distrito equivocado). Se revisaron a mano con el
     nombre de la obra, la entidad y las coordenadas; cada una con su motivo en CORRECCIONES.
Las obras que no se resuelven quedan con ubigeo vacio. Todas las que no coinciden exacto se listan en docs/nombres_sin_ubigeo.csv
con el metodo usado, para revisarlas.

Ademas corrige la ubicacion: el departamento pasa a ser el del ubigeo (hay 2 obras que INFOBRAS publica en Huancavelica y
son de Ica) y lo que publica INFOBRAS queda en departamento_infobras. Provincia y distrito no se tocan: para el nombre oficial
de la ubicacion usar el ubigeo (por ejemplo con escale_infraestructura).
"""
import re
import unicodedata

import pandas as pd

import config
import utils

SALIDA_REVISION = config.DOCS / "nombres_sin_ubigeo.csv"

# provincia como la escribe INFOBRAS -> como la escribe ESCALE (ya normalizadas)
PROVINCIAS_EQUIVALENTES = {"PROV. CONST. DEL CALLAO": "CALLAO", "P. C. DEL CALLAO": "CALLAO"}

# codigo_infobras -> (ubigeo, motivo). Revisado a mano el 2026-11-08 con las fichas leidas el 2026-10-27.
CORRECCIONES = {
    "543835": ("110401", "INFOBRAS dice HUANCAVELICA-PALPA-PALPA, pero Palpa es provincia de Ica: el nombre de la obra dice "
                         "'DISTRITO DE PALPA ... DEPARTAMENTO DE ICA', la entidad es la Municipalidad Provincial de Palpa y las "
                         "coordenadas caen en Ica"),
    "135083": ("110111", "INFOBRAS dice HUANCAVELICA-HUANCAVELICA-RIO SANTIAGO (Rio Santiago es de Amazonas), pero la obra es "
                         "'DISTRITO DE SANTIAGO - ICA - ICA', la entidad es la Municipalidad Distrital de Santiago-Ica y las "
                         "coordenadas caen en Ica"),
    "510315": ("050401", "INFOBRAS dice distrito CHAVIN DE HUANTAR (es de Ancash); el nombre de la obra dice 'DISTRITO DE HUANTA, "
                         "PROVINCIA HUANTA' y las coordenadas caen en Huanta"),
    "156196": ("050401", "INFOBRAS dice distrito CHAVIN DE HUANTAR (es de Ancash); la obra abarca los distritos de Huanta e Iguain "
                         "(provincia de Huanta). Se asigna Huanta: la mayoria de las I.E. y las coordenadas estan en Huanta"),
}


def normalizar(texto):
    """Mayusculas, sin tildes ni enies, sin guion bajo, sin el nombre alterno entre parentesis y sin espacios dobles."""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return None
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    t = re.sub(r"\(.*?\)", " ", t.replace("_", " "))
    return re.sub(r"\s+", " ", t).strip()


def maestro_distritos():
    """Una fila por ubigeo con sus nombres en ESCALE (el anio mas reciente) y los nombres normalizados."""
    escale = pd.read_csv(config.PROCESSED / "escale_infraestructura.csv", dtype={"ubigeo": str}, encoding="utf-8")
    maestro = (escale.sort_values("anio").drop_duplicates("ubigeo", keep="last")[["ubigeo", "departamento", "provincia", "distrito"]])
    for col in ("departamento", "provincia", "distrito"):
        maestro[f"{col}_n"] = maestro[col].map(normalizar)
    repetidos = maestro.duplicated(["departamento_n", "provincia_n", "distrito_n"], keep=False)
    if repetidos.any():   # dos ubigeos con el mismo nombre normalizado: no se podria decidir cual
        raise ValueError(f"nombres normalizados repetidos en ESCALE: {maestro[repetidos].ubigeo.tolist()}")
    return maestro


def asignar_ubigeo(fichas, maestro=None):
    """Devuelve una copia de fichas con ubigeo (texto de 6 digitos) y ubigeo_metodo (exacto, normalizado, manual o sin_ubigeo)."""
    maestro = maestro_distritos() if maestro is None else maestro
    fichas = fichas.drop(columns=[c for c in ("ubigeo", "ubigeo_metodo") if c in fichas], errors="ignore").copy()
    if "departamento_infobras" in fichas:   # volver a lo que publica INFOBRAS antes de emparejar (asi repetir la corrida da lo mismo)
        fichas["departamento"] = fichas["departamento_infobras"]
        fichas = fichas.drop(columns="departamento_infobras")
    fichas["codigo_infobras"] = fichas["codigo_infobras"].astype(str)
    claves = ["departamento", "provincia", "distrito"]

    exacto = fichas[["codigo_infobras"] + claves].merge(maestro[["ubigeo"] + claves], on=claves, how="left")
    ubigeo = dict(zip(exacto.codigo_infobras, exacto.ubigeo))
    metodo = {c: "exacto" for c, u in ubigeo.items() if isinstance(u, str)}

    norm = fichas[["codigo_infobras"] + claves].copy()
    for col in claves:
        norm[f"{col}_n"] = norm[col].map(normalizar)
    norm["provincia_n"] = norm.provincia_n.replace(PROVINCIAS_EQUIVALENTES)
    norm = norm.merge(maestro[["ubigeo", "departamento_n", "provincia_n", "distrito_n"]],
                      on=["departamento_n", "provincia_n", "distrito_n"], how="left")
    for c, u in zip(norm.codigo_infobras, norm.ubigeo):
        if c not in metodo and isinstance(u, str):
            ubigeo[c], metodo[c] = u, "normalizado"

    validos = set(maestro.ubigeo)
    for c, (u, _) in CORRECCIONES.items():
        if u not in validos:
            raise ValueError(f"correccion manual de {c}: el ubigeo {u} no existe en ESCALE")
        if c in ubigeo and c not in metodo:
            ubigeo[c], metodo[c] = u, "manual"

    fichas["ubigeo"] = fichas.codigo_infobras.map(lambda c: ubigeo.get(c) if c in metodo else None)
    fichas["ubigeo_metodo"] = fichas.codigo_infobras.map(lambda c: metodo.get(c, "sin_ubigeo"))
    # columnas nuevas justo despues de distrito
    orden = list(fichas.columns.drop(["ubigeo", "ubigeo_metodo"]))
    pos = orden.index("distrito") + 1 if "distrito" in orden else len(orden)
    return corregir_departamento(fichas[orden[:pos] + ["ubigeo", "ubigeo_metodo"] + orden[pos:]])


def corregir_departamento(fichas):
    """departamento y cod_departamento = los del ubigeo; el departamento que publica INFOBRAS queda en departamento_infobras."""
    por_codigo = {codigo: nombre for nombre, codigo in config.DEPARTAMENTOS.items()}
    fichas = fichas.copy()
    fichas["departamento_infobras"] = fichas["departamento"]
    del_ubigeo = fichas["ubigeo"].str[:2].map(por_codigo)
    fichas["departamento"] = del_ubigeo.fillna(fichas["departamento_infobras"])
    fichas["cod_departamento"] = fichas["departamento"].map(config.DEPARTAMENTOS)
    # departamento_infobras va justo despues de ubigeo_metodo
    orden = [c for c in fichas.columns if c != "departamento_infobras"]
    pos = orden.index("ubigeo_metodo") + 1
    return fichas[orden[:pos] + ["departamento_infobras"] + orden[pos:]]


def tabla_revision(fichas, maestro):
    """Obras que no coincidieron exacto: como se resolvieron y con que nombres de ESCALE."""
    revisar = fichas[fichas.ubigeo_metodo != "exacto"][["codigo_infobras", "departamento_infobras", "provincia", "distrito", "ubigeo", "ubigeo_metodo"]]
    revisar = revisar.rename(columns={"departamento_infobras": "departamento"})
    nombres = maestro.set_index("ubigeo")[["departamento", "provincia", "distrito"]].add_suffix("_escale")
    revisar = revisar.join(nombres, on="ubigeo")
    motivo = {"normalizado": "difiere solo en tildes, enie, guion bajo, nombre alterno entre parentesis o provincia del Callao",
              "sin_ubigeo": "no se encontro el distrito: revisar a mano"}
    revisar["motivo"] = [CORRECCIONES[c][1] if m == "manual" else motivo[m] for c, m in zip(revisar.codigo_infobras, revisar.ubigeo_metodo)]
    return revisar.sort_values(["ubigeo_metodo", "codigo_infobras"])


def agregar_ubigeo(fichas):
    """Asigna el ubigeo a la tabla de fichas, escribe docs/nombres_sin_ubigeo.csv y devuelve la tabla con las columnas nuevas."""
    maestro = maestro_distritos()
    fichas = asignar_ubigeo(fichas, maestro)
    tabla_revision(fichas, maestro).to_csv(SALIDA_REVISION, index=False, encoding="utf-8-sig")
    conteo = fichas.ubigeo_metodo.value_counts().to_dict()
    utils.log(f"UBIGEO: {fichas.ubigeo.notna().sum()} de {len(fichas)} obras con ubigeo {conteo}; revision en {SALIDA_REVISION.name}")
    if (fichas.ubigeo_metodo == "sin_ubigeo").any():
        utils.log(f"AVISO UBIGEO: obras sin ubigeo: {fichas.loc[fichas.ubigeo_metodo == 'sin_ubigeo', 'codigo_infobras'].tolist()}")
    return fichas

### `datos_personales` — Datos personales (limpieza y auditoría)
Datos personales: limpieza y auditoria de los comentarios libres de INFOBRAS.

In [ ]:
%%modulo datos_personales
"""Datos personales: limpieza y auditoria de los comentarios libres de INFOBRAS.

Las entidades escriben a veces el nombre de su residente o supervisor en los comentarios de una obra. La regla del proyecto es no
guardar nombres de personas, asi que:
  * utils.omitir_nombres filtra cada comentario en cuanto se lee (scraper_infobras.leer_detalle_avance);
  * limpiar() vuelve a pasar el filtro por lo ya guardado (caches y tablas); se puede repetir sin cambiar nada si ya esta limpio;
  * auditar() busca nombres en TODAS las tablas finales con patrones propios y devuelve lo que encuentre; el pipeline falla si hay alguno.
"""
import json
import re

import pandas as pd

import config
import utils

CACHES = (config.INFOBRAS_FICHAS, config.INFOBRAS_AMPLIADAS)
_NOMBRE = r"[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÑáéíóúñ]{2,}"
# titulo + dos palabras con inicial mayuscula (se excluye «don»: sale en calles como «Av. Don Bosco»)
_TITULO_NOMBRE = re.compile(rf"\b(?:(?i:ing|arq|abog|econ|lic|sr|sra|srta|cpc|bach)\.?|(?i:dra?)\.)\s+{_NOMBRE}\s+{_NOMBRE}")
# cargo + dos palabras en minuscula con inicial mayuscula
_CARGO_NOMBRE = re.compile(rf"(?<!(?i:consorcio) )\b(?i:residente|supervisor|supervisora|inspector|inspectora|proyectista)(?:\s+de\s+obra)?[,:]?\s+"
                           rf"(?P<nombre>[A-ZÁÉÍÓÚÑ][a-záéíóúñ]{{2,}}\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]{{2,}})")
_NO_PERSONA = ("consorcio", "empresa", "ejecutor", "municipalidad", "gobierno", "unidad", "proyecto", "contratista", "entidad")
_DNI = re.compile(r"(?<![\d.,])\d{8}(?![\d.,])")


def _limpiar_json(objeto):
    """Aplica omitir_nombres a todo campo 'comentario*' (anidado o no). Devuelve cuantos textos cambio."""
    cambios = 0
    if isinstance(objeto, dict):
        for clave, valor in objeto.items():
            if clave.startswith("comentario") and isinstance(valor, str):
                nuevo = utils.omitir_nombres(valor)
                if nuevo != valor:
                    objeto[clave], cambios = nuevo, cambios + 1
            else:
                cambios += _limpiar_json(valor)
    elif isinstance(objeto, list):
        for elemento in objeto:
            cambios += _limpiar_json(elemento)
    return cambios


def limpiar():
    """Quita nombres de personas de las caches (jsonl) y de los CSV con columnas 'comentario*'. Devuelve el total de textos cambiados."""
    total = 0
    for ruta in CACHES:
        if not ruta.exists():
            continue
        filas = [json.loads(linea) for linea in ruta.read_text(encoding="utf-8").splitlines() if linea.strip()]
        cambios = sum(_limpiar_json(f) for f in filas)
        if cambios:
            ruta.write_text("\n".join(json.dumps(f, ensure_ascii=False) for f in filas) + "\n", encoding="utf-8")
        total += cambios
    for ruta in sorted(config.PROCESSED.glob("*.csv")):
        cabecera = pd.read_csv(ruta, nrows=0, encoding="utf-8-sig").columns
        columnas = [c for c in cabecera if c.startswith("comentario")]
        if not columnas:
            continue
        tabla = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        cambios = 0
        for c in columnas:
            nuevo = tabla[c].map(utils.omitir_nombres)
            cambios += int((nuevo != tabla[c]).sum())
            tabla[c] = nuevo
        if cambios:
            tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
        total += cambios
    utils.log(f"DATOS PERSONALES: {total} texto(s) con nombres de personas reemplazados por {utils.OMITIDO}")
    return total


def auditar(carpetas=None):
    """Busca nombres de personas en las columnas de texto largo de todos los CSV de las carpetas. Devuelve un DataFrame de hallazgos."""
    carpetas = carpetas or [config.PROCESSED, config.COMPONENTES, config.FINAL]
    hallazgos = []
    for carpeta in carpetas:
        for ruta in sorted(carpeta.glob("*.csv")):
            tabla = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
            for columna in tabla.columns:
                if tabla[columna].str.len().max() <= 25:
                    continue
                for texto in tabla[columna].drop_duplicates():
                    for patron, tipo in ((_TITULO_NOMBRE, "titulo + nombre"), (_CARGO_NOMBRE, "cargo + nombre")):
                        m = patron.search(texto)
                        if not m:
                            continue
                        if tipo == "cargo + nombre" and m.group("nombre").split()[0].lower() in _NO_PERSONA:
                            continue
                        hallazgos.append({"tabla": ruta.name, "columna": columna, "tipo": tipo, "texto": texto[max(0, m.start() - 30):m.end() + 30]})
                    if columna.startswith("comentario") and _DNI.search(texto):
                        hallazgos.append({"tabla": ruta.name, "columna": columna, "tipo": "numero de 8 digitos (posible DNI)", "texto": texto[:80]})
    return pd.DataFrame(hallazgos, columns=["tabla", "columna", "tipo", "texto"])


def auditoria_o_falla():
    """Registra la auditoria; si hay hallazgos, deja el detalle en docs/auditoria_datos_personales.csv y lanza un error."""
    hallazgos = auditar()
    ruta = config.DOCS / "auditoria_datos_personales.csv"
    hallazgos.to_csv(ruta, index=False, encoding="utf-8-sig")
    if len(hallazgos):
        utils.log(f"AUDITORIA DATOS PERSONALES: {len(hallazgos)} hallazgo(s), ver {ruta.name}")
        raise ValueError(f"se encontraron {len(hallazgos)} posibles nombres de personas en las tablas finales (docs/{ruta.name})")
    utils.log("AUDITORIA DATOS PERSONALES: 0 hallazgos en las tablas finales")
    return hallazgos

### `nlp_causas` — NLP de las causas de paralización
NLP de las causas de paralizacion: del texto libre (comentarios de INFOBRAS, causal de la Contraloria, problematica del MEF)

In [ ]:
%%modulo nlp_causas
"""NLP de las causas de paralizacion: del texto libre (comentarios de INFOBRAS, causal de la Contraloria, problematica del MEF)
a un grupo de causa por obra.

Pasos (todo con reglas simples y un modelo de apoyo; nada de cajas negras):
  1. normalizar: minusculas, sin tildes, sin fechas ni frases de relleno ("No se registraron comentarios").
  2. puntuar: cada grupo de config.CAUSAS_REGLAS suma segun cuantos patrones distintos aparecen (maximo 2 puntos por grupo, asi
     ningun grupo gana solo por tener mas patrones). Se pueden detectar varias causas en un mismo texto.
  3. combinar las tres fuentes de una obra con pesos (la causal oficial pesa mas que un comentario viejo).
     Los grupos de ESTADO (obra concluida, sin restricciones) solo se asignan cuando no hay ninguna causa real.
  4. modelo de apoyo (TF-IDF + regresion logistica con class_weight="balanced"): se entrena con los textos que las reglas clasifican
     sin ambiguedad y solo se usa para textos donde las reglas no encontraron nada. Se mide con validacion cruzada.
  5. macro-categoria (config.CAUSAS_MACRO) para el dashboard: pocas categorias de tamano parecido.
"""
import re
import unicodedata

import numpy as np
import pandas as pd

import config
from utils import log

ESTADO = config.CAUSAS_ESTADO
ORDEN = list(config.CAUSAS_REGLAS)
CAUSAS = [g for g in ORDEN if g not in ESTADO]
UMBRAL_MODELO = 0.60          # probabilidad minima para aceptar la prediccion del modelo
MINIMO_POR_CLASE = 8          # clases con menos ejemplos limpios no entran al modelo (se quedan solo con reglas)
_REGEX = {g: [re.compile(p) for p in patrones] for g, patrones in config.CAUSAS_REGLAS.items()}
_RELLENO = [re.compile(p) for p in config.FRASES_SIN_INFORMACION]


# ---------------- 1) normalizar ----------------
def normalizar(texto):
    """Minusculas, sin tildes (la enie pasa a n), sin fechas ni direcciones web y con espacios simples."""
    if texto is None or (isinstance(texto, float) and np.isnan(texto)):
        return ""
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"\[\d{1,2}/\d{1,2}/\d{2,4}\]", " ", t)          # [27/10/2026] al inicio de la problematica del MEF
    t = re.sub(r"\d{1,2}/\d{1,2}/\d{2,4}", " ", t)
    t = re.sub(r"https?://\S+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def quitar_relleno(norm):
    for r in _RELLENO:
        norm = r.sub(" ", norm)
    return re.sub(r"\s+", " ", norm).strip()


def sin_contenido(texto):
    """True si el texto no dice nada de la causa (vacio, frases tipicas o solo palabras vacias)."""
    limpio = quitar_relleno(normalizar(texto))
    palabras = [p for p in re.findall(r"[a-z]{3,}", limpio) if p not in config.PALABRAS_SIN_INFORMACION]
    return not palabras


# ---------------- 2) puntuar un texto ----------------
_NEGADORES = re.compile(r"\b(sin|no|ni|ningun\w*|permit\w+|favorable\w*|descart\w+)\b[^.;]{0,30}$")


_NEGADORES_DESPUES = re.compile(r"^[^.;]{0,35}\b(permit\w+|favorable\w*|no (impid|afect|limit)\w*|sin (afectar|impedir)\w*)")


def _aparece(regex, norm, comprobar_negacion):
    """True si el patron aparece. En causas «de presencia» (clima, conflictos...) se descartan las apariciones precedidas de
    «no», «sin» o «permiten» en la misma frase («las condiciones climatologicas permiten su ejecucion» no es una causa)."""
    for m in regex.finditer(norm):
        if not comprobar_negacion:
            return True
        antes, despues = norm[max(0, m.start() - 45):m.start()], norm[m.end():m.end() + 45]
        if not _NEGADORES.search(antes) and not _NEGADORES_DESPUES.search(despues):
            return True
    return False


def puntuar(norm):
    """grupo -> puntaje (1 punto por el primer patron que aparece, +0.5 por cada patron distinto adicional, maximo 2)."""
    puntajes = {}
    for grupo, regs in _REGEX.items():
        k = sum(1 for r in regs if _aparece(r, norm, grupo in config.CAUSAS_PRESENCIA))
        if k:
            puntajes[grupo] = min(1 + 0.5 * (k - 1), 2.0)
    return puntajes


# ---------------- 3) combinar las fuentes de una obra ----------------
def analizar(fuentes):
    """fuentes: lista de (nombre_fuente, texto, peso). Devuelve un diccionario con la causa principal y sus datos de apoyo."""
    total, por_fuente, hay_texto = {}, {}, False
    for nombre, texto, peso in fuentes:
        norm = quitar_relleno(normalizar(texto))
        if not norm or sin_contenido(texto):
            continue
        hay_texto = True
        for grupo, puntaje in puntuar(norm).items():
            total[grupo] = total.get(grupo, 0) + puntaje * peso
            por_fuente.setdefault(grupo, {})
            por_fuente[grupo][nombre] = por_fuente[grupo].get(nombre, 0) + puntaje * peso
    causas = {g: s for g, s in total.items() if g not in ESTADO}
    pool = causas or total                       # los grupos de estado solo cuentan si no hay causa real
    if not pool:
        return {"principal": "otros" if hay_texto else "sin_informacion", "secundarias": [], "fuente": "", "confianza": "", "metodo": "reglas"}
    mejor = max(pool.values())
    principal = next(g for g in ORDEN if pool.get(g) == mejor)
    secundarias = [g for g in ORDEN if g != principal and pool.get(g, 0) >= 0.5 * mejor][:2]
    aporte = por_fuente[principal]
    return {"principal": principal, "secundarias": secundarias, "fuente": max(aporte, key=aporte.get),
            "confianza": "alta" if len(aporte) >= 2 else "media", "metodo": "reglas"}


def clasificar_causa(causal, comentario=None):
    """Grupo de causa de un par (causal declarada, comentario). Se mantiene con este nombre porque lo usan otros modulos."""
    return analizar([("causal", causal, 3.0), ("comentario", comentario, 1.0)])["principal"]


def macro(grupo):
    return config.CAUSAS_MACRO.get(grupo, "sin_clasificar")


# ---------------- 4) modelo de apoyo ----------------
def _modelo():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(class_weight="balanced", max_iter=3000, C=2.0))


def ejemplos_limpios(textos):
    """Textos unicos que las reglas clasifican sin ambiguedad: una sola causa real. Devuelve (textos normalizados, etiquetas)."""
    vistos, X, y = set(), [], []
    for texto in textos:
        norm = quitar_relleno(normalizar(texto))
        if not norm or norm in vistos or sin_contenido(texto):
            continue
        vistos.add(norm)
        causas = [g for g in puntuar(norm) if g not in ESTADO]
        if len(causas) == 1:
            X.append(norm)
            y.append(causas[0])
    return X, y


def entrenar(textos):
    """Entrena el modelo con los ejemplos limpios (solo clases con MINIMO_POR_CLASE ejemplos) y mide con validacion cruzada.
    Devuelve (modelo, metricas) donde metricas es una tabla por clase con precision, recall, f1 y cantidad de ejemplos."""
    from sklearn.metrics import classification_report
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    X, y = ejemplos_limpios(textos)
    cuenta = pd.Series(y).value_counts()
    validas = set(cuenta[cuenta >= MINIMO_POR_CLASE].index)
    X, y = [x for x, g in zip(X, y) if g in validas], [g for g in y if g in validas]
    modelo = _modelo()
    pliegues = int(min(5, pd.Series(y).value_counts().min()))
    prediccion = cross_val_predict(modelo, X, y, cv=StratifiedKFold(pliegues, shuffle=True, random_state=2026))
    reporte = classification_report(y, prediccion, output_dict=True, zero_division=0)
    metricas = pd.DataFrame(reporte).T.reset_index().rename(columns={"index": "clase", "f1-score": "f1", "support": "ejemplos"})
    metricas["ejemplos"] = metricas.ejemplos.round(0).astype(int)
    for c in ("precision", "recall", "f1"):
        metricas[c] = metricas[c].round(3)
    modelo.fit(X, y)
    log(f"NLP causas: modelo entrenado con {len(X)} textos ({len(validas)} clases); validacion cruzada de {pliegues} pliegues: "
        f"macro-F1 = {reporte['macro avg']['f1-score']:.3f}, exactitud = {reporte['accuracy']:.3f}")
    return modelo, metricas


def predecir(modelo, texto):
    """(grupo, probabilidad) del modelo para un texto, o ('', 0) si esta vacio."""
    norm = quitar_relleno(normalizar(texto))
    if not norm or sin_contenido(texto):
        return "", 0.0
    prob = modelo.predict_proba([norm])[0]
    i = int(np.argmax(prob))
    return str(modelo.classes_[i]), float(prob[i])


# ---------------- 5) clasificar toda la tabla maestra ----------------
def textos_de_infobras(paralizaciones):
    """codigo_infobras -> lista de (texto, peso) con los comentarios unicos; los del ultimo tramo de paralizacion pesan 2 y los demas 0.5."""
    p = paralizaciones.copy()
    p["texto"] = (p.comentario_fisico.fillna("") + " " + p.comentario_valorizado.fillna("")).str.strip()
    p["tramo_n"] = pd.to_numeric(p.tramo, errors="coerce").fillna(0)
    p = p[~p.texto.map(sin_contenido)]
    salida = {}
    for codigo, g in p.groupby("codigo_infobras"):
        ultimo = g.tramo_n.max()
        vistos, lista = set(), []
        for texto, tramo in sorted(zip(g.texto, g.tramo_n), key=lambda x: -x[1]):
            clave = normalizar(texto)
            if clave in vistos:
                continue
            vistos.add(clave)
            lista.append((texto, 2.0 if tramo == ultimo else 0.5))
        salida[codigo] = lista
    return salida


def clasificar_maestra(m, paralizaciones, mef_completo):
    """Agrega a la tabla maestra las columnas de causa. m necesita: codigo_infobras, causal_oficial_contraloria, mef_ult_problema.
    Devuelve (m_con_causas, metricas_del_modelo, detalle_por_fuente)."""
    infobras = textos_de_infobras(paralizaciones)
    # el modelo aprende de todos los textos libres disponibles: comentarios de INFOBRAS y problematica de todo el Formato 12-B del MEF
    textos_modelo = [t for lista in infobras.values() for t, _ in lista] + list(mef_completo.get("ult_problema", pd.Series(dtype=str)).fillna(""))
    modelo, metricas = entrenar(textos_modelo)

    filas = []
    for _, r in m.iterrows():
        comentarios = infobras.get(r.codigo_infobras, [])
        fuentes = [("infobras_comentario", t, w) for t, w in comentarios]
        fuentes += [("contraloria_causal", r.causal_oficial_contraloria, 3.0), ("mef_problema", r.mef_ult_problema, 1.5)]
        res = analizar(fuentes)
        # causa por fuente (solo para comparar las fuentes entre si)
        g_infobras = analizar([("x", t, w) for t, w in comentarios])["principal"] if comentarios else "sin_informacion"
        g_contraloria = analizar([("x", r.causal_oficial_contraloria, 1.0)])["principal"]
        g_mef = analizar([("x", r.mef_ult_problema, 1.0)])["principal"]
        if res["principal"] == "otros":
            # las reglas no reconocieron nada: se prueba el modelo solo con los textos libres (no con la causal oficial "OTROS")
            libres = [t for t, _ in comentarios] + [r.mef_ult_problema]
            mejor = max((predecir(modelo, t) for t in libres), key=lambda x: x[1], default=("", 0.0))
            if mejor[0] and mejor[1] >= UMBRAL_MODELO:
                res = {"principal": mejor[0], "secundarias": [], "fuente": "texto_libre", "confianza": "baja", "metodo": "modelo"}
        filas.append({"causa_grupo_infobras": g_infobras, "causa_grupo_contraloria": g_contraloria, "causa_grupo_mef": g_mef,
                      "causa_grupo_final": res["principal"], "causa_macro": macro(res["principal"]),
                      "causas_secundarias": "|".join(res["secundarias"]), "causa_fuente": res["fuente"] or "sin_causa_clara",
                      "causa_confianza": res["confianza"], "causa_metodo": res["metodo"]})
    nuevas = pd.DataFrame(filas, index=m.index)
    m = m.drop(columns=[c for c in nuevas.columns if c in m]).join(nuevas)
    return m, metricas

### `texto_nlp` — NLP del texto de los informes (citas y términos)
Fase 4 del proyecto PDF (opcional): texto citable y clasificacion de causas por palabras clave.

In [ ]:
%%modulo texto_nlp
"""Fase 4 del proyecto PDF (opcional): texto citable y clasificacion de causas por palabras clave.

Solo se usan expresiones regulares y conteos (sin spaCy ni scikit-learn): alcanza para buscar frases con 'educacion',
contar terminos y asignar un grupo de causa por palabras clave, y las reglas se pueden leer y corregir a mano en config.py.
No se afirman relaciones de causa-efecto: solo se describe lo que dice el texto.
"""
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

import config
import tablas_finales
from leer_pdf import log


def sin_tildes(texto):
    return unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()


# ---------------- texto narrativo de los informes ----------------
def parrafos_de_pagina(texto):
    """Une las lineas de la pagina en un solo texto narrativo, dejando fuera las lineas que son filas de tabla (muchos numeros)."""
    lineas = []
    for linea in texto.splitlines():
        linea = linea.strip()
        letras = sum(c.isalpha() for c in linea)
        digitos = sum(c.isdigit() for c in linea)
        if letras < 8 or digitos > 0.25 * (letras + digitos):
            continue
        lineas.append(linea)
    unido = " ".join(lineas)
    return re.sub(r"(\w)- (\w)", r"\1\2", unido)   # une palabras cortadas con guion al final de linea


def oraciones(texto):
    """Oraciones del texto narrativo: con palabras suficientes y sin ser texto de celdas en mayusculas ni pies de tabla."""
    salida = []
    for o in re.split(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÑ•▪])", texto):
        o = o.strip()
        letras = [c for c in o if c.isalpha()]
        if (len(o.split()) >= config.CITAS_MIN_PALABRAS and not o.startswith(("FUENTE", "Fuente", "ELABORACI")) and letras
                and sum(c.isupper() for c in letras) <= 0.3 * len(letras)):
            salida.append(o)
    return salida


def paginas_de_listas():
    """{(archivo, pagina)} de las paginas con listas de obras (tablas de 'principales obras'), cuyo texto no es narrativa."""
    catalogo = pd.read_csv(config.CATALOGO_TABLAS, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    lista = catalogo[catalogo.tipo.isin(["principales_obras_departamento", "continuacion"]) & (catalogo.estado == "guardada")]
    paginas = set()
    for _, f in lista.iterrows():
        paginas |= {(f.archivo, p) for p in range(int(f.pagina), int(f.pagina_fin) + 1)}
    return paginas


def paginas_de_informes():
    """(informe, numero_de_pagina, texto) de las paginas narrativas de los informes de la Contraloria."""
    excluidas = paginas_de_listas()
    for informe in tablas_finales.informes_contraloria():
        for archivo in sorted(informe["carpeta"].glob("texto_p*.txt")):
            pagina = int(archivo.stem.split("_p")[1])
            if (informe["archivo"], pagina) not in excluidas:
                yield informe, pagina, archivo.read_text(encoding="utf-8")


# ---------------- citas de educacion ----------------
def citas_educacion():
    """Oraciones de los informes que mencionan educacion o escuelas, con archivo y pagina para poder citarlas."""
    patron = re.compile(config.CITAS_REGEX, re.I)
    filas, vistas = [], set()
    for informe, pagina, texto in paginas_de_informes():
        for oracion in oraciones(parrafos_de_pagina(texto)):
            m = patron.search(oracion)
            clave = (informe["archivo"], oracion)
            if not m or clave in vistas:
                continue
            vistas.add(clave)
            filas.append({"anio": informe["anio"], "fecha_corte": informe["fecha_corte"], "fuente_pdf": informe["archivo"],
                          "pagina": pagina, "termino": m.group(0).lower(), "cita": oracion[:config.CITAS_MAX_CARACTERES]})
    return pd.DataFrame(filas, columns=["anio", "fecha_corte", "fuente_pdf", "pagina", "termino", "cita"])


# ---------------- frecuencia de terminos ----------------
def frecuencia_terminos():
    """Terminos y pares de terminos mas frecuentes de la narrativa de los informes, por anio."""
    conteos = {}
    for informe, pagina, texto in paginas_de_informes():
        palabras = [p for p in re.findall(r"[a-zñ]{4,}", sin_tildes(parrafos_de_pagina(texto)))]
        utiles = [p if p not in config.STOPWORDS_ES else None for p in palabras]
        por_anio = conteos.setdefault(informe["anio"], {"termino": Counter(), "par": Counter()})
        por_anio["termino"].update(p for p in utiles if p)
        por_anio["par"].update(f"{a} {b}" for a, b in zip(utiles, utiles[1:]) if a and b and a != b)   # sin pares como "costo costo"
    filas = []
    n_anios = len(conteos)
    for tipo in ("termino", "par"):
        # TF-IDF por anio: frecuencia relativa en el anio x log(anios / anios en que aparece). Sube los terminos propios de un anio
        # y baja los que aparecen siempre (gobierno, inversion), que no distinguen un informe de otro.
        en_anios = Counter(t for c in conteos.values() for t in c[tipo])
        for anio, c in sorted(conteos.items()):
            total = sum(c[tipo].values()) or 1
            puntajes = {t: (n / total) * np.log(n_anios / en_anios[t]) for t, n in c[tipo].items() if n >= 3}
            for termino in sorted(puntajes, key=puntajes.get, reverse=True)[:config.TERMINOS_POR_ANIO]:
                filas.append({"anio": anio, "tipo": tipo, "termino": termino, "frecuencia": c[tipo][termino],
                              "tfidf": round(puntajes[termino] * 1000, 3), "anios_con_termino": en_anios[termino]})
    return pd.DataFrame(filas)


# ---------------- clasificacion de causas por palabras clave ----------------
# La clasificacion de causas vive en nlp_causas.py (reglas, combinacion de fuentes y modelo de apoyo); aqui se re-exporta para no romper llamadas.
from nlp_causas import clasificar_causa, sin_contenido   # noqa: E402,F401


# ---------------- Fase 4: ejecutar ----------------
# Causales oficiales de la Contraloria y el grupo que deberian recibir (sirve para comprobar que las reglas funcionan)
PRUEBA_REGLAS = {
    "FALTA DE RECURSOS FINANCIEROS Y LIQUIDEZ": "recursos_financieros", "INCUMPLIMIENTO DE CONTRATO": "incumplimiento_contractual",
    "DISCREPANCIAS, CONTROVERSIAS Y ARBITRAJE": "controversias_arbitraje", "EVENTOS CLIMATICOS": "clima",
    "DEFICIENCIA EN EL EXPEDIENTE TECNICO": "expediente_tecnico", "CONFLICTOS SOCIALES": "conflictos_sociales",
    "DISPONIBILIDAD DE TERRENO": "terreno_saneamiento", "FALTA DE PERMISOS, LICENCIAS Y AUTORIZACIONES": "permisos_licencias",
    "INTERFERENCIAS": "interferencias", "ABANDONO DE OBRA": "incumplimiento_contractual",
    "INCUMPLIMIENTO DEL PAGO DE VALORIZACIONES U OTROS": "recursos_financieros",
    "DESABASTECIMIENTO SOSTENIDO DE MATERIALES": "materiales_logistica", "PANDEMIA COVID-19": "pandemia"}


def clasificar_fichas(fichas):
    """Agrega causa_grupo y causa_grupo_origen a la tabla de fichas de INFOBRAS."""
    comentarios = (fichas.comentario_fisico.fillna("") + " " + fichas.comentario_valorizado.fillna("")).str.strip()
    fichas = fichas.drop(columns=[c for c in ("causa_grupo", "causa_grupo_origen") if c in fichas])
    fichas["causa_grupo"] = [clasificar_causa(c, k) for c, k in zip(fichas.causal_paralizacion, comentarios)]

    def origen(causal, comentario, grupo):
        if grupo in ("otros", "sin_informacion"):
            return grupo
        return "causal" if clasificar_causa(causal, None) not in ("otros", "sin_informacion") else "comentario"

    fichas["causa_grupo_origen"] = [origen(c, k, g) for c, k, g in zip(fichas.causal_paralizacion, comentarios, fichas.causa_grupo)]
    return fichas, comentarios


def ejecutar_fase4():
    """Guarda citas_educacion.csv y frecuencia_terminos.csv, y agrega causa_grupo a fichas_infobras.csv con su validacion."""
    citas = citas_educacion()
    citas.to_csv(config.CITAS_EDUCACION, index=False, encoding="utf-8-sig")
    terminos = frecuencia_terminos()
    terminos.to_csv(config.FRECUENCIA_TERMINOS, index=False, encoding="utf-8-sig")
    log(f"Fase 4: {len(citas)} citas sobre educacion, {len(terminos)} filas de frecuencia de terminos")

    # las reglas deben reproducir el grupo de las causales oficiales
    fallas = [(c, esperado, clasificar_causa(c)) for c, esperado in PRUEBA_REGLAS.items() if clasificar_causa(c) != esperado]
    log(f"Fase 4: prueba de las reglas con {len(PRUEBA_REGLAS)} causales oficiales: {len(PRUEBA_REGLAS) - len(fallas)} correctas"
        + (f"; fallan {fallas}" if fallas else ""))

    if not config.INFOBRAS_SALIDA.exists():
        log("Fase 4: no hay fichas_infobras.csv todavia; se omite la clasificacion de causas")
        return citas, terminos, None
    fichas = pd.read_csv(config.INFOBRAS_SALIDA, dtype={"codigo_infobras": str, "cui": str, "ubigeo": str, "cod_departamento": str}, encoding="utf-8-sig")
    fichas["cod_departamento"] = fichas.cod_departamento.astype(str).str.zfill(2)   # codigo de 2 digitos como texto (se perdia el cero: "1" en vez de "01")
    fichas, comentarios = clasificar_fichas(fichas)
    import ubigeo_obras   # A1: agrega ubigeo y ubigeo_metodo; se recalcula en cada corrida
    fichas = ubigeo_obras.agregar_ubigeo(fichas)
    fichas.to_csv(config.INFOBRAS_SALIDA, index=False, encoding="utf-8-sig")
    porcentaje_otros = round(100 * (fichas.causa_grupo == "otros").mean(), 1)
    porcentaje_sin = round(100 * (fichas.causa_grupo == "sin_informacion").mean(), 1)
    log(f"Fase 4: {len(fichas)} fichas clasificadas; 'otros' = {porcentaje_otros} %, 'sin_informacion' = {porcentaje_sin} %; "
        f"grupos: {fichas.causa_grupo.value_counts().to_dict()}")
    muestra = fichas.assign(comentarios=comentarios.str[:300]).sample(min(30, len(fichas)), random_state=2026)
    muestra[["codigo_infobras", "causal_paralizacion", "comentarios", "causa_grupo", "causa_grupo_origen"]].to_csv(
        config.DOCS / "muestra_validacion_causas.csv", index=False, encoding="utf-8-sig")
    return citas, terminos, fichas

### `dashboard` — CSV único para el dashboard
CSV unico del proyecto (lo usa el dashboard HTML): data/final/dashboard_obras.csv

In [ ]:
%%modulo dashboard
"""CSV unico del proyecto (lo usa el dashboard HTML): data/final/dashboard_obras.csv

Una sola tabla en formato largo. La columna `tipo_registro` dice que representa cada fila; hay que filtrar por ella al armar cada grafico
(si no, se mezclan niveles distintos y se cuenta dos veces):

  obra_infobras           429 obras de Educacion/Cultura leidas en INFOBRAS (foto de nov-2026): una fila por obra, todos los datos
  obra_corte_contraloria  obras de Educacion en cada corte de la Contraloria (anexos Excel, sep-2023 a jun-2026): una fila por obra y corte
  departamento_corte      Educacion por departamento y corte (13 cortes, sep-2023 a jun-2026, anexos Excel): cifras ya sumadas
  pais_total_corte        total nacional (todos los sectores) por corte (21 cortes, jun-2022 a jun-2026), segun el informe PDF
  pais_educacion_corte    sector Educacion a nivel pais por corte (con la cifra corregida con los anexos Excel)
  causal_corte            causales oficiales de paralizacion por corte (cantidad de obras y monto), con su grupo de causa
  contexto_region         una fila por region ENLA con ESCALE y ENLA (datos de contexto)

Medidas comunes a todos los tipos: n_obras, n_obras_paralizadas, costo_actualizado_soles, saldo_inversion_soles.
"""
import numpy as np
import pandas as pd

import config
import contraloria_anexo03
import escale_ampliado
import nlp_causas
import utils

TIPOS = ["obra_infobras", "obra_corte_contraloria", "departamento_corte", "pais_total_corte", "pais_educacion_corte", "causal_corte", "contexto_region"]

# (columna, tipo, descripcion, unidad)
DICCIONARIO = [
    ("tipo_registro", "texto", "Qué representa la fila: obra_infobras, obra_corte_contraloria, departamento_corte, pais_total_corte, pais_educacion_corte, causal_corte o contexto_region. Filtrar siempre por este campo.", "categoría"),
    ("fecha_corte", "fecha AAAA-MM-DD", "Fecha del dato: corte del informe de la Contraloría, o fecha de lectura de INFOBRAS (2026-11-11) para obra_infobras. Vacía en contexto_region.", "fecha"),
    ("anio", "entero", "Año de fecha_corte.", "año"),
    ("fuente_dato", "texto", "Archivo o fuente de donde sale la fila (PDF, Excel, INFOBRAS, ENLA/ESCALE).", "texto"),
    ("alertas_calidad", "texto", "Avisos sobre el dato de esa fila, separados por «|»; vacío = sin avisos. Los valores NO se corrigen (vienen así de la fuente): se marcan para filtrarlos o explicarlos. obra_infobras: sin_coordenadas, sin_expediente_tecnico, cui_no_asociado, cui_sin_datos_mef, cui_en_varias_obras, paralizacion_anterior_al_inicio, avance_infobras_vs_mef_difiere_30_puntos, devengado_mayor_al_costo, ejecucion_mayor_al_expediente, causa_sin_clasificar. obra_corte_contraloria: costo_cero_o_vacio_en_fuente, saldo_mayor_que_costo, sin_provincia_ni_distrito.", "texto"),
    ("codigo_infobras", "texto", "Código de la obra en INFOBRAS (como texto).", "código"),
    ("cui", "texto", "Código Único de Inversión; «NO ASOCIADO» si la obra no tiene.", "código"),
    ("nombre_obra", "texto", "Nombre de la obra (solo obra_infobras).", "texto"),
    ("entidad", "texto", "Entidad ejecutora.", "texto"),
    ("cod_departamento", "texto", "Código INEI del departamento, 2 dígitos como texto (Callao = 07).", "código"),
    ("departamento", "texto", "Departamento en MAYÚSCULAS sin tildes.", "texto"),
    ("provincia", "texto", "Provincia en MAYÚSCULAS sin tildes.", "texto"),
    ("distrito", "texto", "Distrito en MAYÚSCULAS sin tildes.", "texto"),
    ("ubigeo", "texto", "Ubigeo de 6 dígitos como texto (solo obra_infobras).", "código"),
    ("region_enla", "texto", "Región según la ENLA (Lima se divide en LIMA METROPOLITANA y LIMA PROVINCIAS; Callao aparte).", "categoría"),
    ("latitud", "decimal", "Latitud (grados decimales) de la ficha de INFOBRAS; vacía si INFOBRAS no tiene coordenadas de la obra.", "grados"),
    ("longitud", "decimal", "Longitud (grados decimales) de la ficha de INFOBRAS.", "grados"),
    ("latitud_mapa", "decimal", "Latitud para el mapa (solo obra_infobras): la de INFOBRAS o, si falta, una aproximada de ESCALE. Es la que usa el mapa del dashboard, junto con coordenada_origen.", "grados"),
    ("longitud_mapa", "decimal", "Longitud para el mapa (mismo criterio que latitud_mapa).", "grados"),
    ("coordenada_origen", "texto", "infobras (exacta), escale_local_obra (local escolar que declara el CUI de la obra) o centro_distrito_escale (mediana de los colegios del distrito): estas dos son aproximadas.", "categoría"),
    ("sector", "texto", "Sector: EDUCACION para las obras de INFOBRAS y de la Contraloría; TODOS en pais_total_corte.", "categoría"),
    ("nivel_gobierno", "texto", "Nivel de gobierno de la entidad (Contraloría).", "categoría"),
    ("origen", "texto", "Por qué la obra está en obra_infobras: paralizada en INFOBRAS el 27-oct, o en la lista de la Contraloría de jun-2026 y no paralizada en esa fecha.", "categoría"),
    ("estado_infobras", "texto", "Estado de la obra en INFOBRAS (PARALIZADA, EN EJECUCIÓN, FINALIZADO, SIN EJECUCIÓN).", "categoría"),
    ("paralizada_hoy", "lógico", "TRUE si el estado en INFOBRAS es PARALIZADA (solo obra_infobras).", "lógico"),
    ("modalidad", "texto", "Modalidad de ejecución (INFOBRAS o Contraloría).", "categoría"),
    ("tipo_obra", "texto", "Tipo de obra.", "categoría"),
    ("tipo_infraestructura", "texto", "Solo obra_infobras. Tipo de infraestructura según el nombre de la obra (reglas; si el nombre no lo dice, el tipo de obra de INFOBRAS): educacion_basica (colegios, inicial, CEBA, CETPRO), educacion_superior (universidades e institutos), deportiva_recreativa, cultural_patrimonio u otra. INFOBRAS agrupa Educación y Cultura. Para el objetivo del proyecto (instituciones educativas) filtrar educacion_basica.", "categoría"),
    ("evento_contraloria", "texto", "Evento de la obra en el corte de la Contraloría (p. ej. PROCESO NORMAL, REINICIO).", "categoría"),
    ("causal_oficial", "texto", "Causal de paralización declarada: la del corte en obra_corte_contraloria, la del último corte en obra_infobras, el texto de la causal en causal_corte.", "texto"),
    ("causa_grupo_final", "texto", "Causa principal (NLP) en 14 grupos: recursos_financieros, incumplimiento_contractual, expediente_tecnico, controversias_arbitraje, clima, conflictos_sociales, terreno_saneamiento, permisos_licencias, interferencias, materiales_logistica, personal_clave, pandemia, obra_concluida_o_en_marcha, sin_restricciones (más otros y sin_informacion). En las filas de la Contraloría se deduce de la causal oficial.", "categoría"),
    ("causa_macro", "texto", "Causa en 8 macro-categorías para gráficos: financiera, contractual_y_legal, tecnica, logistica, entorno_social_y_territorial, fuerza_mayor, sin_causa_de_paralizacion, sin_clasificar.", "categoría"),
    ("causas_secundarias", "texto", "Hasta 2 causas adicionales detectadas en los textos de la obra, separadas por «|» (solo obra_infobras).", "categoría"),
    ("causa_fuente", "texto", "Fuente que más aportó a la causa principal: infobras_comentario, contraloria_causal, mef_problema, texto_libre o sin_causa_clara (solo obra_infobras).", "categoría"),
    ("causa_confianza", "texto", "alta (2 o más fuentes coinciden), media (una fuente), baja (solo modelo) o vacío (solo obra_infobras).", "categoría"),
    ("causa_metodo", "texto", "reglas o modelo (solo obra_infobras).", "categoría"),
    ("causa_grupo_infobras", "texto", "Causa según solo los comentarios de INFOBRAS (solo obra_infobras).", "categoría"),
    ("causa_grupo_contraloria", "texto", "Causa según solo la causal oficial de la Contraloría (solo obra_infobras).", "categoría"),
    ("causa_grupo_mef", "texto", "Causa según solo la problemática registrada en el MEF (solo obra_infobras).", "categoría"),
    ("n_obras", "entero", "Medida: cantidad de obras que representa la fila (1 en las filas por obra; la cifra del informe en las agregadas). En pais_total_corte son todas las obras del país.", "obras"),
    ("n_obras_paralizadas", "entero", "Medida: obras paralizadas que representa la fila (en obra_infobras es 1 solo si paralizada_hoy; en los demás tipos es igual a n_obras porque las listas de la Contraloría son de obras paralizadas).", "obras"),
    ("costo_actualizado_soles", "decimal", "Medida: costo actualizado de la inversión (soles).", "soles"),
    ("saldo_inversion_soles", "decimal", "Medida: saldo por invertir (soles).", "soles"),
    ("monto_expediente_tecnico", "decimal", "Monto del expediente técnico (INFOBRAS, soles).", "soles"),
    ("monto_ejecucion_acumulado", "decimal", "Monto de ejecución acumulado (INFOBRAS, soles).", "soles"),
    ("avance_fisico_pct", "decimal", "Avance físico % (INFOBRAS para obra_infobras; Contraloría para obra_corte_contraloria).", "%"),
    ("pct_obras", "decimal", "% de las obras que representa la causal (causal_corte) o % de Educación sobre el país (pais_educacion_corte).", "%"),
    ("n_obras_estricto", "entero", "Obras de Educación con la definición estricta del anexo (departamento_corte).", "obras"),
    ("costo_estricto_soles", "decimal", "Costo de las obras con la definición estricta (departamento_corte).", "soles"),
    ("educacion_corregida", "lógico", "TRUE si la cifra de Educación del PDF se corrigió con los anexos Excel (pais_educacion_corte).", "lógico"),
    ("fuente_obras_educacion", "texto", "De dónde sale la cifra de obras de Educación del país: pdf o excel (pais_educacion_corte).", "categoría"),
    ("fecha_inicio", "fecha AAAA-MM-DD", "Fecha de inicio de la obra (INFOBRAS).", "fecha"),
    ("fecha_fin_prevista", "fecha AAAA-MM-DD", "Fecha de fin prevista (INFOBRAS).", "fecha"),
    ("fecha_paralizacion", "fecha AAAA-MM-DD", "Fecha de paralización del último tramo (INFOBRAS).", "fecha"),
    ("dias_paralizado", "entero", "Valor «días paralizado» del detalle del último avance paralizado de INFOBRAS. OJO: en casi todas las obras vale 30 o 31 (los días de ese mes del informe), no el tiempo total; para el tiempo total usar dias_desde_paralizacion.", "días"),
    ("dias_desde_paralizacion", "entero", "Solo obra_infobras. Días entre fecha_paralizacion y la lectura de INFOBRAS (11-nov-2026), solo en las obras paralizadas hoy: cuánto tiempo lleva detenida la obra en su paralización actual.", "días"),
    ("meses_paralizado", "entero", "Meses con estado paralizado en el historial de avances.", "meses"),
    ("n_tramos_paralizacion", "entero", "Cantidad de tramos de paralización distintos.", "tramos"),
    ("n_cortes_contraloria", "entero", "Cantidad de cortes (de los 13 con base obra por obra) en que aparece la obra.", "cortes"),
    ("reingresos_contraloria", "entero", "Veces que la obra salió y volvió a entrar a la lista de la Contraloría.", "veces"),
    ("en_ultimo_corte_contraloria", "lógico", "TRUE si la obra está en la lista de jun-2026.", "lógico"),
    ("estado_mef", "texto", "Estado de la inversión en el Formato 12-B del MEF.", "categoría"),
    ("devengado_total_soles", "decimal", "Devengado acumulado según el MEF (soles).", "soles"),
    ("pim_anio_actual_soles", "decimal", "Presupuesto institucional modificado del año actual (MEF).", "soles"),
    ("pct_devengado_del_costo", "decimal", "Devengado total / costo actualizado × 100 (puede pasar 100 por diferencias entre fuentes).", "%"),
    ("avance_fisico_mef_pct", "decimal", "Avance físico % que reporta el MEF.", "%"),
    ("mef_ult_problema", "texto", "Última problemática registrada en el MEF (nombres de personas omitidos).", "texto"),
    ("escale_locales_publicos", "entero", "Locales públicos del distrito (ESCALE).", "locales"),
    ("escale_pct_3_servicios", "decimal", "% de locales del distrito con los 3 servicios básicos.", "%"),
    ("escale_pct_rural", "decimal", "% de locales rurales del distrito.", "%"),
    ("escale_pct_riesgo_estructural", "decimal", "% de locales con riesgo estructural.", "%"),
    ("escale_pct_estructura_mal_estado", "decimal", "% de locales con estructura en mal estado.", "%"),
    ("region_escale_locales_publicos", "entero", "Locales públicos de la región (suma de distritos, ESCALE).", "locales"),
    ("region_escale_pct_3_servicios", "decimal", "% de locales de la región con los 3 servicios básicos.", "%"),
    ("region_escale_pct_rural", "decimal", "% de locales rurales de la región.", "%"),
    ("region_escale_pct_riesgo_estructural", "decimal", "% de locales de la región con riesgo estructural.", "%"),
    ("region_escale_pct_estructura_mal_estado", "decimal", "% de locales de la región con estructura en mal estado.", "%"),
    ("region_enla_4p_lectura_satisfactorio", "decimal", "% de estudiantes de 4.º primaria de la región en nivel Satisfactorio en Lectura (último año ENLA).", "%"),
    ("region_enla_4p_matematica_satisfactorio", "decimal", "% en nivel Satisfactorio en Matemática, 4.º primaria.", "%"),
    ("region_enla_2s_lectura_satisfactorio", "decimal", "% en nivel Satisfactorio en Lectura, 2.º secundaria.", "%"),
    ("region_enla_2s_matematica_satisfactorio", "decimal", "% en nivel Satisfactorio en Matemática, 2.º secundaria.", "%"),
]
# ESCALE ampliado (distrito, 2025) y vinculo de la obra con un local en construccion: se insertan despues de los indicadores de ESCALE
_ampliado = [("escale_" + c, t, d + " (distrito, Censo 2025)", u) for c, t, d, u in escale_ampliado.DICCIONARIO if c not in ("anio", "ubigeo", "n_locales_publicos_ampliado")]
_vinculo = [("escale_n_locales_obra", "entero", "Locales que declaran en el Censo 2025 estar en construcción con el CUI de esta obra (0 = ninguno).", "locales"),
            ("escale_codlocal", "texto", "Código del primer local de ESCALE que declara el CUI de esta obra.", "código")]
_anexo03 = [(c, t, d + " Vacío = la obra no figura en el anexo 03 (corte mar-2026).", u) for c, t, d, u in contraloria_anexo03.DICCIONARIO
            if c not in ("codigo_infobras", "corte_anexo03")]
_pos_sc = [c for c, *_ in DICCIONARIO].index("reingresos_contraloria") + 1
DICCIONARIO = DICCIONARIO[:_pos_sc] + _anexo03 + DICCIONARIO[_pos_sc:]
_pos = [c for c, *_ in DICCIONARIO].index("escale_pct_estructura_mal_estado") + 1
DICCIONARIO = DICCIONARIO[:_pos] + _vinculo + _ampliado + DICCIONARIO[_pos:]
# ENLA: todos los grados que ofrece SICRECE (ultimo anio de cada uno); region_enla_{grado}_anio dice de que anio es cada dato
GRADOS_ENLA = [("2p", "2.º primaria"), ("4p", "4.º primaria"), ("6p", "6.º primaria"), ("2s", "2.º secundaria")]
_enla = [("region_enla_" + suf + "_anio", "entero", f"Último año de la ENLA de {gr} disponible para la región.", "año") for suf, gr in GRADOS_ENLA]
for suf, gr in GRADOS_ENLA:
    if suf not in ("4p", "2s"):
        _enla += [(f"region_enla_{suf}_lectura_satisfactorio", "decimal", f"% en nivel Satisfactorio en Lectura, {gr} (último año ENLA).", "%"),
                  (f"region_enla_{suf}_matematica_satisfactorio", "decimal", f"% en nivel Satisfactorio en Matemática, {gr}.", "%")]
DICCIONARIO += _enla
COLUMNAS = [c for c, *_ in DICCIONARIO]


def _fecha_iso(s):
    return pd.to_datetime(s, errors="coerce").dt.strftime("%Y-%m-%d")


def _contexto_region(resumen):
    """Columnas region_* a partir de resumen_region (una fila por region ENLA)."""
    r = resumen.copy()
    pares = {"escale_locales_publicos": "region_escale_locales_publicos", "escale_pct_3_servicios": "region_escale_pct_3_servicios",
             "escale_pct_rural": "region_escale_pct_rural", "escale_pct_riesgo_estructural": "region_escale_pct_riesgo_estructural",
             "escale_pct_estructura_mal_estado": "region_escale_pct_estructura_mal_estado",
             "enla_4p_lectura_satisfactorio": "region_enla_4p_lectura_satisfactorio", "enla_4p_matematica_satisfactorio": "region_enla_4p_matematica_satisfactorio",
             "enla_2s_lectura_satisfactorio": "region_enla_2s_lectura_satisfactorio", "enla_2s_matematica_satisfactorio": "region_enla_2s_matematica_satisfactorio"}
    for suf, _ in GRADOS_ENLA:
        for sufijo in ("anio",) + (() if suf in ("4p", "2s") else ("lectura_satisfactorio", "matematica_satisfactorio")):
            pares[f"enla_{suf}_{sufijo}"] = f"region_enla_{suf}_{sufijo}"
    return r[["region_enla", "cod_departamento"] + list(pares)].rename(columns=pares)


def alertas(d):
    """Texto con los avisos de calidad de cada fila (ver el diccionario). No cambia ningun valor: solo marca lo que viene dudoso de la fuente."""
    n = lambda c: pd.to_numeric(d[c], errors="coerce")
    obra, corte = d.tipo_registro == "obra_infobras", d.tipo_registro == "obra_corte_contraloria"
    cui_ok = d.cui.astype(str).str.fullmatch(r"\d+").fillna(False)
    repetidos = d[obra & cui_ok].cui.value_counts()
    reglas = {
        "sin_coordenadas": obra & d.latitud.isna(),
        "sin_expediente_tecnico": obra & ~(n("monto_expediente_tecnico") > 0),
        "cui_no_asociado": obra & ~cui_ok,
        "cui_sin_datos_mef": obra & cui_ok & d.estado_mef.isna(),
        "cui_en_varias_obras": obra & cui_ok & d.cui.map(repetidos).gt(1).fillna(False),
        "paralizacion_anterior_al_inicio": obra & (pd.to_datetime(d.fecha_paralizacion, errors="coerce") < pd.to_datetime(d.fecha_inicio, errors="coerce")),
        "avance_infobras_vs_mef_difiere_30_puntos": obra & ((n("avance_fisico_pct") - n("avance_fisico_mef_pct")).abs() > 30),
        "devengado_mayor_al_costo": obra & (n("pct_devengado_del_costo") > 100),
        "ejecucion_mayor_al_expediente": obra & (n("monto_ejecucion_acumulado") > n("monto_expediente_tecnico") * 1.0001),
        "causa_sin_clasificar": obra & d.causa_grupo_final.isin(["otros", "sin_informacion"]),
        "costo_cero_o_vacio_en_fuente": corte & ~(n("costo_actualizado_soles") > 0),
        "saldo_mayor_que_costo": corte & (n("saldo_inversion_soles") > n("costo_actualizado_soles") * 1.0001),
        "sin_provincia_ni_distrito": corte & (d.provincia.isna() | d.distrito.isna()),
    }
    texto = pd.Series("", index=d.index)
    for nombre, marca in reglas.items():
        marca = marca.fillna(False)
        texto = texto.where(~marca, texto + np.where(texto == "", "", "|") + nombre)
    return texto.replace("", np.nan)


def construir(m, resumen, serie_pais, serie_dep, causales, region_fn):
    """Une las tablas del producto final en el CSV unico. m = obras_maestra, resumen = resumen_region, serie_* y causales = tablas finales."""
    ctx = _contexto_region(resumen)
    cols_region = [c for c in ctx.columns if c.startswith("region_") and c != "region_enla"]
    partes = []

    # 1) obras de INFOBRAS (una fila por obra)
    a = m.copy()
    a["tipo_registro"] = "obra_infobras"
    a["fecha_corte"] = _fecha_iso(a.fecha_lectura_infobras)
    a["fuente_dato"] = "INFOBRAS (lectura 11-nov-2026) + Contraloria + MEF + ESCALE"
    a["sector"] = "EDUCACION"
    a["causal_oficial"] = a.causal_oficial_contraloria
    a["n_obras"] = 1
    a["n_obras_paralizadas"] = a.paralizada_hoy.astype(int)
    a = a.merge(ctx.drop(columns="cod_departamento"), on="region_enla", how="left")
    partes.append(a)

    # 2) obras de Educacion en cada corte de la Contraloria (anexos Excel)
    oc = pd.read_csv(config.PROCESSED / "contraloria_obras_corte.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    oc = oc[oc.sector_agrupado.str.contains("EDUC")].copy()
    oc["tipo_registro"] = "obra_corte_contraloria"
    oc["fecha_corte"] = _fecha_iso(oc.corte)
    oc["fuente_dato"] = oc.fuente_excel
    oc["sector"] = "EDUCACION"
    oc["modalidad"] = oc.modalidad_ejecucion
    oc["evento_contraloria"] = oc.evento
    oc["causal_oficial"] = oc.causal_paralizacion
    oc["causa_grupo_final"] = [nlp_causas.clasificar_causa(c) for c in oc.causal_paralizacion]
    oc["causa_macro"] = oc.causa_grupo_final.map(nlp_causas.macro)
    oc["avance_fisico_pct"] = pd.to_numeric(oc.avance_fisico, errors="coerce")
    oc["region_enla"] = [region_fn(d, p) for d, p in zip(oc.departamento, oc.provincia)]
    for c in ("costo_actualizado_soles", "saldo_inversion_soles"):
        oc[c] = pd.to_numeric(oc[c], errors="coerce")
    oc["n_obras"] = 1
    oc["n_obras_paralizadas"] = 1
    oc = oc.merge(ctx.drop(columns="cod_departamento"), on="region_enla", how="left")
    partes.append(oc)

    # 3) Educacion por departamento y corte
    d = serie_dep.rename(columns={"obras_educacion": "n_obras", "costo_educacion_soles": "costo_actualizado_soles", "obras_educacion_estricto": "n_obras_estricto",
                                  "costo_educacion_estricto_soles": "costo_estricto_soles", "fuente_excel": "fuente_dato"}).copy()
    d["tipo_registro"] = "departamento_corte"
    d["fecha_corte"] = _fecha_iso(d.fecha_corte)
    d["sector"] = "EDUCACION"
    d["n_obras_paralizadas"] = d.n_obras
    partes.append(d)

    # 4) pais: total y Educacion
    p = serie_pais.copy()
    pt = pd.DataFrame({"tipo_registro": "pais_total_corte", "fecha_corte": _fecha_iso(p.fecha_corte), "sector": "TODOS", "n_obras": p.obras_pais,
                       "n_obras_paralizadas": p.obras_pais, "costo_actualizado_soles": p.costo_pais_soles, "saldo_inversion_soles": p.saldo_pais_soles,
                       "fuente_dato": p.fuente_pdf})
    pe = pd.DataFrame({"tipo_registro": "pais_educacion_corte", "fecha_corte": _fecha_iso(p.fecha_corte), "sector": "EDUCACION", "n_obras": p.obras_educacion,
                       "n_obras_paralizadas": p.obras_educacion, "costo_actualizado_soles": p.costo_educacion_pdf_soles, "pct_obras": p.pct_educacion_obras,
                       "educacion_corregida": p.educacion_corregida, "fuente_obras_educacion": p.fuente_obras_educacion, "fuente_dato": p.fuente_pdf})
    partes += [pt, pe]

    # 5) causales por corte
    c = causales.rename(columns={"causal": "causal_oficial", "causa_grupo": "causa_grupo_final", "obras_paralizadas": "n_obras_paralizadas",
                                 "fuente_pdf": "fuente_dato"}).copy()
    c["tipo_registro"] = "causal_corte"
    c["fecha_corte"] = _fecha_iso(c.fecha_corte)
    c["n_obras"] = c.n_obras_paralizadas
    c["sector"] = "TODOS"
    partes.append(c)

    # 6) contexto por region
    x = ctx.copy()
    x["tipo_registro"] = "contexto_region"
    x["fuente_dato"] = "ESCALE (Minedu) y ENLA (Minedu)"
    partes.append(x)

    partes = [t.reindex(columns=COLUMNAS) for t in partes]   # agrega de una vez las columnas que faltan (vacias)
    out = pd.concat(partes, ignore_index=True)
    out["anio"] = pd.to_datetime(out.fecha_corte, errors="coerce").dt.year.astype("Int64")
    out["alertas_calidad"] = alertas(out)
    for col in ("n_obras", "n_obras_paralizadas", "n_obras_estricto", "dias_paralizado", "dias_desde_paralizacion", "meses_paralizado", "n_tramos_paralizacion", "n_cortes_contraloria",
                "reingresos_contraloria", "escale_locales_publicos", "region_escale_locales_publicos"):
        out[col] = pd.to_numeric(out[col], errors="coerce").round().astype("Int64")
    # el departamento de las series viene como texto; todas las columnas de codigo quedan como texto
    out["tipo_registro"] = pd.Categorical(out.tipo_registro, categories=TIPOS, ordered=True)
    out = out.sort_values(["tipo_registro", "fecha_corte", "cod_departamento", "codigo_infobras", "causal_oficial"], na_position="last", kind="stable")
    out["tipo_registro"] = out.tipo_registro.astype(str)
    for col in ("paralizada_hoy", "en_ultimo_corte_contraloria", "educacion_corregida"):    # logicos como True/False (vacio si no aplica)
        out[col] = out[col].map({True: "True", False: "False", 1.0: "True", 0.0: "False", "True": "True", "False": "False"})
    for col in out.columns:
        if str(out[col].dtype).startswith("float"):
            out[col] = out[col].round(6 if col in ("latitud", "longitud", "latitud_mapa", "longitud_mapa") else 4)
    return out.reset_index(drop=True)


def validar(d, m, serie_pais, serie_dep):
    """Comprobaciones del CSV unico; devuelve lista de (comprobacion, valor, esperado, ok)."""
    c = []
    obra = d[d.tipo_registro == "obra_infobras"]
    c.append(("dashboard: tipos de registro presentes (7)", d.tipo_registro.nunique(), 7, d.tipo_registro.nunique() == 7))
    c.append(("dashboard: filas obra_infobras = obras_maestra", len(obra), len(m), len(obra) == len(m)))
    c.append(("dashboard: obras paralizadas hoy = obras_maestra", int(obra.n_obras_paralizadas.sum()), int(m.paralizada_hoy.sum()), int(obra.n_obras_paralizadas.sum()) == int(m.paralizada_hoy.sum())))
    c.append(("dashboard: codigo_infobras sin repetidos en obra_infobras", int(obra.codigo_infobras.duplicated().sum()), 0, not obra.codigo_infobras.duplicated().any()))
    oc = d[d.tipo_registro == "obra_corte_contraloria"]
    rep = int(oc.duplicated(["fecha_corte", "codigo_infobras"]).sum())
    c.append(("dashboard: obra_corte_contraloria sin obra repetida en un corte", rep, 0, rep == 0))
    # las obras de Educacion por corte (suma de obras) deben cuadrar con la serie por departamento en los 13 cortes de Excel
    por_corte = oc.groupby("fecha_corte").size()
    dep = d[d.tipo_registro == "departamento_corte"].groupby("fecha_corte").n_obras.sum()
    cuadran = int(sum(abs(int(por_corte.get(f, -1)) - int(dep.get(f, -2))) <= 2 for f in por_corte.index))
    c.append(("dashboard: obras por corte (Excel) = serie por departamento (tolerancia 2 obras)", cuadran, len(por_corte), cuadran == len(por_corte)))
    sp = d[d.tipo_registro == "pais_educacion_corte"].n_obras.sum()
    c.append(("dashboard: serie pais Educacion = tabla serie_pais_educacion", int(sp), int(serie_pais.obras_educacion.sum()), int(sp) == int(serie_pais.obras_educacion.sum())))
    lat, lon = obra.latitud.dropna(), obra.longitud.dropna()
    fuera = int(((lat > 0.1) | (lat < -18.6)).sum() + ((lon > -68.4) | (lon < -81.4)).sum())
    c.append(("dashboard: coordenadas dentro del Peru", fuera, 0, fuera == 0))
    c.append(("dashboard: obras con coordenadas (informativo: al menos 90 %)", int(obra.latitud.notna().sum()), len(obra), obra.latitud.notna().mean() >= 0.90))
    lat_m, lon_m = obra.latitud_mapa.dropna(), obra.longitud_mapa.dropna()
    fuera_m = int(((lat_m > 0.1) | (lat_m < -18.6)).sum() + ((lon_m > -68.4) | (lon_m < -81.4)).sum())
    c.append(("dashboard: coordenadas para el mapa dentro del Peru", fuera_m, 0, fuera_m == 0))
    c.append(("dashboard: obras con coordenadas para el mapa (INFOBRAS o aproximadas; informativo: al menos 90 %)", int(obra.latitud_mapa.notna().sum()),
              len(obra), obra.latitud_mapa.notna().mean() >= 0.90))
    cods = d.cod_departamento.dropna().astype(str)
    c.append(("dashboard: cod_departamento de 2 digitos", int(cods.str.fullmatch(r"\d{2}").sum()), len(cods), bool(cods.str.fullmatch(r"\d{2}").all())))
    ubi = obra.ubigeo.dropna().astype(str)
    c.append(("dashboard: ubigeo de 6 digitos", int(ubi.str.fullmatch(r"\d{6}").sum()), len(ubi), bool(ubi.str.fullmatch(r"\d{6}").all())))
    c.append(("dashboard: sin filas completamente repetidas", int(d.duplicated().sum()), 0, not d.duplicated().any()))
    return c


def escribir(d):
    config.FINAL.mkdir(parents=True, exist_ok=True)
    ruta = config.FINAL / "dashboard_obras.csv"
    d.to_csv(ruta, index=False, encoding="utf-8-sig", na_rep="", lineterminator="\n")
    diccionario = pd.DataFrame(DICCIONARIO, columns=["columna", "tipo", "descripcion", "unidad"])
    diccionario["rellena_en"] = [", ".join(sorted(d.loc[d[c].notna(), "tipo_registro"].unique())) for c in diccionario.columna]
    diccionario.to_csv(config.FINAL / "dashboard_obras_diccionario.csv", index=False, encoding="utf-8-sig", lineterminator="\n")
    utils.log(f"DASHBOARD: {ruta.name} con {len(d)} filas y {d.shape[1]} columnas; filas por tipo: {d.tipo_registro.value_counts().to_dict()}")
    return ruta

### `producto_final` — Producto final
Producto final: arma las tablas del tablero con todo lo que extrajo el proyecto y las une en UN SOLO CSV (dashboard.py -> data/final/dashboard_obras.csv).

In [ ]:
%%modulo producto_final
"""Producto final: arma las tablas del tablero con todo lo que extrajo el proyecto y las une en UN SOLO CSV (dashboard.py -> data/final/dashboard_obras.csv).

Las cinco tablas intermedias (y producto_final.xlsx, una hoja por tabla + diccionario) quedan en data/interim/componentes:
  obras_maestra          una fila por obra de Educacion/Cultura (429): INFOBRAS + ubicacion con ubigeo + Contraloria + MEF + ESCALE
  resumen_region         una fila por region ENLA (26): obras paralizadas, causas, Contraloria, ESCALE (distritos) y ENLA
  serie_pais_educacion   cifras oficiales por corte: pais y sector Educacion (con la cifra corregida donde el PDF estaba mal)
  serie_educacion_departamento  obras de Educacion por departamento y corte (anexos Excel de la Contraloria)
  causales_por_corte     causales oficiales por corte, con el grupo de causa del proyecto
Cada tabla se valida (docs/validacion_final.csv) y su diccionario esta en DICCIONARIO_FINAL (se agrega a docs/diccionario.csv).
Nada de aqui inventa datos: todo sale de las tablas de data/processed; los campos derivados estan descritos.
"""
import re
import unicodedata

import numpy as np
import pandas as pd

import config
import contraloria_anexo03
import dashboard
import descarga_mef
import escale_ampliado
import escale_obras
import nlp_causas
import texto_nlp
import utils

GRUPOS_CAUSA = list(config.CAUSAS_REGLAS) + ["otros", "sin_informacion"]
MACROS_CAUSA = sorted(set(config.CAUSAS_MACRO.values()))
ULTIMO_CORTE = "2026-06-30"   # se recalcula en tiempo de ejecucion; esta constante solo documenta el valor actual
FUENTE_FINAL = "Producto final (derivado de las tablas de data/processed)"


def leer(nombre, carpeta=None):
    return pd.read_csv((carpeta or config.PROCESSED) / f"{nombre}.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")


def num(serie):
    return pd.to_numeric(serie, errors="coerce")


def booleano(serie):
    return serie.astype(str).str.lower().isin(["true", "1", "si", "sí"])


def sin_tildes(texto):
    return unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper().strip()


def region_enla(departamento, provincia):
    """Region como la usa ENLA: Lima se parte en Lima Metropolitana (provincia de Lima) y Lima Provincias."""
    if departamento == "LIMA":
        return "LIMA METROPOLITANA" if provincia == "LIMA" else "LIMA PROVINCIAS"
    return departamento


# ======================================================================================================
# Diccionario de las tablas finales: (columna, tipo, descripcion, unidad)
# ======================================================================================================
GRADOS_ENLA_EXTRA = [("2p", "2.° primaria"), ("6p", "6.° primaria")]   # ademas de 4.° primaria y 2.° secundaria (5.° secundaria: SICRECE no devuelve datos)
DICCIONARIO_ENLA_EXTRA = [fila for suf, gr in GRADOS_ENLA_EXTRA for fila in (
    (f"enla_{suf}_anio", "entero", f"Último año de la ENLA de {gr} disponible para la región.", "año"),
    (f"enla_{suf}_lectura_satisfactorio", "decimal", f"% en nivel Satisfactorio en Lectura, {gr}.", "%"),
    (f"enla_{suf}_matematica_satisfactorio", "decimal", f"% en nivel Satisfactorio en Matemática, {gr}.", "%"))]
DICCIONARIO_ANEXO03 = [(c, t, d + " Vacío = la obra no figura en el anexo 03 (corte mar-2026).", u) for c, t, d, u in contraloria_anexo03.DICCIONARIO
                       if c not in ("codigo_infobras", "corte_anexo03")]
DICCIONARIO_ESCALE_AMPLIADO = [("escale_" + c, t, d + " (distrito, Censo 2025)", u) for c, t, d, u in escale_ampliado.DICCIONARIO
                               if c not in ("anio", "ubigeo", "n_locales_publicos_ampliado")]
DICCIONARIO_FINAL = {
    "dashboard_obras": dashboard.DICCIONARIO,
    "escale_distrito_ampliado": escale_ampliado.DICCIONARIO,
    "escale_coordenadas_apoyo": [
        ("tipo", "texto", "local_obra (coordenada del local de ESCALE 2025 que declara el CUI de una obra) o centro_distrito (mediana de las coordenadas de los colegios del distrito, Padrón 2025).", "categoría"),
        ("clave", "texto", "CUI de la obra (si tipo = local_obra) o ubigeo de 6 dígitos del distrito (si tipo = centro_distrito).", "código"),
        ("lat", "decimal", "Latitud (grados decimales).", "grados"),
        ("lon", "decimal", "Longitud (grados decimales).", "grados"),
    ],
    "contraloria_servicios_control_obra": contraloria_anexo03.DICCIONARIO,
    "obras_maestra": [
        ("codigo_infobras", "texto", "Código de la obra en INFOBRAS (llave de la tabla).", "código"),
        ("cui", "texto", "Código Único de Inversiones.", "código"),
        ("nombre_obra", "texto", "Nombre de la obra.", "texto"),
        ("entidad", "texto", "Entidad pública responsable de la obra.", "texto"),
        ("origen", "texto", "Por qué la obra está en la tabla: infobras_paralizada_2026-10-27 (paralizada en INFOBRAS el 27-oct) o contraloria_educacion_2026-06 (Educación/Cultura en la lista de la Contraloría de jun-2026 y no paralizada en INFOBRAS el 27-oct).", "categoría"),
        ("cod_departamento", "texto de 2 dígitos", "Código INEI del departamento del ubigeo.", "código"),
        ("departamento", "texto", "Departamento del ubigeo, en mayúsculas y sin tildes (corregido: 2 obras que INFOBRAS publica en Huancavelica son de Ica).", "texto"),
        ("provincia", "texto", "Provincia como la publica INFOBRAS (mayúsculas, sin tildes). Para el nombre oficial usar el ubigeo.", "texto"),
        ("distrito", "texto", "Distrito como lo publica INFOBRAS (mayúsculas, sin tildes). Para el nombre oficial usar el ubigeo.", "texto"),
        ("ubigeo", "texto de 6 dígitos", "Ubigeo INEI del distrito de la obra (asignado por nombre contra ESCALE; 4 obras a mano).", "código"),
        ("ubigeo_metodo", "texto", "Cómo se asignó el ubigeo: exacto, normalizado o manual.", "categoría"),
        ("region_enla", "texto", "Región ENLA de la obra (Lima se parte en LIMA METROPOLITANA, la provincia de Lima, y LIMA PROVINCIAS). Llave para unir con resumen_region.", "texto"),
        ("latitud", "decimal", "Latitud de la obra según INFOBRAS (vacía en las obras sin coordenadas).", "grados"),
        ("longitud", "decimal", "Longitud de la obra según INFOBRAS.", "grados"),
        ("latitud_mapa", "decimal", "Latitud para el mapa: la de INFOBRAS; si INFOBRAS no la tiene, la del local de ESCALE que declara el CUI de la obra o, si no, el centro de los colegios del distrito (Padrón ESCALE 2025). Ver coordenada_origen.", "grados"),
        ("longitud_mapa", "decimal", "Longitud para el mapa (mismo criterio que latitud_mapa).", "grados"),
        ("coordenada_origen", "texto", "De dónde sale latitud_mapa/longitud_mapa: infobras (exacta), escale_local_obra (aproximada: local escolar que declara el CUI de la obra), centro_distrito_escale (aproximada: mediana de los colegios del distrito; en las obras que sí tienen coordenadas queda a ~3 km de mediana de la real) o vacío.", "categoría"),
        ("estado_infobras", "texto", "Estado de la obra en su ficha de INFOBRAS el día de la lectura (PARALIZADA, EN EJECUCIÓN, FINALIZADO, SIN EJECUCIÓN).", "categoría"),
        ("paralizada_hoy", "lógico", "TRUE si estado_infobras es PARALIZADA.", "lógico"),
        ("fecha_lectura_infobras", "fecha AAAA-MM-DD", "Día en que se leyó la ficha (2026-11-11).", "fecha"),
        ("modalidad", "texto", "Modalidad de ejecución según INFOBRAS.", "texto"),
        ("naturaleza", "texto", "Naturaleza de la obra según INFOBRAS.", "texto"),
        ("tipo_obra", "texto", "Tipo de obra según INFOBRAS.", "texto"),
        ("tipo_infraestructura", "texto", "Tipo de infraestructura según el nombre de la obra (reglas; si el nombre no lo dice, el tipo de obra de INFOBRAS): educacion_basica (colegios, inicial, CEBA, CETPRO), educacion_superior (universidades e institutos), deportiva_recreativa, cultural_patrimonio u otra. INFOBRAS agrupa Educación y Cultura.", "categoría"),
        ("monto_expediente_tecnico", "decimal", "Monto del expediente técnico (listado de INFOBRAS).", "soles"),
        ("monto_ejecucion_acumulado", "decimal", "Monto de ejecución acumulado según INFOBRAS.", "soles"),
        ("avance_fisico_pct", "decimal", "Avance físico del encabezado de la ficha de INFOBRAS.", "%"),
        ("fecha_inicio", "fecha AAAA-MM-DD", "Fecha de inicio de la obra.", "fecha"),
        ("fecha_fin_prevista", "fecha AAAA-MM-DD", "Fecha de fin prevista de la obra.", "fecha"),
        ("n_avances_infobras", "entero", "Número de avances mensuales registrados en INFOBRAS.", "avances"),
        ("meses_paralizado", "entero", "Meses en estado Paralizado en la tabla de avances de INFOBRAS.", "meses"),
        ("n_tramos_paralizacion", "entero", "Número de tramos seguidos de meses paralizados (más de uno = se reactivó y volvió a paralizar).", "tramos"),
        ("primer_mes_paralizado", "texto AAAA-MM", "Primer mes en estado Paralizado.", "mes"),
        ("ultimo_mes_paralizado", "texto AAAA-MM", "Último mes en estado Paralizado.", "mes"),
        ("fecha_paralizacion", "fecha AAAA-MM-DD", "Fecha de paralización del último avance paralizado.", "fecha"),
        ("dias_paralizado", "entero", "Valor «días paralizado» del detalle del último avance paralizado de INFOBRAS. OJO: en casi todas las obras vale 30 o 31 (los días de ese mes del informe), no el tiempo total; para el tiempo total usar dias_desde_paralizacion.", "días"),
        ("dias_desde_paralizacion", "entero", "Días entre fecha_paralizacion y la lectura de INFOBRAS (11-nov-2026), solo en las obras paralizadas hoy: cuánto tiempo lleva detenida la obra en su paralización actual.", "días"),
        ("causa_grupo_infobras", "texto", "Grupo de causa de los comentarios de INFOBRAS de la obra (todos los meses paralizados leídos; los del último tramo pesan más). sin_informacion si no hay comentario útil.", "categoría"),
        ("causal_oficial_contraloria", "texto", "Causal de paralización de la Contraloría en el último corte en que aparece la obra (puede ser antigua).", "texto"),
        ("causa_grupo_contraloria", "texto", "Grupo de causa asignado a la causal oficial con las mismas reglas (la causal «OTROS» queda en otros).", "categoría"),
        ("causa_grupo_mef", "texto", "Grupo de causa asignado al texto de la última problemática registrada en el MEF (mef_ult_problema), con las mismas reglas.", "categoría"),
        ("causa_grupo_final", "texto", "Causa principal de la obra (NLP): se suman los puntajes de las tres fuentes con pesos (causal oficial 3, comentarios del último tramo 2, problemática del MEF 1.5, comentarios de tramos anteriores 0.5). Los grupos de estado (obra_concluida_o_en_marcha, sin_restricciones) solo se asignan si no hay ninguna causa real. otros = hay texto sin causa reconocible o la causal oficial es «OTROS»; sin_informacion = no hay texto.", "categoría"),
        ("causa_macro", "texto", "Agrupación de causa_grupo_final en 8 categorías para el tablero: financiera, contractual_y_legal, tecnica, logistica, entorno_social_y_territorial, fuerza_mayor, sin_causa_de_paralizacion, sin_clasificar.", "categoría"),
        ("causas_secundarias", "texto", "Hasta 2 causas más que también aparecen en los textos (con al menos la mitad del puntaje de la principal), separadas por «|». Vacío si no hay.", "categoría"),
        ("causa_fuente", "texto", "Fuente que más aportó a la causa principal: infobras_comentario, contraloria_causal, mef_problema, texto_libre (modelo) o sin_causa_clara.", "categoría"),
        ("causa_confianza", "texto", "alta = al menos dos fuentes distintas apuntan a la misma causa; media = una sola fuente; baja = solo la sugirió el modelo; vacío = sin causa.", "categoría"),
        ("causa_metodo", "texto", "reglas = palabras clave; modelo = TF-IDF + regresión logística (solo cuando las reglas no encontraron nada y la probabilidad es ≥ 0.60).", "categoría"),
        ("en_ultimo_corte_contraloria", "lógico", "TRUE si la obra está en la lista de la Contraloría del último corte (jun-2026).", "lógico"),
        ("primer_corte_contraloria", "fecha AAAA-MM-DD", "Primer corte (entre los 13 con base obra por obra, sep-2023 a jun-2026) en que aparece la obra.", "fecha"),
        ("ultimo_corte_contraloria", "fecha AAAA-MM-DD", "Último corte en que aparece la obra.", "fecha"),
        ("n_cortes_contraloria", "entero", "Cantidad de cortes (de 13) en que aparece la obra.", "cortes"),
        ("reingresos_contraloria", "entero", "Veces que la obra salió de la lista y volvió a entrar.", "veces"),
        ("primer_corte_censurado", "lógico", "TRUE si la obra ya estaba en el primer corte (sep-2023): su fecha de entrada real es anterior y no se conoce.", "lógico"),
        ("nivel_gobierno", "texto", "Nivel de gobierno de la entidad según la Contraloría (último corte).", "categoría"),
        ("sector_agrupado", "texto", "Sector agrupado según la Contraloría (último corte).", "categoría"),
        ("evento_contraloria", "texto", "Evento registrado en INFOBRAS según la Contraloría (último corte).", "categoría"),
        ("costo_actualizado_soles", "decimal", "Costo actualizado de la inversión (Invierte.pe) en el último corte.", "soles"),
        ("saldo_inversion_soles", "decimal", "Saldo de inversión por ejecutar en el último corte.", "soles"),
        ("estado_mef", "texto", "Estado de la inversión en el Formato 12-B del MEF.", "categoría"),
        ("ultimo_devengado", "texto AAAAMM", "Periodo del último devengado de la inversión registrado en el SIAF (MEF).", "periodo"),
        ("devengado_total_soles", "decimal", "Devengado acumulado hasta el año anterior más el del año actual (MEF).", "soles"),
        ("devengado_anio_actual_soles", "decimal", "Devengado del año actual (MEF).", "soles"),
        ("pim_anio_actual_soles", "decimal", "Presupuesto Institucional Modificado del año actual (MEF).", "soles"),
        ("pct_devengado_del_costo", "decimal", "devengado_total_soles / costo_actualizado_soles. Puede pasar de 100 si los costos no son comparables.", "%"),
        ("avance_fisico_mef_pct", "decimal", "Avance físico declarado en el Formato 12-B (MEF).", "%"),
        ("mef_ult_problema", "texto", "Última problemática registrada en el Formato 12-B (texto libre de la entidad).", "texto"),
        ("escale_anio", "entero", "Año del Censo Educativo usado para los indicadores del distrito.", "año"),
        ("escale_locales_publicos", "entero", "Locales educativos públicos del distrito (ESCALE).", "locales"),
        ("escale_pct_3_servicios", "decimal", "% de locales públicos del distrito con agua, desagüe y electricidad de red pública.", "%"),
        ("escale_pct_rural", "decimal", "% de locales públicos del distrito en zona rural.", "%"),
        ("escale_pct_riesgo_estructural", "decimal", "% de locales con al menos una edificación en riesgo estructural.", "%"),
        ("escale_pct_estructura_mal_estado", "decimal", "% de locales con estructura en mal estado.", "%"),
        ("escale_n_locales_obra", "entero", "Locales que declaran en el Censo 2025 estar en construcción con el CUI de esta obra (0 = ninguno).", "locales"),
        ("escale_codlocal", "texto", "Código del primer local de ESCALE que declara el CUI de esta obra.", "código"),
        *DICCIONARIO_ESCALE_AMPLIADO,
        *DICCIONARIO_ANEXO03,
    ],
    "resumen_region": [
        ("region_enla", "texto", "Región ENLA (26: los 24 departamentos sin Lima + Callao + LIMA METROPOLITANA + LIMA PROVINCIAS). Llave.", "texto"),
        ("cod_departamento", "texto de 2 dígitos", "Código INEI del departamento (Lima Metropolitana y Provincias = 15).", "código"),
        ("obras_en_tabla", "entero", "Obras de obras_maestra en la región.", "obras"),
        ("obras_paralizadas_hoy", "entero", "Obras con estado PARALIZADA en INFOBRAS el 2026-11-11.", "obras"),
        ("monto_expediente_paralizadas", "decimal", "Suma del expediente técnico de las obras paralizadas hoy.", "soles"),
        ("monto_ejecutado_paralizadas", "decimal", "Suma del monto de ejecución acumulado de las obras paralizadas hoy.", "soles"),
        ("dias_paralizado_mediana", "decimal", "Mediana de días paralizada de las obras paralizadas hoy.", "días"),
        ("avance_fisico_mediano", "decimal", "Mediana del avance físico de las obras paralizadas hoy.", "%"),
        *[(f"paralizadas_causa_{g}", "entero", f"Obras paralizadas hoy cuyo causa_grupo_final es {g}.", "obras") for g in GRUPOS_CAUSA],
        ("contraloria_obras_educacion", "entero", "Obras del sector Educación en la lista de la Contraloría del último corte.", "obras"),
        ("contraloria_costo_educacion_soles", "decimal", "Costo actualizado de esas obras.", "soles"),
        ("escale_anio", "entero", "Año del Censo Educativo usado (el más reciente).", "año"),
        ("escale_locales_publicos", "entero", "Locales educativos públicos de la región.", "locales"),
        ("escale_pct_3_servicios", "decimal", "% de locales con agua, desagüe y electricidad de red pública (ponderado por locales con dato).", "%"),
        ("escale_pct_rural", "decimal", "% de locales rurales (ponderado por locales).", "%"),
        ("escale_pct_riesgo_estructural", "decimal", "% de locales con riesgo estructural (ponderado por locales con dato de edificaciones).", "%"),
        ("escale_pct_estructura_mal_estado", "decimal", "% de locales con estructura en mal estado (ponderado igual).", "%"),
        ("enla_4p_anio", "entero", "Último año de la ENLA de 4.° primaria disponible para la región.", "año"),
        ("enla_4p_lectura_satisfactorio", "decimal", "% en nivel Satisfactorio en Lectura, 4.° primaria.", "%"),
        ("enla_4p_matematica_satisfactorio", "decimal", "% en nivel Satisfactorio en Matemática, 4.° primaria.", "%"),
        ("enla_2s_anio", "entero", "Último año de la ENLA de 2.° secundaria disponible para la región.", "año"),
        ("enla_2s_lectura_satisfactorio", "decimal", "% en nivel Satisfactorio en Lectura, 2.° secundaria.", "%"),
        ("enla_2s_matematica_satisfactorio", "decimal", "% en nivel Satisfactorio en Matemática, 2.° secundaria.", "%"),
        *DICCIONARIO_ENLA_EXTRA,
    ],
    "serie_pais_educacion": [
        ("fecha_corte", "fecha AAAA-MM-DD", "Fecha de corte del informe de la Contraloría.", "fecha"),
        ("anio", "entero", "Año de la fecha de corte.", "año"),
        ("obras_pais", "entero", "Obras paralizadas en el país.", "obras"),
        ("costo_pais_soles", "decimal", "Costo actualizado de las obras paralizadas del país.", "soles"),
        ("saldo_pais_soles", "decimal", "Saldo de inversión por ejecutar del país.", "soles"),
        ("obras_educacion_pdf", "entero", "Obras del sector Educación según la tabla de sectores del PDF.", "obras"),
        ("costo_educacion_pdf_soles", "decimal", "Costo actualizado del sector Educación según el PDF.", "soles"),
        ("obras_educacion", "entero", "Obras del sector Educación a usar: la del anexo Excel donde el PDF está mal (set-2023 y set-2025) y la del PDF en el resto.", "obras"),
        ("educacion_corregida", "lógico", "TRUE si obras_educacion difiere de la cifra del PDF porque el PDF está mal (set-2023 y set-2025).", "lógico"),
        ("fuente_obras_educacion", "texto", "De dónde sale obras_educacion: el anexo Excel de la Contraloría o el informe PDF (cuando no hay Excel para contrastar).", "texto"),
        ("pct_educacion_obras", "decimal", "obras_educacion / obras_pais.", "%"),
        ("fuente_pdf", "texto", "PDF del informe.", "texto"),
    ],
    "serie_educacion_departamento": [
        ("fecha_corte", "fecha AAAA-MM-DD", "Fecha de corte del informe.", "fecha"),
        ("anio", "entero", "Año de la fecha de corte.", "año"),
        ("cod_departamento", "texto de 2 dígitos", "Código INEI del departamento.", "código"),
        ("departamento", "texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        ("obras_educacion", "entero", "Obras paralizadas del sector Educación (agrupación del informe).", "obras"),
        ("costo_educacion_soles", "decimal", "Costo actualizado de esas obras.", "soles"),
        ("obras_educacion_estricto", "entero", "Obras cuyo sector es exactamente EDUCACION (sin Cultura).", "obras"),
        ("costo_educacion_estricto_soles", "decimal", "Costo actualizado de esas obras.", "soles"),
        ("fuente_excel", "texto", "Anexo Excel de la Contraloría del que se calcula.", "texto"),
    ],
    "causales_por_corte": [
        ("fecha_corte", "fecha AAAA-MM-DD", "Fecha de corte del informe.", "fecha"),
        ("anio", "entero", "Año de la fecha de corte.", "año"),
        ("causal", "texto", "Causal de paralización declarada por la entidad (el catálogo cambió entre informes).", "texto"),
        ("causa_grupo", "texto", "Grupo de causa del proyecto asignado a esa causal (mismas reglas que las obras).", "categoría"),
        ("causa_macro", "texto", "Macro-categoría del grupo de causa (financiera, contractual_y_legal, tecnica, logistica, entorno_social_y_territorial, fuerza_mayor, sin_causa_de_paralizacion, sin_clasificar).", "categoría"),
        ("obras_paralizadas", "entero", "Obras con esa causal.", "obras"),
        ("pct_obras", "decimal", "Porcentaje de las obras del corte, como lo publica el informe.", "%"),
        ("costo_actualizado_soles", "decimal", "Costo actualizado de esas obras.", "soles"),
        ("saldo_inversion_soles", "decimal", "Saldo de inversión de esas obras.", "soles"),
        ("fuente_pdf", "texto", "PDF del informe.", "texto"),
        ("pagina", "entero", "Página del PDF.", "página"),
    ],
}


# ======================================================================================================
# Construccion
# ======================================================================================================
# ---- tipo de infraestructura (INFOBRAS mezcla colegios, universidades, estadios y sitios arqueologicos en "Educacion/Cultura") ----
def _sin_tildes(t): return unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode().upper()
REGLAS_INFRAESTRUCTURA = [
    ("educacion_superior", r"UNIVERSID|UNIVERSITARI|\bFACULTAD|\bUNI\b|\bUNMSM\b|\bUNFV\b|\bUNSAAC\b|EDUCACION SUPERIOR|\bIEST?P\b|\bISTP?\b|INSTITUTO (DE EDUCACION SUPERIOR|SUPERIOR|PEDAGOGICO|TECNOLOGICO)|PEDAGOGICA|CEPREUNI|SENCICO|ESCUELAS? (DE FORMACION )?PROFESIONAL|ESCUELA SUPERIOR|CARRERA PROFESIONAL|\bI\.?\s?E\.?\s?S\.?\s?T\.?\s?P"),
    ("educacion_basica", r"INSTITUCI\w* EDUCATIV|\bI\.\s?E\b\.?|\bI\.E\.[IPS]\b|\bIE[IPS]?\b|\bI\.?E\.?\s?N[°ºRO.]*\s*\d|IEGECOM|\bCEI\b|\bPRONOEI|COLEGIO|CENTRO EDUCATIVO|LOCAL ESCOLAR|CODIGO (DE )?LOCAL|ESCUELA|JARDIN (DE NINOS|INFANTIL|N)|\bCUNA|\bCEBA\b|\bCETPRO|INFRAESTRUCTURA EDUCATIVA|SERVICIOS? EDUCATIVO|SERVICIO DE EDUCACION (INICIAL|PRIMARIA|SECUNDARIA|BASICA)|EDUCACION (INICIAL|PRIMARIA|SECUNDARIA|BASICA)|\bINICIAL\b|\bPRIMARIA\b|\bSECUNDARIA\b|AULAS? (PEDAGOGIC|EDUCATIV)"),
    ("deportiva_recreativa", r"DEPORT|ESTADIO|COLISEO|\bLOSA|\bLOZA|POLIDEPORTIV|GRAS+ SINTETIC|PISCINA|RECREA|PARQUE ZONAL|PARQUE RECREATIV|PLAZA DE TOROS|CANCHA|CAMPO MENOR|AREAS VERDES|ESPACIOS? PUBLICOS?"),
    ("cultural_patrimonio", r"CULTUR|MUSEO|BIBLIOTECA|ARQUEOLOG|MONUMENT|\bHUACA|PREHISPANIC|SITIO HISTORICO|ANFITEATRO|TEATRO|INTERPRETACION|PATRIMONIO|PUESTA EN VALOR"),
]
INFRAESTRUCTURA_POR_TIPO_OBRA = {"CENTRO EDUCATIVO/COLEGIO": "educacion_basica", "AULAS": "educacion_basica", "CUNA/JARDIN": "educacion_basica",
                 "INSTITUTO": "educacion_superior", "ESCUELA PROFESIONAL": "educacion_superior", "COMPLEJO DEPORTIVO": "deportiva_recreativa",
                 "ESTADIO": "deportiva_recreativa", "LOSA DEPORTIVA": "deportiva_recreativa", "COLISEO": "deportiva_recreativa",
                 "POLIDEPORTIVO": "deportiva_recreativa", "COMPLEJO RECREACIONAL": "deportiva_recreativa", "CASA Y CENTRO DE CULTURA": "cultural_patrimonio"}
def tipo_infraestructura(nombre, tipo_obra=""):
    """Clasifica la obra por su nombre (reglas en orden; la primera que coincide gana) y, si el nombre no dice nada, por el tipo de obra de
    INFOBRAS. INFOBRAS agrupa Educacion y Cultura: aqui se separa lo escolar de lo universitario, deportivo y cultural."""
    t = _sin_tildes(nombre)
    for tipo, patron in REGLAS_INFRAESTRUCTURA:
        if re.search(patron, t):
            return tipo
    return INFRAESTRUCTURA_POR_TIPO_OBRA.get(_sin_tildes(tipo_obra).strip(), "otra")


def coordenadas_mapa(m, con_cui):
    """Agrega latitud_mapa, longitud_mapa y coordenada_origen. No toca latitud/longitud (son las de INFOBRAS).

    Orden: 1) INFOBRAS; 2) local(es) de ESCALE 2025 que declaran el CUI de la obra (mediana de sus coordenadas, loc_lineal_1);
    3) centro del distrito = mediana de las coordenadas de los colegios del distrito (Padron 2025). Lee data/raw sin modificarlo.
    """
    import extraccion
    dentro = lambda t: t[t.lat.between(-18.6, 0.1) & t.lon.between(-81.4, -68.4)]
    la, lo = num(m.latitud), num(m.longitud)
    origen = pd.Series(np.where(la.notna() & lo.notna(), "infobras", ""), index=m.index)
    try:
        loc = utils.leer_dbf(extraccion.dbf_de_zip("escale", 2025, config.ESCALE_ZIP_LINEAL), columnas=["CODLOCAL", "LATITUD", "LONGITUD"])
        loc = dentro(loc.assign(lat=num(loc.LATITUD), lon=num(loc.LONGITUD)))
        loc = con_cui.merge(loc, left_on="codlocal", right_on="CODLOCAL").groupby("cui")[["lat", "lon"]].median()
        pad = utils.leer_dbf(extraccion.dbf_de_zip("escale", 2025, config.ESCALE_ZIP_PADRON), columnas=["CODGEO", "NLAT_IE", "NLONG_IE"])
        cen = dentro(pad.assign(lat=num(pad.NLAT_IE), lon=num(pad.NLONG_IE))).groupby("CODGEO")[["lat", "lon"]].median()
        apoyo = pd.concat([loc.reset_index().rename(columns={"cui": "clave"}).assign(tipo="local_obra"),
                           cen.reset_index().rename(columns={"CODGEO": "clave"}).assign(tipo="centro_distrito")])[["tipo", "clave", "lat", "lon"]]
        apoyo[["lat", "lon"]] = apoyo[["lat", "lon"]].round(6)
        apoyo.to_csv(config.PROCESSED / "escale_coordenadas_apoyo.csv", index=False, encoding="utf-8-sig")   # se reusa si no hay datos crudos
    except (FileNotFoundError, KeyError, ValueError) as e:
        ruta_apoyo = config.PROCESSED / "escale_coordenadas_apoyo.csv"
        if ruta_apoyo.exists():
            utils.log(f"COORDENADAS: no hay datos crudos de ESCALE ({type(e).__name__}); se usa escale_coordenadas_apoyo.csv ya calculado")
            apoyo = pd.read_csv(ruta_apoyo, dtype={"tipo": str, "clave": str}, encoding="utf-8-sig")
            loc = apoyo[apoyo.tipo == "local_obra"].set_index("clave")[["lat", "lon"]]
            cen = apoyo[apoyo.tipo == "centro_distrito"].set_index("clave")[["lat", "lon"]]
        else:
            utils.log(f"AVISO COORDENADAS: no se pudo leer ESCALE ({type(e).__name__}: {e}); el mapa usa solo las coordenadas de INFOBRAS")
            loc = cen = pd.DataFrame(columns=["lat", "lon"])
    for nombre, tabla, clave in (("escale_local_obra", loc, "cui"), ("centro_distrito_escale", cen, "ubigeo")):
        la2, lo2 = m[clave].map(tabla.lat), m[clave].map(tabla.lon)
        usar = (la.isna() | lo.isna()) & la2.notna() & lo2.notna()
        la, lo, origen = la.where(~usar, la2), lo.where(~usar, lo2), origen.where(~usar, nombre)
    m["latitud_mapa"], m["longitud_mapa"] = la.round(6), lo.round(6)
    m["coordenada_origen"] = origen.replace("", np.nan)
    utils.log(f"COORDENADAS: {origen.replace('', 'sin_coordenadas').value_counts().to_dict()}")
    return m


def tabla_maestra():
    fi = leer("infobras_fichas_ampliadas")
    pan = leer("contraloria_obras_panel")
    oc = leer("contraloria_obras_corte")
    par = leer("infobras_paralizaciones")
    f198 = leer("fichas_infobras")
    mef = leer("mef_f12b_obras") if (config.PROCESSED / "mef_f12b_obras.csv").exists() else pd.DataFrame(columns=["cui"])
    esc = leer("escale_infraestructura")
    eobra = leer("escale_obras_en_local") if (config.PROCESSED / "escale_obras_en_local.csv").exists() else pd.DataFrame(columns=["cui", "codlocal"])

    m = fi.rename(columns={"estado": "estado_infobras", "fecha_lectura": "fecha_lectura_infobras", "n_avances": "n_avances_infobras",
                           "n_paralizados": "meses_paralizado"}).copy()
    m["paralizada_hoy"] = m.estado_infobras.str.upper() == "PARALIZADA"
    m["region_enla"] = [region_enla(d, p) for d, p in zip(m.departamento, m.provincia)]

    # tramos de paralizacion (de la tabla de avances paralizados de INFOBRAS)
    par = par.assign(mes_txt=par.anio + "-" + par.mes.str.zfill(2))
    tramos = par.groupby("codigo_infobras").agg(n_tramos_paralizacion=("tramo", "nunique"), primer_mes_paralizado=("mes_txt", "min"),
                                                ultimo_mes_paralizado=("mes_txt", "max"))
    m = m.join(tramos, on="codigo_infobras")

    # Contraloria: panel (cuantos cortes, reingresos) + ultimo registro de cada obra
    pan = pan.rename(columns={"primer_corte": "primer_corte_contraloria", "ultimo_corte": "ultimo_corte_contraloria",
                              "n_cortes": "n_cortes_contraloria", "reingresos": "reingresos_contraloria",
                              "sigue_en_ultimo_corte": "en_ultimo_corte_contraloria", "causal_paralizacion_ultimo": "causal_oficial_contraloria",
                              "nivel_gobierno_ultimo": "nivel_gobierno", "sector_agrupado_ultimo": "sector_agrupado"})
    m = m.merge(pan[["codigo_infobras", "primer_corte_contraloria", "ultimo_corte_contraloria", "n_cortes_contraloria", "reingresos_contraloria",
                     "en_ultimo_corte_contraloria", "primer_corte_censurado", "causal_oficial_contraloria", "nivel_gobierno", "sector_agrupado"]],
                on="codigo_infobras", how="left")
    ultimo = oc.sort_values("corte").groupby("codigo_infobras").tail(1)[["codigo_infobras", "evento", "costo_actualizado_soles", "saldo_inversion_soles"]]
    m = m.merge(ultimo.rename(columns={"evento": "evento_contraloria"}), on="codigo_infobras", how="left")


    # MEF (devengado por CUI)
    columnas_mef = ["cui", "estado_mef", "ultimo_devengado", "devengado_total_soles", "devengado_anio_actual_soles", "pim_anio_actual_soles",
                    "avance_fisico_mef_pct", "ult_problema"]
    for c in columnas_mef:
        if c not in mef:
            mef[c] = ""
    m = m.merge(mef[columnas_mef].rename(columns={"ult_problema": "mef_ult_problema"}), on="cui", how="left")
    # causas: NLP sobre los tres textos (comentarios de INFOBRAS de todas las obras, causal oficial y problematica del MEF)
    m, metricas_nlp = nlp_causas.clasificar_maestra(m, par, mef)
    metricas_nlp.to_csv(config.DOCS / "nlp_causas_metricas.csv", index=False, encoding="utf-8-sig")
    costo = num(m.costo_actualizado_soles)
    m["pct_devengado_del_costo"] = np.where(costo > 0, (100 * num(m.devengado_total_soles) / costo).round(2), np.nan)

    # ESCALE: indicadores del distrito (anio mas reciente de cada distrito) y local que declara el CUI
    esc = esc.assign(anio_n=num(esc.anio)).sort_values("anio_n").drop_duplicates("ubigeo", keep="last")
    esc = esc.rename(columns={"anio": "escale_anio", "n_locales_publicos": "escale_locales_publicos", "pct_3_servicios": "escale_pct_3_servicios",
                              "pct_rural": "escale_pct_rural", "pct_riesgo_estructural": "escale_pct_riesgo_estructural",
                              "pct_estructura_mal_estado": "escale_pct_estructura_mal_estado"})
    m = m.merge(esc[["ubigeo", "escale_anio", "escale_locales_publicos", "escale_pct_3_servicios", "escale_pct_rural",
                     "escale_pct_riesgo_estructural", "escale_pct_estructura_mal_estado"]], on="ubigeo", how="left")
    amp = leer("escale_distrito_ampliado").drop(columns=["anio", "n_locales_publicos_ampliado"])   # indicadores extra de ESCALE 2025 por distrito
    m = m.merge(amp.rename(columns={c: "escale_" + c for c in amp.columns if c != "ubigeo"}), on="ubigeo", how="left")
    sc = leer("contraloria_servicios_control_obra").drop(columns="corte_anexo03")   # servicios de control por obra (anexo 03 de la Contraloria)
    m = m.merge(sc, on="codigo_infobras", how="left")
    con_cui = eobra[eobra.cui != ""]
    por_cui = con_cui.groupby("cui").agg(escale_n_locales_obra=("codlocal", "nunique"), escale_codlocal=("codlocal", "first"))
    m = m.join(por_cui, on="cui")
    m["escale_n_locales_obra"] = m.escale_n_locales_obra.fillna(0)
    m = coordenadas_mapa(m, con_cui)
    # Tiempo total paralizada: el campo "dias paralizado" de INFOBRAS es casi siempre 30/31 (dias del mes del informe), no el total
    lectura, desde = pd.to_datetime(m.fecha_lectura_infobras, errors="coerce"), pd.to_datetime(m.fecha_paralizacion, errors="coerce")
    m["dias_desde_paralizacion"] = (lectura - desde).dt.days.where(m.paralizada_hoy & desde.notna() & (desde <= lectura))
    m["tipo_infraestructura"] = [tipo_infraestructura(n, t) for n, t in zip(m.nombre_obra, m.tipo_obra)]
    utils.log(f"TIPO DE INFRAESTRUCTURA: {m.tipo_infraestructura.value_counts().to_dict()}")

    columnas = [c for c, *_ in DICCIONARIO_FINAL["obras_maestra"]]
    faltan = [c for c in columnas if c not in m]
    if faltan:
        raise KeyError(f"obras_maestra: faltan columnas {faltan}")
    m = m[columnas].copy()
    for c in ("latitud", "longitud", "latitud_mapa", "longitud_mapa", "monto_expediente_tecnico", "monto_ejecucion_acumulado", "avance_fisico_pct", "costo_actualizado_soles",
              "saldo_inversion_soles", "devengado_total_soles", "devengado_anio_actual_soles", "pim_anio_actual_soles", "avance_fisico_mef_pct",
              "escale_pct_3_servicios", "escale_pct_rural", "escale_pct_riesgo_estructural", "escale_pct_estructura_mal_estado"):
        m[c] = num(m[c])
    for c, *_ in DICCIONARIO_ESCALE_AMPLIADO + DICCIONARIO_ANEXO03:
        m[c] = num(m[c])
    for c in ("n_avances_infobras", "meses_paralizado", "n_tramos_paralizacion", "dias_paralizado", "dias_desde_paralizacion", "n_cortes_contraloria", "reingresos_contraloria",
              "escale_locales_publicos", "escale_anio", "escale_n_locales_obra"):
        m[c] = num(m[c]).round().astype("Int64")
    for c in ("en_ultimo_corte_contraloria", "primer_corte_censurado"):
        m[c] = booleano(m[c])
    return m.sort_values(["origen", "codigo_infobras"]).reset_index(drop=True)


def _ponderado(grupo, columna, peso):
    w = grupo[peso].astype(float)
    ok = grupo[columna].notna() & (w > 0)
    return round(float((grupo.loc[ok, columna] * w[ok]).sum() / w[ok].sum()), 2) if w[ok].sum() > 0 else np.nan


def resumen_region(m):
    niveles = leer("enla_niveles_logro")
    regiones = niveles[["region_clave", "cod_departamento"]].drop_duplicates().rename(columns={"region_clave": "region_enla"})
    out = regiones.copy().set_index("region_enla")
    g = m.groupby("region_enla")
    out["obras_en_tabla"] = g.size()
    hoy = m[m.paralizada_hoy]
    gh = hoy.groupby("region_enla")
    out["obras_paralizadas_hoy"] = gh.size()
    out["monto_expediente_paralizadas"] = gh.monto_expediente_tecnico.sum()
    out["monto_ejecutado_paralizadas"] = gh.monto_ejecucion_acumulado.sum()
    out["dias_paralizado_mediana"] = gh.dias_paralizado.apply(lambda s: s.astype(float).median())
    out["avance_fisico_mediano"] = gh.avance_fisico_pct.median()
    por_causa = hoy.groupby(["region_enla", "causa_grupo_final"]).size().unstack(fill_value=0)
    for grupo in GRUPOS_CAUSA:
        out[f"paralizadas_causa_{grupo}"] = por_causa[grupo] if grupo in por_causa else 0
    # Contraloria: Educacion en el ultimo corte (de la base obra por obra)
    oc = leer("contraloria_obras_corte")
    oc = oc[(oc.corte == oc.corte.max()) & (oc.sector_agrupado == "EDUCACION")].copy()
    oc["region_enla"] = [region_enla(d, p) for d, p in zip(oc.departamento, oc.provincia)]
    out["contraloria_obras_educacion"] = oc.groupby("region_enla").size()
    out["contraloria_costo_educacion_soles"] = oc.assign(c=num(oc.costo_actualizado_soles)).groupby("region_enla").c.sum()
    # ESCALE: distritos del anio mas reciente agregados a la region
    esc = leer("escale_infraestructura")
    esc = esc[esc.anio == esc.anio.max()].copy()
    for c in ("n_locales_publicos", "n_locales_sin_dato", "pct_3_servicios", "pct_rural", "n_locales_con_dato_edif", "pct_riesgo_estructural", "pct_estructura_mal_estado"):
        esc[c] = num(esc[c])
    esc["con_dato"] = esc.n_locales_publicos - esc.n_locales_sin_dato
    esc["region_enla"] = [region_enla(d, p) for d, p in zip(esc.departamento, esc.provincia)]
    ge = esc.groupby("region_enla")
    out["escale_anio"] = int(esc.anio.max())
    out["escale_locales_publicos"] = ge.n_locales_publicos.sum()
    out["escale_pct_3_servicios"] = ge.apply(lambda x: _ponderado(x, "pct_3_servicios", "con_dato"))
    out["escale_pct_rural"] = ge.apply(lambda x: _ponderado(x, "pct_rural", "n_locales_publicos"))
    out["escale_pct_riesgo_estructural"] = ge.apply(lambda x: _ponderado(x, "pct_riesgo_estructural", "n_locales_con_dato_edif"))
    out["escale_pct_estructura_mal_estado"] = ge.apply(lambda x: _ponderado(x, "pct_estructura_mal_estado", "n_locales_con_dato_edif"))
    # ENLA: ultimo anio de cada grado
    for sufijo, grado in (("2p", "2.° primaria"), ("4p", "4.° primaria"), ("6p", "6.° primaria"), ("2s", "2.° secundaria")):
        n = niveles[niveles.grado == grado].assign(anio_n=lambda d: num(d.anio))
        ultimo = n[n.anio_n == n.anio_n.max()]
        out[f"enla_{sufijo}_anio"] = int(n.anio_n.max())
        for area in ("lectura", "matematica"):
            out[f"enla_{sufijo}_{area}_satisfactorio"] = ultimo[ultimo.area == area].set_index("region_clave").satisfactorio.pipe(num)
    columnas = [c for c, *_ in DICCIONARIO_FINAL["resumen_region"]]
    out = out.reset_index()
    for c in columnas:
        if c not in out:
            raise KeyError(f"resumen_region: falta {c}")
    out = out[columnas]
    cuentas = [c for c in columnas if c.startswith(("obras_", "paralizadas_causa_"))] + ["contraloria_obras_educacion"]
    out[cuentas] = out[cuentas].fillna(0).astype(int)
    return out.sort_values("region_enla").reset_index(drop=True)


def serie_pais_educacion():
    c = leer("cifras_oficiales_contraloria")
    nac = c[c.ambito == "nacional"].set_index("fecha_corte")
    edu = c[(c.ambito == "sector") & (c.categoria == "EDUCACION")].set_index("fecha_corte")
    s = pd.DataFrame({"anio": nac.anio, "obras_pais": num(nac.obras_paralizadas), "costo_pais_soles": num(nac.costo_actualizado_soles),
                      "saldo_pais_soles": num(nac.saldo_inversion_soles), "obras_educacion_pdf": num(edu.obras_paralizadas),
                      "costo_educacion_pdf_soles": num(edu.costo_actualizado_soles), "fuente_pdf": nac.fuente_pdf})
    corregida = num(edu.obras_educacion_corregida)
    s["obras_educacion"] = corregida.fillna(s.obras_educacion_pdf)
    s["educacion_corregida"] = s.obras_educacion != s.obras_educacion_pdf
    s["fuente_obras_educacion"] = edu.fuente_educacion_corregida
    s["pct_educacion_obras"] = (100 * s.obras_educacion / s.obras_pais).round(2)
    s = s.reset_index()
    s["anio"] = num(s.anio).astype(int)
    for col in ("obras_pais", "obras_educacion_pdf", "obras_educacion"):
        s[col] = s[col].astype(int)
    return s[[c for c, *_ in DICCIONARIO_FINAL["serie_pais_educacion"]]].sort_values("fecha_corte").reset_index(drop=True)


def serie_educacion_departamento():
    d = leer("contraloria_educacion_departamento")
    for c in ("obras_educacion", "obras_educacion_estricto"):
        d[c] = num(d[c]).astype(int)
    for c in ("costo_educacion_soles", "costo_educacion_estricto_soles"):
        d[c] = num(d[c])
    d["anio"] = num(d.anio).astype(int)
    return d[[c for c, *_ in DICCIONARIO_FINAL["serie_educacion_departamento"]]].sort_values(["fecha_corte", "cod_departamento"]).reset_index(drop=True)


def causales_por_corte():
    c = leer("contraloria_causales")
    c["causa_grupo"] = [texto_nlp.clasificar_causa(x) for x in c.causal]
    c["causa_macro"] = c.causa_grupo.map(nlp_causas.macro)
    c["anio"] = num(c.anio).astype(int)
    c["obras_paralizadas"] = num(c.obras_paralizadas).astype(int)
    for col in ("pct_obras", "costo_actualizado_soles", "saldo_inversion_soles"):
        c[col] = num(c[col])
    c["pagina"] = num(c.pagina).astype(int)
    return c[[x for x, *_ in DICCIONARIO_FINAL["causales_por_corte"]]].sort_values(["fecha_corte", "causal"]).reset_index(drop=True)


# ======================================================================================================
# Validacion
# ======================================================================================================
def validar(m, r, serie, edu, causales):
    checks = []

    def agregar(nombre, valor, esperado, ok=None, tolerancia=0.0):
        diferencia = None if valor is None or esperado is None else float(valor) - float(esperado)
        paso = bool(ok) if ok is not None else (diferencia is not None and abs(diferencia) <= tolerancia)
        checks.append({"tabla": nombre.split(":")[0], "comprobacion": nombre.split(":", 1)[1].strip(), "valor": valor, "esperado": esperado, "ok": paso})

    fi = leer("infobras_fichas_ampliadas")
    f198 = leer("fichas_infobras")
    agregar("obras_maestra: filas = fichas ampliadas de INFOBRAS", len(m), len(fi))
    agregar("obras_maestra: codigo_infobras sin repetidos", int(m.codigo_infobras.duplicated().sum()), 0)
    agregar("obras_maestra: contiene las 198 obras paralizadas del 27-oct", int(m.codigo_infobras.isin(f198.codigo_infobras).sum()), len(f198))
    agregar("obras_maestra: ubigeo de 6 digitos en todas", int(m.ubigeo.astype(str).str.fullmatch(r"\d{6}").sum()), len(m))
    agregar("obras_maestra: cod_departamento de 2 digitos y = ubigeo[:2]", int((m.cod_departamento.astype(str) == m.ubigeo.astype(str).str[:2]).sum()), len(m))
    agregar("obras_maestra: region_enla es una de las 26 regiones ENLA", int(m.region_enla.isin(r.region_enla).sum()), len(m))
    agregar("obras_maestra: suma del expediente tecnico = fichas ampliadas", round(m.monto_expediente_tecnico.sum(), 2), round(num(fi.monto_expediente_tecnico).sum(), 2), tolerancia=0.01)
    agregar("obras_maestra: obras paralizadas hoy = estado PARALIZADA en las fichas", int(m.paralizada_hoy.sum()), int((fi.estado.str.upper() == "PARALIZADA").sum()))
    agregar("obras_maestra: todas las obras tienen causal oficial o grupo de causa", int(m.causa_grupo_final.notna().sum()), len(m))
    agregar("obras_maestra: grupos de causa y macro-categorias validos", int((m.causa_grupo_final.isin(GRUPOS_CAUSA) & m.causa_macro.isin(MACROS_CAUSA)).sum()), len(m))
    sin_causa = int(m.causa_grupo_final.isin(["otros", "sin_informacion"]).sum())
    agregar("obras_maestra: obras sin causa reconocible (otros + sin_informacion), como maximo 10 %", sin_causa, int(0.10 * len(m)), ok=sin_causa <= 0.10 * len(m))
    tipos = {"educacion_basica", "educacion_superior", "deportiva_recreativa", "cultural_patrimonio", "otra"}
    agregar("obras_maestra: tipo de infraestructura asignado y valido en todas", int(m.tipo_infraestructura.isin(tipos).sum()), len(m))
    agregar("obras_maestra: obras paralizadas hoy con fecha de paralizacion (para dias_desde_paralizacion)", int(m.dias_desde_paralizacion.notna().sum()),
            int(m.paralizada_hoy.sum()), ok=m.dias_desde_paralizacion.notna().sum() >= 0.9 * m.paralizada_hoy.sum())
    agregar("obras_maestra: obras de educacion basica (informativo)", int((m.tipo_infraestructura == "educacion_basica").sum()), len(m), ok=True)
    mayor = m.causa_macro.value_counts(normalize=True).max()
    agregar("obras_maestra: ninguna macro-categoria de causa supera el 50 % de las obras", round(float(mayor), 3), 0.5, ok=mayor <= 0.5)
    metricas_nlp = pd.read_csv(config.DOCS / "nlp_causas_metricas.csv", encoding="utf-8-sig")
    f1 = float(metricas_nlp.loc[metricas_nlp.clase == "macro avg", "f1"].iloc[0])
    agregar("NLP causas: macro-F1 del modelo de apoyo (validacion cruzada) de al menos 0.75", f1, 0.75, ok=f1 >= 0.75)
    agregar("obras_maestra: avance fisico entre 0 y 100", int(m.avance_fisico_pct.between(0, 100).sum()), int(m.avance_fisico_pct.notna().sum()))
    validas = m.cui.astype(str).str.fullmatch(r"\d+")   # 15 obras tienen CUI "NO ASOCIADO": no se pueden cruzar
    con_mef = int((validas & m.estado_mef.notna()).sum())
    agregar("obras_maestra: obras con CUI valido que aparecen en el Formato 12-B del MEF (informativo: al menos 80 %)", con_mef, int(validas.sum()), ok=con_mef >= 0.8 * validas.sum())
    agregar("resumen_region: 26 regiones", len(r), 26)
    agregar("resumen_region: obras_en_tabla suma = obras_maestra", int(r.obras_en_tabla.sum()), len(m))
    agregar("resumen_region: obras paralizadas hoy suma = obras_maestra", int(r.obras_paralizadas_hoy.sum()), int(m.paralizada_hoy.sum()))
    causas = [c for c in r.columns if c.startswith("paralizadas_causa_")]
    agregar("resumen_region: causas de las paralizadas suman las paralizadas", int(r[causas].sum().sum()), int(r.obras_paralizadas_hoy.sum()))
    ultimo_edu = int(serie.sort_values("fecha_corte").obras_educacion.iloc[-1])
    agregar("resumen_region: obras de Educacion de la Contraloria suman las del ultimo corte de la serie oficial", int(r.contraloria_obras_educacion.sum()), ultimo_edu)
    esc = leer("escale_infraestructura")
    esc = esc[esc.anio == esc.anio.max()]
    agregar("resumen_region: locales publicos de ESCALE suman el total del censo mas reciente", int(r.escale_locales_publicos.sum()), int(num(esc.n_locales_publicos).sum()))
    pcts = [c for c in r.columns if c.endswith(("_pct_3_servicios", "_pct_rural", "_pct_riesgo_estructural", "_pct_estructura_mal_estado", "_satisfactorio"))]
    agregar("resumen_region: porcentajes entre 0 y 100", int(((r[pcts] >= 0) & (r[pcts] <= 100)).sum().sum()), int(r[pcts].notna().sum().sum()))
    agregar("serie_pais_educacion: 21 cortes", len(serie), 21)
    agregar("serie_pais_educacion: Educacion corregida solo en set-2023 y set-2025", int(serie.educacion_corregida.sum()), 2)
    agregar("serie_pais_educacion: obras de Educacion <= obras del pais", int((serie.obras_educacion <= serie.obras_pais).sum()), len(serie))
    agregar("serie_educacion_departamento: 13 cortes", edu.fecha_corte.nunique(), 13)
    por_corte = edu.groupby("fecha_corte").obras_educacion.sum()
    base = serie.set_index("fecha_corte").obras_educacion.reindex(por_corte.index)
    coinciden = int((por_corte == base).sum())
    agregar("serie_educacion_departamento: la suma por corte coincide con la serie del pais (informativo: al menos 11 de 13 cortes)", coinciden, len(por_corte), ok=coinciden >= 11)
    agregar("causales_por_corte: 21 cortes", causales.fecha_corte.nunique(), 21)
    sumas = causales.groupby("fecha_corte").obras_paralizadas.sum()
    agregar("causales_por_corte: suma de causales = obras del pais en cada corte (cortes que cuadran)", int((sumas.reindex(serie.fecha_corte).values == serie.obras_pais.values).sum()), 21, ok=int((sumas.reindex(serie.fecha_corte).values == serie.obras_pais.values).sum()) >= 19)
    return pd.DataFrame(checks)


# ======================================================================================================
# Orquestacion
# ======================================================================================================
def escribir_excel(tablas):
    ruta = config.COMPONENTES / "producto_final.xlsx"
    filas_dic = [(t, c, tipo, desc, unidad) for t, cols in DICCIONARIO_FINAL.items() for c, tipo, desc, unidad in cols]
    dic = pd.DataFrame(filas_dic, columns=["tabla", "columna", "tipo", "descripcion", "unidad"])
    leeme = pd.DataFrame({"contenido": [
        "Producto final del proyecto «Monitor de obras de infraestructura educativa paralizadas» (USMP, Data Analytics).",
        "Hojas: obras_maestra (una fila por obra), resumen_region (26 regiones ENLA), serie_pais_educacion, serie_educacion_departamento, causales_por_corte y diccionario.",
        "INFOBRAS se leyó el 2026-11-11 (y 2026-10-27 para los comentarios); Contraloría hasta el corte 2026-06-30; MEF el día de la descarga; ESCALE y ENLA hasta 2025.",
        "Los códigos (ubigeo, CUI, cod_departamento) son texto: en Excel, no convertirlos a número. Limitaciones: docs/hallazgos_pdf.md y docs/producto_final.md."]})
    with pd.ExcelWriter(ruta, engine="openpyxl") as escritor:
        leeme.to_excel(escritor, sheet_name="LEEME", index=False)
        for nombre, tabla in tablas.items():
            tabla.to_excel(escritor, sheet_name=nombre[:31], index=False)
        dic.to_excel(escritor, sheet_name="diccionario", index=False)
        for hoja in escritor.book.worksheets:
            for columna in hoja.columns:
                ancho = max(len(str(c.value)) if c.value is not None else 0 for c in list(columna)[:200])
                hoja.column_dimensions[columna[0].column_letter].width = min(max(ancho + 2, 10), 60)
            hoja.freeze_panes = "A2"
    return ruta


def construir():
    """Arma y guarda las tablas finales; devuelve (tablas, validacion). Lanza un error si falla alguna comprobacion obligatoria."""
    config.FINAL.mkdir(parents=True, exist_ok=True)
    config.COMPONENTES.mkdir(parents=True, exist_ok=True)
    escale_obras.construir()
    escale_ampliado.construir()
    contraloria_anexo03.construir()
    cuis = set(leer("infobras_fichas_ampliadas").cui) | set(leer("contraloria_obras_corte").cui)
    descarga_mef.construir_tabla(cuis)
    m = tabla_maestra()
    r = resumen_region(m)
    serie, edu, causales = serie_pais_educacion(), serie_educacion_departamento(), causales_por_corte()
    tablas = {"obras_maestra": m, "resumen_region": r, "serie_pais_educacion": serie, "serie_educacion_departamento": edu, "causales_por_corte": causales}
    validacion = validar(m, r, serie, edu, causales)
    for nombre, tabla in tablas.items():
        tabla.round(4).to_csv(config.COMPONENTES / f"{nombre}.csv", index=False, encoding="utf-8-sig", na_rep="")
    # el entregable: un solo CSV para el dashboard con todas las tablas anteriores
    unico = dashboard.construir(m, r, serie, edu, causales, region_enla)
    extra = pd.DataFrame(dashboard.validar(unico, m, serie, edu), columns=["comprobacion", "valor", "esperado", "ok"])
    extra.insert(0, "tabla", "dashboard_obras")
    extra["comprobacion"] = extra.comprobacion.str.replace("dashboard: ", "", regex=False)
    validacion = pd.concat([validacion, extra], ignore_index=True)
    dashboard.escribir(unico)
    tablas["dashboard_obras"] = unico
    validacion.to_csv(config.DOCS / "validacion_final.csv", index=False, encoding="utf-8-sig")
    ruta = escribir_excel(tablas)
    malas = validacion[~validacion.ok]
    utils.log(f"PRODUCTO FINAL: obras_maestra {len(m)} filas, resumen_region {len(r)}, serie_pais {len(serie)}, serie_educacion_departamento {len(edu)}, "
              f"causales {len(causales)}; {len(validacion)} comprobaciones ({len(malas)} fallan); Excel {ruta.name}")
    for _, f in malas.iterrows():
        utils.log(f"AVISO PRODUCTO FINAL: no pasa '{f.tabla}: {f.comprobacion}' (valor {f.valor}, esperado {f.esperado})")
    if len(malas):
        raise ValueError(f"{len(malas)} comprobaciones del producto final no pasan (docs/validacion_final.csv)")
    return tablas, validacion

### `auditoria_calidad` — Auditoría de calidad
Auditoria de calidad del CSV unico del dashboard (data/final/dashboard_obras.csv).

In [ ]:
%%modulo auditoria_calidad
"""Auditoria de calidad del CSV unico del dashboard (data/final/dashboard_obras.csv).

Revisa: estructura (columnas = diccionario, sin filas repetidas), texto (espacios, caracteres raros, nulos disfrazados, nombres de lugares en
MAYUSCULAS sin tildes), codigos (departamento de 2 digitos, ubigeo de 6), rangos (sin negativos, porcentajes de 0 a 100, coordenadas dentro del Peru),
fechas (formato, coherencia), consistencia entre campos y entre tipos de registro, campos obligatorios por tipo y posibles nombres de personas.
Resultado: docs/auditoria_calidad.csv con nivel «ERROR» (el proceso se detiene) o «aviso» (dato dudoso que viene de la fuente y ya esta marcado
en la columna alertas_calidad).
"""
import re
import unicodedata

import numpy as np
import pandas as pd

import config
import utils

MARCAS_NULAS = ["nan", "none", "null", "n/a", "na", "-", "--", "s/d", "#n/a"]
OBLIGATORIOS = {
    "obra_infobras": ["codigo_infobras", "cui", "nombre_obra", "entidad", "cod_departamento", "departamento", "provincia", "distrito", "ubigeo", "region_enla",
                      "estado_infobras", "paralizada_hoy", "fecha_corte", "n_obras", "causa_grupo_final", "causa_macro", "avance_fisico_pct", "escale_locales_publicos"],
    "obra_corte_contraloria": ["codigo_infobras", "fecha_corte", "departamento", "region_enla", "causal_oficial", "causa_grupo_final", "nivel_gobierno"],
    "departamento_corte": ["fecha_corte", "cod_departamento", "departamento", "n_obras", "costo_actualizado_soles"],
    "pais_total_corte": ["fecha_corte", "n_obras", "costo_actualizado_soles", "saldo_inversion_soles"],
    "pais_educacion_corte": ["fecha_corte", "n_obras", "costo_actualizado_soles", "pct_obras"],
    "causal_corte": ["fecha_corte", "causal_oficial", "causa_grupo_final", "causa_macro", "n_obras", "costo_actualizado_soles"],
}
PATRON_NOMBRE = re.compile(r"\b(ing|arq|abog|econ|lic|sr|sra|srta|cpc|dr|dra)\.?\s+[A-Za-zÁÉÍÓÚÑáéíóúñ]{3,}\s+[A-Za-zÁÉÍÓÚÑáéíóúñ]{3,}", re.I)


def auditar(escribir=True):
    """Devuelve un DataFrame con los hallazgos (nivel, tema, detalle); lo guarda en docs/auditoria_calidad.csv."""
    d = pd.read_csv(config.FINAL / "dashboard_obras.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    dic = pd.read_csv(config.FINAL / "dashboard_obras_diccionario.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
    num = lambda s: pd.to_numeric(s, errors="coerce")
    hall = []
    nota = lambda nivel, tema, detalle: hall.append((nivel, tema, detalle))
    tipo = lambda t: d[d.tipo_registro == t]

    # 1) estructura
    if list(d.columns) != list(dic.columna):
        nota("ERROR", "estructura", "las columnas del CSV no coinciden con el diccionario")
    if d.duplicated().any():
        nota("ERROR", "estructura", f"{int(d.duplicated().sum())} filas repetidas")
    for c in d.columns:
        if (d[c] == "").all():
            nota("ERROR", "columna vacia", c)

    # 2) texto
    for c in d.columns:
        v = d[c][d[c] != ""]
        if v.str.contains(r"^\s|\s$").any():
            nota("ERROR", "espacios", f"{c}: valores con espacios al inicio o al final")
        if v.str.contains(r"[\x00-\x08\x0b\x0c\x0e-\x1f�]").any():
            nota("ERROR", "caracteres de control", c)
        if v.str.contains("Ã.|â€").any():
            nota("ERROR", "mojibake", c)
        if v.str.lower().isin(MARCAS_NULAS).any():
            nota("ERROR", "nulo disfrazado", c)
    for c in ("departamento", "provincia", "distrito", "region_enla"):
        v = d[c][d[c] != ""].drop_duplicates()
        mal = v[v.map(lambda x: x != x.upper() or any(unicodedata.combining(ch) for ch in unicodedata.normalize("NFD", x)))]
        if len(mal):
            nota("ERROR", "nombres de lugares", f"{c}: {mal.head(5).tolist()}")

    # 3) codigos
    cod = d[d.cod_departamento != ""]
    if (~cod.cod_departamento.str.fullmatch(r"\d{2}")).any():
        nota("ERROR", "cod_departamento", "no tiene 2 digitos")
    ub = d[d.ubigeo != ""]
    if (~ub.ubigeo.str.fullmatch(r"\d{6}")).any() or (ub.ubigeo.str[:2] != ub.cod_departamento).any():
        nota("ERROR", "ubigeo", "no tiene 6 digitos o su inicio no es el cod_departamento")
    mapa = d[(d.cod_departamento != "") & (d.departamento != "")][["cod_departamento", "departamento"]].drop_duplicates()
    if mapa.cod_departamento.duplicated().any() or mapa.departamento.duplicated().any():
        nota("ERROR", "cod_departamento", "un codigo con varios nombres o al reves")
    reg = d[(d.cod_departamento != "") & (d.region_enla != "")][["cod_departamento", "region_enla"]].drop_duplicates().groupby("cod_departamento").size()
    if (reg[reg > 1].index != "15").any():
        nota("ERROR", "region_enla", "un departamento (que no es Lima) cae en mas de una region")

    # 4) rangos
    for c in d.columns:
        if d[c].str.fullmatch(r"-?\d+(\.\d+)?").sum() > 0 and c not in ("latitud", "longitud", "latitud_mapa", "longitud_mapa") and not c.startswith(("cod_", "ubigeo", "codigo_", "cui", "escale_codlocal")):
            x = num(d[c]).dropna()
            if (x < 0).any():
                nota("ERROR", "negativos", c)
            if ("pct" in c or c.endswith("satisfactorio")) and c != "pct_devengado_del_costo" and ((x < 0) | (x > 100)).any():
                nota("ERROR", "porcentaje fuera de 0-100", c)
    ob, oc = tipo("obra_infobras"), tipo("obra_corte_contraloria")
    lat, lon = num(ob.latitud).dropna(), num(ob.longitud).dropna()
    if ((lat > 0.1) | (lat < -18.6)).any() or ((lon > -68.4) | (lon < -81.4)).any():
        nota("ERROR", "coordenadas", "fuera del Peru")
    if ((ob.latitud == "") != (ob.longitud == "")).any():
        nota("ERROR", "coordenadas", "obras con solo latitud o solo longitud")
    lat_m, lon_m = num(ob.latitud_mapa).dropna(), num(ob.longitud_mapa).dropna()   # coordenadas para el mapa (INFOBRAS o aproximadas de ESCALE)
    if ((lat_m > 0.1) | (lat_m < -18.6)).any() or ((lon_m > -68.4) | (lon_m < -81.4)).any():
        nota("ERROR", "coordenadas", "latitud_mapa/longitud_mapa fuera del Peru")
    if ((ob.latitud_mapa == "") != (ob.longitud_mapa == "")).any():
        nota("ERROR", "coordenadas", "obras con solo latitud_mapa o solo longitud_mapa")

    # 5) fechas
    for c in ("fecha_corte", "fecha_inicio", "fecha_fin_prevista", "fecha_paralizacion"):
        v = d[c][d[c] != ""]
        f = pd.to_datetime(v, errors="coerce", format="%Y-%m-%d")
        if f.isna().any():
            nota("ERROR", "fechas", f"{c}: valores que no son AAAA-MM-DD")
        elif (f.dt.year < 1990).any() or (f.dt.year > 2030).any():
            nota("ERROR", "fechas", f"{c}: fuera de 1990-2030")
    con_f = d[d.fecha_corte != ""]
    if (con_f.anio != con_f.fecha_corte.str[:4]).any():
        nota("ERROR", "anio", "distinto del ano de fecha_corte")
    fi, ff = pd.to_datetime(ob.fecha_inicio.replace("", np.nan)), pd.to_datetime(ob.fecha_fin_prevista.replace("", np.nan))
    if (ff < fi).any():
        nota("aviso", "fechas de obra", f"{int((ff < fi).sum())} obras con fecha_fin_prevista anterior al inicio")

    # 6) consistencia
    if ((ob.estado_infobras.str.upper() == "PARALIZADA") != (ob.paralizada_hoy == "True")).any():
        nota("ERROR", "estado", "paralizada_hoy no coincide con estado_infobras")
    if (num(ob.n_obras_paralizadas) != (ob.paralizada_hoy == "True").astype(int)).any():
        nota("ERROR", "n_obras_paralizadas", "no coincide con paralizada_hoy")
    if (ob.causa_grupo_final == "").any() or (ob.causa_macro == "").any():
        nota("ERROR", "causa", "obras sin causa")
    tot, edu = tipo("pais_total_corte").set_index("fecha_corte"), tipo("pais_educacion_corte").set_index("fecha_corte")
    if (num(edu.n_obras) > num(tot.n_obras).reindex(edu.index)).any():
        nota("ERROR", "pais", "Educacion mayor que el total del pais")
    suma = tipo("causal_corte").groupby("fecha_corte").n_obras.apply(lambda s: num(s).sum())
    dif = (suma - num(tot.n_obras).reindex(suma.index)).abs()
    for corte in dif[dif > 0].index:
        nota("aviso", "causales vs total", f"{corte}: la suma de causales difiere del total del pais en {int(dif[corte])} obras (error del propio informe)")
    if tipo("departamento_corte").duplicated(["fecha_corte", "cod_departamento"]).any() or oc.duplicated(["fecha_corte", "codigo_infobras"]).any() or ob.duplicated("codigo_infobras").any():
        nota("ERROR", "llaves", "obra o departamento repetido dentro de un corte")
    por_corte = oc.groupby("fecha_corte").size()
    por_dep = tipo("departamento_corte").groupby("fecha_corte").n_obras.apply(lambda s: num(s).sum())
    for corte in por_corte.index:
        if abs(int(por_corte[corte]) - int(por_dep.get(corte, -999))) > 2:
            nota("ERROR", "obras por corte", f"{corte}: {por_corte[corte]} obras por obra vs {por_dep.get(corte)} en la serie por departamento")
    # avisos que vienen de la fuente (ya marcados en alertas_calidad)
    if (num(oc.costo_actualizado_soles) <= 0).any() or oc.costo_actualizado_soles.eq("").any():
        nota("aviso", "costos", f"{int((~(num(oc.costo_actualizado_soles) > 0)).sum())} filas de la Contraloria con costo 0 o vacio en la fuente")
    if (num(oc.saldo_inversion_soles) > num(oc.costo_actualizado_soles) * 1.0001).any():
        nota("aviso", "saldo > costo", f"{int((num(oc.saldo_inversion_soles) > num(oc.costo_actualizado_soles) * 1.0001).sum())} filas de la Contraloria")
    sin_alerta = int((ob.alertas_calidad == "").sum())
    nota("aviso", "alertas_calidad", f"{len(ob) - sin_alerta} de {len(ob)} obras de INFOBRAS tienen al menos una alerta (ver columna alertas_calidad)")

    # 7) obligatorios
    for t, cols in OBLIGATORIOS.items():
        g = tipo(t)
        for c in cols:
            n = int((g[c] == "").sum())
            if n:
                nota("aviso", "obligatorio vacio", f"{t}.{c}: {n} de {len(g)}")
    ctx = tipo("contexto_region")
    if (ctx[["region_enla", "cod_departamento"] + [c for c in d.columns if c.startswith("region_")]] == "").any().any():
        nota("ERROR", "contexto_region", "campos vacios en las 26 regiones")

    # 8) posibles nombres de personas
    for c in ("nombre_obra", "entidad", "mef_ult_problema", "causal_oficial"):
        marca = d[c].map(lambda t: bool(PATRON_NOMBRE.search(t)) and "[NOMBRE OMITIDO]" not in t)
        if marca.any():
            nota("ERROR", "posible nombre de persona", f"{c}: {int(marca.sum())} textos")

    h = pd.DataFrame(hall, columns=["nivel", "tema", "detalle"])
    if escribir:
        h.to_csv(config.DOCS / "auditoria_calidad.csv", index=False, encoding="utf-8-sig")
    utils.log(f"AUDITORIA DE CALIDAD: {len(d)} filas x {d.shape[1]} columnas; {int((h.nivel == 'ERROR').sum())} errores, {int((h.nivel == 'aviso').sum())} avisos (docs/auditoria_calidad.csv)")
    return h


def auditoria_o_falla():
    """Falla (para que el pipeline lo avise) si hay algun hallazgo de nivel ERROR."""
    h = auditar()
    errores = h[h.nivel == "ERROR"]
    if len(errores):
        raise ValueError(f"{len(errores)} errores en el CSV unico: {errores[['tema', 'detalle']].head(5).values.tolist()}")
    return h

### `diccionario` — Diccionario (ESCALE y ENLA)
Diccionario de datos de las dos tablas finales (se escribe en docs/diccionario.csv y en los Excel).

In [ ]:
%%modulo diccionario
"""Diccionario de datos de las dos tablas finales (se escribe en docs/diccionario.csv y en los Excel)."""
import pandas as pd

import config
import utils

TABLAS = ("escale_infraestructura", "enla_regiones")
ESCALE = "ESCALE (Censo Educativo): lineal 1, servicios básicos, padrón y edificaciones"
SICRECE = "SICRECE (UMC, MINEDU), acceso público"

# (tabla, columna, tipo, descripción, unidad, fuente, notas)
DICCIONARIO = [
    ("escale_infraestructura", "anio", "entero", "Año del Censo Educativo.", "año", ESCALE, ""),
    ("escale_infraestructura", "ubigeo", "texto de 6 dígitos",
     "Código de ubigeo del distrito (departamento, provincia y distrito).", "código", ESCALE,
     "Es texto para no perder el cero inicial. En Excel, importar la columna como texto (si no, 010101 se ve 10101)."),
    ("escale_infraestructura", "departamento", "texto",
     "Departamento, como viene en la base (mayúsculas y sin tildes).", "texto", ESCALE, "Callao figura como departamento."),
    ("escale_infraestructura", "provincia", "texto", "Provincia, como viene en la base.", "texto", ESCALE, ""),
    ("escale_infraestructura", "distrito", "texto", "Distrito, como viene en la base.", "texto", ESCALE, ""),
    ("escale_infraestructura", "n_locales_publicos", "entero",
     "Locales educativos públicos (de gestión directa o privada), activos en el padrón y que respondieron el censo.",
     "locales", ESCALE,
     "Con pocos locales los porcentajes son muy escalonados: con 2 locales solo pueden dar 0, 50 o 100."),
    ("escale_infraestructura", "n_locales_sin_dato", "entero",
     "De los anteriores, locales de los que no se puede saber si tienen agua, desagüe y electricidad de red pública "
     "(sin respuesta o respuesta ambigua).", "locales", ESCALE,
     "En 2023 son muchos (8,5 % de los locales públicos): es un hueco de la base publicada."),
    ("escale_infraestructura", "pct_3_servicios", "decimal de 0 a 100",
     "Porcentaje de locales con agua, desagüe y electricidad de red pública, calculado solo con los locales con dato.",
     "%", ESCALE,
     "Vacío si ningún local del distrito tiene dato. En 2025 la electricidad «de terceros» también cuenta como red pública (efecto de 0,18 puntos)."),
    ("escale_infraestructura", "pct_sin_3_servicios", "decimal de 0 a 100", "100 menos pct_3_servicios.", "%", ESCALE, ""),
    ("escale_infraestructura", "pct_rural", "decimal de 0 a 100",
     "Porcentaje de locales públicos ubicados en zona rural (según el padrón).", "%", ESCALE, ""),
    ("escale_infraestructura", "n_locales_con_dato_edif", "entero",
     "Locales públicos con al menos una edificación en uso que respondió la pregunta de riesgo estructural. "
     "Es la base de las dos columnas siguientes.", "locales", ESCALE, ""),
    ("escale_infraestructura", "pct_riesgo_estructural", "decimal de 0 a 100",
     "Porcentaje de esos locales con al menos una edificación en uso que declara riesgo de colapso o riesgo estructural.",
     "%", ESCALE,
     "2024 no es comparable con los otros años (se declara mucho menos). Vacío si no hay locales con dato."),
    ("escale_infraestructura", "pct_estructura_mal_estado", "decimal de 0 a 100",
     "Porcentaje de esos locales con al menos una edificación en uso con muros, columnas, vigas o estructura de techos en mal estado.",
     "%", ESCALE,
     "No equivale a «reparación total»: esa categoría no existe en el censo. Vacío si no hay locales con dato."),
    ("enla_regiones", "region", "texto",
     "Región según la DRE (Dirección Regional de Educación): 23 departamentos sin Lima, Callao, Lima Metropolitana y Lima Provincias (26 en total).",
     "texto", SICRECE,
     "Para unir con la tabla de ESCALE se asume: Lima Metropolitana = departamento LIMA, provincia LIMA; Lima Provincias = el resto de provincias de LIMA."),
    ("enla_regiones", "anio", "entero", "Año de la evaluación.", "año", SICRECE, ""),
    ("enla_regiones", "grado", "texto", "Grado evaluado: «2.° primaria», «4.° primaria», «6.° primaria», «2.° secundaria» o «5.° secundaria».", "texto", SICRECE, ""),
    ("enla_regiones", "evaluacion", "texto", "Tipo de evaluación: MUESTRAL o CENSAL.", "texto", SICRECE,
     "En 2024 el 4.° de primaria es censal (todas las escuelas); los demás años son muestrales: no son estrictamente comparables."),
    ("enla_regiones", "lectura_satisfactorio", "decimal de 0 a 100",
     "Porcentaje de estudiantes en nivel Satisfactorio en Lectura.", "%", SICRECE,
     "Los otros niveles de logro (previo al inicio, en inicio, en proceso) están en los archivos de data/raw/enla."),
    ("enla_regiones", "matematica_satisfactorio", "decimal de 0 a 100",
     "Porcentaje de estudiantes en nivel Satisfactorio en Matemática.", "%", SICRECE,
     "Los otros niveles de logro están en los archivos de data/raw/enla."),
]


def construir_diccionario():
    """Devuelve el diccionario como tabla; los años de cada columna se calculan con los datos de data/processed."""
    tablas = {}
    for nombre in TABLAS:
        ruta = config.PROCESSED / f"{nombre}.csv"
        if ruta.exists():
            tablas[nombre] = pd.read_csv(ruta, dtype={"ubigeo": str})
    filas = []
    for tabla, columna, tipo, descripcion, unidad, fuente, notas in DICCIONARIO:
        df = tablas.get(tabla)
        anios = ""
        if df is not None and columna in df:
            anios = ", ".join(str(a) for a in sorted(df.loc[df[columna].notna(), "anio"].unique()))
        filas.append({"tabla": tabla, "columna": columna, "tipo": tipo, "descripcion": descripcion,
                      "unidad": unidad, "fuente": fuente, "anios": anios, "notas": notas})
    return pd.DataFrame(filas)


def escribir_diccionario():
    """Escribe docs/diccionario.csv y devuelve la tabla."""
    import diccionario_pdf   # las tablas del proyecto PDF/INFOBRAS se agregan al mismo diccionario
    tabla = pd.concat([construir_diccionario(), diccionario_pdf.construir_diccionario_pdf()], ignore_index=True)
    ruta = config.DOCS / "diccionario.csv"
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    utils.log(f"DICCIONARIO: escrito {ruta.name} ({len(tabla)} columnas)")
    return tabla

### `exportar` — Exportar a Excel
Exporta cada tabla final a Excel (.xlsx) con una hoja de datos y una hoja de diccionario.

In [ ]:
%%modulo exportar
"""Exporta cada tabla final a Excel (.xlsx) con una hoja de datos y una hoja de diccionario."""
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

import config
import diccionario
import utils

COLUMNAS_LARGAS = {"descripcion": 70, "notas": 70, "fuente": 45}   # ancho de las columnas de texto largo del diccionario


def dar_formato(hoja, tabla, anchos=None):
    """Encabezado en negrita y con color, primera fila fija, filtros y ancho de columna razonable."""
    anchos = anchos or {}
    for celda in hoja[1]:
        celda.font = Font(bold=True, color="FFFFFF")
        celda.fill = PatternFill("solid", fgColor="1F4E78")
        celda.alignment = Alignment(vertical="center", wrap_text=True)
    hoja.freeze_panes = "A2"
    hoja.auto_filter.ref = hoja.dimensions
    for i, columna in enumerate(tabla.columns, start=1):
        ancho = anchos.get(columna) or min(max(len(str(columna)), int(tabla[columna].astype(str).str.len().max())) + 2, 32)
        hoja.column_dimensions[get_column_letter(i)].width = ancho
        if columna in anchos:
            for fila in hoja.iter_rows(min_row=2, min_col=i, max_col=i):
                fila[0].alignment = Alignment(wrap_text=True, vertical="top")


def escribir_excel():
    """Genera data/processed/<tabla>.xlsx: hoja «datos» (números como números, ubigeo como texto) y hoja «diccionario»."""
    dic = diccionario.construir_diccionario()
    for tabla in diccionario.TABLAS:
        ruta_csv = config.PROCESSED / f"{tabla}.csv"
        if not ruta_csv.exists():
            continue
        datos = pd.read_csv(ruta_csv, dtype={"ubigeo": str})
        hoja_dic = dic[dic.tabla == tabla].drop(columns="tabla")
        ruta = config.PROCESSED / f"{tabla}.xlsx"
        with pd.ExcelWriter(ruta, engine="openpyxl") as escritor:
            datos.to_excel(escritor, sheet_name="datos", index=False)
            hoja_dic.to_excel(escritor, sheet_name="diccionario", index=False)
            dar_formato(escritor.sheets["datos"], datos)
            dar_formato(escritor.sheets["diccionario"], hoja_dic, COLUMNAS_LARGAS)
        utils.log(f"EXCEL: escrito {ruta.name} ({len(datos)} filas, {len(hoja_dic)} columnas en el diccionario)")

### `diccionario_pdf` — Diccionario (PDF, final y dashboard)
Diccionario de las tablas del proyecto PDF/INFOBRAS. Se agrega a docs/diccionario.csv junto al de ESCALE y ENLA.

In [ ]:
%%modulo diccionario_pdf
"""Diccionario de las tablas del proyecto PDF/INFOBRAS. Se agrega a docs/diccionario.csv junto al de ESCALE y ENLA.

Cada columna de cada CSV debe tener descripcion aqui: si falta alguna, construir_diccionario_pdf() falla a proposito.
"""
import re

import pandas as pd

import config

CGR = "Informes de obras paralizadas, Contraloría General de la República (gob.pe)"
CGR_XLSX = "Anexos Excel de los informes de la Contraloría (base de obras paralizadas)"
INFOBRAS = "INFOBRAS, Contraloría (ficha pública de cada obra)"
ENLA_PDF = "Reporte nacional de la Evaluación Muestral 2022, MINEDU-UMC"
PROCEDENCIA = "Informe PDF"

# columna -> (tipo, descripcion, unidad). Las columnas con el mismo nombre en varias tablas se describen una sola vez.
COMUNES = {
    "anio": ("entero", "Año (en la tablas de la Contraloría, el año de la fecha de corte del informe).", "año"),
    "fecha_corte": ("fecha AAAA-MM-DD", "Fecha de corte de los datos del informe (fin del mes o trimestre que indica su título).", "fecha"),
    "cod_departamento": ("texto de 2 dígitos", "Código INEI del departamento (Callao = 07; Lima Metropolitana y Provincias = 15).", "código"),
    "fuente_pdf": ("texto", "Nombre del archivo PDF de data/raw de donde sale la cifra.", "texto"),
    "pagina": ("entero", "Página del PDF donde está la cifra (la primera página si la tabla sigue en la siguiente).", "página"),
    "tabla_n": ("entero", "Número de la tabla dentro de esa página (2023-2026: orden de aparición; en 2022 y ENLA: el número del título).", "número"),
    "obras_paralizadas": ("entero", "Número de obras paralizadas.", "obras"),
    "pct_obras": ("decimal", "Porcentaje que representa del total de obras paralizadas, tal como lo publica el informe.", "%"),
    "pct_costo": ("decimal", "Porcentaje que representa del costo actualizado total, tal como lo publica el informe.", "%"),
    "pct_saldo": ("decimal", "Porcentaje que representa del saldo de inversión total, tal como lo publica el informe.", "%"),
    "costo_actualizado_soles": ("decimal", "Costo actualizado de la inversión (según Invierte.pe) de esas obras.", "soles"),
    "saldo_inversion_soles": ("decimal", "Saldo de inversión por ejecutar (costo actualizado menos lo devengado) de esas obras.", "soles"),
}

POR_TABLA = {
    "cifras_oficiales_contraloria": (CGR, {
        "ambito": ("texto", "Qué representa la fila: nacional (total del país), departamento o sector.", "categoría"),
        "categoria": ("texto", "NACIONAL, nombre del departamento o nombre del sector, en mayúsculas y sin tildes.", "texto"),
        "obras_g_nacional": ("entero", "Solo en filas de sector: obras de entidades del Gobierno Nacional.", "obras"),
        "obras_g_regional": ("entero", "Solo en filas de sector: obras de entidades de Gobiernos Regionales.", "obras"),
        "obras_g_local": ("entero", "Solo en filas de sector: obras de entidades de Gobiernos Locales.", "obras"),
        "unidad_original": ("texto", "Unidad en que venía el monto en el PDF: soles, o millones de soles con 1 decimal (mapas de set y dic 2022).", "texto"),
        "coincide_excel": ("lógico", "TRUE/FALSE si el número de obras coincide con el anexo Excel del mismo corte; vacío si ese corte no tiene anexo con base de obras.", "lógico"),
        "observacion": ("texto", "Aviso sobre la cifra (no coincide con el Excel, o precisión reducida).", "texto"),
        "obras_educacion_corregida": ("entero", "Solo en filas de sector Educación: obras de Educación con la cifra corregida. En los 13 cortes con anexo Excel es la del Excel (corrige set-2023: 324 → 283 y set-2025: 73 → 196); en los demás cortes es la del PDF.", "obras"),
        "fuente_educacion_corregida": ("texto", "De dónde sale obras_educacion_corregida: anexo Excel, o informe PDF cuando el corte no tiene anexo para contrastar.", "texto"),
    }),
    "contraloria_causales": (CGR, {
        "causal": ("texto", "Causal de paralización declarada por la entidad, en mayúsculas y sin tildes. El catálogo de causales cambió entre informes.", "texto"),
    }),
    "contraloria_distribuciones": (CGR, {
        "dimension": ("texto", "Tabla de origen: nivel_gobierno, rango_costo o rango_avance_fisico.", "categoría"),
        "categoria": ("texto", "Categoría dentro de la dimensión (por ejemplo G. LOCAL, '> 1 - 3 MILLONES', '20 - 30 %').", "texto"),
    }),
    "contraloria_educacion_departamento": (CGR_XLSX, {
        "departamento": ("texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        "obras_educacion": ("entero", "Obras paralizadas del sector Educación con el mismo agrupamiento de sectores del informe (en 2026-06 incluye EDUCACION/CULTURA).", "obras"),
        "costo_educacion_soles": ("decimal", "Costo actualizado de esas obras (según Invierte.pe).", "soles"),
        "obras_educacion_estricto": ("entero", "Obras cuyo sector es exactamente EDUCACION (sin Cultura).", "obras"),
        "costo_educacion_estricto_soles": ("decimal", "Costo actualizado de esas obras.", "soles"),
        "fuente_excel": ("texto", "Anexo Excel de data/raw/contraloria del que se calcula.", "texto"),
        "hoja": ("texto", "Hoja del Excel con la base de obras.", "texto"),
    }),
    "enla_pdf": (ENLA_PDF, {
        "region": ("texto", "Región ENLA en mayúsculas y sin tildes (Lima Metropolitana y Lima Provincias van por separado).", "texto"),
        "grado": ("texto", "Grado evaluado: 2.°, 4.° o 6.° de primaria; 2.° o 5.° de secundaria.", "texto"),
        "area": ("texto", "Área evaluada: lectura o matematica.", "texto"),
        "pct_satisfactorio": ("decimal de 0 a 100", "Porcentaje de estudiantes en nivel Satisfactorio.", "%"),
    }),
    "fichas_infobras": (INFOBRAS, {
        "codigo_infobras": ("texto", "Código de la obra en INFOBRAS.", "código"),
        "cui": ("texto", "Código Único de Inversiones.", "código"),
        "nombre_obra": ("texto", "Nombre de la obra.", "texto"),
        "entidad": ("texto", "Entidad responsable de la obra (entidad pública).", "texto"),
        "departamento": ("texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        "provincia": ("texto", "Provincia en mayúsculas y sin tildes.", "texto"),
        "distrito": ("texto", "Distrito en mayúsculas y sin tildes.", "texto"),
        "ubigeo": ("texto de 6 dígitos", "Ubigeo INEI del distrito de la obra, tomado de escale_infraestructura (src/ubigeo_obras.py). Puede contradecir departamento/provincia/distrito cuando INFOBRAS los trae mal (ubigeo_metodo = manual).", "código"),
        "ubigeo_metodo": ("texto", "Cómo se asignó el ubigeo: exacto (mismos nombres que ESCALE), normalizado (difieren solo en tildes, eñe, guion bajo, nombre alterno o provincia del Callao), manual (ubicación de INFOBRAS corregida a mano; motivo en docs/nombres_sin_ubigeo.csv) o sin_ubigeo.", "categoría"),
        "estado": ("texto", "Estado de la obra en la ficha (PARALIZADA en todas las filas).", "texto"),
        "modalidad": ("texto", "Modalidad de ejecución.", "texto"),
        "naturaleza": ("texto", "Naturaleza de la obra.", "texto"),
        "tipo_obra": ("texto", "Tipo de obra.", "texto"),
        "monto_expediente_tecnico": ("decimal", "Monto del expediente técnico (del listado de INFOBRAS).", "soles"),
        "monto_ejecucion_acumulado": ("decimal", "Monto de ejecución acumulado.", "soles"),
        "avance_fisico_pct": ("decimal", "Porcentaje de avance físico del encabezado de la ficha.", "%"),
        "fecha_inicio": ("fecha AAAA-MM-DD", "Fecha de inicio de la obra.", "fecha"),
        "fecha_fin_prevista": ("fecha AAAA-MM-DD", "Fecha de fin prevista de la obra.", "fecha"),
        "periodo_ultimo_avance": ("texto", "Año / mes del último avance registrado.", "texto"),
        "n_avances": ("entero", "Número de avances mensuales registrados.", "avances"),
        "periodo_avance_paralizado": ("texto", "Año / mes del último avance en estado Paralizado.", "texto"),
        "causal_paralizacion": ("texto", "Causal de paralización registrada por la entidad. Puede ser genérica («Catalogo de Obra», «-»).", "texto"),
        "fecha_paralizacion": ("fecha AAAA-MM-DD", "Fecha de paralización del último avance paralizado.", "fecha"),
        "dias_paralizado": ("entero", "Días paralizada según el último avance paralizado.", "días"),
        "comentario_fisico": ("texto", "Comentario libre de la entidad sobre el avance físico.", "texto"),
        "comentario_valorizado": ("texto", "Comentario libre de la entidad sobre el avance valorizado.", "texto"),
        "latitud": ("decimal", "Latitud de la obra.", "grados"),
        "longitud": ("decimal", "Longitud de la obra.", "grados"),
        "avance_fisico_tabla": ("decimal", "Avance físico real del último periodo en la tabla de avances (para contrastar con el del encabezado).", "%"),
        "estado_listado": ("texto", "Estado que da el listado exportado por el buscador.", "texto"),
        "url_ficha": ("texto", "Dirección de la ficha pública.", "texto"),
        "fecha_lectura": ("fecha AAAA-MM-DD", "Día en que se leyó la ficha.", "fecha"),
        "causa_grupo": ("texto", "Grupo de causa asignado por reglas de palabras clave (config.CAUSAS_REGLAS) a partir de la causal y los comentarios; «otros» si ninguna regla aplica.", "categoría"),
        "causa_grupo_origen": ("texto", "De dónde salió el grupo: causal (la causal declarada tuvo coincidencias), comentario, sin_informacion u otros.", "categoría"),
    }),
    "citas_educacion": (CGR, {
        "termino": ("texto", "Palabra clave encontrada en la oración (educación, escuela, etc.).", "texto"),
        "cita": ("texto", "Oración literal del informe (limpia de saltos de línea). Citar con fuente_pdf y página.", "texto"),
    }),
    "frecuencia_terminos": (CGR, {
        "tipo": ("texto", "termino (palabra) o par (dos palabras seguidas).", "categoría"),
        "termino": ("texto", "Palabra o par de palabras, sin tildes y en minúsculas.", "texto"),
        "frecuencia": ("entero", "Veces que aparece en la narrativa de los informes de ese año.", "veces"),
        "tfidf": ("decimal", "Puntaje TF-IDF x 1000: frecuencia relativa en el año por log(años totales / años en que aparece). Alto = término propio de ese año; se listan los 40 de mayor puntaje (con al menos 3 apariciones).", "puntaje"),
        "anios_con_termino": ("entero", "Número de años (de los informes disponibles) en cuya narrativa aparece el término; si es igual al total de años, no distingue un año de otro.", "años"),
    }),
}

CGR_EXCEL = "Anexos Excel de los informes de la Contraloría (base de obras paralizadas)"
POR_TABLA_ANEXOS = {
    "contraloria_obras_corte": (CGR_EXCEL, {
        "corte": ("fecha AAAA-MM-DD", "Fecha de corte del anexo del que sale la fila (13 cortes con base obra por obra: sep-2023 a jun-2026).", "fecha"),
        "codigo_infobras": ("texto", "Código de la obra en INFOBRAS. Clave para unir con fichas_infobras.", "código"),
        "cui": ("texto", "Código Único de Inversiones.", "código"),
        "entidad": ("texto", "Entidad pública responsable de la obra, en mayúsculas y sin tildes.", "texto"),
        "evento": ("texto", "Evento registrado en INFOBRAS: PROCESO NORMAL, RECONSTRUCCION CON CAMBIOS, REACTIVACION ECONOMICA o EMBLEMATICAS.", "categoría"),
        "modalidad_ejecucion": ("texto", "Modalidad de ejecución (POR CONTRATA, ADMINISTRACION DIRECTA, etc.; los rótulos CONTRATA y APP se unificaron).", "categoría"),
        "tipo_obra": ("texto", "Tipo de obra (nivel 1) tal como lo trae el anexo; puede diferir de sector.", "categoría"),
        "tipo_obra_nivel2": ("texto", "Tipo de obra, nivel 2.", "categoría"),
        "tipo_obra_nivel3": ("texto", "Tipo de obra, nivel 3.", "categoría"),
        "nivel_gobierno": ("texto", "Nivel de gobierno de la entidad: G. NACIONAL, G. REGIONAL o G. LOCAL.", "categoría"),
        "sector": ("texto", "Sector de la obra (SECTOR del anexo; EDUCACION y EDUCACION/CULTURA son distintos).", "categoría"),
        "sector_agrupado": ("texto", "Sector con el agrupamiento del informe (columna SECTOR 2; si el corte no la trae, EDUCACION/CULTURA se cuenta como EDUCACION).", "categoría"),
        "causal_codigo": ("texto", "Número de la causal oficial (1 a 15); vacío en las causales nuevas que el anexo trae sin número.", "código"),
        "causal_paralizacion": ("texto", "Causal de paralización oficial (sin el número).", "texto"),
        "avance_fisico": ("decimal de 0 a 100", "Avance físico de la obra según INFOBRAS en el corte.", "%"),
        "rango_avance_fisico": ("texto", "Rango de avance físico del anexo.", "categoría"),
        "anio_inicio": ("entero", "Año de inicio de la obra.", "año"),
        "plazo_ejecucion_dias": ("entero", "Plazo de ejecución.", "días"),
        "departamento": ("texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        "provincia": ("texto", "Provincia en mayúsculas y sin tildes.", "texto"),
        "distrito": ("texto", "Distrito en mayúsculas y sin tildes.", "texto"),
        "rango_inversion_mll": ("texto", "Rango de inversión (millones de soles) del anexo.", "categoría"),
        "fuente_excel": ("texto", "Anexo Excel de data/raw/contraloria del que sale la fila.", "texto"),
        "hoja": ("texto", "Hoja del Excel con la base de obras.", "texto"),
    }),
    "contraloria_obras_panel": (CGR_EXCEL, {
        "codigo_infobras": ("texto", "Código de la obra en INFOBRAS (una fila por obra).", "código"),
        "primer_corte": ("fecha AAAA-MM-DD", "Primer corte (de los 13 con base) en que la obra aparece en la lista. Si es sep-2023 (primer_corte_censurado) la obra puede estar desde antes.", "fecha"),
        "ultimo_corte": ("fecha AAAA-MM-DD", "Último corte en que la obra aparece.", "fecha"),
        "n_cortes": ("entero", "Número de cortes en que aparece (de 13).", "cortes"),
        "sigue_en_ultimo_corte": ("lógico", "TRUE si la obra está en el último corte (jun-2026).", "lógico"),
        "cortes_faltantes_intermedios": ("entero", "Cortes entre el primero y el último en que la obra no aparece.", "cortes"),
        "reingresos": ("entero", "Veces que la obra salió de la lista y volvió a entrar.", "veces"),
        "primer_corte_censurado": ("lógico", "TRUE si primer_corte es el primer corte del panel (sep-2023): no se sabe desde cuándo está.", "lógico"),
        "cui_ultimo": ("texto", "CUI en el último corte en que aparece.", "código"),
        "sector_agrupado_ultimo": ("texto", "Sector agrupado en el último corte en que aparece.", "categoría"),
        "nivel_gobierno_ultimo": ("texto", "Nivel de gobierno en el último corte en que aparece.", "categoría"),
        "causal_paralizacion_ultimo": ("texto", "Causal de paralización en el último corte en que aparece.", "texto"),
        "cod_departamento_ultimo": ("texto de 2 dígitos", "Código INEI del departamento en el último corte en que aparece.", "código"),
        "departamento_ultimo": ("texto", "Departamento en el último corte en que aparece.", "texto"),
        "provincia_ultimo": ("texto", "Provincia en el último corte en que aparece.", "texto"),
        "distrito_ultimo": ("texto", "Distrito en el último corte en que aparece.", "texto"),
        "avance_primer_corte": ("decimal de 0 a 100", "Avance físico en el primer corte en que aparece.", "%"),
        "avance_ultimo_corte": ("decimal de 0 a 100", "Avance físico en el último corte en que aparece.", "%"),
        "fuente": ("texto", "Tabla de la que se construye el panel.", "texto"),
    }),
    "contraloria_pim": (CGR, {
        "anio_pim": ("entero", "Año del presupuesto (PIM) al que se refiere la tabla.", "año"),
        "ambito": ("texto", "nacional o departamento.", "categoría"),
        "categoria": ("texto", "NACIONAL o nombre del departamento (mayúsculas, sin tildes).", "texto"),
        "obras_con_pim": ("entero", "Obras paralizadas con PIM asignado.", "obras"),
        "pct_obras_con_pim": ("decimal", "Porcentaje de las obras que tienen PIM, tal como lo publica el informe.", "%"),
        "obras_sin_pim": ("entero", "Obras paralizadas sin PIM.", "obras"),
        "pct_obras_sin_pim": ("decimal", "Porcentaje de las obras que no tienen PIM.", "%"),
        "obras_total": ("entero", "Obras paralizadas (con + sin PIM).", "obras"),
        "costo_con_pim_soles": ("decimal", "Solo en la fila nacional: costo actualizado de las obras con PIM.", "soles"),
        "costo_sin_pim_soles": ("decimal", "Solo en la fila nacional: costo actualizado de las obras sin PIM.", "soles"),
        "costo_total_soles": ("decimal", "Solo en la fila nacional: costo actualizado total.", "soles"),
        "pim_soles": ("decimal", "Solo en la fila nacional: PIM asignado a las obras que lo tienen.", "soles"),
    }),
    "contraloria_modalidad_ejecucion": (CGR, {
        "tabla": ("texto", "Tabla del informe: por_modalidad (todas las modalidades) o administracion_directa_y_contrata (solo esas dos).", "categoría"),
        "categoria": ("texto", "Modalidad de ejecución, en mayúsculas y sin tildes (CONTRATA y APP unificados).", "categoría"),
        "nivel_gobierno": ("texto", "NACIONAL, REGIONAL, LOCAL o TOTAL. El orden de las columnas se decide con los totales del informe, no con el rótulo del PDF.", "categoría"),
        "obras_paralizadas": ("entero", "Obras paralizadas de esa categoría y nivel; vacío si el PDF dejó la celda vacía.", "obras"),
        "orden_niveles": ("texto", "verificado = el orden de las columnas de nivel coincide con los totales por nivel de gobierno del mismo informe.", "texto"),
        "pct_obras": ("decimal", "Solo en filas TOTAL: porcentaje de obras, tal como lo publica el informe (algunos informes no lo traen).", "%"),
        "pct_costo": ("decimal", "Solo en filas TOTAL: porcentaje del costo, tal como lo publica el informe.", "%"),
    }),
    "contraloria_evento_infobras": (CGR, {
        "tabla": ("texto", "Siempre por_evento.", "categoría"),
        "categoria": ("texto", "Evento registrado en INFOBRAS (sin la nota al pie).", "categoría"),
        "nivel_gobierno": ("texto", "NACIONAL, REGIONAL, LOCAL o TOTAL. El orden de las columnas se decide con los totales del informe: en el PDF de jun-2026 el rótulo dice Nacional-Regional-Local pero las cifras van en Local-Nacional-Regional.", "categoría"),
        "obras_paralizadas": ("entero", "Obras paralizadas de ese evento y nivel.", "obras"),
        "orden_niveles": ("texto", "verificado = el orden coincide con los totales por nivel de gobierno del mismo informe.", "texto"),
        "pct_obras": ("decimal", "No viene en la tabla de eventos (vacío).", "%"),
        "pct_costo": ("decimal", "Solo en filas TOTAL: porcentaje del costo.", "%"),
    }),
    "contraloria_servicios_control_total": (CGR, {
        "tipo_servicio": ("texto", "Tipo de servicio de control: PREVIO, SIMULTANEO o POSTERIOR.", "categoría"),
        "cantidad": ("entero", "Servicios de control efectuados a obras paralizadas.", "servicios"),
        "pct": ("decimal", "Porcentaje del total de servicios, tal como lo publica el informe.", "%"),
    }),
    "contraloria_servicios_control_nivel_gobierno": (CGR, {
        "tipo_servicio": ("texto", "Tipo de servicio de control: PREVIO, SIMULTANEO o POSTERIOR.", "categoría"),
        "nivel_gobierno": ("texto", "NACIONAL, REGIONAL, LOCAL o TOTAL.", "categoría"),
        "cantidad": ("entero", "Servicios de control a obras de ese nivel de gobierno.", "servicios"),
        "pct": ("decimal", "Porcentaje que publica el informe (de la columna de nivel o del total, según el informe).", "%"),
        "orden_niveles": ("texto", "Cómo se fijó el orden de las columnas de nivel: verificado con la narración del informe o según el encabezado del PDF.", "texto"),
    }),
    "contraloria_servicios_control_departamento": (CGR, {
        "departamento": ("texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        "control_previo": ("entero", "Servicios de control previo (una celda vacía en el PDF se guarda como 0).", "servicios"),
        "control_simultaneo": ("entero", "Servicios de control simultáneo; un servicio hecho a obras de varios departamentos se cuenta en cada uno (nota (*) del informe).", "servicios"),
        "control_posterior": ("entero", "Servicios de control posterior.", "servicios"),
        "total_servicios": ("entero", "Total de servicios del departamento; la suma de los departamentos supera el total nacional por la nota (*).", "servicios"),
    }),
    "contraloria_principales_obras": (CGR_EXCEL, {
        "orden_en_hoja": ("entero", "Orden de la obra en la hoja del anexo (dentro de cada departamento, por nivel de gobierno y costo).", "número"),
        "cui": ("texto", "Código Único de Inversiones.", "código"),
        "codigo_infobras": ("texto", "Código de la obra en INFOBRAS. Los anexos de 2025 en adelante no lo traen: se completa con la base obra por obra del mismo corte (ver codigo_infobras_de_base).", "código"),
        "descripcion_obra": ("texto", "Nombre de la inversión (proyecto público).", "texto"),
        "entidad": ("texto", "Entidad pública responsable.", "texto"),
        "nivel_gobierno": ("texto", "G. NACIONAL, G. REGIONAL o G. LOCAL.", "categoría"),
        "modalidad_ejecucion": ("texto", "Modalidad de ejecución.", "categoría"),
        "causal_codigo": ("texto", "Número de la causal oficial; vacío si el anexo no lo trae.", "código"),
        "causal_paralizacion": ("texto", "Causal de paralización oficial.", "texto"),
        "departamento": ("texto", "Departamento en mayúsculas y sin tildes.", "texto"),
        "codigo_infobras_de_base": ("lógico", "TRUE si codigo_infobras se completó desde la base obra por obra (contraloria_obras_corte) porque la hoja no lo traía.", "lógico"),
        "es_lista_completa": ("lógico", "TRUE en mar-2024, cuya hoja de principales obras trae la lista completa de obras (2 267 filas) y no solo las principales.", "lógico"),
        "fuente_excel": ("texto", "Anexo Excel de data/raw/contraloria del que sale la fila.", "texto"),
        "hoja": ("texto", "Hoja del Excel con la lista de principales obras.", "texto"),
    }),
}
POR_TABLA.update(POR_TABLA_ANEXOS)
# columnas cuyo nombre depende de los datos (una por corte): tabla -> (patron, (tipo, descripcion, unidad))
COLUMNAS_POR_PATRON = {
    "contraloria_obras_panel": (r"^avance_\d{4}-\d{2}-\d{2}$", ("decimal de 0 a 100", "Avance físico de la obra en el corte que indica el nombre de la columna; vacío si la obra no está en ese corte.", "%")),
}

# causa_grupo_origen y causa_grupo se agregan a fichas_infobras en la Fase 4; los demas usan COMUNES

# --- pasada ampliada por INFOBRAS (A4 + B1 + B2), src/scraper_infobras.py ---
_FICHA = POR_TABLA["fichas_infobras"][1]
_PERIODO = {
    "codigo_infobras": _FICHA["codigo_infobras"], "cui": _FICHA["cui"],
    "periodo": ("texto", "Año / mes del avance, como lo escribe INFOBRAS (por ejemplo «2026 / SEPTIEMBRE»).", "texto"),
    "anio": ("entero", "Año del avance (de la columna periodo).", "año"),
    "mes": ("entero", "Mes del avance, de 1 a 12 (de la columna periodo).", "mes"),
    "codigo_avance": ("texto", "Código del avance en INFOBRAS (enlace Mapa/Avance?codigo=…); vacío si la fila no tiene enlace.", "código"),
    "fecha_lectura": ("fecha AAAA-MM-DD", "Día en que se leyó la ficha en la pasada ampliada (no es el 2026-10-27 de fichas_infobras).", "fecha"),
}
POR_TABLA["infobras_avances"] = (INFOBRAS, {
    **_PERIODO,
    "fisico_programado": ("decimal", "Avance físico programado acumulado al mes, según la tabla «Avances de obra».", "%"),
    "fisico_real": ("decimal", "Avance físico real acumulado al mes, según la tabla «Avances de obra».", "%"),
    "estado": ("texto", "Estado de la obra en ese mes (En Ejecución, Paralizado, etc.), tal como lo escribe INFOBRAS.", "texto"),
})
POR_TABLA["infobras_paralizaciones"] = (INFOBRAS, {
    **_PERIODO,
    "tramo": ("entero", "Número del tramo de paralización de la obra: meses en estado Paralizado seguidos en la tabla de avances (1 = el más antiguo).", "número"),
    "posicion_tramo": ("texto", "Lugar del mes en su tramo: unico, inicio, medio o fin.", "categoría"),
    "detalle_leido": ("lógico", "TRUE si se abrió la página de detalle del avance. Con config.INFOBRAS_DETALLE_PARALIZACIONES = por_tramo solo se abren el primer y el último mes de cada tramo; los meses «medio» quedan sin detalle.", "lógico"),
    "causal_paralizacion": ("texto", "Causal registrada en el detalle del avance. Puede ser genérica («Catalogo de Obra», «-»).", "texto"),
    "fecha_paralizacion": ("fecha AAAA-MM-DD", "Fecha de paralización registrada en el detalle del avance.", "fecha"),
    "dias_paralizado": ("entero", "Días paralizada según el detalle del avance.", "días"),
    "comentario_fisico": _FICHA["comentario_fisico"], "comentario_valorizado": _FICHA["comentario_valorizado"],
})
POR_TABLA["infobras_fichas_ampliadas"] = (INFOBRAS, {
    **{c: _FICHA[c] for c in ("codigo_infobras", "cui", "nombre_obra", "entidad", "departamento", "provincia", "distrito", "ubigeo",
                              "ubigeo_metodo", "modalidad", "naturaleza", "tipo_obra", "monto_ejecucion_acumulado", "avance_fisico_pct",
                              "fecha_inicio", "fecha_fin_prevista", "periodo_ultimo_avance", "n_avances", "periodo_avance_paralizado",
                              "causal_paralizacion", "fecha_paralizacion", "dias_paralizado", "latitud", "longitud",
                              "avance_fisico_tabla", "url_ficha")},
    "fecha_lectura": _PERIODO["fecha_lectura"],
    "origen": ("texto", "Por qué se leyó: infobras_paralizada_2026-10-27 (las 198 de fichas_infobras) o contraloria_educacion_2026-06 (obras de Educación/Cultura del anexo de junio de 2026 que no estaban como Paralizada en INFOBRAS, tarea A4).", "categoría"),
    "estado": ("texto", "Estado de la obra en la ficha el día de la lectura (PARALIZADA, EN EJECUCIÓN, FINALIZADO, etc.).", "texto"),
    "estado_listado_todos": ("texto", "Estado según el listado de Educación con todos los estados exportado del buscador (data/raw/infobras).", "texto"),
    "monto_expediente_tecnico": ("decimal", "Monto del expediente técnico, del listado de Educación con todos los estados.", "soles"),
    "n_paralizados": ("entero", "Número de avances mensuales en estado Paralizado (filas de la obra en infobras_paralizaciones).", "avances"),
})
# --- ubicacion corregida por ubigeo y tramos de paralizacion ---
_DEP_INFOBRAS = ("texto", "Departamento tal como lo publica INFOBRAS (en mayúsculas y sin tildes). Difiere de departamento en 2 obras (135083 y 543835: INFOBRAS dice Huancavelica y son de Ica).", "texto")
POR_TABLA["fichas_infobras"][1]["departamento_infobras"] = _DEP_INFOBRAS
POR_TABLA["infobras_fichas_ampliadas"][1]["departamento_infobras"] = _DEP_INFOBRAS
for _t in ("fichas_infobras", "infobras_fichas_ampliadas"):
    POR_TABLA[_t][1]["departamento"] = ("texto", "Departamento del ubigeo (corregido), en mayúsculas y sin tildes. El que publica INFOBRAS está en departamento_infobras.", "texto")
    POR_TABLA[_t][1]["cod_departamento"] = ("texto de 2 dígitos", "Código INEI del departamento del ubigeo (Callao = 07). Texto, con cero a la izquierda.", "código")
POR_TABLA["infobras_paralizaciones"][1]["causal_paralizacion_tramo"] = ("texto", "Causal del tramo de paralización: la del detalle leído, repetida en los meses del medio del tramo (que no se abren).", "texto")
POR_TABLA["infobras_paralizaciones"][1]["fecha_paralizacion_tramo"] = ("fecha AAAA-MM-DD", "Fecha de paralización del tramo, repetida en los meses del medio del tramo.", "fecha")
# --- C3: los 4 niveles de logro de SICRECE ---
POR_TABLA["enla_niveles_logro"] = ("SICRECE (UMC, MINEDU), acceso público", {
    "region": ("texto", "Región ENLA como la escribe SICRECE (con tildes; Lima Metropolitana y Lima Provincias van separadas).", "texto"),
    "region_clave": ("texto", "Región en mayúsculas y sin tildes, para unir con otras tablas.", "texto"),
    "evaluacion": ("texto", "MUESTRAL o CENSAL (la ENLA 2024 de 4.° primaria fue censal).", "categoría"),
    "grado": ("texto", "Grado evaluado: 2.°, 4.° o 6.° de primaria; 2.° o 5.° de secundaria.", "texto"),
    "area": ("texto", "Área evaluada: lectura o matematica.", "texto"),
    "previo_al_inicio": ("decimal de 0 a 100", "% de estudiantes en el nivel Previo al inicio.", "%"),
    "en_inicio": ("decimal de 0 a 100", "% de estudiantes en el nivel En inicio.", "%"),
    "en_proceso": ("decimal de 0 a 100", "% de estudiantes en el nivel En proceso.", "%"),
    "satisfactorio": ("decimal de 0 a 100", "% de estudiantes en el nivel Satisfactorio (igual al de enla_regiones).", "%"),
    "archivo_json": ("texto", "Archivo de data/raw/enla del que sale (procedencia).", "texto"),
})
# --- MEF (Formato 12-B) y locales de ESCALE en construccion ---
POR_TABLA["mef_f12b_obras"] = ("MEF, Portal de Datos Abiertos: Formato 12-B de inversiones", {
    "cui": ("texto", "Código Único de Inversiones (CODIGO_UNICO del MEF).", "código"),
    "estado_mef": ("texto", "Estado de la inversión en el MEF (ACTIVO, DESACTIVADO TEMPORAL, CERRADO, ...).", "categoría"),
    "primer_devengado": ("texto AAAAMM", "Periodo del primer devengado registrado en el SIAF.", "periodo"),
    "ultimo_devengado": ("texto AAAAMM", "Periodo del último devengado registrado en el SIAF.", "periodo"),
    "devengado_hasta_anio_anterior_soles": ("decimal", "Devengado acumulado hasta el año anterior al del archivo.", "soles"),
    "devengado_anio_actual_soles": ("decimal", "Devengado del año del archivo.", "soles"),
    "pim_anio_actual_soles": ("decimal", "Presupuesto Institucional Modificado del año del archivo.", "soles"),
    "avance_fisico_mef_pct": ("decimal", "Avance físico declarado en el Formato 12-B.", "%"),
    "avance_ejecucion_mef_pct": ("decimal", "Avance de ejecución general declarado en el Formato 12-B.", "%"),
    "ult_problema": ("texto", "Última problemática registrada (texto libre de la entidad; se omiten los nombres de personas). No se guardan el estado situacional ni los próximos pasos: son narrativos y traen nombres de ingenieros.", "texto"),
    "fecha_ult_act_f12b": ("fecha", "Fecha de la última actualización del Formato 12-B.", "fecha"),
    "ult_periodo_reg_f12b": ("texto", "Último periodo de registro del Formato 12-B.", "periodo"),
    "inicio_ejec_fisica": ("fecha", "Fecha de inicio de ejecución física registrada en el Formato 12-B.", "fecha"),
    "culmina_ejec_fisica": ("fecha", "Fecha de término de ejecución física registrada en el Formato 12-B.", "fecha"),
    "devengado_total_soles": ("decimal", "devengado_hasta_anio_anterior_soles + devengado_anio_actual_soles.", "soles"),
    "archivo_mef": ("texto", "Archivo de data/raw/mef del que sale (procedencia).", "texto"),
})
POR_TABLA["escale_obras_en_local"] = ("ESCALE (Censo Educativo 2025): archivo de local, pregunta 2.1", {
    "anio": ("entero", "Año del Censo Educativo (solo 2025 trae el CUI de la obra).", "año"),
    "codlocal": ("texto", "Código del local educativo.", "código"),
    "ubigeo": ("texto de 6 dígitos", "Ubigeo del local (CODGEO).", "código"),
    "departamento": ("texto", "Departamento del local, como lo escribe ESCALE (mayúsculas).", "texto"),
    "provincia": ("texto", "Provincia del local.", "texto"),
    "distrito": ("texto", "Distrito del local.", "texto"),
    "tipo_construccion": ("texto", "TOTAL o PARCIAL: si la construcción/reconstrucción es de todo el local o de parte.", "categoría"),
    "cui": ("texto", "CUI de la obra que el local declara en el Censo (vacío si no lo informó).", "código"),
})
# --- producto final: el diccionario de cada tabla vive en producto_final.py junto a su construccion ---
import producto_final   # noqa: E402  (se importa aqui para no repetir las descripciones)
for _tabla, _columnas in producto_final.DICCIONARIO_FINAL.items():
    POR_TABLA[_tabla] = (producto_final.FUENTE_FINAL, {c: (tipo, desc, unidad) for c, tipo, desc, unidad in _columnas})
TABLAS_PDF = tuple(POR_TABLA)


def construir_diccionario_pdf():
    """Diccionario (tabla, columna, ...) de las tablas del proyecto PDF; los años se calculan con los datos."""
    filas = []
    for tabla, (fuente, columnas) in POR_TABLA.items():
        ruta = next((c / f"{tabla}.csv" for c in (config.PROCESSED, config.COMPONENTES, config.FINAL) if (c / f"{tabla}.csv").exists()), None)
        if ruta is None:
            continue
        datos = pd.read_csv(ruta, dtype={"cod_departamento": str, "codigo_infobras": str, "cui": str, "ubigeo": str}, encoding="utf-8-sig")
        patron = COLUMNAS_POR_PATRON.get(tabla)
        por_patron = lambda c: patron[1] if patron and re.match(patron[0], c) else None
        sin_describir = [c for c in datos.columns if c not in columnas and c not in COMUNES and por_patron(c) is None]
        if sin_describir:
            raise KeyError(f"{tabla}: columnas sin descripcion en diccionario_pdf.py: {sin_describir}")
        for columna in datos.columns:
            tipo, descripcion, unidad = columnas.get(columna) or por_patron(columna) or COMUNES[columna]
            fuente_col = PROCEDENCIA if columna in ("fuente_pdf", "pagina", "tabla_n") else fuente
            anios = ""
            if "anio" in datos and datos[columna].notna().any():
                anios = ", ".join(str(a) for a in sorted(datos.loc[datos[columna].notna(), "anio"].unique()))
            elif "fecha_lectura" in datos:
                anios = str(datos.fecha_lectura.iloc[0])[:4]
            filas.append({"tabla": tabla, "columna": columna, "tipo": tipo, "descripcion": descripcion, "unidad": unidad,
                          "fuente": fuente_col, "anios": anios, "notas": ""})
    return pd.DataFrame(filas)

### `inventario_datos` — Inventario de datos
Inventario de toda la data del proyecto: que tablas hay, cuantas filas, que periodo cubren y que necesidades cubren.

In [ ]:
%%modulo inventario_datos
"""Inventario de toda la data del proyecto: que tablas hay, cuantas filas, que periodo cubren y que necesidades cubren.

Las cifras se leen de los archivos (nada se escribe a mano salvo la descripcion de cada tabla y el cuadro de necesidades).
Salida: docs/inventario_datos.csv y docs/inventario_datos.md
"""
import pandas as pd

import config
import utils

# tabla -> (fuente, que es, una fila por, columna de periodo)
TABLAS = {
    "escale_infraestructura": ("ESCALE (Censo Educativo)", "Infraestructura de locales educativos publicos: servicios basicos, ruralidad, riesgo estructural", "distrito y año", "anio"),
    "enla_regiones": ("SICRECE (MINEDU-UMC)", "% de estudiantes en nivel Satisfactorio en Lectura y Matematica", "region, año, grado y evaluacion", "anio"),
    "mef_f12b_obras": ("MEF (Portal de Datos Abiertos)", "Devengado, PIM, ultimo devengado y problematica de cada obra (por CUI), del Formato 12-B", "CUI", "fecha_ult_act_f12b"),
    "escale_obras_en_local": ("ESCALE (Censo Educativo 2025)", "Locales que declaran estar en construccion, con el CUI de la obra", "local", "anio"),
    "dashboard_obras": ("PRODUCTO FINAL", "EL CSV UNICO del proyecto (alimenta el dashboard HTML): obras, cortes de la Contraloria, series por departamento y pais, causales y contexto (filtrar por tipo_registro)", "tipo_registro (obra, corte, departamento...)", "fecha_corte"),
    "obras_maestra": ("PRODUCTO FINAL", "Una fila por obra de Educacion/Cultura (429): INFOBRAS + ubigeo + Contraloria + MEF + ESCALE + causa", "obra", "fecha_lectura_infobras"),
    "resumen_region": ("PRODUCTO FINAL", "Una fila por region ENLA (26): obras paralizadas, causas, Contraloria, ESCALE y ENLA", "region", "escale_anio"),
    "serie_pais_educacion": ("PRODUCTO FINAL", "Cifras oficiales por corte: pais y sector Educacion (con la cifra corregida)", "corte", "fecha_corte"),
    "serie_educacion_departamento": ("PRODUCTO FINAL", "Obras de Educacion por departamento y corte (anexos Excel)", "corte y departamento", "fecha_corte"),
    "causales_por_corte": ("PRODUCTO FINAL", "Causales oficiales por corte con el grupo de causa del proyecto", "corte y causal", "fecha_corte"),
    "enla_niveles_logro": ("SICRECE (MINEDU-UMC)", "Los 4 niveles de logro (previo al inicio, en inicio, en proceso, satisfactorio) en Lectura y Matematica", "region, año, grado, evaluacion y area", "anio"),
    "enla_pdf": ("PDF de la EM 2022 (MINEDU-UMC)", "Lo mismo que enla_regiones, leido del PDF; trae tambien 2019", "region, año, grado y area", "anio"),
    "fichas_infobras": ("INFOBRAS (Contraloria)", "Obras educativas en estado Paralizada: ubicacion, entidad, montos, avance, causal, dias paralizada, grupo de causa", "obra", "fecha_lectura"),
    "cifras_oficiales_contraloria": ("Informes de obras paralizadas (Contraloria)", "Cifras oficiales: total nacional, por sector (incluye Educacion) y por departamento", "corte y ambito (pais, sector, departamento)", "fecha_corte"),
    "contraloria_causales": ("Informes de obras paralizadas (Contraloria)", "Causales de paralizacion declaradas por las entidades, con obras y montos", "corte y causal", "fecha_corte"),
    "contraloria_distribuciones": ("Informes de obras paralizadas (Contraloria)", "Nivel de gobierno, rango de costo y rango de avance fisico", "corte y categoria", "fecha_corte"),
    "contraloria_educacion_departamento": ("Anexos Excel (Contraloria)", "Obras de Educacion paralizadas por departamento", "corte y departamento", "fecha_corte"),
    "contraloria_obras_corte": ("Anexos Excel (Contraloria)", "Datos por obra y corte: causal oficial, nivel de gobierno, sector, avance fisico, tipo, año de inicio, plazo, provincia, distrito, evento, CUI", "obra y corte (13 cortes)", "corte"),
    "contraloria_obras_panel": ("Anexos Excel (Contraloria)", "Panel historico por obra: primer y ultimo corte, numero de cortes, si sigue, reingresos y avance por corte", "obra", "ultimo_corte"),
    "contraloria_pim": ("Informes de obras paralizadas (Contraloria)", "Obras con y sin presupuesto (PIM) por departamento y total nacional", "corte y departamento (o nacional)", "fecha_corte"),
    "contraloria_modalidad_ejecucion": ("Informes de obras paralizadas (Contraloria)", "Obras y costo por modalidad de ejecucion y nivel de gobierno", "corte, tabla, modalidad y nivel", "fecha_corte"),
    "contraloria_evento_infobras": ("Informes de obras paralizadas (Contraloria)", "Obras y costo por evento registrado en INFOBRAS y nivel de gobierno", "corte, evento y nivel", "fecha_corte"),
    "contraloria_servicios_control_total": ("Informes de obras paralizadas (Contraloria)", "Servicios de control efectuados por tipo (previo, simultaneo, posterior)", "corte y tipo de servicio", "fecha_corte"),
    "contraloria_servicios_control_nivel_gobierno": ("Informes de obras paralizadas (Contraloria)", "Servicios de control por tipo y nivel de gobierno", "corte, tipo y nivel", "fecha_corte"),
    "contraloria_servicios_control_departamento": ("Informes de obras paralizadas (Contraloria)", "Servicios de control por departamento y tipo (2023 en adelante)", "corte y departamento", "fecha_corte"),
    "contraloria_principales_obras": ("Anexos Excel (Contraloria)", "Principales obras paralizadas por departamento (entidad, costo, causal, modalidad)", "obra y corte", "fecha_corte"),
    "citas_educacion": ("Informes de obras paralizadas (Contraloria)", "Oraciones de los informes sobre educacion, con archivo y pagina", "oracion", "fecha_corte"),
    "frecuencia_terminos": ("Informes de obras paralizadas (Contraloria)", "Terminos y pares de terminos mas frecuentes por año", "año, tipo y termino", "anio"),
    # --- pasada ampliada por INFOBRAS (A4 + B1 + B2) ---
    "infobras_avances": ("INFOBRAS (Contraloria)", "Avances mensuales de cada obra: fisico programado y real, estado del mes (B1)", "obra y mes", "anio"),
    "infobras_paralizaciones": ("INFOBRAS (Contraloria)", "Meses en estado Paralizado de cada obra, agrupados en tramos, con causal, fecha y dias (B2)", "obra y mes paralizado", "anio"),
    "infobras_fichas_ampliadas": ("INFOBRAS (Contraloria)", "Ficha actual de las 198 obras paralizadas y de las obras de Educacion de la Contraloria que ya no estan como Paralizada, con su estado (A4) y ubigeo", "obra", "fecha_lectura"),
}

# necesidad del proyecto -> (estado, donde esta, comentario). Estado: SI / PARCIAL / NO
# Cifras de la corrida del 2026-11-12 (INFOBRAS leido el 27-oct y el 11-nov-2026; Contraloria hasta jun-2026; MEF: foto mensual)
NECESIDADES = [
    ("Lista de obras educativas paralizadas (estado, monto, ubicacion)", "SI", "obras_maestra (infobras_fichas_ampliadas)",
     "429 obras: las 198 paralizadas en INFOBRAS al 27-oct + las de Educacion de la Contraloria (jun-2026); estado leido el 11-nov-2026 (179 siguen paralizadas). Es una foto, no una serie"),
    ("Serie historica de obras paralizadas por obra", "PARCIAL", "contraloria_obras_corte, contraloria_obras_panel; infobras_avances",
     "Contraloria: 13 cortes (sep-2023 a jun-2026); antes solo cifras agregadas de los PDF. INFOBRAS: avance mensual de cada una de las 429 obras"),
    ("Cifras oficiales para validar la base (pais, Educacion, departamento)", "SI", "cifras_oficiales_contraloria",
     "21 cortes; INFOBRAS y la Contraloria son poblaciones distintas (198 vs 341 obras de Educacion, 110 en comun; ver hallazgos_pdf.md)"),
    ("Causas de paralizacion", "SI", "obras_maestra.causa_grupo_final / causa_macro; contraloria_causales",
     "Por obra: NLP con 3 fuentes (causal oficial, comentarios de INFOBRAS, problematica del MEF); 417 de 429 con causa. Comentarios y causal oficial coinciden en 49 % de las obras con ambas"),
    ("Montos (costo actualizado, saldo, expediente tecnico)", "SI", "cifras_oficiales_contraloria, contraloria_obras_corte, obras_maestra",
     "98 filas de la Contraloria con costo 0 o vacio y 9 con saldo mayor que el costo (marcadas, sin corregir)"),
    ("Gasto devengado por obra (MEF)", "PARCIAL", "mef_f12b_obras, obras_maestra",
     "Formato 12-B por CUI: 361 de las 414 obras con CUI valido aparecen; 15 tienen CUI NO ASOCIADO"),
    ("Tiempo de paralizacion", "SI", "infobras_paralizaciones, obras_maestra",
     "Las 429 obras: meses paralizados agrupados en tramos, con fecha y dias (8 739 meses; detalle leido del primer y ultimo mes de cada tramo). Los PDF no lo traen"),
    ("Nivel de gobierno", "PARCIAL", "contraloria_obras_corte; contraloria_distribuciones",
     "Por obra solo en las obras de la lista de la Contraloria; en INFOBRAS solo el nombre de la entidad"),
    ("Departamento / provincia / distrito de cada obra", "SI", "obras_maestra (ubigeo, latitud, longitud)",
     "Ubigeo INEI en las 429 (4 corregidas a mano: INFOBRAS traia mal la ubicacion). Coordenadas en 412; 17 no las tienen en INFOBRAS"),
    ("Infraestructura escolar por distrito", "SI", "escale_infraestructura, escale_distrito_ampliado",
     "5 indicadores 2022 a 2025; 16 indicadores mas solo para 2025"),
    ("Aprendizajes (ENLA) por region", "SI", "enla_regiones, enla_niveles_logro, enla_pdf",
     "2022 a 2025 (+2019 en enla_pdf); solo por region (no hay UGEL ni distrito); SICRECE no devolvio 3 combinaciones de año y grado"),
    ("Texto citable de los informes", "PARCIAL", "citas_educacion", "Pocas citas: los informes hablan poco de educacion"),
    ("Presupuesto asignado (PIM) por departamento", "PARCIAL", "contraloria_pim", "Solo 4 cortes (set-2025 a jun-2026): los informes anteriores no traen la tabla"),
    ("Datos de ESCALE o ENLA de 2026", "NO", "-", "Aun no publicados")]


def _periodo(tabla, columna):
    if columna not in tabla or tabla[columna].dropna().empty:
        return ""
    valores = tabla[columna].dropna().astype(str)
    if columna == "anio":
        return f"{valores.min()} a {valores.max()}"
    return f"{valores.min()} a {valores.max()}" if valores.min() != valores.max() else valores.min()


def inventario_tablas():
    filas = []
    for nombre, (fuente, que_es, grano, col_periodo) in TABLAS.items():
        carpeta = config.COMPONENTES if fuente == "PRODUCTO FINAL" else config.PROCESSED
        if nombre == "dashboard_obras":
            carpeta = config.FINAL
        ruta = carpeta / f"{nombre}.csv"
        archivo = ruta.relative_to(config.RAIZ).as_posix()
        if not ruta.exists():
            filas.append({"tabla": nombre, "fuente": fuente, "que_es": que_es, "una_fila_por": grano, "filas": 0, "columnas": 0,
                          "periodo": "NO EXISTE", "archivo": archivo})
            continue
        datos = pd.read_csv(ruta, encoding="utf-8-sig", low_memory=False)
        filas.append({"tabla": nombre, "fuente": fuente, "que_es": que_es, "una_fila_por": grano, "filas": len(datos),
                      "columnas": datos.shape[1], "periodo": _periodo(datos, col_periodo), "archivo": archivo})
    return pd.DataFrame(filas)


def inventario_raw():
    """Archivos descargados por fuente (data/raw)."""
    filas = []
    if not config.RAW.exists():   # modo --desde-tablas (por ejemplo en GitHub): los datos crudos no se publican
        return pd.DataFrame(columns=["fuente", "archivos", "mb", "anios"])
    for carpeta in sorted(p for p in config.RAW.iterdir() if p.is_dir()):
        archivos = [a for a in carpeta.rglob("*") if a.is_file()]
        anios = sorted({a.parent.name for a in archivos if a.parent.name.isdigit()})
        filas.append({"fuente": carpeta.name, "archivos": len(archivos), "mb": round(sum(a.stat().st_size for a in archivos) / 1024 / 1024, 1),
                      "anios": ", ".join(anios)})
    return pd.DataFrame(filas)


def _md(tabla):
    columnas = list(tabla.columns)
    lineas = ["| " + " | ".join(columnas) + " |", "|" + "|".join("---" for _ in columnas) + "|"]
    lineas += ["| " + " | ".join("" if pd.isna(v) else str(v) for v in fila) + " |" for fila in tabla.itertuples(index=False)]
    return "\n".join(lineas)


def construir():
    """Escribe docs/inventario_datos.csv y .md; devuelve (tablas, raw, necesidades)."""
    tablas, raw = inventario_tablas(), inventario_raw()
    necesidades = pd.DataFrame(NECESIDADES, columns=["necesidad", "estado", "donde", "comentario"])
    tablas.to_csv(config.DOCS / "inventario_datos.csv", index=False, encoding="utf-8-sig")
    texto = f"""# Inventario de datos del proyecto

Generado automaticamente por `src/inventario_datos.py` ({pd.Timestamp.today():%Y-%m-%d}). Cifras leidas de los archivos.

## Tablas finales (data/processed)

{_md(tablas.drop(columns='archivo'))}

## Archivos descargados (data/raw, nunca se modifican)

{_md(raw)}

## Necesidades del proyecto y que cubrimos

{_md(necesidades)}

SI = cubierto; PARCIAL = hay algo pero con limites (ver comentario); NO = no tenemos.
Detalle de columnas: `docs/diccionario.csv`. Validaciones y limites: `docs/hallazgos_pdf.md`.
"""
    (config.DOCS / "inventario_datos.md").write_text(texto, encoding="utf-8")
    utils.log(f"INVENTARIO DE DATOS: {len(tablas)} tablas, {int(raw.archivos.sum())} archivos en data/raw; escrito docs/inventario_datos.md")
    return tablas, raw, necesidades

### `main` — Orquestación de ESCALE/ENLA/Contraloría/INFOBRAS
Punto de entrada. Ejemplo: python src/main.py --anio-inicio 2022 --solo escale

In [ ]:
%%modulo main
"""Punto de entrada. Ejemplo: python src/main.py --anio-inicio 2022 --solo escale

Orden: descarga (ESCALE, ENLA, Contraloria e INFOBRAS) -> extraccion a tablas -> validacion -> diccionario de datos -> Excel.
"""
import argparse

import config
import descarga_mef
import diccionario
import exportar
import extraccion
import scraper_contraloria
import scraper_enla
import scraper_escale
import scraper_infobras
import utils


def leer_argumentos():
    p = argparse.ArgumentParser(description="Descarga automatizada de ESCALE y ENLA y extraccion a tablas")
    p.add_argument("--anio-inicio", type=int, default=config.ANIO_INICIO, help="primer anio a descargar")
    p.add_argument("--solo", choices=["escale", "enla", "contraloria", "infobras", "mef"], help="ejecutar solo una fuente")
    p.add_argument("--forzar", action="store_true", help="descargar de nuevo aunque ya exista")
    p.add_argument("--headless", action="store_true", help="navegador sin ventana (por defecto es visible)")
    return p.parse_args()


def descargar(args):
    """Corre los scrapers. Si una fuente falla, se registra y se sigue con la otra."""
    fuentes = {"escale": scraper_escale, "enla": scraper_enla, "contraloria": scraper_contraloria,
               "infobras": scraper_infobras, "mef": descarga_mef}
    resultados = []
    for nombre, modulo in fuentes.items():
        if args.solo and args.solo != nombre:
            continue
        if getattr(args, "fuentes", None) and nombre not in args.fuentes:
            continue
        try:
            resultados.append(modulo.ejecutar(args.anio_inicio, args.forzar, args.headless))
        except Exception as error:
            utils.log(f"{nombre.upper()}: error inesperado, se continua con la siguiente fuente: {type(error).__name__} {error}")
            resultado = utils.nuevo_resultado(nombre.upper())
            resultado["errores"].append(("fuente completa", type(error).__name__))
            resultados.append(resultado)
    utils.imprimir_resumen(resultados)


def extraer(args):
    """Convierte lo descargado en data/processed y valida las tablas."""
    if args.solo in ("contraloria", "infobras"):
        return   # esas fuentes se procesan con src/main_pdf.py
    escale = enla = None
    try:
        if args.solo in (None, "escale"):
            escale, _ = extraccion.construir_escale(args.anio_inicio)
        if args.solo in (None, "enla"):
            enla = extraccion.construir_enla()
            extraccion.construir_enla_niveles()
        extraccion.validar(escale, enla)
        diccionario.escribir_diccionario()
        exportar.escribir_excel()
    except Exception as error:
        utils.log(f"EXTRACCION: fallo ({type(error).__name__}: {error}). Los archivos descargados siguen en data/raw.")


def main():
    args = leer_argumentos()
    utils.log(f"Inicio de corrida: anio_inicio={args.anio_inicio}, solo={args.solo}, forzar={args.forzar}, headless={args.headless}")
    descargar(args)
    extraer(args)
    utils.log("Fin de corrida")


if __name__ == "__main__":
    main()

### `main_pdf` — Orquestación del proyecto PDF
Proyecto PDF: inventario, extraccion de tablas y texto de los PDF descargados.

In [ ]:
%%modulo main_pdf
"""Proyecto PDF: inventario, extraccion de tablas y texto de los PDF descargados.

Uso (desde la carpeta del proyecto):  .venv/Scripts/python.exe src/main_pdf.py [--forzar] [--hasta N]
Un documento ya procesado no se vuelve a procesar salvo con --forzar.
"""
import argparse
import sys

import contraloria_excel
import leer_pdf
import diccionario
import tablas_finales
import tablas_pdf
import texto_nlp
from leer_pdf import log


def ejecutar(hasta=5, forzar=False):
    """Corre las fases 1 a `hasta` del proyecto PDF (usado tambien por pipeline_completo.py)."""
    log("=" * 60)
    log(f"main_pdf: inicio (forzar={forzar}, hasta fase {hasta})")
    leer_pdf.construir_inventario(forzar)
    if hasta >= 2:
        catalogo, procesados, omitidos = tablas_pdf.extraer_todo(forzar)
        log(f"Fase 2: documentos procesados {procesados}, omitidos (ya hechos) {omitidos}; filas en el catalogo {len(catalogo)}")
    if hasta >= 3:
        tablas_finales.ejecutar_fase3()
        contraloria_excel.ejecutar_anexos_excel()   # A2, A3, B3, B4, B6 y B7: tablas por obra, panel, PIM, modalidad/evento/servicios y principales obras
    if hasta >= 4:
        texto_nlp.ejecutar_fase4()
        import scraper_infobras   # (A4+B1+B2): rearma las tablas de la pasada ampliada desde su cache; sin cache no hace nada
        scraper_infobras.construir_csv_ampliados()
    if hasta >= 5:
        diccionario.escribir_diccionario()
    log("main_pdf: fin")


def main():
    parser = argparse.ArgumentParser(description="Inventario y extraccion de tablas de los PDF")
    parser.add_argument("--forzar", action="store_true", help="volver a procesar todo aunque ya este hecho")
    parser.add_argument("--hasta", type=int, default=5, help="ultima fase a ejecutar (1 inventario, 2 tablas, 3 tablas finales, 4 texto, 5 diccionario)")
    argumentos = parser.parse_args()
    ejecutar(argumentos.hasta, argumentos.forzar)


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    main()

### `dashboard_html` — Dashboard en HTML
Dashboard en HTML: un solo archivo que se abre en cualquier navegador, sin instalar nada y sin internet.

In [ ]:
%%modulo dashboard_html
"""Dashboard en HTML: un solo archivo que se abre en cualquier navegador, sin instalar nada y sin internet.

Lee los productos ya validados del pipeline (no recalcula nada por su cuenta):
  data/final/dashboard_obras.csv     obras, series de la Contraloria, causales y contexto por region
  docs/validacion_final.csv          comprobaciones del producto final
  docs/auditoria_calidad.csv         auditoria de calidad del CSV unico
  docs/registro_fuentes.csv          archivos descargados por fuente
  docs/inventario_datos.csv          tablas del proyecto (filas, periodo)
  docs/limites_y_calidad.md          limites y calidad del dato
y escribe resultados/dashboard.html (Plotly incluido dentro del archivo).

Pestanas: Resumen, Mapa, Causas, Regiones (pregunta guia), Obras (ficha), Proceso (como se construyo) y Limites.
Solo muestra obras de educacion basica (colegios): las deportivas, culturales y de educacion superior se excluyen del analisis y se listan en Proceso.
Filtros comunes: estado (por defecto paralizadas hoy), region y causa.
"""
import html
import json
import re
from datetime import datetime

import numpy as np
import pandas as pd

import config
import utils

SALIDA = config.RAIZ / "resultados" / "dashboard.html"
# silueta de Sudamerica para el mapa (Natural Earth via el paquete sane-topojson, licencia MIT): va dentro del HTML para que funcione sin internet
TOPOJSON = config.RAIZ / "src" / "recursos" / "south-america_50m.json"
RECURSOS = config.RAIZ / "src" / "recursos"
# limites de departamentos y provincias y capitales (github.com/juaneladio/peru-geojson, licencia MPL 2.0; ver src/recursos/FUENTE_peru-geojson.txt)
PERU_DEPARTAMENTOS, PERU_PROVINCIAS, PERU_CIUDADES = RECURSOS / "peru_departamentos.geojson", RECURSOS / "peru_provincias.geojson", RECURSOS / "peru_ciudades.json"
PLOTLY_JS = config.RAIZ / "src" / "recursos" / "plotly.min.js"   # plotly.js (licencia MIT) va con el proyecto: no hace falta instalar plotly

COLUMNAS_OBRA = ["codigo_infobras", "cui", "nombre_obra", "entidad", "departamento", "provincia", "distrito", "ubigeo", "region_enla",
                 "latitud_mapa", "longitud_mapa", "coordenada_origen", "tipo_infraestructura", "estado_infobras", "paralizada_hoy", "origen",
                 "causa_macro", "causa_grupo_final", "causa_fuente", "causal_oficial", "nivel_gobierno", "modalidad",
                 "monto_expediente_tecnico", "costo_actualizado_soles", "devengado_total_soles", "avance_fisico_pct",
                 "fecha_inicio", "fecha_paralizacion", "dias_paralizado", "dias_desde_paralizacion", "n_tramos_paralizacion", "alertas_calidad", "escale_pct_3_servicios"]
COLUMNAS_REGION = ["region_enla", "region_escale_locales_publicos", "region_escale_pct_3_servicios", "region_escale_pct_rural",
                   "region_escale_pct_riesgo_estructural", "region_enla_4p_lectura_satisfactorio", "region_enla_4p_matematica_satisfactorio",
                   "region_enla_2s_lectura_satisfactorio", "region_enla_2s_matematica_satisfactorio", "region_enla_4p_anio", "region_enla_2s_anio"]
NUMERICAS = ["latitud_mapa", "longitud_mapa", "monto_expediente_tecnico", "costo_actualizado_soles", "devengado_total_soles",
             "avance_fisico_pct", "dias_paralizado", "dias_desde_paralizacion", "n_tramos_paralizacion", "escale_pct_3_servicios", "n_obras", "costo_actualizado_soles",
             "pct_obras"] + COLUMNAS_REGION[1:]


def _registros(tabla):
    """DataFrame -> lista de dicts lista para JSON (NaN -> null; numeros como numeros)."""
    t = tabla.copy()
    for c in t.columns:
        if c in NUMERICAS:
            t[c] = pd.to_numeric(t[c], errors="coerce").round(4)
    t = t.astype(object).where(t.notna(), None)
    return t.to_dict(orient="records")


def _md_a_html(texto):
    """Markdown sencillo (titulos, listas, negrita, codigo) -> HTML. Suficiente para docs/limites_y_calidad.md."""
    salida, en_lista = [], False
    for linea in texto.splitlines():
        l = html.escape(linea.rstrip())
        l = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", l)
        l = re.sub(r"`(.+?)`", r"<code>\1</code>", l)
        if l.startswith("- "):
            if not en_lista:
                salida.append("<ul>")
                en_lista = True
            salida.append(f"<li>{l[2:]}</li>")
            continue
        if en_lista:
            salida.append("</ul>")
            en_lista = False
        if l.startswith("## "):
            salida.append(f"<h3>{l[3:]}</h3>")
        elif l.startswith("# "):
            continue          # el titulo lo pone la pestana
        elif l.strip():
            salida.append(f"<p>{l}</p>")
    if en_lista:
        salida.append("</ul>")
    return "\n".join(salida)


def _proceso(dash, validacion, calidad, fuentes, inventario, n_diccionario):
    """Pasos del pipeline con los numeros que salen de esta corrida (para la pestana Proceso)."""
    filas = dict(zip(inventario.tabla, pd.to_numeric(inventario.filas, errors="coerce")))
    disponibles = fuentes[fuentes.estado.isin(["ok", "omitido"])].drop_duplicates("archivo")   # el registro acumula una fila por corrida
    por_fuente = disponibles.groupby("fuente").size().to_dict()
    errores_calidad = int((calidad.nivel.str.upper() == "ERROR").sum()) if len(calidad) else 0
    avisos_calidad = int((calidad.nivel.str.lower() == "aviso").sum()) if len(calidad) else 0
    obras = dash[dash.tipo_registro == "obra_infobras"]
    n = lambda tabla: int(filas.get(tabla, 0) or 0)
    return [
        {"paso": "1. Descarga", "detalle": "Portales públicos: ESCALE, SICRECE (ENLA), Contraloría, MEF e INFOBRAS. Lo ya descargado se salta.",
         "cifra": f"{len(disponibles)} archivos", "extra": ", ".join(f"{k}: {v}" for k, v in sorted(por_fuente.items()))},
        {"paso": "2. ESCALE y ENLA", "detalle": "Infraestructura escolar por distrito y aprendizajes por región.",
         "cifra": f"{n('escale_infraestructura'):,} filas", "extra": f"ENLA: {n('enla_regiones')} filas por región y grado"},
        {"paso": "3. Datos personales", "detalle": "Se quitan nombres de personas de los textos libres antes de guardar.", "cifra": "Limpieza", "extra": "se comprueba en el paso 7"},
        {"paso": "4. PDF, anexos e INFOBRAS", "detalle": "Cifras oficiales de los informes de la Contraloría y lectura de las fichas HTML de INFOBRAS (semiestructurado).",
         "cifra": f"{len(obras)} obras", "extra": f"{n('infobras_avances'):,} avances mensuales · {n('contraloria_obras_corte'):,} filas obra-corte"},
        {"paso": "5. Producto final", "detalle": "Une todo por obra (CUI, ubigeo) y por región; clasifica la causa (reglas + NLP) y el tipo de infraestructura.",
         "cifra": f"{len(dash):,} filas", "extra": f"{int(validacion.ok.astype(str).eq('True').sum())} de {len(validacion)} comprobaciones pasan"},
        {"paso": "6. Diccionario", "detalle": "Cada columna de cada tabla con su tipo, descripción y unidad.", "cifra": f"{n_diccionario} columnas", "extra": "docs/diccionario.csv"},
        {"paso": "7. Auditoría de datos personales", "detalle": "Busca nombres, DNI y teléfonos en las tablas finales.", "cifra": "0 hallazgos", "extra": ""},
        {"paso": "8. Inventario", "detalle": "Qué tablas hay, de qué fuente y qué período cubren.", "cifra": f"{len(inventario)} tablas", "extra": ""},
        {"paso": "9. Auditoría de calidad", "detalle": "Tipos, rangos, códigos y coordenadas del CSV único.", "cifra": f"{errores_calidad} errores", "extra": f"{avisos_calidad} avisos (ver Límites)"},
        {"paso": "10. Dashboard", "detalle": "Este archivo: se genera solo con los productos validados.", "cifra": "dashboard.html", "extra": ""},
    ]


def _distritos(regiones_validas):
    """Todos los distritos del pais con sus indicadores de ESCALE (ultimo anio) y su region ENLA (Lima se separa en Metropolitana y Provincias)."""
    e = pd.read_csv(config.PROCESSED / "escale_infraestructura.csv", dtype={"ubigeo": str}, encoding="utf-8-sig")
    e = e[e.anio == e.anio.max()].copy()
    e["region_enla"] = np.where(e.departamento != "LIMA", e.departamento, np.where(e.provincia == "LIMA", "LIMA METROPOLITANA", "LIMA PROVINCIAS"))
    fuera = sorted(set(e.region_enla) - set(regiones_validas))
    if fuera:
        utils.log(f"AVISO DASHBOARD HTML: regiones de ESCALE que no estan en ENLA: {fuera}")
    cols = ["ubigeo", "region_enla", "provincia", "distrito", "n_locales_publicos", "pct_3_servicios", "pct_rural", "pct_riesgo_estructural"]
    for c in cols[4:]:
        e[c] = pd.to_numeric(e[c], errors="coerce").round(1)
    return e[cols].astype(object).where(e[cols].notna(), None).to_dict(orient="records"), int(e.anio.max())


MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]


def _cobertura(obras, serie, regiones, lectura):
    """Desde que anio hay informacion de cada fuente (texto corto para el encabezado)."""
    colegios = obras[obras.tipo_infraestructura == "educacion_basica"]
    anio = lambda col: pd.to_datetime(colegios[col], errors="coerce").dt.year
    corte = pd.to_datetime(serie.fecha_corte, errors="coerce")
    mes = lambda f: f"{MESES[f.month - 1]}-{f.year}"
    escale = pd.read_csv(config.PROCESSED / "escale_infraestructura.csv", usecols=["anio"], encoding="utf-8-sig").anio
    enla = pd.to_numeric(regiones[[c for c in regiones.columns if c.startswith("region_enla_") and c.endswith("_anio")]].stack(), errors="coerce")
    return (f"Información: obras iniciadas desde {int(anio('fecha_inicio').min())} y paralizaciones registradas desde {int(anio('fecha_paralizacion').min())} "
            f"(INFOBRAS, leído el {lectura}) · Contraloría de {mes(corte.min())} a {mes(corte.max())} · ESCALE {int(escale.min())}–{int(escale.max())} · "
            f"ENLA {int(enla.min())}–{int(enla.max())} · MEF: Formato 12-B (foto mensual).")


def _archivos_embebidos():
    """CSV de descarga comprimidos dentro del HTML: el tablero no depende de su carpeta."""
    import base64, gzip
    def uno(ruta, nombre):
        return {"nombre": nombre, "ruta": str(ruta.relative_to(config.RAIZ)).replace("\\", "/"),
                "datos": base64.b64encode(gzip.compress(ruta.read_bytes(), 9)).decode("ascii")}
    return json.dumps({"csv": uno(config.FINAL / "dashboard_obras.csv", "dashboard_obras.csv"),
                       "diccionario": uno(config.DOCS / "diccionario.csv", "diccionario.csv")})


def _punto_etiqueta(geom):
    """Centroide del poligono mas grande (lon, lat): donde va el nombre de la region o provincia."""
    poligonos = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    def area_centro(anillo):
        a = cx = cy = 0.0
        for (x0, y0), (x1, y1) in zip(anillo, anillo[1:] + anillo[:1]):
            c = x0 * y1 - x1 * y0; a += c; cx += (x0 + x1) * c; cy += (y0 + y1) * c
        return (abs(a) / 2, cx / (3 * a), cy / (3 * a)) if a else (0, anillo[0][0], anillo[0][1])
    _, x, y = max((area_centro(pol[0]) for pol in poligonos), key=lambda t: t[0])
    return round(x, 3), round(y, 3)


# ajustes a mano donde el centroide queda mal ubicado o se cruza con otro nombre (lon, lat)
AJUSTE_ETIQUETA = {"LIMA": (-76.55, -11.45), "ICA": (-75.45, -14.45), "TUMBES": (-80.5, -3.85), "LAMBAYEQUE": (-80.15, -6.45), "CAJAMARCA": (-78.6, -6.95),
                   "AMAZONAS": (-78.05, -5.3), "MOQUEGUA": (-70.95, -16.75), "TACNA": (-70.35, -17.6), "PASCO": (-75.4, -10.45),
                   "HUANCAVELICA": (-75.05, -12.85), "AYACUCHO": (-74.15, -14.1), "APURIMAC": (-72.95, -14.25), "CUSCO": (-71.95, -12.85),
                   "LA LIBERTAD": (-78.25, -7.95), "ANCASH": (-77.6, -9.45), "PIURA": (-80.25, -5.15)}


def _cartografia():
    """Regiones y provincias del Peru (GeoJSON), puntos para sus nombres y ciudades principales: el mapa no necesita internet."""
    if not (PERU_DEPARTAMENTOS.exists() and PERU_PROVINCIAS.exists()):
        utils.log("AVISO DASHBOARD HTML: faltan los limites del Peru en src/recursos; el mapa se dibuja sin regiones")
        return None
    dep = json.loads(PERU_DEPARTAMENTOS.read_text(encoding="utf-8"))
    prov = json.loads(PERU_PROVINCIAS.read_text(encoding="utf-8"))
    etiquetas_dep = []
    for f in dep["features"]:
        nombre = f["properties"]["departamento"]
        lon, lat = AJUSTE_ETIQUETA.get(nombre) or _punto_etiqueta(f["geometry"])
        if nombre != "CALLAO":   # es muy pequeno: lo nombra la ciudad
            etiquetas_dep.append({"nombre": nombre, "lon": lon, "lat": lat})
    etiquetas_prov = [dict(zip(("lon", "lat"), _punto_etiqueta(f["geometry"])), nombre=f["properties"]["provincia"], departamento=f["properties"]["departamento"]) for f in prov["features"]]
    ciudades = json.loads(PERU_CIUDADES.read_text(encoding="utf-8")) if PERU_CIUDADES.exists() else []
    return {"departamentos": dep, "provincias": prov, "etiquetas_dep": etiquetas_dep, "etiquetas_prov": etiquetas_prov, "ciudades": ciudades}


def construir():
    dash = pd.read_csv(config.FINAL / "dashboard_obras.csv", dtype=str, encoding="utf-8-sig", low_memory=False)
    validacion = pd.read_csv(config.DOCS / "validacion_final.csv", dtype=str, encoding="utf-8-sig")
    calidad = pd.read_csv(config.DOCS / "auditoria_calidad.csv", dtype=str, encoding="utf-8-sig")
    fuentes = pd.read_csv(config.DOCS / "registro_fuentes.csv", dtype=str, encoding="utf-8-sig")
    inventario = pd.read_csv(config.DOCS / "inventario_datos.csv", dtype=str, encoding="utf-8-sig")
    limites = (config.DOCS / "limites_y_calidad.md").read_text(encoding="utf-8") if (config.DOCS / "limites_y_calidad.md").exists() else ""

    tipo = dash.tipo_registro
    obras = dash.loc[tipo == "obra_infobras", COLUMNAS_OBRA].copy()
    obras["paralizada_hoy"] = obras.paralizada_hoy.astype(str).str.lower().eq("true")
    # Nombres del distrito y la provincia segun el ubigeo (ESCALE/INEI): en 4 obras INFOBRAS trae mal la ubicacion y el ubigeo ya esta corregido
    escale = pd.read_csv(config.PROCESSED / "escale_infraestructura.csv", dtype={"ubigeo": str}, encoding="utf-8-sig", usecols=["ubigeo", "anio", "provincia", "distrito"])
    nombres = escale.sort_values("anio").drop_duplicates("ubigeo", keep="last").set_index("ubigeo")
    for col in ("distrito", "provincia"):
        obras[col] = obras.ubigeo.map(nombres[col]).fillna(obras[col])
    serie = dash.loc[tipo.isin(["pais_educacion_corte", "pais_total_corte"]), ["tipo_registro", "fecha_corte", "n_obras", "costo_actualizado_soles", "pct_obras"]]
    causales = (dash.loc[tipo == "causal_corte"].assign(n=lambda d: pd.to_numeric(d.n_obras, errors="coerce"))
                .groupby(["fecha_corte", "causa_macro"], as_index=False).n.sum())
    regiones = dash.loc[tipo == "contexto_region", COLUMNAS_REGION]
    validacion = validacion.assign(**{c: validacion[c].str.replace(r"\.0$", "", regex=True) for c in ("valor", "esperado")})
    distritos, anio_escale = _distritos(regiones.region_enla.tolist())
    regiones_todo = dash.loc[tipo == "contexto_region"]
    datos = {
        "geo": _cartografia(),
        "cobertura": _cobertura(obras, serie, regiones_todo, dash.loc[tipo == "obra_infobras", "fecha_corte"].dropna().max()),
        "obras": _registros(obras), "serie": _registros(serie), "causales": causales.rename(columns={"n": "n_obras"}).to_dict(orient="records"),
        "regiones": _registros(regiones),
        "distritos": distritos, "anio_escale": anio_escale,
        "validacion": validacion[["tabla", "comprobacion", "valor", "esperado", "ok"]].fillna("").to_dict(orient="records"),
        "calidad": calidad.fillna("").to_dict(orient="records"),
        "proceso": _proceso(dash, validacion, calidad, fuentes, inventario, len(pd.read_csv(config.DOCS / "diccionario.csv", dtype=str, encoding="utf-8-sig"))),
        "generado": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "url_ficha": f"{config.INFOBRAS_BASE}/Mapa/Sumario?ObraId=",
    }
    js_datos = json.dumps(datos, ensure_ascii=False).replace("</", "<\\/")
    if PLOTLY_JS.exists():
        plotly_js = PLOTLY_JS.read_text(encoding="utf-8")
    else:   # respaldo: la libreria plotly, si esta instalada
        import plotly.offline
        plotly_js = plotly.offline.get_plotlyjs()
    if TOPOJSON.exists():
        topo = '{"south-america_50m": ' + TOPOJSON.read_text(encoding="utf-8") + "}"
    else:   # sin el archivo, Plotly descarga la silueta de internet (el mapa solo se ve con conexion)
        utils.log(f"AVISO DASHBOARD HTML: falta {TOPOJSON}; el mapa necesitara internet para verse")
        topo = "{}"
    pagina = (PLANTILLA.replace("/*PLOTLY*/", plotly_js)
              .replace("/*TOPOJSON*/", topo)
              .replace("/*ARCHIVOS*/", _archivos_embebidos())
              .replace("/*DATOS*/", js_datos)
              .replace("<!--LIMITES-->", _md_a_html(limites)))
    SALIDA.parent.mkdir(parents=True, exist_ok=True)
    SALIDA.write_text(pagina, encoding="utf-8")
    basica = obras[(obras.tipo_infraestructura == "educacion_basica") & obras.paralizada_hoy]
    utils.log(f"DASHBOARD HTML: {SALIDA.relative_to(config.RAIZ)} ({SALIDA.stat().st_size / 1e6:.1f} MB); "
              f"{len(obras)} obras, {len(basica)} de educacion basica paralizadas hoy en {basica.ubigeo.nunique()} distritos")
    return SALIDA


PLANTILLA = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Monitor de obras de infraestructura educativa paralizadas · Perú</title>
<style>
:root {
  color-scheme: light;
  --surface-0: #f6f9fd; --surface-1: #ffffff; --border: #dde6f2; --grid: #edf1f7;
  --text-primary: #0f1d2e; --text-secondary: #46566b; --text-muted: #77849a;
  --series-1: #2a78d6; --series-2: #eb6834; --series-3: #1baf7a; --other: #a9b0bb;
  --good: #0ca30c; --critical: #d03b3b; --accent-soft: #e4edfa;
  --azul: #3a72c4; --azul-osc: #2a5ba8; --azul-claro: #7d9fd6; --azul-palido: #b8c8e6; --naranja: #f7a21b;
  --rojo: #ee6a72; --ambar: #f3c04c; --verde: #74c47a;
  --c-amarillo: #f5b915; --c-verde: #3fb36b; --c-rojo: #ec5466; --c-naranja: #f7891f; --c-celeste: #3a9ad9;
  --acento: #f7891f; --acento-tx: #fff;
  --sombra: 0 3px 10px rgba(58, 114, 196, .12);
}
@media (prefers-color-scheme: dark) {
  :root:where(:not([data-theme="light"])) {
    color-scheme: dark;
    --surface-0: #0f1620; --surface-1: #172230; --border: #263446; --grid: #22303f;
    --text-primary: #eef2f7; --text-secondary: #b8c3d1; --text-muted: #8996a8;
    --series-1: #3987e5; --series-2: #d95926; --series-3: #199e70; --other: #5f6b7a;
    --good: #0ca30c; --critical: #e66767; --accent-soft: #1d2d42;
    --azul: #3a72c4; --azul-osc: #24508f; --azul-claro: #6f90cc; --azul-palido: #4a6696;
    --rojo: #ee6a72; --ambar: #e9b843; --verde: #63b46a; --sombra: none;
  }
}
* { box-sizing: border-box; }
[hidden] { display: none !important; }
body { margin: 0; font-family: "Segoe UI", system-ui, -apple-system, sans-serif; background: var(--surface-0); color: var(--text-primary); }
svg.ic { width: 26px; height: 26px; fill: none; stroke: currentColor; stroke-width: 1.8; stroke-linecap: round; stroke-linejoin: round; flex: none; }
/* ---------- portada ---------- */
.portada { min-height: 100vh; display: grid; grid-template-columns: 300px 1fr 170px; gap: 28px; padding: 26px 34px; align-items: start; }
.portada .menu { display: flex; flex-direction: column; gap: 12px; }
.portada .menu button { display: flex; align-items: center; gap: 14px; text-align: left; border: 0; border-radius: 10px; padding: 14px 16px; font-size: 16px; color: var(--tx, #fff);
  background: var(--bg, var(--azul)); box-shadow: inset 0 0 0 3px rgba(255,255,255,.55), var(--sombra); cursor: pointer; transition: transform .15s, background .15s; }
.portada .menu button:hover { transform: translateX(6px); filter: brightness(1.06); box-shadow: inset 0 0 0 3px #fff, 0 4px 14px rgba(0,0,0,.15); }
.portada .menu button small { display: block; font-size: 12px; opacity: .85; }
.portada .marca { display: flex; align-items: center; gap: 10px; margin-bottom: 6px; }
.portada .marca .sello { background: var(--azul); color: #fff; font-weight: 800; font-size: 26px; letter-spacing: 1px; padding: 6px 14px; border-radius: 6px; }
.portada .marca small { color: var(--text-secondary); font-size: 12px; line-height: 1.3; }
.portada .centro { text-align: center; }
.portada h1 { margin: 6px 0 2px; font-size: 30px; font-style: italic; font-weight: 500; color: var(--azul); }
.portada .gancho { font-size: 18px; color: var(--text-secondary); margin: 4px auto 6px; max-width: 760px; }
.portada .gancho b { color: var(--rojo); font-size: 22px; }
#rueda { width: min(470px, 90%); height: auto; display: block; margin: 0 auto; }
#rueda .seg { cursor: pointer; transition: transform .2s, filter .2s; transform-origin: 250px 250px; }
#rueda .seg:hover { transform: scale(1.04); filter: brightness(1.08); }
.cifras { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin-top: 10px; }
.cifra-p { background: var(--surface-1); border-radius: 12px; padding: 14px 10px; box-shadow: var(--sombra); border-top: 5px solid var(--c); }
.cifra-p svg.ic { width: 34px; height: 34px; color: var(--c); }
.cifra-p .v { font-size: 26px; font-weight: 750; margin-top: 4px; }
.cifra-p .e { font-size: 12px; color: var(--text-secondary); }
.portada .derecha { display: flex; flex-direction: column; gap: 26px; align-items: center; padding-top: 120px; }
.portada .derecha a { display: flex; flex-direction: column; align-items: center; gap: 6px; color: var(--c, var(--azul)); text-decoration: none; font-size: 13px; text-align: center; }
.portada .derecha a svg.ic { width: 54px; height: 54px; stroke-width: 1.6; color: var(--c); }
.portada .derecha a { color: var(--text-secondary) !important; font-weight: 600; }
.portada .derecha a:hover { filter: brightness(.85); }
.portada .caja { border: 1px solid var(--border); background: var(--surface-1); padding: 10px 12px; font-size: 12px; color: var(--text-secondary); margin-top: 18px; border-radius: 6px; }
/* ---------- paginas internas ---------- */
.banda { background: linear-gradient(90deg, var(--azul), #4f8ad8); color: #fff; padding: 14px 28px 0; display: grid; grid-template-columns: 1fr auto; gap: 8px 20px; align-items: center; }
.banda .marca { font-size: 12px; opacity: .8; letter-spacing: .4px; text-transform: uppercase; }
.banda h1 { margin: 2px 0 0; font-size: 26px; font-weight: 600; }
.inicio-btn { grid-row: span 2; width: 86px; height: 86px; border-radius: 50%; border: 0; background: #fff; color: var(--azul); cursor: pointer; display: flex; flex-direction: column; align-items: center; justify-content: center; font-size: 12px; font-weight: 700; box-shadow: 0 2px 10px rgba(0,0,0,.2); }
.inicio-btn svg.ic { width: 34px; height: 34px; }
.inicio-btn:hover { background: var(--naranja); color: #fff; }
.pildoras { display: flex; gap: 8px; flex-wrap: wrap; padding: 10px 0 14px; }
.pildoras button { border: 0; border-radius: 999px; padding: 9px 22px; font-size: 14px; background: #e8edf4; color: #1f2d3d; cursor: pointer; }
.pildoras button.activa { background: var(--naranja); color: #fff; font-weight: 700; box-shadow: 0 2px 6px rgba(0,0,0,.2); }
nav { display: flex; gap: 6px; padding: 10px 28px; background: var(--surface-1); border-bottom: 1px solid var(--border); flex-wrap: wrap; }
nav button { display: flex; align-items: center; gap: 6px; border: 1px solid var(--border); background: var(--surface-1); padding: 6px 12px; border-radius: 8px; font-size: 13px; color: var(--text-secondary); cursor: pointer; }
nav button svg.ic { width: 18px; height: 18px; }
nav button.activa { background: var(--acento); border-color: var(--acento); color: var(--acento-tx); font-weight: 700; }
nav button:hover { border-color: var(--acento); }
.cuerpo { display: grid; grid-template-columns: 250px minmax(0, 1fr) 190px; gap: 18px; padding: 18px 28px 40px; align-items: start; }
.cuerpo.ancho { grid-template-columns: minmax(0, 1fr); }
.cuerpo.ancho .panel, .cuerpo.ancho .lateral { display: none; }
.panel { background: var(--surface-1); border-radius: 12px; padding: 14px; box-shadow: var(--sombra); border-top: 5px solid var(--acento); position: sticky; top: 10px; }
.panel h3 { margin: 0 0 10px; text-align: center; font-size: 17px; font-weight: 600; }
.panel label, label.en-linea { display: flex; flex-direction: column; font-size: 12px; color: var(--text-secondary); gap: 4px; margin-bottom: 10px; }
label.en-linea { flex-direction: row; align-items: center; gap: 8px; margin: 0; font-size: 13px; white-space: nowrap; }
label.en-linea select { width: auto; }
.panel select, label.en-linea select { font-size: 14px; padding: 7px 8px; border: 1px solid var(--border); border-radius: 8px; background: var(--surface-1); color: var(--text-primary); width: 100%; }
.que-muestra { background: var(--azul); color: #fff; border-radius: 10px; padding: 12px; margin: 12px 0; font-size: 13px; line-height: 1.45; }
.que-muestra b { display: block; font-size: 14px; margin-bottom: 4px; color: #ffe08a; }
.borrar { width: 100%; border: 0; border-radius: 8px; padding: 10px; background: #a9b4c4; color: #fff; font-size: 15px; font-weight: 700; cursor: pointer; }
.borrar:hover { background: var(--c-rojo); }
.panel .nota { font-size: 11px; color: var(--text-muted); margin-top: 10px; }
.lateral { display: flex; flex-direction: column; gap: 12px; position: sticky; top: 10px; }
.kpi { background: var(--surface-1); border-radius: 10px; padding: 14px 12px; text-align: center; box-shadow: var(--sombra); border-bottom: 5px solid var(--c, var(--azul)); }
.kpi:first-child { background: var(--c-naranja); color: #fff; border-bottom-color: #d96f08; }
.kpi:first-child .etiqueta { color: #fff7e6; }
.kpi .valor { font-size: 24px; font-weight: 700; line-height: 1.1; color: inherit; }
.kpi .etiqueta { font-size: 12px; color: var(--text-secondary); margin-top: 4px; }
.pestana { display: none; }
.pestana.activa { display: block; }
.fila { display: grid; gap: 16px; grid-template-columns: repeat(auto-fit, minmax(340px, 1fr)); margin-bottom: 16px; }
.tarjeta { background: var(--surface-1); border-radius: 12px; padding: 16px; box-shadow: var(--sombra); border-top: 5px solid var(--acento); }
.tarjeta h2 { margin: 0 0 2px; font-size: 16px; font-weight: 650; color: var(--azul); text-align: center; }
.tarjeta .sub { margin: 0 0 8px; font-size: 12px; color: var(--text-muted); text-align: center; }
.grafico { width: 100%; height: 360px; }
.grafico.alto { height: 560px; }
.fuente { font-size: 11px; color: var(--text-muted); margin-top: 6px; }
.semaforo { display: flex; gap: 14px; flex-wrap: wrap; align-items: center; font-size: 13px; color: var(--text-secondary); margin-left: auto; }
.semaforo span { display: inline-flex; align-items: center; gap: 6px; }
.semaforo i { display: inline-block; width: 46px; height: 10px; border-radius: 6px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
th, td { text-align: left; padding: 7px 8px; border-bottom: 1px solid var(--border); vertical-align: top; }
th { color: var(--acento-tx); background: var(--acento); font-weight: 600; position: sticky; top: 0; cursor: pointer; }
td.num, th.num { text-align: right; font-variant-numeric: tabular-nums; }
tbody tr:nth-child(even) { background: rgba(0, 0, 0, .025); }
tbody tr:hover { background: var(--accent-soft); cursor: pointer; }
.tabla-scroll { max-height: 560px; overflow: auto; }
.ficha { display: none; margin-bottom: 16px; border-top-color: var(--c-rojo) !important; }
.ficha.visible { display: block; }
.ficha h2 { text-align: left; }
.ficha dl { display: grid; grid-template-columns: 220px 1fr; gap: 6px 14px; margin: 8px 0 0; font-size: 13px; }
.ficha dt { color: var(--text-secondary); }
.ficha dd { margin: 0; }
.estado { display: inline-block; padding: 2px 8px; border-radius: 999px; font-size: 12px; border: 1px solid var(--border); }
.ok { color: var(--good); font-weight: 600; }
.no { color: var(--critical); font-weight: 600; }
.flujo { display: grid; gap: 10px; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); }
.paso { background: var(--surface-1); border-radius: 10px; padding: 12px; box-shadow: var(--sombra); border-left: 5px solid var(--naranja); }
.paso b { display: block; font-size: 14px; }
.paso .cifra { font-size: 20px; font-weight: 700; margin: 6px 0 2px; color: var(--c-naranja); }
.paso:nth-child(4n+2) { border-left-color: var(--c-verde); } .paso:nth-child(4n+3) { border-left-color: var(--c-amarillo); } .paso:nth-child(4n+4) { border-left-color: var(--c-rojo); }
.paso small { color: var(--text-secondary); font-size: 12px; display: block; }
.aviso { background: #fff7e0; border-left: 5px solid var(--c-amarillo); border-radius: 8px; padding: 10px 12px; font-size: 13px; color: var(--text-secondary); margin-bottom: 14px; }
.texto-limites { max-width: 900px; font-size: 14px; line-height: 1.55; margin: 0 auto; }
.texto-limites h2 { text-align: left; }
.texto-limites h3 { margin: 22px 0 6px; font-size: 16px; color: var(--azul); }
code { font-size: 12px; background: var(--surface-0); padding: 1px 4px; border-radius: 4px; }
@media (max-width: 1150px) {
  .portada, .cuerpo { grid-template-columns: 1fr; }
  .portada .derecha { flex-direction: row; padding-top: 0; justify-content: center; }
  .panel, .lateral { position: static; }
  .lateral { flex-direction: row; flex-wrap: wrap; } .lateral .kpi { flex: 1 1 150px; }
  .cifras { grid-template-columns: repeat(2, 1fr); }
}
</style>
</head>
<body>
<section id="inicio" class="portada">
  <div>
    <div class="marca"><span class="sello">MOEP</span><small>Monitor de Obras<br>Escolares Paralizadas · Perú</small></div>
    <div class="menu" id="menu-portada"></div>
    <div class="caja" id="cobertura"></div>
  </div>
  <div class="centro">
    <h1>¿Qué colegios tienen obras paralizadas?</h1>
    <p class="gancho" id="gancho"></p>
    <svg id="rueda" viewBox="0 0 500 500" role="img" aria-label="Las cuatro preguntas del monitor"></svg>
    <div class="cifras" id="cifras-portada"></div>
  </div>
  <div class="derecha">
    <a style="--c:#3fb36b" href="#" onclick="descargar('csv');return false" title="CSV único del proyecto (data/final/dashboard_obras.csv)"><span data-ic="descarga"></span>Descargar datos (CSV)</a>
    <a style="--c:#f5b915" href="#" onclick="descargar('diccionario');return false" title="Qué significa cada columna (docs/diccionario.csv)"><span data-ic="libro"></span>Diccionario de datos</a>
    <a style="--c:#f7891f" href="#" onclick="mostrar('proceso');return false"><span data-ic="base"></span>Fuentes y proceso</a>
    <a style="--c:#ec5466" href="#" onclick="mostrar('limites');return false"><span data-ic="info"></span>Límites del dato</a>
  </div>
</section>

<div id="app" hidden>
<div class="banda">
  <div><div class="marca">Monitor de obras escolares paralizadas · Perú</div><h1 id="titulo-pagina"></h1></div>
  <button class="inicio-btn" onclick="mostrar('inicio')"><span data-ic="casa"></span>INICIO</button>
  <div class="pildoras" id="pildoras">
    <button data-e="hoy" class="activa">Paralizadas hoy</button>
    <button data-e="ya_no">Ya no paralizadas</button>
    <button data-e="todas">Todas las obras escolares</button>
  </div>
</div>
<nav id="nav"></nav>
<div class="cuerpo" id="cuerpo">
<aside class="panel" id="panel">
  <h3>Filtros</h3>
  <select id="f-estado" hidden><option value="hoy">hoy</option><option value="ya_no">ya_no</option><option value="todas">todas</option></select>
  <label>Región<select id="f-region"></select></label>
  <label>Distrito<select id="f-distrito" disabled><option value="">Elija primero una región</option></select></label>
  <label>Causa principal<select id="f-causa"></select></label>
  <label>Año de paralización<select id="f-anio"></select></label>
  <div class="que-muestra"><b>¿Qué muestra esta página?</b><span id="explica"></span></div>
  <button class="borrar" id="borrar">Borrar filtros</button>
  <div class="nota" id="nota-filtro"></div>
</aside>
<main>

<section class="pestana" id="p-ranking">
  <div class="tarjeta" style="margin-bottom:16px;display:flex;gap:16px;align-items:center;flex-wrap:wrap">
    <label class="en-linea">Ordenar por <select id="f-orden-ranking">
      <option value="n">Número de obras</option><option value="monto">Monto del expediente técnico (S/ millones)</option><option value="anios">Años paralizada (mediana)</option>
    </select></label>
    <div class="semaforo" id="leyenda-semaforo"></div>
  </div>
  <div class="fila">
    <div class="tarjeta"><h2>Región</h2><p class="sub">Color = cuántos años llevan paralizadas sus obras (mediana) · clic en una región para filtrarla</p><div class="grafico" id="g-rank-region"></div></div>
    <div class="tarjeta"><h2>Distrito (los 25 primeros)</h2><p class="sub">Mismo color · clic en un distrito para ver sus obras</p><div class="grafico" id="g-rank-distrito"></div></div>
  </div>
  <div class="fuente">Fuente: INFOBRAS (lectura del 11-nov-2026) y Contraloría. El tiempo paralizada se mide desde la fecha de paralización; solo existe para las obras paralizadas hoy (las demás van en gris).</div>
</section>

<section class="pestana" id="p-resumen">
  <div class="fila">
    <div class="tarjeta"><h2>Obras de Educación paralizadas en el país, por corte</h2><p class="sub">Número de obras · lista oficial de la Contraloría (todo el sector Educación)</p><div class="grafico" id="g-serie"></div><div class="fuente">Fuente: Contraloría, informes de obras paralizadas (jun-2022 a jun-2026). Cuenta <b>todo el sector Educación</b>, incluidas obras que no son colegios (la fuente no permite separarlas). No depende de los filtros.</div></div>
    <div class="tarjeta"><h2>Regiones con más obras (según filtros)</h2><p class="sub">Número de obras</p><div class="grafico" id="g-top-regiones"></div><div class="fuente">Fuente: INFOBRAS (lectura del 11-nov-2026) y Contraloría.</div></div>
  </div>
  <div class="fila">
    <div class="tarjeta"><h2>Obras por año en que se paralizaron (según filtros)</h2><p class="sub">Número de obras de colegios · año de la fecha de paralización registrada en INFOBRAS · no depende del filtro de año</p><div class="grafico" id="g-anio-paralizacion"></div><div class="fuente">Fuente: INFOBRAS, detalle del último avance paralizado.</div></div>
    <div class="tarjeta"><h2>Obras por año de inicio (según filtros)</h2><p class="sub">Número de obras de colegios · año de inicio de la obra</p><div class="grafico" id="g-anio-inicio"></div><div class="fuente">Fuente: INFOBRAS, datos generales de la obra.</div></div>
  </div>
</section>

<section class="pestana" id="p-mapa">
  <div class="fila" style="grid-template-columns: 1fr">
  <div class="tarjeta">
    <h2>Dónde están las obras</h2><p class="sub">Cada punto es una obra · color = causa principal · ◆ = ubicación aproximada (centro del distrito o local escolar de ESCALE) · clic en un punto para ver la ficha</p>
    <div class="grafico" id="g-mapa" style="height:760px"></div>
    <div class="fuente">Fuente: coordenadas de INFOBRAS; 17 obras sin coordenadas se ubican de forma aproximada con ESCALE 2025.</div>
  </div>
  <div class="tarjeta"><h2>Distritos con más obras (según filtros)</h2><p class="sub">Haga clic en un distrito para ver sus obras</p><div class="tabla-scroll" style="max-height:640px"><table id="t-distritos"></table></div></div>
  </div>
</section>

<section class="pestana" id="p-causas">
  <div class="fila">
    <div class="tarjeta"><h2>Causa principal de la paralización (según filtros)</h2><p class="sub">Número de obras por macro-categoría</p><div class="grafico" id="g-causa-macro"></div><div class="fuente">Fuente: causal oficial de la Contraloría, comentarios de INFOBRAS y problemática del MEF, clasificados con reglas (NLP).</div></div>
    <div class="tarjeta"><h2>Causa detallada (según filtros)</h2><p class="sub">Número de obras por grupo de causa</p><div class="grafico" id="g-causa-grupo"></div><div class="fuente">Fuente: igual que el gráfico anterior.</div></div>
  </div>
  <div class="tarjeta"><h2>Causas a lo largo del tiempo (todos los sectores)</h2><p class="sub">Número de obras paralizadas por macro-categoría de causa, por corte · contractual, financiera y técnica (las más frecuentes en las obras escolares); el resto y lo que el informe no clasifica, agrupado</p><div class="grafico" id="g-causa-tiempo"></div><div class="fuente">Fuente: Contraloría, cuadro de causales de cada informe. No depende de los filtros.</div></div>
</section>

<section class="pestana" id="p-regiones">
  <div class="aviso"><b>Pregunta guía:</b> ¿las regiones con más obras educativas paralizadas son también las de mayores brechas de infraestructura y aprendizaje? Se compara cada región con su número de obras <b>por cada 100 locales escolares públicos</b> (para no favorecer a las regiones grandes). Son 26 regiones: es una <b>asociación</b>, no prueba que una cosa cause la otra.</div>
  <div class="fila" style="grid-template-columns: 1fr">
    <div class="tarjeta">
      <h2 id="t-dispersion">Obras por cada 100 locales frente al indicador</h2>
      <p class="sub"><label>Indicador: <select id="f-indicador"></select></label> · <span id="rho"></span></p>
      <div class="grafico alto" id="g-dispersion"></div>
      <div class="fuente">Fuentes: obras según filtros (INFOBRAS/Contraloría); locales y servicios: ESCALE 2025; aprendizajes: ENLA (último año disponible por grado).</div>
    </div>
    <div class="tarjeta"><h2>Tabla por región</h2><p class="sub">Clic en un encabezado para ordenar · clic en una región para ver sus distritos</p><div class="tabla-scroll"><table id="t-regiones"></table></div></div>
  </div>
  <div class="tarjeta" id="c-distritos">
    <h2 id="h-distritos-region">Distritos de la región</h2>
    <p class="sub" id="s-distritos-region"></p>
    <div class="tabla-scroll"><table id="t-distritos-region"></table></div>
    <div class="fuente" id="f-distritos-region"></div>
  </div>
</section>

<section class="pestana" id="p-obras">
  <div class="tarjeta ficha" id="ficha"></div>
  <div class="tarjeta">
    <h2>Obras (según filtros)</h2>
    <p class="sub"><input id="buscar" placeholder="Buscar por nombre, distrito, colegio o código..." style="width:340px;padding:6px 8px;border:1px solid var(--border);border-radius:6px;background:var(--surface-1);color:var(--text-primary)"> · haga clic en una fila para ver la ficha · <span id="n-obras"></span></p>
    <div class="tabla-scroll"><table id="t-obras"></table></div>
  </div>
</section>

<section class="pestana" id="p-proceso">
  <div class="aviso">Todo se genera con <b>un solo comando</b> (<code>ejecutar.bat</code> o <code>python src\pipeline_completo.py</code>). El detalle paso a paso, con el código y sus resultados, está en <code>notebooks/proyecto_completo.ipynb</code>. Las cifras de este tablero salen de esa misma corrida. <b>Generado: <span id="generado"></span>.</b></div>
  <div class="tarjeta" style="margin-bottom:16px">
    <h2>De las obras leídas a las obras del análisis</h2>
    <p class="sub">INFOBRAS agrupa Educación y Cultura. Como el objetivo son las instituciones educativas (y ESCALE y ENLA miden colegios y alumnos), el análisis se limita a la educación básica. Las demás obras siguen en los datos (columna <code>tipo_infraestructura</code>).</p>
    <div id="embudo"></div>
    <div class="fila" style="margin-top:12px">
      <div><h2 style="font-size:14px">Obras excluidas del análisis, por tipo</h2><table id="t-excluidas"></table></div>
      <div class="sub" style="align-self:center">La clasificación usa reglas sobre el nombre de la obra (por ejemplo «I.E.», «colegio», «local escolar» → colegio; «estadio», «losa deportiva» → deportiva; «universidad», «instituto superior» → educación superior; «museo», «arqueológico» → cultural). Si el nombre no lo dice, se usa el tipo de obra de INFOBRAS. Puede haber errores puntuales: la lista completa está abajo para revisarla.</div>
    </div>
    <details style="margin-top:10px"><summary style="cursor:pointer;font-size:13px">Ver la lista de obras excluidas</summary><div class="tabla-scroll" style="max-height:420px;margin-top:8px"><table id="t-excluidas-lista"></table></div></details>
  </div>
  <div class="flujo" id="flujo"></div>
  <div class="tarjeta" style="margin-top:16px"><h2>Auditoría de calidad del CSV único</h2><p class="sub">Los errores detienen el pipeline; los avisos se explican en Límites</p><table id="t-calidad"></table></div>
  <div class="tarjeta" style="margin-top:16px"><h2>Comprobaciones del producto final</h2><p class="sub" id="resumen-validacion"></p><div class="tabla-scroll"><table id="t-validacion"></table></div></div>
</section>

<section class="pestana" id="p-limites">
  <div class="tarjeta texto-limites"><h2>Límites y calidad del dato</h2><!--LIMITES--></div>
</section>
</main>
<aside class="lateral" id="kpis"></aside>
</div>
</div>

<script>const ARCHIVOS = /*ARCHIVOS*/;
async function descargar(clave) {
  // los CSV van dentro del HTML (gzip + base64) para que la descarga funcione aunque el tablero se abra fuera del proyecto
  const a = ARCHIVOS[clave];
  try {
    const bytes = Uint8Array.from(atob(a.datos), c => c.charCodeAt(0));
    const blob = await new Response(new Blob([bytes]).stream().pipeThrough(new DecompressionStream("gzip"))).blob();
    const url = URL.createObjectURL(new Blob([blob], {type: "text/csv;charset=utf-8"}));
    const enlace = document.createElement("a"); enlace.href = url; enlace.download = a.nombre;
    document.body.appendChild(enlace); enlace.click(); enlace.remove(); setTimeout(() => URL.revokeObjectURL(url), 5000);
  } catch (e) { alert("Este navegador no pudo preparar la descarga. El archivo está en el proyecto: " + a.ruta); }
}
</script>
<script>window.PlotlyGeoAssets = {topojson: /*TOPOJSON*/};</script>
<script>/*PLOTLY*/</script>
<script>
const D = /*DATOS*/;
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const ETIQ_TIPO = {educacion_basica: "Educación básica (colegios)", educacion_superior: "Educación superior", deportiva_recreativa: "Deportiva y recreativa", cultural_patrimonio: "Cultural y patrimonio", otra: "Otra"};
const ETIQ_MACRO = {financiera: "Financiera", contractual_y_legal: "Contractual y legal", tecnica: "Técnica", logistica: "Logística", fuerza_mayor: "Fuerza mayor", entorno_social_y_territorial: "Entorno social y territorial", sin_causa_de_paralizacion: "Sin causa de paralización", sin_clasificar: "Sin clasificar"};
const COLOR_CAUSA = {contractual_y_legal: "--series-1", financiera: "--series-2", tecnica: "--series-3"};
const colorCausa = k => css(COLOR_CAUSA[k] || "--other");
const etq = (m, k) => m[k] || (k ? k.replaceAll("_", " ") : "—");
const fmt = (x, d = 0) => (x === null || x === undefined || isNaN(x)) ? "—" : Number(x).toLocaleString("es-PE", {maximumFractionDigits: d, minimumFractionDigits: d});
const soles = x => (x === null || x === undefined || isNaN(x)) ? "—" : "S/ " + fmt(x / 1e6, 1) + " mill.";
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;"}[c]));
document.getElementById("cobertura").textContent = D.cobertura;
const ICON = {
  casa: '<path d="M3 11l9-8 9 8"/><path d="M5 9.5V21h5v-6h4v6h5V9.5"/>',
  ranking: '<path d="M3 21h18"/><rect x="4" y="12" width="4" height="9"/><rect x="10" y="6" width="4" height="15"/><rect x="16" y="9" width="4" height="12"/>',
  reloj: '<circle cx="12" cy="12" r="9"/><path d="M12 7v5l3.5 2"/>',
  pin: '<path d="M12 22s7-7.4 7-13a7 7 0 10-14 0c0 5.6 7 13 7 13z"/><circle cx="12" cy="9" r="2.6"/>',
  alerta: '<path d="M10.3 4l-8 14a2 2 0 001.7 3h16a2 2 0 001.7-3l-8-14a2 2 0 00-3.4 0z"/><path d="M12 9.5v4.5M12 17.5v.01"/>',
  capas: '<path d="M3 6.5l6-3 6 3 6-3v14l-6 3-6-3-6 3z"/><path d="M9 3.5v14M15 6.5v14"/>',
  lista: '<rect x="5" y="4" width="14" height="17" rx="2"/><path d="M9 4V2.5h6V4M8.5 10h7M8.5 14h7M8.5 18h4"/>',
  engranes: '<circle cx="6" cy="6" r="3"/><circle cx="18" cy="18" r="3"/><path d="M9 6h5a4 4 0 014 4v5"/><path d="M15 18h-5a4 4 0 01-4-4V9"/>',
  info: '<circle cx="12" cy="12" r="9"/><path d="M12 11v6M12 7.5v.01"/>',
  descarga: '<path d="M7 18a4.5 4.5 0 01-.6-9A6 6 0 0118 8a4 4 0 01-1 10"/><path d="M12 11v9M8.5 16.5L12 20l3.5-3.5"/>',
  libro: '<path d="M4 5a2 2 0 012-2h13v15H6a2 2 0 00-2 2z"/><path d="M4 20a2 2 0 002 2h13v-4M8 7h7M8 11h5"/>',
  base: '<ellipse cx="12" cy="5" rx="8" ry="3"/><path d="M4 5v14c0 1.7 3.6 3 8 3s8-1.3 8-3V5M4 12c0 1.7 3.6 3 8 3s8-1.3 8-3"/>',
  escuela: '<path d="M2.5 10L12 4l9.5 6"/><path d="M5 9v12h14V9M10 21v-5h4v5M12 8.5v2.5"/>',
  monedas: '<ellipse cx="12" cy="6" rx="7" ry="3"/><path d="M5 6v6c0 1.7 3.1 3 7 3s7-1.3 7-3V6M5 12v6c0 1.7 3.1 3 7 3s7-1.3 7-3v-6"/>',
};
const ic = k => `<svg class="ic" viewBox="0 0 24 24" aria-hidden="true">${ICON[k]}</svg>`;
document.querySelectorAll("[data-ic]").forEach(e => e.outerHTML = ic(e.dataset.ic));
const PAGINAS = [
  ["resumen", "Panorama", "reloj", "¿Desde cuándo están paralizadas?", "Cuántas obras de colegios hay con los filtros, cuánto dinero comprometen y en qué año se paralizaron. La línea muestra cómo cambió el total del país en cada informe de la Contraloría.", "#3fb36b", "#fff"],
  ["ranking", "Ranking", "ranking", "Ranking de regiones y distritos", "Regiones y distritos ordenados por número de obras, por monto o por años paralizadas. El color es un semáforo: rojo = sus obras llevan más de 3 años detenidas.", "#f5b915", "#3a2a00"],
  ["mapa", "Mapa", "pin", "¿Dónde están las obras?", "Cada punto es una obra de colegio; su color indica la causa de la paralización. Haga clic en una región para acercarse y ver sus provincias, y en un punto para ver la ficha de la obra. «Borrar filtros» vuelve al Perú completo.", "#f7891f", "#fff"],
  ["causas", "Causas", "alerta", "¿Por qué se paralizaron?", "Las causas se clasificaron leyendo el texto oficial de la Contraloría, los comentarios de INFOBRAS y la problemática del MEF. Problemas de contrato y de dinero son los más comunes.", "#ec5466", "#fff"],
  ["regiones", "Regiones y distritos", "capas", "Regiones, distritos y brechas", "¿Las regiones con más obras paralizadas son las de peor infraestructura y aprendizaje? Elija un indicador. Al elegir una región aparecen todos sus distritos.", "#3fb36b", "#fff"],
  ["obras", "Obras", "lista", "Ficha de cada obra", "Busque una obra por nombre, distrito o código. Haga clic en una fila para ver su ficha completa y el enlace a INFOBRAS.", "#f5b915", "#3a2a00"],
  ["proceso", "Proceso", "engranes", "Cómo se construyó este monitor", "", "#f7891f", "#fff"],
  ["limites", "Límites", "info", "Límites y calidad del dato", "", "#ec5466", "#fff"],
];
document.getElementById("menu-portada").innerHTML = PAGINAS.map(p => `<button style="--bg:${p[5]};--tx:${p[6]}" onclick="mostrar('${p[0]}')">${ic(p[2])}<span>${p[1]}<small>${p[3]}</small></span></button>`).join("");
document.getElementById("nav").innerHTML = PAGINAS.map(p => `<button data-p="${p[0]}">${ic(p[2])}${p[1]}</button>`).join("");
function portada() {
  const hoy = COLEGIOS.filter(o => o.paralizada_hoy), anios = hoy.map(o => o.anios_paralizada).filter(x => x !== null);
  const masDos = hoy.filter(o => o.anios_paralizada > 2).length, de10 = Math.round(10 * masDos / hoy.length);
  document.getElementById("gancho").innerHTML = `<b>${de10} de cada 10</b> colegios con una obra paralizada llevan <b>más de 2 años</b> esperando que se reanude.`;
  const cif = (c, i, v, e) => `<div class="cifra-p" style="--c:${c}">${ic(i)}<div class="v">${v}</div><div class="e">${e}</div></div>`;
  document.getElementById("cifras-portada").innerHTML =
    cif("#f7891f", "escuela", fmt(hoy.length), "obras de colegios paralizadas hoy") + cif("#3fb36b", "pin", fmt(new Set(hoy.map(o => o.ubigeo)).size), "distritos afectados") +
    cif("#f5b915", "monedas", soles(suma(hoy.map(o => o.monto_expediente_tecnico))), "comprometidos (expediente técnico)") + cif("#ec5466", "reloj", fmt(mediana(anios), 1) + " años", "paralizadas (mediana)");
  const segs = [["mapa", ["¿Dónde?"], "#f7891f"], ["ranking", ["¿Cuánto?"], "#f5b915"], ["resumen", ["¿Desde", "cuándo?"], "#3fb36b"], ["causas", ["¿Por qué?"], "#ec5466"]];
  const R = 225, r = 125, cx = 250, cy = 250, pt = (rad, a) => [cx + rad * Math.cos(a), cy + rad * Math.sin(a)];
  document.getElementById("rueda").innerHTML = segs.map((sg, i) => {
    const a0 = -Math.PI / 2 + i * Math.PI / 2 + 0.035, a1 = a0 + Math.PI / 2 - 0.07, m = (a0 + a1) / 2;
    const [x0, y0] = pt(R, a0), [x1, y1] = pt(R, a1), [x2, y2] = pt(r, a1), [x3, y3] = pt(r, a0), [tx, ty] = pt((R + r) / 2, m);
    return `<g class="seg" onclick="mostrar('${sg[0]}')"><path d="M${x0},${y0} A${R},${R} 0 0 1 ${x1},${y1} L${x2},${y2} A${r},${r} 0 0 0 ${x3},${y3} Z" fill="${sg[2]}"/>
      <text x="${tx}" y="${ty - (sg[1].length - 1) * 12}" text-anchor="middle" dominant-baseline="middle" fill="${i === 1 ? "#3a2a00" : "#fff"}" font-size="21" font-weight="700">${sg[1].map((t, k) => `<tspan x="${tx}" dy="${k ? 24 : 0}">${t}</tspan>`).join("")}</text></g>`;
  }).join("") + `<circle cx="250" cy="250" r="112" fill="var(--surface-1)"/>
    <text x="250" y="232" text-anchor="middle" font-size="64" font-weight="800" fill="var(--azul)">${hoy.length}</text>
    <text x="250" y="268" text-anchor="middle" font-size="16" fill="var(--text-secondary)">colegios con obra</text>
    <text x="250" y="290" text-anchor="middle" font-size="16" fill="var(--text-secondary)">paralizada hoy</text>`;
}

// ---------- filtros ----------
const opciones = (id, valores, etiquetas, todas) => {
  const s = document.getElementById(id);
  s.innerHTML = (todas ? `<option value="">${todas}</option>` : "") + valores.map(v => `<option value="${esc(v)}">${esc(etiquetas ? etq(etiquetas, v) : v)}</option>`).join("");
};
const contar = (arr, k) => arr.reduce((m, o) => (m[o[k]] = (m[o[k]] || 0) + 1, m), {});
const porFrecuencia = (arr, k) => Object.entries(contar(arr, k)).sort((a, b) => b[1] - a[1]).map(e => e[0]).filter(v => v && v !== "null");
const anioDe = f => f ? Number(String(f).slice(0, 4)) || null : null;
D.obras.forEach(o => { o.anio_inicio = anioDe(o.fecha_inicio); o.anio_paralizacion = anioDe(o.fecha_paralizacion); o.anios_paralizada = o.dias_desde_paralizacion === null ? null : o.dias_desde_paralizacion / 365.25; });
const COLEGIOS = D.obras.filter(o => o.tipo_infraestructura === "educacion_basica");   // el analisis es solo de colegios
opciones("f-region", [...new Set(COLEGIOS.map(o => o.region_enla))].filter(Boolean).sort(), null, "Todas las regiones");
opciones("f-causa", porFrecuencia(COLEGIOS, "causa_macro"), ETIQ_MACRO, "Todas las causas");
opciones("f-anio", [...new Set(COLEGIOS.map(o => o.anio_paralizacion))].filter(Boolean).sort().reverse().map(String), null, "Todos los años");

function poblarDistritos() {
  const r = document.getElementById("f-region").value, s = document.getElementById("f-distrito");
  if (!r) { s.innerHTML = '<option value="">Elija primero una región</option>'; s.disabled = true; return; }
  const vistos = {}; COLEGIOS.filter(o => o.region_enla === r).forEach(o => vistos[o.ubigeo] = `${o.distrito} (${o.provincia})`);
  s.innerHTML = '<option value="">Todos los distritos con obras</option>' + Object.entries(vistos).sort((a, b) => a[1].localeCompare(b[1])).map(([u, n]) => `<option value="${esc(u)}">${esc(n)}</option>`).join("");
  s.disabled = false;
}
function filtradas(sinRegion = false, sinAnio = false) {
  const e = document.getElementById("f-estado").value, r = document.getElementById("f-region").value, c = document.getElementById("f-causa").value,
        d = sinRegion ? "" : document.getElementById("f-distrito").value, a = sinAnio ? "" : document.getElementById("f-anio").value;
  return COLEGIOS.filter(o => (e === "todas" || (e === "hoy" ? o.paralizada_hoy : !o.paralizada_hoy))
                          && (sinRegion || !r || o.region_enla === r) && (!d || o.ubigeo === d) && (!c || o.causa_macro === c)
                          && (!a || String(o.anio_paralizacion) === a));
}
function elegirRegion(r) { document.getElementById("f-region").value = r; poblarDistritos(); actualizar(); }

// ---------- utilidades de graficos ----------
function base(extra = {}) {
  return Object.assign({
    paper_bgcolor: "rgba(0,0,0,0)", plot_bgcolor: "rgba(0,0,0,0)", font: {family: "Segoe UI, system-ui, sans-serif", size: 12, color: css("--text-secondary")},
    margin: {l: 50, r: 16, t: 10, b: 40}, hoverlabel: {bgcolor: css("--surface-1"), bordercolor: css("--border"), font: {color: css("--text-primary")}},
    xaxis: {gridcolor: css("--grid"), zeroline: false, linecolor: css("--border")}, yaxis: {gridcolor: css("--grid"), zeroline: false, linecolor: css("--border")},
  }, extra);
}
const CONFIG = {displaylogo: false, responsive: true, locale: "es", modeBarButtonsToRemove: ["lasso2d", "select2d", "autoScale2d"]};
const dibujar = (id, trazas, layout) => Plotly.react(id, trazas, base(layout), CONFIG);
function barrasH(id, conteo, etiquetas, titulo, color = "--c-naranja") {
  const pares = Object.entries(conteo).sort((a, b) => a[1] - b[1]);
  dibujar(id, [{type: "bar", orientation: "h", x: pares.map(p => p[1]), y: pares.map(p => etq(etiquetas || {}, p[0])), marker: {color: css(color)},
               text: pares.map(p => p[1]), textposition: "outside", cliponaxis: false, hovertemplate: "%{y}: %{x} obras<extra></extra>"}],
          {margin: {l: 190, r: 40, t: 10, b: 30}, xaxis: {gridcolor: css("--grid"), title: titulo || "Número de obras"}, bargap: 0.35});
}
const mediana = a => { const s = a.filter(x => x !== null && !isNaN(x)).sort((x, y) => x - y); if (!s.length) return null; const m = Math.floor(s.length / 2); return s.length % 2 ? s[m] : (s[m - 1] + s[m]) / 2; };
const suma = a => a.reduce((s, x) => s + (x || 0), 0);

// ---------- Cifras de la derecha ----------
function kpis(obras) {
  const anios = obras.map(o => o.anios_paralizada).filter(x => x !== null), distritos = new Set(obras.map(o => o.ubigeo)).size;
  const masDos = obras.filter(o => o.anios_paralizada > 2).length;
  const kpi = (v, e, c) => `<div class="kpi" style="--c:${c}"><div class="valor">${v}</div><div class="etiqueta">${e}</div></div>`;
  document.getElementById("kpis").innerHTML =
    kpi(fmt(obras.length), "obras de colegios con los filtros", "") + kpi(fmt(distritos), "distritos", "#3fb36b") +
    kpi(soles(suma(obras.map(o => o.monto_expediente_tecnico))), "expediente técnico", "#f5b915") +
    kpi(anios.length ? fmt(mediana(anios), 1) + " años" : "—", anios.length ? "paralizadas (mediana)" : "tiempo paralizada: solo para las paralizadas hoy", "#ec5466") +
    kpi(anios.length ? fmt(masDos) : "—", "llevan más de 2 años paralizadas", "#f7891f");
}

// ---------- Ranking ----------
const semaforo = a => a === null || a === undefined ? css("--other") : a > 3 ? css("--rojo") : a >= 1 ? css("--ambar") : css("--verde");
document.getElementById("leyenda-semaforo").innerHTML = [["--rojo", "Más de 3 años"], ["--ambar", "De 1 a 3 años"], ["--verde", "Menos de 1 año"], ["--other", "Sin dato (no paralizada hoy)"]]
  .map(([c, t]) => `<span><i style="background:var(${c})"></i>${t}</span>`).join("");
function ranking(obras) {
  const met = document.getElementById("f-orden-ranking").value;
  const agrupar = (clave, nombre) => { const g = {}; obras.forEach(o => { const k = o[clave]; g[k] = g[k] || {clave: k, nombre: nombre(o), n: 0, monto: 0, anios: []};
    g[k].n++; g[k].monto += o.monto_expediente_tecnico || 0; if (o.anios_paralizada !== null) g[k].anios.push(o.anios_paralizada); });
    return Object.values(g).map(x => ({...x, med: mediana(x.anios)})); };
  const valor = x => met === "monto" ? x.monto / 1e6 : met === "anios" ? x.med : x.n;
  const titulo = {n: "Número de obras", monto: "Expediente técnico (S/ millones)", anios: "Años paralizada (mediana)"}[met];
  const dib = (id, lista, top, alClic) => {
    const l = lista.filter(x => valor(x) !== null).sort((a, b) => valor(b) - valor(a) || b.n - a.n).slice(0, top).reverse();
    const div = document.getElementById(id); div.style.height = Math.max(300, l.length * 26 + 70) + "px";
    dibujar(id, [{type: "bar", orientation: "h", x: l.map(valor), y: l.map(x => x.nombre), marker: {color: l.map(x => semaforo(x.med))},
      text: l.map(x => fmt(valor(x), met === "n" ? 0 : 1)), textposition: "outside", cliponaxis: false, customdata: l.map(x => x.clave),
      hovertext: l.map(x => `<b>${esc(x.nombre)}</b><br>${x.n} obras · ${soles(x.monto)}<br>${x.med === null ? "Sin obras paralizadas hoy" : "Paralizadas hace " + fmt(x.med, 1) + " años (mediana)"}`),
      hovertemplate: "%{hovertext}<extra></extra>"}],
      {margin: {l: 210, r: 50, t: 6, b: 44}, xaxis: {gridcolor: css("--grid"), title: titulo, rangemode: "tozero"}, yaxis: {automargin: true}, bargap: 0.28});
    if (div.removeAllListeners) div.removeAllListeners("plotly_click");
    div.on("plotly_click", ev => alClic(ev.points[0].customdata));
  };
  dib("g-rank-region", agrupar("region_enla", o => o.region_enla), 26, elegirRegion);
  dib("g-rank-distrito", agrupar("ubigeo", o => `${o.distrito} (${o.region_enla})`), 25, u => {
    const o = COLEGIOS.find(x => x.ubigeo === u); document.getElementById("f-region").value = o.region_enla; poblarDistritos();
    document.getElementById("f-distrito").value = u; mostrar("obras"); });
}

// ---------- Resumen ----------
function resumen(obras) {
  const reg = contar(obras, "region_enla"); const top = Object.fromEntries(Object.entries(reg).sort((a, b) => b[1] - a[1]).slice(0, 12));
  barrasH("g-top-regiones", top, null, null, "--c-naranja");
  barrasAnio("g-anio-paralizacion", filtradas(false, true), "anio_paralizacion", "Año en que se paralizó");
  barrasAnio("g-anio-inicio", obras, "anio_inicio", "Año de inicio de la obra");
}
function barrasAnio(id, obras, campo, titulo) {
  const c = contar(obras.filter(o => o[campo]), campo), anios = Object.keys(c).map(Number).sort((a, b) => a - b);
  const sel = document.getElementById("f-anio").value;
  dibujar(id, [{type: "bar", x: anios, y: anios.map(a => c[a]), text: anios.map(a => c[a]), textposition: "outside", cliponaxis: false,
    marker: {color: anios.map(a => campo === "anio_paralizacion" && sel && String(a) === sel ? css("--c-rojo") : css(campo === "anio_paralizacion" ? "--c-verde" : "--c-amarillo"))},
    hovertemplate: "%{x}: %{y} obras<extra></extra>"}],
    {xaxis: {gridcolor: "rgba(0,0,0,0)", title: titulo, dtick: 1, tickangle: -45}, yaxis: {gridcolor: css("--grid"), title: "Obras", rangemode: "tozero"}, bargap: 0.3, margin: {l: 50, r: 16, t: 20, b: 60}});
  const g = document.getElementById(id);
  if (campo === "anio_paralizacion") { if (g.removeAllListeners) g.removeAllListeners("plotly_click"); g.on("plotly_click", ev => { const f = document.getElementById("f-anio"); f.value = f.value === String(ev.points[0].x) ? "" : String(ev.points[0].x); actualizar(); }); }
}
function serie() {
  const edu = D.serie.filter(s => s.tipo_registro === "pais_educacion_corte").sort((a, b) => a.fecha_corte.localeCompare(b.fecha_corte));
  dibujar("g-serie", [{type: "scatter", mode: "lines+markers", x: edu.map(s => s.fecha_corte), y: edu.map(s => s.n_obras), line: {color: css("--c-rojo"), width: 2.5},
    marker: {size: 8, color: css("--c-rojo"), line: {color: css("--surface-1"), width: 2}},
    customdata: edu.map(s => [s.pct_obras, s.costo_actualizado_soles]),
    hovertemplate: "%{x|%b %Y}: %{y} obras de Educación<br>%{customdata[0]:.1f}% del total del país<br>Costo: S/ %{customdata[1]:,.0f}<extra></extra>"}],
    {yaxis: {gridcolor: css("--grid"), title: "Obras", rangemode: "tozero"}, xaxis: {gridcolor: css("--grid")}, hovermode: "closest"});
}

// ---------- Mapa ----------
// colores del mapa (estilo de los mapas de calles: mar celeste, Peru en verde claro, la region elegida en amarillo)
const MAPA = {mar: "#aadaff", peru: "#d4ecd5", vecinos: "#eeefe9", borde: "#7f948a", elegida: "#fbe7a1", provincia: "#f6d77a", texto: "#3f5247", ciudad: "#1f2d26"};
const titulo = t => t.toLowerCase().replace(/(^|[\s-])\S/g, m => m.toUpperCase());
const depDe = r => r.startsWith("LIMA") ? "LIMA" : r;
function encuadre(features) {
  let x0 = 999, x1 = -999, y0 = 999, y1 = -999;
  const ver = c => { if (typeof c[0] === "number") { x0 = Math.min(x0, c[0]); x1 = Math.max(x1, c[0]); y0 = Math.min(y0, c[1]); y1 = Math.max(y1, c[1]); } else c.forEach(ver); };
  features.forEach(f => ver(f.geometry.coordinates));
  const px = Math.max(0.35, (x1 - x0) * 0.12), py = Math.max(0.35, (y1 - y0) * 0.12);
  return {lon: [x0 - px, x1 + px], lat: [y0 - py, y1 + py]};
}
function llenar(r, div) {
  const w = div.clientWidth || 900, h = div.clientHeight || 760, y = lat => Math.log(Math.tan(Math.PI / 4 + lat * Math.PI / 360));
  const alto = y(r.lat[1]) - y(r.lat[0]), ancho = (r.lon[1] - r.lon[0]) * Math.PI / 180, cx = (r.lon[0] + r.lon[1]) / 2;
  if (ancho / alto < w / h) { const nuevo = alto * w / h * 180 / Math.PI; return {lat: r.lat, lon: [cx - nuevo / 2, cx + nuevo / 2]}; }
  const nuevoAlto = ancho * h / w, cy = (y(r.lat[0]) + y(r.lat[1])) / 2, inv = v => (2 * Math.atan(Math.exp(v)) - Math.PI / 2) * 180 / Math.PI;
  return {lon: r.lon, lat: [inv(cy - nuevoAlto / 2), inv(cy + nuevoAlto / 2)]};
}
function mapa(obras) {
  const macros = Object.keys(COLOR_CAUSA);
  const grupos = [...macros, "otras"], colores = grupos.map(colorCausa);
  const region = document.getElementById("f-region").value, G = D.geo;
  const trazas = [];
  let rango = {lat: [-18.6, 0.2], lon: [-81.6, -68.4]};
  if (G) {
    const nDep = contar(obras, "region_enla"), porDep = {};
    Object.entries(nDep).forEach(([r, n]) => porDep[depDe(r)] = (porDep[depDe(r)] || 0) + n);
    const depSel = region ? depDe(region) : null;
    const feats = G.departamentos.features;
    // 1) regiones del Peru
    trazas.push({type: "choropleth", geojson: G.departamentos, featureidkey: "properties.departamento", locations: feats.map(f => f.properties.departamento),
      z: feats.map(f => f.properties.departamento === depSel ? 1 : 0), zmin: 0, zmax: 1, colorscale: [[0, MAPA.peru], [1, MAPA.elegida]], showscale: false,
      marker: {line: {color: MAPA.borde, width: 1}}, customdata: feats.map(f => porDep[f.properties.departamento] || 0),
      hovertemplate: "<b>%{location}</b><br>%{customdata} obras de colegios con los filtros<extra></extra>", showlegend: false});
    // 2) con una region elegida: sus provincias y zoom
    if (depSel) {
      let provs = G.provincias.features.filter(f => f.properties.departamento === depSel);
      if (region === "LIMA METROPOLITANA") provs = provs.filter(f => f.properties.provincia === "LIMA");
      if (region === "LIMA PROVINCIAS") provs = provs.filter(f => f.properties.provincia !== "LIMA");
      const nProv = contar(obras.filter(o => o.region_enla === region), "provincia");
      trazas.push({type: "choropleth", geojson: {type: "FeatureCollection", features: provs}, featureidkey: "properties.cod", locations: provs.map(f => f.properties.cod),
        z: provs.map(f => nProv[f.properties.provincia] ? 1 : 0), zmin: 0, zmax: 1, colorscale: [[0, MAPA.elegida], [1, MAPA.provincia]], showscale: false,
        marker: {line: {color: "#ffffff", width: 1.6}}, text: provs.map(f => titulo(f.properties.provincia)), customdata: provs.map(f => nProv[f.properties.provincia] || 0),
        hovertemplate: "<b>Provincia de %{text}</b><br>%{customdata} obras de colegios<extra></extra>", showlegend: false});
      const ep = G.etiquetas_prov.filter(e => provs.some(f => f.properties.provincia === e.nombre && f.properties.departamento === e.departamento));
      trazas.push({type: "scattergeo", mode: "text", lon: ep.map(e => e.lon), lat: ep.map(e => e.lat), text: ep.map(e => titulo(e.nombre)),
        textfont: {size: 11, color: "#7a6a2c", family: "Segoe UI, sans-serif"}, hoverinfo: "skip", showlegend: false});
      const callao = region === "LIMA METROPOLITANA" ? G.departamentos.features.filter(f => f.properties.departamento === "CALLAO") : [];
      rango = encuadre([...provs, ...callao]);
    }
    rango = llenar(rango, document.getElementById("g-mapa"));
    // 3) nombres de las regiones (la elegida se nombra por sus provincias)
    const ed = G.etiquetas_dep.filter(e => e.nombre !== depSel && (!depSel || (e.lon > rango.lon[0] && e.lon < rango.lon[1] && e.lat > rango.lat[0] && e.lat < rango.lat[1])));
    trazas.push({type: "scattergeo", mode: "text", lon: ed.map(e => e.lon), lat: ed.map(e => e.lat), text: ed.map(e => e.nombre.replace(" ", "<br>")),
      textfont: {size: depSel ? 13 : 11, color: MAPA.texto, family: "Segoe UI Semibold, Segoe UI, sans-serif"}, hoverinfo: "skip", showlegend: false});
    if (depSel) trazas.push({type: "scattergeo", mode: "text", lon: [(rango.lon[0] + rango.lon[1]) / 2], lat: [rango.lat[1] - (rango.lat[1] - rango.lat[0]) * 0.04],
      text: [`<b>${region}</b>`], textfont: {size: 16, color: MAPA.ciudad}, hoverinfo: "skip", showlegend: false});
    // 4) paises vecinos y el mar
    const fuera = [["ECUADOR", -78.4, -1.2], ["COLOMBIA", -72.6, -0.7], ["BRASIL", -70.1, -7.2], ["BOLIVIA", -68.85, -11.6], ["CHILE", -69.75, -18.45], ["Océano Pacífico", -79.6, -12.8]]
      .filter(([, x, y]) => x > rango.lon[0] && x < rango.lon[1] && y > rango.lat[0] && y < rango.lat[1]);
    trazas.push({type: "scattergeo", mode: "text", lon: fuera.map(f => f[1]), lat: fuera.map(f => f[2]), text: fuera.map(f => f[0].startsWith("Océano") ? `<i>${f[0]}</i>` : f[0]),
      textfont: {size: 12, color: fuera.map(f => f[0].startsWith("Océano") ? "#3b7bbf" : "#8b918c")}, hoverinfo: "skip", showlegend: false});
    // 5) ciudades
    const ci = G.ciudades.filter(c => c.lon > rango.lon[0] && c.lon < rango.lon[1] && c.lat > rango.lat[0] && c.lat < rango.lat[1] && (depSel || c.ciudad === "LIMA"));   // vista del pais: solo Lima; al acercarse, las ciudades
    trazas.push({type: "scattergeo", mode: "markers+text", lon: ci.map(c => c.lon), lat: ci.map(c => c.lat), text: ci.map(c => titulo(c.ciudad)), textposition: "middle right",
      textfont: {size: ci.map(c => c.ciudad === "LIMA" ? 14 : c.capital_region ? 12 : 11), color: MAPA.ciudad},
      marker: {size: ci.map(c => c.ciudad === "LIMA" ? 8 : 6), color: "#ffffff", line: {color: MAPA.ciudad, width: 1.5}}, hovertemplate: "%{text}<extra></extra>", showlegend: false});
  }
  // 6) las obras, coloreadas por causa
  grupos.forEach((g, i) => {
    const sel = obras.filter(o => o.latitud_mapa !== null && (g === "otras" ? !macros.includes(o.causa_macro) : o.causa_macro === g));
    trazas.push({type: "scattergeo", mode: "markers", name: g === "otras" ? "Otras causas" : etq(ETIQ_MACRO, g), lat: sel.map(o => o.latitud_mapa), lon: sel.map(o => o.longitud_mapa),
      marker: {size: region ? 12 : 10, color: colores[i], opacity: 0.92, line: {color: "#ffffff", width: 1.5}, symbol: sel.map(o => o.coordenada_origen === "infobras" ? "circle" : "diamond")},
      text: sel.map(o => `<b>${esc(o.nombre_obra).slice(0, 90)}…</b><br>${esc(o.distrito)}, ${esc(o.provincia)} (${esc(o.region_enla)})<br>Estado: ${esc(o.estado_infobras)}${o.anios_paralizada !== null ? " · paralizada hace " + fmt(o.anios_paralizada, 1) + " años" : ""}<br>Expediente: ${soles(o.monto_expediente_tecnico)}<br>Causa: ${esc(etq(ETIQ_MACRO, o.causa_macro))}${o.coordenada_origen !== "infobras" ? "<br><i>Ubicación aproximada</i>" : ""}`),
      hovertemplate: "%{text}<extra></extra>", customdata: sel.map(o => o.codigo_infobras)});
  });
  dibujar("g-mapa", trazas, {margin: {l: 0, r: 0, t: 0, b: 0}, paper_bgcolor: MAPA.mar,
    legend: {orientation: "h", y: 0.01, yanchor: "bottom", x: 0.01, bgcolor: "rgba(255,255,255,.88)", bordercolor: MAPA.borde, borderwidth: 1, font: {color: "#1f2d26"}},
    geo: {resolution: 50, scope: "south america", fitbounds: false, lataxis: {range: rango.lat}, lonaxis: {range: rango.lon}, projection: {type: "mercator"},
          showcountries: true, countrycolor: "#b9bdb5", countrywidth: 1, showcoastlines: false, showland: true, landcolor: MAPA.vecinos,
          showocean: true, oceancolor: MAPA.mar, showlakes: true, lakecolor: MAPA.mar, showrivers: false, bgcolor: MAPA.mar, showframe: false}});
  const g = document.getElementById("g-mapa");
  if (g.removeAllListeners) g.removeAllListeners("plotly_click");
  g.on("plotly_click", ev => {
    const pt = ev.points[0];
    if (pt.data.type === "choropleth" && pt.data.featureidkey === "properties.departamento") {   // clic en una region: filtrarla
      const r = pt.location === "LIMA" ? "LIMA PROVINCIAS" : pt.location;
      if ([...document.getElementById("f-region").options].some(o => o.value === r)) elegirRegion(r);
      return; }
    if (pt.data.type === "scattergeo" && pt.customdata !== undefined && pt.customdata !== null) { mostrar("obras"); verFicha(pt.customdata); } });
  const porDistrito = {};
  obras.forEach(o => { const k = o.ubigeo; porDistrito[k] = porDistrito[k] || {distrito: o.distrito, provincia: o.provincia, region: o.region_enla, n: 0, monto: 0, anios: []};
    porDistrito[k].n++; porDistrito[k].monto += o.monto_expediente_tecnico || 0; if (o.anios_paralizada !== null) porDistrito[k].anios.push(o.anios_paralizada); });
  const lista = Object.values(porDistrito).sort((a, b) => b.n - a.n || b.monto - a.monto);
  document.getElementById("t-distritos").innerHTML = "<thead><tr><th>Distrito</th><th>Región</th><th class='num'>Obras</th><th class='num'>Expediente</th><th class='num'>Años paralizada (mediana)</th></tr></thead><tbody>" +
    lista.map(d => `<tr data-d="${esc(d.distrito)}"><td>${esc(d.distrito)}<br><small style="color:var(--text-muted)">${esc(d.provincia)}</small></td><td>${esc(d.region)}</td><td class="num">${d.n}</td><td class="num">${soles(d.monto)}</td><td class="num">${d.anios.length ? fmt(mediana(d.anios), 1) : "—"}</td></tr>`).join("") + "</tbody>";
  document.querySelectorAll("#t-distritos tbody tr").forEach(tr => tr.onclick = () => {
    const o = obras.find(x => x.distrito === tr.dataset.d); if (!o) return;
    document.getElementById("f-region").value = o.region_enla; poblarDistritos(); document.getElementById("f-distrito").value = o.ubigeo; mostrar("obras"); });
}

// ---------- Causas ----------
function causas(obras) {
  barrasH("g-causa-macro", contar(obras, "causa_macro"), ETIQ_MACRO, null, "--c-rojo");
  const grupos = {}; Object.entries(contar(obras, "causa_grupo_final")).forEach(([k, v]) => { const t = (k || "").replaceAll("_", " "); grupos[t.charAt(0).toUpperCase() + t.slice(1)] = v; });
  barrasH("g-causa-grupo", grupos, {}, null, "--c-naranja");
}
function causasTiempo() {
  const tot = {}; D.causales.forEach(c => tot[c.causa_macro] = (tot[c.causa_macro] || 0) + c.n_obras);
  const top = Object.keys(COLOR_CAUSA);
  const fechas = [...new Set(D.causales.map(c => c.fecha_corte))].sort();
  const colores = [...top, "otras"].map(colorCausa);
  const trazas = [...top, "otras"].map((g, i) => ({type: "scatter", mode: "lines+markers", name: g === "otras" ? "Otras o sin clasificar" : etq(ETIQ_MACRO, g), x: fechas,
    y: fechas.map(f => suma(D.causales.filter(c => c.fecha_corte === f && (g === "otras" ? !top.includes(c.causa_macro) : c.causa_macro === g)).map(c => c.n_obras))),
    line: {color: colores[i], width: 2}, marker: {size: 8, color: colores[i], line: {color: css("--surface-1"), width: 2}},
    hovertemplate: "%{x|%b %Y}: %{y:,} obras<extra>%{fullData.name}</extra>"}));
  dibujar("g-causa-tiempo", trazas, {yaxis: {gridcolor: css("--grid"), title: "Obras paralizadas", rangemode: "tozero"}, legend: {orientation: "h", y: 1.12, font: {color: css("--text-primary")}}, hovermode: "x unified"});
}

// ---------- Regiones ----------
const INDICADORES = {
  region_enla_4p_lectura_satisfactorio: "ENLA 4.º primaria · Lectura satisfactorio (%)", region_enla_4p_matematica_satisfactorio: "ENLA 4.º primaria · Matemática satisfactorio (%)",
  region_enla_2s_lectura_satisfactorio: "ENLA 2.º secundaria · Lectura satisfactorio (%)", region_enla_2s_matematica_satisfactorio: "ENLA 2.º secundaria · Matemática satisfactorio (%)",
  region_escale_pct_3_servicios: "Locales públicos con agua, desagüe y luz (%)", region_escale_pct_riesgo_estructural: "Locales con riesgo estructural (%)", region_escale_pct_rural: "Locales rurales (%)"};
opciones("f-indicador", Object.keys(INDICADORES), INDICADORES);
function rangos(a) { const idx = a.map((v, i) => [v, i]).sort((x, y) => x[0] - y[0]); const r = Array(a.length); let i = 0;
  while (i < idx.length) { let j = i; while (j + 1 < idx.length && idx[j + 1][0] === idx[i][0]) j++; for (let k = i; k <= j; k++) r[idx[k][1]] = (i + j) / 2 + 1; i = j + 1; } return r; }
function spearman(x, y) { const rx = rangos(x), ry = rangos(y), n = x.length, mx = suma(rx) / n, my = suma(ry) / n;
  let a = 0, b = 0, c = 0; for (let i = 0; i < n; i++) { a += (rx[i] - mx) * (ry[i] - my); b += (rx[i] - mx) ** 2; c += (ry[i] - my) ** 2; } return a / Math.sqrt(b * c); }
let ordenRegion = {col: "por100", asc: false};
function regiones() {
  const obras = filtradas(true), conteo = contar(obras, "region_enla"), ind = document.getElementById("f-indicador").value;
  const filas = D.regiones.map(r => ({...r, n: conteo[r.region_enla] || 0, por100: 100 * (conteo[r.region_enla] || 0) / r.region_escale_locales_publicos}));
  const validas = filas.filter(f => f[ind] !== null && !isNaN(f[ind]) && isFinite(f.por100));
  const rho = spearman(validas.map(f => f.por100), validas.map(f => f[ind]));
  document.getElementById("rho").innerHTML = `Correlación de Spearman: <b>${fmt(rho, 2)}</b> (n = ${validas.length} regiones; de −1 a +1)`;
  const sel = document.getElementById("f-region").value;
  const maxX = Math.max(...validas.map(f => f.por100), 0.1);
  dibujar("g-dispersion", [{type: "scatter", mode: "markers+text", x: validas.map(f => f.por100), y: validas.map(f => f[ind]), text: validas.map(f => f.n > 0 ? f.region_enla : ""),
    hovertext: validas.map(f => f.region_enla), textposition: "top center", textfont: {size: 10, color: css("--text-secondary")},
    marker: {size: validas.map(f => f.region_enla === sel ? 14 : 10), color: validas.map(f => f.region_enla === sel ? css("--c-rojo") : css("--c-verde")), line: {color: css("--surface-1"), width: 2}},
    customdata: validas.map(f => [f.n, f.region_escale_locales_publicos]),
    hovertemplate: "<b>%{hovertext}</b><br>%{customdata[0]} obras · %{customdata[1]:,} locales públicos<br>%{x:.2f} obras por 100 locales<br>Indicador: %{y:.1f}<extra></extra>"}],
    {xaxis: {gridcolor: css("--grid"), title: "Obras por cada 100 locales escolares públicos (regiones sin obras: en 0, sin nombre)", range: [-0.04 * maxX, maxX * 1.12]}, yaxis: {gridcolor: css("--grid"), title: INDICADORES[ind]}, margin: {l: 60, r: 20, t: 10, b: 50}});
  const cols = [["region_enla", "Región", 0], ["n", "Obras", 0], ["por100", "Por 100 locales", 2], ["region_escale_pct_3_servicios", "% con 3 servicios", 1],
                ["region_enla_4p_lectura_satisfactorio", "4.º prim. lectura %", 1], ["region_enla_4p_matematica_satisfactorio", "4.º prim. matem. %", 1]];
  const o = ordenRegion; filas.sort((a, b) => { const x = a[o.col], y = b[o.col]; return (typeof x === "string" ? x.localeCompare(y) : (x ?? -1) - (y ?? -1)) * (o.asc ? 1 : -1); });
  document.getElementById("t-regiones").innerHTML = "<thead><tr>" + cols.map(c => `<th class="${c[2] || c[0] === "n" ? "num" : ""}" data-c="${c[0]}">${c[1]}</th>`).join("") + "</tr></thead><tbody>" +
    filas.map(f => "<tr>" + cols.map(c => `<td class="${c[0] === "region_enla" ? "" : "num"}">${c[0] === "region_enla" ? esc(f[c[0]]) : fmt(f[c[0]], c[2])}</td>`).join("") + "</tr>").join("") + "</tbody>";
  document.querySelectorAll("#t-regiones th").forEach(th => th.onclick = () => { ordenRegion = {col: th.dataset.c, asc: ordenRegion.col === th.dataset.c ? !ordenRegion.asc : false}; regiones(); });
  document.querySelectorAll("#t-regiones tbody tr").forEach((tr, i) => tr.onclick = () => elegirRegion(filas[i].region_enla));
  const g = document.getElementById("g-dispersion");
  if (g.removeAllListeners) g.removeAllListeners("plotly_click");
  g.on("plotly_click", ev => elegirRegion(ev.points[0].hovertext));
  distritosRegion(obras, sel);
}
let ordenDistrito = {col: "n", asc: false};
function distritosRegion(obras, region) {
  const tit = document.getElementById("h-distritos-region"), sub = document.getElementById("s-distritos-region"), tab = document.getElementById("t-distritos-region"),
        fte = document.getElementById("f-distritos-region");
  if (!region) {
    tit.textContent = "Distritos de la región"; tab.innerHTML = ""; fte.textContent = "";
    sub.innerHTML = "Elija una región en el filtro de arriba, en la tabla o en el gráfico para ver <b>todos sus distritos</b>, con y sin obras, y su infraestructura escolar.";
    return;
  }
  const n = contar(obras.filter(o => o.region_enla === region), "ubigeo");
  const filas = D.distritos.filter(d => d.region_enla === region).map(d => ({...d, n: n[d.ubigeo] || 0}));
  const con = filas.filter(f => f.n > 0), sin = filas.filter(f => f.n === 0);
  const med = (a, k) => mediana(a.map(f => f[k]));
  tit.textContent = `Distritos de ${region} (${filas.length})`;
  sub.innerHTML = `<b>${con.length}</b> distritos con obras de colegios según los filtros (${fmt(suma(con.map(f => f.n)))} obras). ` +
    `Locales públicos con agua, desagüe y luz (mediana): <b>${fmt(med(con, "pct_3_servicios"), 1)} %</b> en los distritos con obras frente a <b>${fmt(med(sin, "pct_3_servicios"), 1)} %</b> en los demás. ` +
    `Clic en un distrito con obras para verlas.`;
  const o = ordenDistrito;
  filas.sort((a, b) => { const x = a[o.col], y = b[o.col]; return (typeof x === "string" ? x.localeCompare(y) : (x ?? -1) - (y ?? -1)) * (o.asc ? 1 : -1) || b.n - a.n || (a.pct_3_servicios ?? 0) - (b.pct_3_servicios ?? 0); });
  const cols = [["distrito", "Distrito"], ["provincia", "Provincia"], ["n", "Obras", 0], ["n_locales_publicos", "Locales públicos", 0], ["pct_3_servicios", "% con agua, desagüe y luz", 1],
                ["pct_rural", "% rurales", 1], ["pct_riesgo_estructural", "% con riesgo estructural", 1]];
  tab.innerHTML = "<thead><tr>" + cols.map(c => `<th class="${c.length > 2 ? "num" : ""}" data-c="${c[0]}">${c[1]}</th>`).join("") + "</tr></thead><tbody>" +
    filas.map(f => `<tr data-u="${esc(f.ubigeo)}" style="${f.n ? "" : "color:var(--text-muted)"}">` + cols.map(c => c.length > 2 ? `<td class="num">${c[0] === "n" && f.n ? "<b>" + f.n + "</b>" : fmt(f[c[0]], c[2])}</td>` : `<td>${esc(f[c[0]])}</td>`).join("") + "</tr>").join("") + "</tbody>";
  fte.textContent = `Fuente: obras según filtros (INFOBRAS/Contraloría); indicadores de los locales escolares públicos: ESCALE ${D.anio_escale}. En gris, distritos sin obras de colegios con los filtros elegidos.`;
  tab.querySelectorAll("th").forEach(th => th.onclick = () => { ordenDistrito = {col: th.dataset.c, asc: ordenDistrito.col === th.dataset.c ? !ordenDistrito.asc : false}; distritosRegion(obras, region); });
  tab.querySelectorAll("tbody tr").forEach(tr => tr.onclick = () => { if (!n[tr.dataset.u]) return; document.getElementById("f-distrito").value = tr.dataset.u; mostrar("obras"); });
}

// ---------- Obras y ficha ----------
let ordenObras = {col: "dias_desde_paralizacion", asc: false};
function tablaObras(obras) {
  const q = document.getElementById("buscar").value.trim().toLowerCase();
  let lista = q ? obras.filter(o => [o.nombre_obra, o.distrito, o.provincia, o.region_enla, o.codigo_infobras, o.cui, o.entidad].join(" ").toLowerCase().includes(q)) : obras.slice();
  const o = ordenObras; lista.sort((a, b) => { const x = a[o.col], y = b[o.col]; return (typeof x === "string" ? (x || "").localeCompare(y || "") : (x ?? -1) - (y ?? -1)) * (o.asc ? 1 : -1); });
  document.getElementById("n-obras").textContent = `${lista.length} obras`;
  const cols = [["nombre_obra", "Obra"], ["distrito", "Distrito"], ["region_enla", "Región"], ["estado_infobras", "Estado"], ["anio_inicio", "Inicio"], ["fecha_paralizacion", "Paralizada desde"], ["dias_desde_paralizacion", "Años paralizada"],
                ["monto_expediente_tecnico", "Expediente (S/)"], ["causa_macro", "Causa principal"]];
  document.getElementById("t-obras").innerHTML = "<thead><tr>" + cols.map(c => `<th class="${["dias_desde_paralizacion", "monto_expediente_tecnico", "anio_inicio"].includes(c[0]) ? "num" : ""}" data-c="${c[0]}">${c[1]}</th>`).join("") + "</tr></thead><tbody>" +
    lista.slice(0, 500).map(x => `<tr data-id="${esc(x.codigo_infobras)}"><td>${esc(x.nombre_obra).slice(0, 110)}${x.nombre_obra && x.nombre_obra.length > 110 ? "…" : ""}</td><td>${esc(x.distrito)}</td><td>${esc(x.region_enla)}</td><td>${esc(x.estado_infobras)}</td><td class="num">${x.anio_inicio ?? "—"}</td><td>${esc(x.fecha_paralizacion || "—")}</td><td class="num">${x.anios_paralizada === null ? "—" : fmt(x.anios_paralizada, 1)}</td><td class="num">${fmt(x.monto_expediente_tecnico)}</td><td>${esc(etq(ETIQ_MACRO, x.causa_macro))}</td></tr>`).join("") + "</tbody>";
  document.querySelectorAll("#t-obras th").forEach(th => th.onclick = () => { ordenObras = {col: th.dataset.c, asc: ordenObras.col === th.dataset.c ? !ordenObras.asc : false}; tablaObras(filtradas()); });
  document.querySelectorAll("#t-obras tbody tr").forEach(tr => tr.onclick = () => verFicha(tr.dataset.id));
}
function verFicha(id) {
  const o = D.obras.find(x => x.codigo_infobras === id); if (!o) return;
  const filas = [["Código INFOBRAS / CUI", `${esc(o.codigo_infobras)} / ${esc(o.cui)}`], ["Entidad", esc(o.entidad)], ["Ubicación", `${esc(o.distrito)}, ${esc(o.provincia)}, ${esc(o.departamento)} (ubigeo ${esc(o.ubigeo)})`],
    ["Tipo de infraestructura", esc(etq(ETIQ_TIPO, o.tipo_infraestructura))], ["Estado (INFOBRAS, 11-nov-2026)", `<span class="estado">${esc(o.estado_infobras)}</span>`],
    ["Paralizada desde", `${esc(o.fecha_paralizacion || "—")}${o.anios_paralizada !== null ? " · hace " + fmt(o.anios_paralizada, 1) + " años (" + fmt(o.dias_desde_paralizacion) + " días)" : ""} · ${fmt(o.n_tramos_paralizacion)} tramo(s) de paralización`],
    ["Inicio de la obra", esc(o.fecha_inicio || "—")],
    ["Causa principal", `${esc(etq(ETIQ_MACRO, o.causa_macro))} · ${esc((o.causa_grupo_final || "").replaceAll("_", " "))} (fuente: ${esc(o.causa_fuente || "—")})`],
    ["Causal oficial (Contraloría)", esc(o.causal_oficial || "—")], ["Expediente técnico", soles(o.monto_expediente_tecnico)], ["Costo actualizado (Contraloría)", soles(o.costo_actualizado_soles)],
    ["Devengado (MEF)", soles(o.devengado_total_soles)], ["Avance físico", o.avance_fisico_pct === null ? "—" : fmt(o.avance_fisico_pct, 1) + " %"],
    ["Nivel de gobierno / modalidad", `${esc(o.nivel_gobierno || "—")} / ${esc(o.modalidad || "—")}`],
    ["Locales del distrito con 3 servicios (ESCALE)", o.escale_pct_3_servicios === null ? "—" : fmt(o.escale_pct_3_servicios, 1) + " %"],
    ["Alertas de calidad", esc(o.alertas_calidad || "ninguna")], ["Ficha oficial", `<a href="${D.url_ficha}${encodeURIComponent(o.codigo_infobras)}" target="_blank" rel="noopener">Ver en INFOBRAS</a>`]];
  const f = document.getElementById("ficha");
  f.innerHTML = `<h2>${esc(o.nombre_obra)}</h2><dl>${filas.map(r => `<dt>${r[0]}</dt><dd>${r[1]}</dd>`).join("")}</dl>`;
  f.classList.add("visible"); f.scrollIntoView({behavior: "smooth", block: "start"});
}

// ---------- Proceso ----------
function proceso() {
  document.getElementById("generado").textContent = D.generado;
  const total = D.obras.length, colegios = COLEGIOS.length, hoy = COLEGIOS.filter(o => o.paralizada_hoy), distritos = new Set(hoy.map(o => o.ubigeo)).size;
  const p198 = D.obras.filter(o => o.origen === "infobras_paralizada_2026-10-27").length;
  const etapa = (n, titulo, detalle, ancho) => `<div style="margin:6px 0"><div style="display:flex;justify-content:space-between;font-size:13px"><span><b>${titulo}</b> · <span style="color:var(--text-muted)">${detalle}</span></span><b>${fmt(n)}</b></div><div style="height:14px;border-radius:4px;background:var(--c-naranja);width:${ancho}%;opacity:${0.45 + ancho / 180}"></div></div>`;
  document.getElementById("embudo").innerHTML =
    etapa(total, "Obras leídas de INFOBRAS", `${p198} paralizadas al 27-oct + ${total - p198} de la lista de Educación de la Contraloría (jun-2026)`, 100) +
    etapa(colegios, "Obras de colegios (educación básica)", "se excluyen deportivas, culturales, de educación superior y otras", 100 * colegios / total) +
    etapa(hoy.length, "Colegios con la obra paralizada hoy", `estado PARALIZADA en INFOBRAS el 11-nov-2026 · ${distritos} distritos`, 100 * hoy.length / total);
  const excl = D.obras.filter(o => o.tipo_infraestructura !== "educacion_basica"), porTipo = {};
  excl.forEach(o => { porTipo[o.tipo_infraestructura] = porTipo[o.tipo_infraestructura] || {n: 0, hoy: 0}; porTipo[o.tipo_infraestructura].n++; if (o.paralizada_hoy) porTipo[o.tipo_infraestructura].hoy++; });
  document.getElementById("t-excluidas").innerHTML = "<thead><tr><th>Tipo</th><th class='num'>Obras</th><th class='num'>Paralizadas hoy</th></tr></thead><tbody>" +
    Object.entries(porTipo).sort((a, b) => b[1].n - a[1].n).map(([t, v]) => `<tr><td>${esc(etq(ETIQ_TIPO, t))}</td><td class="num">${v.n}</td><td class="num">${v.hoy}</td></tr>`).join("") +
    `<tr><td><b>Total excluidas</b></td><td class="num"><b>${excl.length}</b></td><td class="num"><b>${excl.filter(o => o.paralizada_hoy).length}</b></td></tr></tbody>`;
  document.getElementById("t-excluidas-lista").innerHTML = "<thead><tr><th>Tipo</th><th>Obra</th><th>Región</th><th>Estado</th></tr></thead><tbody>" +
    excl.sort((a, b) => (a.tipo_infraestructura || "").localeCompare(b.tipo_infraestructura || "")).map(o => `<tr><td>${esc(etq(ETIQ_TIPO, o.tipo_infraestructura))}</td><td>${esc(o.nombre_obra)}</td><td>${esc(o.region_enla)}</td><td>${esc(o.estado_infobras)}</td></tr>`).join("") + "</tbody>";
  document.getElementById("flujo").innerHTML = D.proceso.map(p => `<div class="paso"><b>${esc(p.paso)}</b><div class="cifra">${esc(p.cifra)}</div><small>${esc(p.detalle)}</small><small>${esc(p.extra)}</small></div>`).join("");
  const ok = D.validacion.filter(v => String(v.ok) === "True").length;
  document.getElementById("resumen-validacion").innerHTML = `<span class="${ok === D.validacion.length ? "ok" : "no"}">${ok} de ${D.validacion.length} pasan</span>`;
  document.getElementById("t-validacion").innerHTML = "<thead><tr><th>Tabla</th><th>Comprobación</th><th class='num'>Valor</th><th class='num'>Esperado</th><th>Resultado</th></tr></thead><tbody>" +
    D.validacion.map(v => `<tr><td>${esc(v.tabla)}</td><td>${esc(v.comprobacion)}</td><td class="num">${esc(v.valor)}</td><td class="num">${esc(v.esperado)}</td><td>${String(v.ok) === "True" ? '<span class="ok">✓ pasa</span>' : '<span class="no">✗ falla</span>'}</td></tr>`).join("") + "</tbody>";
  document.getElementById("t-calidad").innerHTML = "<thead><tr><th>Nivel</th><th>Tema</th><th>Detalle</th></tr></thead><tbody>" +
    (D.calidad.length ? D.calidad.map(c => `<tr><td>${esc(c.nivel)}</td><td>${esc(c.tema)}</td><td>${esc(c.detalle)}</td></tr>`).join("") : "<tr><td colspan=3>Sin errores ni avisos</td></tr>") + "</tbody>";
}

// ---------- navegacion y actualizacion ----------
let pestana = "inicio";
function mostrar(p) {
  pestana = p;
  const enInicio = p === "inicio";
  document.getElementById("inicio").hidden = !enInicio; document.getElementById("app").hidden = enInicio;
  window.scrollTo(0, 0);
  if (enInicio) return;
  const pg = PAGINAS.find(x => x[0] === p);
  document.getElementById("titulo-pagina").textContent = pg[3];
  document.documentElement.style.setProperty("--acento", pg[5]); document.documentElement.style.setProperty("--acento-tx", pg[6]);
  document.getElementById("explica").textContent = pg[4];
  document.getElementById("cuerpo").classList.toggle("ancho", ["proceso", "limites"].includes(p));
  document.getElementById("pildoras").style.visibility = ["proceso", "limites"].includes(p) ? "hidden" : "visible";
  document.querySelectorAll("nav button").forEach(b => b.classList.toggle("activa", b.dataset.p === p));
  document.querySelectorAll(".pestana").forEach(s => s.classList.toggle("activa", s.id === "p-" + p));
  actualizar();
}
document.querySelectorAll("nav button").forEach(b => b.onclick = () => mostrar(b.dataset.p));
document.querySelectorAll("#pildoras button").forEach(b => b.onclick = () => {
  document.getElementById("f-estado").value = b.dataset.e;
  document.querySelectorAll("#pildoras button").forEach(x => x.classList.toggle("activa", x === b)); actualizar(); });
document.getElementById("borrar").onclick = () => {
  ["f-region", "f-causa", "f-anio"].forEach(id => document.getElementById(id).value = "");
  document.getElementById("buscar").value = ""; poblarDistritos(); document.querySelector('#pildoras button[data-e="hoy"]').click(); };
function actualizar() {
  if (pestana === "inicio") return;
  const obras = filtradas();
  kpis(obras);
  if (pestana === "ranking") ranking(obras);
  document.getElementById("nota-filtro").innerHTML = "Solo obras de <b>colegios</b> (educación básica). INFOBRAS agrupa Educación y Cultura: las obras deportivas, culturales y universitarias se excluyeron (ver <a href='#' onclick='mostrar(\"proceso\");return false'>Proceso</a>).";
  if (pestana === "resumen") { resumen(obras); serie(); }
  if (pestana === "mapa") mapa(obras);
  if (pestana === "causas") { causas(obras); causasTiempo(); }
  if (pestana === "regiones") regiones();
  if (pestana === "obras") tablaObras(obras);
  if (pestana === "proceso") proceso();
  window.dispatchEvent(new Event("resize"));
}
["f-distrito", "f-causa", "f-anio", "f-indicador", "f-orden-ranking"].forEach(id => document.getElementById(id).onchange = actualizar);
document.getElementById("f-region").onchange = () => { poblarDistritos(); actualizar(); };
document.getElementById("buscar").oninput = () => tablaObras(filtradas());
window.matchMedia("(prefers-color-scheme: dark)").addEventListener("change", actualizar);
portada(); mostrar("inicio");
</script>
</body>
</html>
"""


if __name__ == "__main__":
    construir()

### `pipeline_completo` — Pipeline completo (los 10 pasos)
UN SOLO SCRIPT para reproducir toda la data del proyecto, de principio a fin, hasta las tablas del producto final.

In [ ]:
%%modulo pipeline_completo
"""UN SOLO SCRIPT para reproducir toda la data del proyecto, de principio a fin, hasta las tablas del producto final.

    python src/pipeline_completo.py                        (todo; lo ya descargado o procesado se salta)
    python src/pipeline_completo.py --sin-descarga         (solo procesa lo que ya esta en data/raw)
    python src/pipeline_completo.py --fuentes escale,enla  (descarga solo esas fuentes)
    python src/pipeline_completo.py --forzar               (rehace todo, tarda mucho)
En Jupyter / VS Code: notebooks/proyecto_completo.ipynb llama a ejecutar() paso a paso.

Pasos, en este orden:
  1. Descarga: ESCALE, ENLA (Chrome visible), Contraloria (informes y anexos) y MEF (Formato 12-B, por HTTP). INFOBRAS solo con --fuentes infobras.
  2. Tablas de ESCALE y ENLA (SICRECE), niveles de logro, validacion, Excel y catalogo de variables de ESCALE.
  3. Datos personales: se quitan los nombres de personas de los comentarios ya guardados.
  4. Proyecto PDF, fases 1 a 4: inventario, tablas, cifras oficiales, panel por obra, texto, ubigeo de las obras e INFOBRAS ampliado.
  5. Producto final: NLP de causas, tablas intermedias y el CSV unico del dashboard (data/final/dashboard_obras.csv), con sus comprobaciones.
  6. Diccionario de todas las tablas (docs/diccionario.csv).
  7. Auditoria de datos personales en las tablas finales (falla si encuentra un nombre).
  8. Inventario de datos (docs/inventario_datos.md).
  9. Auditoria de calidad del CSV unico (docs/auditoria_calidad.csv; falla si encuentra algun error).
 10. Dashboard en HTML (resultados/dashboard.html): un solo archivo que se abre en el navegador, sin internet. Al terminar se abre solo (--no-abrir para evitarlo).
Cada paso deja su rastro en docs/log_descargas.txt y docs/log_pdf.txt. Si un paso falla, se avisa y se sigue con el siguiente.
"""
import argparse
import sys
import time
from types import SimpleNamespace

import auditoria_calidad
import catalogo_escale
import config
import dashboard_html
import datos_personales
import diccionario
import inventario_datos
import main
import main_pdf
import producto_final
import utils

FUENTES = ("escale", "enla", "contraloria", "infobras", "mef")      # las que existen
# INFOBRAS es una foto (27-oct y 11-nov-2026): si se volviera a leer, el listado cambiaria (hay obras que dejaron de estar paralizadas)
# y se moverian los numeros del producto. Por eso NO se descarga por defecto: hay que pedirlo con --fuentes infobras.
FUENTES_POR_DEFECTO = ("escale", "enla", "contraloria", "mef")


def leer_argumentos():
    p = argparse.ArgumentParser(description="Reproduce toda la data del proyecto con un solo comando")
    p.add_argument("--fuentes", default=",".join(FUENTES_POR_DEFECTO), help=f"fuentes a descargar, separadas por coma (por defecto: {','.join(FUENTES_POR_DEFECTO)}; INFOBRAS solo si se pide)")
    p.add_argument("--sin-descarga", action="store_true", help="no abrir el navegador ni bajar nada: solo procesar lo ya descargado")
    p.add_argument("--anio-inicio", type=int, default=config.ANIO_INICIO, help="primer año a descargar")
    p.add_argument("--forzar", action="store_true", help="descargar y procesar de nuevo aunque ya este hecho")
    p.add_argument("--headless", action="store_true", help="navegador sin ventana")
    p.add_argument("--desde-tablas", action="store_true", help="solo los pasos 5 a 10 (producto final, diccionario, auditorias y dashboard) con las tablas ya procesadas; no necesita data/raw (es lo que corre GitHub)")
    p.add_argument("--no-abrir", action="store_true", help="no abrir el dashboard en el navegador al terminar")
    args = p.parse_args()
    args.fuentes = [f.strip() for f in args.fuentes.split(",") if f.strip()]
    desconocidas = [f for f in args.fuentes if f not in FUENTES]
    if desconocidas:
        p.error(f"fuente desconocida: {desconocidas}; las validas son {FUENTES}")
    return args


def hay_datos_crudos():
    """True si data/raw existe y tiene archivos (el repositorio de GitHub no los trae: algunos contienen datos personales)."""
    return config.RAW.exists() and any(p.is_file() for p in config.RAW.rglob("*"))


def paso(numero, nombre, funcion):
    """Ejecuta un paso; si falla lo registra y devuelve False, para seguir con el siguiente."""
    utils.log(f"=== PASO {numero}: {nombre} ===")
    inicio = time.time()
    try:
        funcion()
        utils.log(f"=== PASO {numero} terminado en {time.time() - inicio:.0f} s ===")
        return True
    except Exception as error:
        utils.log(f"=== PASO {numero} FALLO ({type(error).__name__}: {error}); se sigue con el siguiente. Los archivos ya descargados siguen en data/raw ===")
        return False


def ejecutar(fuentes=FUENTES_POR_DEFECTO, sin_descarga=False, forzar=False, headless=False, anio_inicio=config.ANIO_INICIO, desde_tablas=False):
    """Corre los 10 pasos. Devuelve la lista de pasos que fallaron (vacia si todo salio bien). Tambien se usa desde Jupyter."""
    utils.log(f"PIPELINE COMPLETO: fuentes={list(fuentes)}, sin_descarga={sin_descarga}, forzar={forzar}, anio_inicio={anio_inicio}")
    # el mismo objeto de argumentos que usa main.py (solo=None significa «todas las fuentes elegidas»)
    a = SimpleNamespace(anio_inicio=anio_inicio, solo=None, forzar=forzar, headless=headless, fuentes=list(fuentes))
    resultados = {}
    if sin_descarga and not desde_tablas and not hay_datos_crudos():
        utils.log("AVISO: no hay datos crudos en data/raw y se pidio no descargar: se usa el modo --desde-tablas (pasos 5 a 10)")
        desde_tablas = True
    if desde_tablas:
        utils.log("MODO DESDE TABLAS: se omiten los pasos 1 a 4 (descarga y datos crudos); se parte de data/processed")
    else:
        if not sin_descarga:
            resultados["descarga"] = paso(1, "descarga de las fuentes", lambda: main.descargar(a))
        resultados["escale_enla"] = paso(2, "tablas de ESCALE y ENLA", lambda: (main.extraer(a), catalogo_escale.construir()))
        resultados["datos_personales"] = paso(3, "limpieza de datos personales en lo ya guardado", datos_personales.limpiar)
        resultados["pdf"] = paso(4, "proyecto PDF (fases 1 a 4; el diccionario de la fase 5 se escribe en el paso 6)", lambda: main_pdf.ejecutar(4, forzar))
    resultados["producto_final"] = paso(5, "producto final (data/final)", producto_final.construir)
    resultados["diccionario"] = paso(6, "diccionario de todas las tablas", diccionario.escribir_diccionario)
    resultados["auditoria"] = paso(7, "auditoria de datos personales", datos_personales.auditoria_o_falla)
    resultados["inventario"] = paso(8, "inventario de datos", inventario_datos.construir)
    resultados["calidad"] = paso(9, "auditoria de calidad del CSV unico (falla si hay algun error)", auditoria_calidad.auditoria_o_falla)
    resultados["dashboard_html"] = paso(10, "dashboard en HTML (resultados/dashboard.html)", dashboard_html.construir)
    fallidos = [k for k, ok in resultados.items() if not ok]
    utils.log("PIPELINE COMPLETO: terminado" + (f", con pasos fallidos: {fallidos}" if fallidos else " sin fallos"))
    return fallidos


def abrir_dashboard():
    """Abre resultados/dashboard.html en el navegador predeterminado (si se genero)."""
    import webbrowser
    if dashboard_html.SALIDA.exists():
        utils.log(f"Abriendo el dashboard: {dashboard_html.SALIDA}")
        webbrowser.open(dashboard_html.SALIDA.resolve().as_uri())
    else:
        utils.log("AVISO: no se encontro resultados/dashboard.html; revise el paso 10")


def main_completo():
    args = leer_argumentos()
    fallidos = ejecutar(args.fuentes, args.sin_descarga, args.forzar, args.headless, args.anio_inicio, args.desde_tablas)
    if not args.no_abrir:
        abrir_dashboard()
    return fallidos


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8")
    sys.exit(1 if main_completo() else 0)

# Parte B — Ejecución paso a paso

Solo el código que **corre el proyecto**, un paso por celda y en orden. No hay tablas de revisión: los resultados quedan en `data/final/dashboard_obras.csv`, en el dashboard `resultados/dashboard.html` y en `docs/`
(`validacion_final.csv`, `auditoria_calidad.csv`, `auditoria_datos_personales.csv`, `nlp_causas_metricas.csv`, `diccionario.csv`, `inventario_datos.md`).

| Paso | Qué hace | Módulos |
|---|---|---|
| 1 | Descarga ESCALE, ENLA (todos los grados que ofrece SICRECE), Contraloría y MEF (INFOBRAS solo si se pide) | `main`, `scraper_*`, `descarga_mef` |
| 2 | Tablas de ESCALE (+ indicadores ampliados) y ENLA (+ niveles de logro) y catálogo de variables | `extraccion`, `escale_ampliado`, `catalogo_escale` |
| 3 | Quita nombres de personas de los comentarios ya guardados | `datos_personales`, `utils` |
| 4 | Proyecto PDF (fases 1 a 4) y anexos Excel de la Contraloría (incluido el anexo 03) | `main_pdf` y sus módulos |
| 5 | Producto final: **NLP de las causas** (reglas + modelo de apoyo + macro-categorías), tablas y **CSV único del dashboard** | `producto_final`, `nlp_causas`, `dashboard` |
| 6 | Diccionario, auditoría de datos personales e inventario | `diccionario`, `datos_personales`, `inventario_datos` |
| 7 | **Auditoría de calidad** del CSV único (se detiene si hay un error) | `auditoria_calidad` |
| 8 | **Dashboard en HTML** con los productos ya validados; se abre solo en el navegador | `dashboard_html` |

## 0. Opciones

In [ ]:
from types import SimpleNamespace

import auditoria_calidad
import catalogo_escale
import config
import dashboard_html
import datos_personales
import diccionario
import escale_ampliado
import inventario_datos
import main
import main_pdf
import pipeline_completo
import producto_final

# ---------------- opciones ----------------
DESCARGAR = False          # True: consulta ESCALE, ENLA, Contraloría y MEF (Chrome visible + internet, ~15 min). False: usa data/raw tal como está
INCLUIR_INFOBRAS = False   # True: ADEMAS vuelve a leer INFOBRAS. Cambia el listado (hay obras que dejaron de estar paralizadas) y mueve los números
FORZAR = False             # True: rehace todo aunque ya esté hecho (muy lento)

# Sin datos crudos (por ejemplo, al bajar el repositorio de GitHub) y sin descargar: se parte de las tablas ya procesadas (solo los pasos 5 a 8)
DESDE_TABLAS = not (DESCARGAR or pipeline_completo.hay_datos_crudos())

FUENTES = list(pipeline_completo.FUENTES_POR_DEFECTO) + (["infobras"] if INCLUIR_INFOBRAS else [])
ARGS = SimpleNamespace(anio_inicio=config.ANIO_INICIO, solo=None, forzar=FORZAR, headless=False, fuentes=FUENTES)
print("Carpeta del proyecto:", config.RAIZ.name)
print("Descargar:", DESCARGAR, "| fuentes:", FUENTES)
print("Modo:", "desde las tablas ya procesadas (pasos 5 a 8)" if DESDE_TABLAS else "completo (pasos 1 a 8)")

## 1. Descarga de las fuentes
ESCALE, ENLA y Contraloría se descargan con Chrome visible (Selenium); el MEF con una petición HTTP. INFOBRAS solo si se pide: es una foto (27-oct y 11-nov-2026) y leerlo de nuevo cambia los números.
Reglas de cortesía: pausa de 2 a 5 s, 3 reintentos, nada en paralelo, y **lo ya descargado se salta**. Los archivos de `data/raw` nunca se modifican.

In [ ]:
if DESCARGAR:
    main.descargar(ARGS)
elif DESDE_TABLAS:
    print("No hay datos crudos en data/raw y DESCARGAR = False: se parte de las tablas ya procesadas (pasos 5 a 8). Para repetir desde los portales: DESCARGAR = True")
else:
    print("DESCARGAR = False: no se consulta ningún portal; se usa lo que hay en data/raw")

## 2. Tablas de ESCALE y ENLA
`extraccion` convierte los archivos crudos en las tablas de ESCALE (por distrito y año) y de ENLA (por región y grado); `escale_ampliado` calcula 16 indicadores más de infraestructura (Censo 2025);
`catalogo_escale` lista las 331 variables de ESCALE con su descripción oficial.

In [ ]:
if not DESDE_TABLAS:
    main.extraer(SimpleNamespace(anio_inicio=config.ANIO_INICIO, solo=None, forzar=FORZAR, headless=False, fuentes=FUENTES))
    catalogo_escale.construir()
    escale_ampliado.construir()

## 3. Datos personales
**La regla del proyecto es no guardar nombres de personas.** `utils.omitir_nombres` los reemplaza por `[NOMBRE OMITIDO]` apenas se lee un comentario; este paso vuelve a pasar el filtro por lo ya guardado (se puede repetir sin cambiar nada).

In [ ]:
if not DESDE_TABLAS:
    datos_personales.limpiar()

## 4. Proyecto PDF (fases 1 a 4) y anexos Excel
Texto y tablas de los PDF (pdfplumber/pypdf), cifras oficiales, causales, panel por obra de los anexos Excel de la Contraloría (incluido el anexo 03), citas y términos de los informes, ubigeo de cada obra
y tablas ampliadas de INFOBRAS. Un PDF ya procesado no se vuelve a procesar.

In [ ]:
if not DESDE_TABLAS:
    main_pdf.ejecutar(4, FORZAR)

## 5. Producto final: NLP de las causas, tablas y CSV único
`producto_final.construir()` clasifica la causa de cada obra con `nlp_causas` y arma las tablas del producto final y el **CSV único** `data/final/dashboard_obras.csv`.

**NLP** (código en la celda `%%modulo nlp_causas` de la Parte A): normaliza los textos de INFOBRAS, la causal oficial de la Contraloría y la problemática del MEF; puntúa 14 grupos de palabras clave (con negación);
combina las fuentes con pesos; deja una causa principal y hasta 2 secundarias; un modelo de apoyo (TF-IDF + regresión logística con pesos por clase) solo interviene si las reglas no encuentran nada;
y agrupa en 8 macro-categorías para el dashboard. Escribe sus métricas en `docs/nlp_causas_metricas.csv`.

Además agrega dos columnas que usa el dashboard: `tipo_infraestructura` (separa los colegios de las obras deportivas, culturales y universitarias, porque INFOBRAS agrupa Educación y Cultura)
y `dias_desde_paralizacion` (tiempo real paralizada, desde la fecha de paralización hasta la lectura de INFOBRAS).

In [ ]:
tablas, validacion = producto_final.construir()

## 6. Diccionario, datos personales e inventario
El diccionario describe cada columna de cada tabla (si falta una descripción, el programa falla a propósito); la auditoría de datos personales se detiene si encuentra un nombre; el inventario lista qué datos hay.

In [ ]:
diccionario.escribir_diccionario()
datos_personales.auditoria_o_falla()
inventario_datos.construir()

## 7. Auditoría de calidad del CSV único
Revisa estructura, texto, códigos, rangos, fechas, consistencia entre tipos de registro y campos obligatorios; escribe `docs/auditoria_calidad.csv` y se detiene si hay un ERROR.

In [ ]:
auditoria_calidad.auditoria_o_falla()

## 8. Dashboard en HTML
`dashboard_html.construir()` arma `resultados/dashboard.html` con los productos ya validados (no recalcula nada): **un solo archivo** que se abre con doble clic en cualquier navegador, sin instalar nada y sin internet.
Abre en una **portada** con las cifras principales y un menú. Páginas: **Panorama** (¿desde cuándo? año por año), **Ranking** (regiones y distritos con semáforo de años paralizadas), **Mapa** (¿dónde?),
**Causas** (¿por qué?), **Regiones y distritos** (pregunta guía con ESCALE y ENLA), **Obras** (ficha de cada obra), **Proceso** (los pasos con sus cifras y las comprobaciones) y **Límites**.
Muestra solo las obras de **colegios** (`tipo_infraestructura = educacion_basica`); las demás se listan en la página Proceso. Al terminar, el dashboard se abre en el navegador.

In [ ]:
ruta = dashboard_html.construir()
print("Dashboard:", ruta)
pipeline_completo.abrir_dashboard()   # lo abre en el navegador predeterminado

## Qué leer después
* `resultados/dashboard.html` — el dashboard (doble clic para abrirlo).
* `docs/producto_final.md` — qué contiene el CSV único, cómo usarlo y los límites que hay que decir al presentar.
* `docs/limites_y_calidad.md` — lo que la data no tiene o no puede afirmar.
* `docs/diccionario.csv` y `data/final/dashboard_obras_diccionario.csv` — qué significa cada columna.